# Qdrant Technical Documentation Search & RAG Research — Retrieval-Improved Variant

This notebook documents the research and development of a technical documentation retrieval system built on Qdrant.

The project has two layers:

1. A retrieval system that satisfies the Qdrant Essentials Final Project requirements.
2. A portfolio extension that turns the retrieval pipeline into an end-to-end RAG system with explicit context engineering and grounded generation.

**Revision status: not executed.** This separate variant keeps the original notebook unchanged and applies retrieval-only improvements: explicit hierarchy-aware embedding text, a fixed 50-candidate hybrid/ColBERT pool, transparent technical-query expansion, deeper failure diagnostics, and a fresh post-freeze held-out set. Restart the kernel and run from the beginning when execution is approved. All project functions remain in this notebook; no external Python modules are required.

## 1. Project Goals

### 1.1 Qdrant Essentials Final Project Requirements

The core project will implement:

- structure-aware chunking of technical documentation;
- dense vector retrieval;
- sparse vector retrieval;
- hybrid dense + sparse retrieval;
- server-side rank fusion using RRF and/or DBSF;
- ColBERT multivector reranking;
- a labeled evaluation dataset with realistic queries;
- retrieval evaluation using Recall@10 and MRR;
- latency evaluation using P50 and P95;
- a target Recall@10 of at least 0.80.

In addition to the Day 6 final-project requirements, the notebook includes a dedicated collection-optimization experiment before the portfolio extension. That experiment evaluates scalar and binary quantization, deferred HNSW indexing during bulk loading, controlled memory-tier placement, oversampling, rescoring, Recall@10, MRR, and P50/P95 latency.


### 1.2 Portfolio Extension

After validating the retrieval pipeline, the project will be extended into a complete RAG system with:

- explicit context construction;
- neighboring-section expansion;
- context deduplication;
- token-budget management;
- source metadata preservation;
- local LLM generation;
- grounded answers with citations;
- abstention when retrieved evidence is insufficient;
- end-to-end RAG evaluation.

### 1.3 Success Criteria

The course core is successful when the selected pipeline reaches Recall@10 >= 0.80 on the held-out queries and reports MRR and P50/P95 latency with a reproducible protocol.

Hybrid fusion and ColBERT are required experiments, not assumed improvements: a simpler retriever may win. Configuration selection uses the research queries only. Fresh held-out queries are materialized after selection and are never used by tuning sweeps. Their relevance labels are explicit section anchors, with incomplete judgments acknowledged.

Quantization must preserve measured Recall@10 and MRR while improving P50/P95 latency; `speedup` may be shown only as a derived efficiency value. Sections 16–19 remain a future RAG extension, outside the completed retrieval core.


## 2. Environment and Configuration

This section initializes the project environment, resolves repository paths, verifies the Qdrant connection, and defines reproducibility settings used throughout the experiments.

Connection settings are read from `.env`: `QDRANT_HOST=qdrant`, `QDRANT_HTTP_PORT=6333`, `QDRANT_GRPC_PORT=6334`, `QDRANT_TIMEOUT_SECONDS=60`, and optional `QDRANT_API_KEY` / `HF_TOKEN`. The explicit 60-second request deadline avoids the client's five-second default terminating legitimate ColBERT and sparse benchmark queries. Keep secrets out of notebook outputs. The server Dockerfile is pinned to `qdrant/qdrant:v1.19.1` without a digest.

The existing GPU base image `huggingface/transformers-all-latest-gpu` has no published version tag in its [Docker Hub tag list](https://hub.docker.com/r/huggingface/transformers-all-latest-gpu/tags). Its working image is left unchanged instead of substituting an untested GPU stack. This remains a reproducibility limitation; runtime package versions are recorded below. No container rebuild or notebook execution is performed by this revision.

### 2.1 Imports

In [1]:
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache, partial
from collections import OrderedDict
from dataclasses import dataclass
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import Request, urlopen
import json
import os
import random
import re
import time
import unicodedata
import uuid
from dotenv import load_dotenv

import frontmatter
from markdown_it import MarkdownIt
from PIL import Image
from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import plotly.express as px

from dotenv import load_dotenv
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient, models
from ranx import Qrels, Run, evaluate
from tqdm import tqdm
from transformers import AutoTokenizer


### 2.2 Project Paths

In [2]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / ".git").exists() and (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Project root could not be found.")


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


Project root: /workspace


### 2.3 Qdrant Connection

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

QDRANT_HOST = os.getenv("QDRANT_HOST", "qdrant")
QDRANT_HTTP_PORT = int(os.getenv("QDRANT_HTTP_PORT", "6333"))
QDRANT_GRPC_PORT = int(os.getenv("QDRANT_GRPC_PORT", "6334"))
QDRANT_TIMEOUT_SECONDS = int(os.getenv("QDRANT_TIMEOUT_SECONDS", "60"))
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

qdrant_client = QdrantClient(
    host=QDRANT_HOST,
    port=QDRANT_HTTP_PORT,
    grpc_port=QDRANT_GRPC_PORT,
    api_key=QDRANT_API_KEY,
    prefer_grpc=True,
    timeout=QDRANT_TIMEOUT_SECONDS,
)

qdrant_client.get_collections()


CollectionsResponse(collections=[CollectionDescription(name='docs_search'), CollectionDescription(name='docs_search_final')])


### 2.4 Reproducibility

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

packages = ["qdrant-client", "fastembed-gpu", "pandas", "plotly", "ranx"]

for package in packages:
    print(f"{package}: {version(package)}")

# Runtime results stay in notebook memory/output; no extra report files are required.
RUN_MANIFEST = {
    "seed": SEED,
    "packages": {name: version(name) for name in packages},
    "python": __import__("sys").version,
    "qdrant": qdrant_client.info().version,
    "transformers": version("transformers"),
    "torch": version("torch"),
}
BENCHMARK_RUNS = {}


def remember_result(name: str, frame: pd.DataFrame):
    """Keep detailed results in this notebook session, without external artifacts."""
    BENCHMARK_RUNS[name] = frame.copy()
    return frame


qdrant-client: 1.19.0
fastembed-gpu: 0.8.0
pandas: 2.3.3
plotly: 7.0.0
ranx: 0.3.21


## 3. Dataset Acquisition

This section acquires a reproducible snapshot of the official Qdrant documentation that will be used as the retrieval corpus.

### 3.1 Qdrant Documentation Source

The corpus is sourced from the official `qdrant/landing_page` repository.

### 3.2 Repository Snapshot

The documentation cache is namespaced by the pinned Git commit, not by content hashes. The repository tree is cached alongside that snapshot. Only files listed in that tree enter the corpus; the legacy unversioned cache is left untouched. Downloads use atomic replacement and expected byte counts to reject partial files.

In [5]:
QDRANT_DOCS_COMMIT = "46e80312568d1e4505917b94c1ef78f4000330aa"
DOCS_PREFIX = "qdrant-landing/content/documentation/"
QDRANT_SNAPSHOT_DIR = RAW_DATA_DIR / "qdrant-docs" / QDRANT_DOCS_COMMIT
QDRANT_DOCS_DIR = QDRANT_SNAPSHOT_DIR / "documentation"
TREE_CACHE_PATH = QDRANT_SNAPSHOT_DIR / "repository-tree.json"
MAX_WORKERS = min(16, max(1, (os.cpu_count() or 1) - 2))
TREE_URL = (
    "https://api.github.com/repos/qdrant/landing_page/"
    f"git/trees/{QDRANT_DOCS_COMMIT}?recursive=1"
)


def fetch_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "qdrant-final-project"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def atomic_write(path: Path, content: bytes):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.name}.{uuid.uuid4().hex}.part")
    try:
        temporary.write_bytes(content)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


if TREE_CACHE_PATH.is_file():
    repository_tree = json.loads(TREE_CACHE_PATH.read_text(encoding="utf-8"))
else:
    repository_tree = json.loads(fetch_bytes(TREE_URL))
    if repository_tree.get("truncated"):
        raise ValueError("GitHub returned an incomplete repository tree.")
    atomic_write(TREE_CACHE_PATH, json.dumps(repository_tree).encode("utf-8"))

if repository_tree.get("truncated"):
    raise ValueError("The cached repository tree is incomplete.")

docs_entries = {
    item["path"]: item
    for item in repository_tree["tree"]
    if item["type"] == "blob"
    and item["path"].startswith(DOCS_PREFIX)
    and Path(item["path"]).suffix.lower() in {".md", ".mdx"}
}
docs_files = sorted(docs_entries)


def get_target_path(repo_path: str) -> Path:
    return QDRANT_DOCS_DIR / Path(repo_path).relative_to(DOCS_PREFIX)


def snapshot_file_ready(repo_path: str) -> bool:
    target = get_target_path(repo_path)
    return target.is_file() and target.stat().st_size == docs_entries[repo_path]["size"]


def download_file(repo_path: str) -> Path:
    url = (
        "https://raw.githubusercontent.com/qdrant/landing_page/" f"{QDRANT_DOCS_COMMIT}/{repo_path}"
    )
    content = fetch_bytes(url)
    if len(content) != docs_entries[repo_path]["size"]:
        raise ValueError(f"Incomplete snapshot download: {repo_path}")
    target = get_target_path(repo_path)
    atomic_write(target, content)
    return target


files_to_download = [path for path in docs_files if not snapshot_file_ready(path)]
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    downloaded_files = list(
        tqdm(
            executor.map(download_file, files_to_download),
            total=len(files_to_download),
            desc="Downloading snapshot",
            unit="file",
        )
    )
assert all(snapshot_file_ready(path) for path in docs_files)
RUN_MANIFEST["source_commit"] = QDRANT_DOCS_COMMIT
print(
    f"Commit: {QDRANT_DOCS_COMMIT}; documents: {len(docs_files):,}; downloaded: {len(downloaded_files):,}"
)


Commit: 46e80312568d1e4505917b94c1ef78f4000330aa; documents: 3,088; downloaded: 0


### 3.3 Documentation File Discovery

The corpus is constructed from the exact file list of the pinned repository tree. Unrelated local files and other commit caches are not scanned.

### 3.4 Corpus Statistics

Basic corpus statistics are collected before parsing and chunking.

In [6]:
local_docs_files = [get_target_path(path) for path in docs_files]

corpus_df = pd.DataFrame(
    {
        "path": [str(path.relative_to(QDRANT_DOCS_DIR)) for path in local_docs_files],
        "extension": [path.suffix.lower() for path in local_docs_files],
        "size_bytes": [path.stat().st_size for path in local_docs_files],
        "section": [
            (
                path.relative_to(QDRANT_DOCS_DIR).parts[0]
                if len(path.relative_to(QDRANT_DOCS_DIR).parts) > 1
                else "_root"
            )
            for path in local_docs_files
        ],
    }
)


In [7]:
retrieval_corpus_df = corpus_df[corpus_df["section"] != "headless"].reset_index(drop=True)

headless_files = (corpus_df["section"] == "headless").sum()

print(
    f"All Markdown files: {len(corpus_df):,}\n"
    f"Headless support files: {headless_files:,}\n"
    f"Retrieval documents: {len(retrieval_corpus_df):,}\n"
    f"Retrieval corpus size: {retrieval_corpus_df['size_bytes'].sum() / 1024 / 1024:.2f} MB\n"
    f"Median retrieval document size: {retrieval_corpus_df['size_bytes'].median() / 1024:.1f} KB"
)

display(retrieval_corpus_df.head(10))


All Markdown files: 3,088
Headless support files: 2,753
Retrieval documents: 335
Retrieval corpus size: 2.47 MB
Median retrieval document size: 4.6 KB


,path,extension,size_bytes,section
0,_index.md,.md,9899,_root
1,capacity-planning.md,.md,16773,_root
2,cloud-account-setup.md,.md,6662,_root
3,cloud-api.md,.md,3020,_root
4,cloud-cli.md,.md,3251,_root
5,cloud-getting-started.md,.md,2897,_root
6,cloud-premium.md,.md,1935,_root
7,cloud-pricing-payments.md,.md,5788,_root
8,cloud-quickstart.md,.md,43484,_root
9,cloud-rbac/_index.md,.md,1430,cloud-rbac


In [8]:
section_counts = (
    retrieval_corpus_df["section"].value_counts().rename_axis("section").reset_index(name="files")
)

fig = px.bar(
    section_counts, x="section", y="files", title="Retrieval Documents by Top-Level Section"
)

fig.show()


## 4. Documentation Parsing

This section converts the raw Markdown documentation into structured records while preserving metadata required for structure-aware chunking and retrieval.

### 4.1 Front Matter Parsing

Qdrant documentation pages use YAML front matter to store page-level metadata such as titles, descriptions, aliases, and ordering information.

The front matter is separated from the Markdown body so that metadata and document content can be processed independently.

In [9]:
parsed_documents = []
parse_errors = []

retrieval_paths = [QDRANT_DOCS_DIR / path for path in retrieval_corpus_df["path"]]

for path in tqdm(retrieval_paths, desc="Parsing front matter", unit="file"):
    try:
        raw_text = path.read_text(encoding="utf-8")
        metadata, content = frontmatter.parse(raw_text)

        parsed_documents.append(
            {
                "path": str(path.relative_to(QDRANT_DOCS_DIR)),
                "metadata": metadata,
                "content": content.strip(),
            }
        )
    except Exception as exc:
        parse_errors.append({"path": str(path.relative_to(QDRANT_DOCS_DIR)), "error": str(exc)})

print(f"Documents parsed: {len(parsed_documents):,}\n" f"Parse errors: {len(parse_errors):,}")


Parsing front matter: 100%|██████████| 335/335 [00:00<00:00, 356.58file/s]

Documents parsed: 335
Parse errors: 0


In [10]:
frontmatter_df = pd.DataFrame(
    {
        "path": [doc["path"] for doc in parsed_documents],
        "title": [doc["metadata"].get("title") for doc in parsed_documents],
        "description": [doc["metadata"].get("description") for doc in parsed_documents],
        "weight": [doc["metadata"].get("weight") for doc in parsed_documents],
        "content_chars": [len(doc["content"]) for doc in parsed_documents],
    }
)

print(
    f"Missing titles: {frontmatter_df['title'].isna().sum():,}\n"
    f"Missing descriptions: {frontmatter_df['description'].isna().sum():,}"
)

display(frontmatter_df.head(10))


Missing titles: 0
Missing descriptions: 10


,path,title,description,weight,content_chars
0,_index.md,Documentation,Official Qdrant documentation for vector searc...,2.0,5358
1,capacity-planning.md,Capacity Planning,Plan Qdrant cluster capacity: estimate RAM and...,115.0,16246
2,cloud-account-setup.md,Account Setup,"Register a Qdrant Cloud account with email, Go...",210.0,6203
3,cloud-api.md,Qdrant Cloud API,Use the Qdrant Cloud API over gRPC or REST/JSO...,245.0,2600
4,cloud-cli.md,Qdrant Cloud CLI,Install and use the qcloud CLI to manage Qdran...,250.0,2877
5,cloud-getting-started.md,Getting Started,Onboarding guide for Qdrant Managed Cloud — se...,205.0,2476
6,cloud-premium.md,Premium Tier,"Qdrant Cloud Premium adds 24/7 support, strong...",260.0,1521
7,cloud-pricing-payments.md,Billing & Payments,Qdrant Cloud billing explained: usage-based pr...,255.0,5302
8,cloud-quickstart.md,Cloud Quickstart,"Create a free Qdrant Cloud cluster, install th...",115.0,42981
9,cloud-rbac/_index.md,Cloud RBAC,Use Cloud RBAC to assign granular permissions ...,215.0,1047


### 4.2 Markdown Structure Extraction

The Markdown body of each document is parsed into structural elements.

At this stage, heading tokens are extracted to inspect the hierarchy of the documentation before defining the final section-aware chunking strategy.

In [11]:
markdown_parser = MarkdownIt("commonmark")

document_tokens = {doc["path"]: markdown_parser.parse(doc["content"]) for doc in parsed_documents}
heading_records = []

for document in tqdm(parsed_documents, desc="Extracting Markdown structure", unit="document"):
    tokens = document_tokens[document["path"]]

    heading_index = 0

    for index, token in enumerate(tokens):
        if token.type != "heading_open":
            continue

        level = int(token.tag[1:])

        if level not in {1, 2, 3}:
            continue

        heading_text = tokens[index + 1].content.strip()

        heading_records.append(
            {
                "path": document["path"],
                "page_title": document["metadata"].get("title"),
                "heading_index": heading_index,
                "level": level,
                "heading": heading_text,
            }
        )

        heading_index += 1

headings_df = pd.DataFrame(heading_records)

print(f"Documents: {len(parsed_documents):,}\n" f"Headings extracted: {len(headings_df):,}")

display(headings_df.head(20))


Extracting Markdown structure: 100%|██████████| 335/335 [00:00<00:00, 77202.85document/s]

Documents: 335
Headings extracted: 2,520


,path,page_title,heading_index,level,heading
0,_index.md,Documentation,0,1,Qdrant Documentation
1,_index.md,Documentation,1,2,Getting Started
2,_index.md,Documentation,2,2,Develop
3,_index.md,Documentation,3,2,Deploy
4,_index.md,Documentation,4,2,Ecosystem
5,_index.md,Documentation,5,2,Tutorials & Examples
6,_index.md,Documentation,6,2,Learn
7,_index.md,Documentation,7,2,API Reference
8,capacity-planning.md,Capacity Planning,0,1,Capacity Planning
9,capacity-planning.md,Capacity Planning,1,2,Calculating RAM and Disk Size


In [12]:
heading_level_counts = (
    headings_df["level"]
    .value_counts()
    .sort_index()
    .rename_axis("level")
    .reset_index(name="headings")
)

display(heading_level_counts)

fig = px.bar(heading_level_counts, x="level", y="headings", title="Markdown Headings by Level")

fig.show()


,level,headings
0,1,313
1,2,1337
2,3,870


In [13]:
all_heading_records = []

for document in tqdm(parsed_documents, desc="Inspecting all heading levels", unit="document"):
    tokens = document_tokens[document["path"]]

    for index, token in enumerate(tokens):
        if token.type != "heading_open":
            continue

        level = int(token.tag[1:])
        heading_text = tokens[index + 1].content.strip()

        all_heading_records.append(
            {"path": document["path"], "level": level, "heading": heading_text}
        )

all_headings_df = pd.DataFrame(all_heading_records)

heading_level_counts = (
    all_headings_df["level"]
    .value_counts()
    .sort_index()
    .rename_axis("level")
    .reset_index(name="headings")
)

documents_with_h1 = set(all_headings_df.loc[all_headings_df["level"] == 1, "path"])

documents_without_h1 = [
    document["path"] for document in parsed_documents if document["path"] not in documents_with_h1
]

display(heading_level_counts)

print(f"Documents without H1: {len(documents_without_h1):,}")
display(pd.DataFrame({"path": documents_without_h1}).head(30))


Inspecting all heading levels: 100%|██████████| 335/335 [00:00<00:00, 78396.02document/s]


,level,headings
0,1,313
1,2,1337
2,3,870
3,4,179
4,5,2


Documents without H1: 39


,path
0,cloud-tab.md
1,cloud-tools/_index.md
2,cloud-tools/pulumi.md
3,cloud-tools/terraform.md
4,data-management/_index.md
5,data-management/confluent.md
6,data-management/fluvio.md
7,data-management/redpanda.md
8,dl-cloud-getting-started.md
9,dl-cloud-interfaces.md


### 4.3 Heading Hierarchy

Markdown headings are converted into an explicit hierarchical structure.

The front matter title is treated as the page-level root, while H1-H6 headings define nested sections within the page. This preserves document structure even for pages that do not contain an explicit H1 heading.

In [14]:
heading_hierarchy_records = []

for document in tqdm(parsed_documents, desc="Building heading hierarchy", unit="document"):
    tokens = document_tokens[document["path"]]

    hierarchy = {}
    heading_index = 0

    for index, token in enumerate(tokens):
        if token.type != "heading_open":
            continue

        level = int(token.tag[1:])
        heading = tokens[index + 1].content.strip()

        hierarchy[level] = heading

        for deeper_level in range(level + 1, 7):
            hierarchy.pop(deeper_level, None)

        heading_path = [hierarchy[current_level] for current_level in sorted(hierarchy)]

        heading_hierarchy_records.append(
            {
                "path": document["path"],
                "page_title": document["metadata"]["title"],
                "heading_index": heading_index,
                "level": level,
                "heading": heading,
                "heading_path": heading_path,
            }
        )

        heading_index += 1

heading_hierarchy_df = pd.DataFrame(heading_hierarchy_records)

print(f"Hierarchy records: {len(heading_hierarchy_df):,}")

display(heading_hierarchy_df.head(20))


Building heading hierarchy: 100%|██████████| 335/335 [00:00<00:00, 44355.45document/s]

Hierarchy records: 2,701


,path,page_title,heading_index,level,heading,heading_path
0,_index.md,Documentation,0,1,Qdrant Documentation,[Qdrant Documentation]
1,_index.md,Documentation,1,2,Getting Started,"[Qdrant Documentation, Getting Started]"
2,_index.md,Documentation,2,2,Develop,"[Qdrant Documentation, Develop]"
3,_index.md,Documentation,3,2,Deploy,"[Qdrant Documentation, Deploy]"
4,_index.md,Documentation,4,2,Ecosystem,"[Qdrant Documentation, Ecosystem]"
5,_index.md,Documentation,5,2,Tutorials & Examples,"[Qdrant Documentation, Tutorials & Examples]"
6,_index.md,Documentation,6,2,Learn,"[Qdrant Documentation, Learn]"
7,_index.md,Documentation,7,2,API Reference,"[Qdrant Documentation, API Reference]"
8,capacity-planning.md,Capacity Planning,0,1,Capacity Planning,[Capacity Planning]
9,capacity-planning.md,Capacity Planning,1,2,Calculating RAM and Disk Size,"[Capacity Planning, Calculating RAM and Disk S..."


### 4.4 URL and Breadcrumb Construction

In [15]:
routing_metadata_df = pd.DataFrame(
    [
        {
            "path": document["path"],
            "slug": document["metadata"].get("slug"),
            "url": document["metadata"].get("url"),
            "aliases": document["metadata"].get("aliases"),
            "breadcrumb": document["metadata"].get("breadcrumb"),
            "hide_in_sidebar": document["metadata"].get("hideInSidebar"),
        }
        for document in parsed_documents
    ]
)

print(
    f"Documents with slug: {routing_metadata_df['slug'].notna().sum():,}\n"
    f"Documents with explicit URL: {routing_metadata_df['url'].notna().sum():,}\n"
    f"Documents with aliases: {routing_metadata_df['aliases'].notna().sum():,}"
)

display(routing_metadata_df[routing_metadata_df[["slug", "url", "aliases"]].notna().any(axis=1)])


Documents with slug: 3
Documents with explicit URL: 0
Documents with aliases: 140


,path,slug,url,aliases,breadcrumb,hide_in_sidebar
1,capacity-planning.md,None,None,"[capacity, /documentation/cloud/capacity-sizin...",None,None
2,cloud-account-setup.md,None,None,[/documentation/cloud/qdrant-cloud-setup/],None,None
3,cloud-api.md,None,None,[/documentation/qdrant-cloud-api/],None,None
5,cloud-getting-started.md,None,None,[/documentation/cloud/getting-started/],None,None
6,cloud-premium.md,None,None,[/documentation/cloud/premium/],None,None
...,...,...,...,...,...,...
329,tutorials-search-engineering/static-embeddings.md,None,None,"[/blog/static-embeddings/, /documentation/data...",None,None
330,tutorials-search-engineering/turbo4-multivecto...,None,None,[/documentation/tutorials-search-engineering/m...,None,None
331,tutorials-search-engineering/using-multivector...,None,None,[/documentation/search-precision/multivector-r...,None,None
333,upgrades.md,None,None,"[/documentation/upgrades, /documentation/opera...",None,None


In [16]:
QDRANT_DOCS_BASE_URL = "https://qdrant.tech/documentation"


def build_page_url(document: dict) -> str:
    path = Path(document["path"])
    metadata = document["metadata"]

    if path.name == "_index.md":
        route_parts = path.parent.parts
    else:
        page_slug = metadata.get("slug") or path.stem
        route_parts = (*path.parent.parts, page_slug)

    route = "/".join(route_parts)

    if not route:
        return f"{QDRANT_DOCS_BASE_URL}/"

    return f"{QDRANT_DOCS_BASE_URL}/{route}/"


page_records = []

for document in parsed_documents:
    page_records.append(
        {
            "path": document["path"],
            "page_title": document["metadata"]["title"],
            "page_url": build_page_url(document),
            "aliases": document["metadata"].get("aliases", []),
        }
    )

pages_df = pd.DataFrame(page_records)

display(pages_df.head(20))


,path,page_title,page_url,aliases
0,_index.md,Documentation,https://qdrant.tech/documentation/,[]
1,capacity-planning.md,Capacity Planning,https://qdrant.tech/documentation/capacity-pla...,"[capacity, /documentation/cloud/capacity-sizin..."
2,cloud-account-setup.md,Account Setup,https://qdrant.tech/documentation/cloud-accoun...,[/documentation/cloud/qdrant-cloud-setup/]
3,cloud-api.md,Qdrant Cloud API,https://qdrant.tech/documentation/cloud-api/,[/documentation/qdrant-cloud-api/]
4,cloud-cli.md,Qdrant Cloud CLI,https://qdrant.tech/documentation/cloud-cli/,[]
5,cloud-getting-started.md,Getting Started,https://qdrant.tech/documentation/cloud-gettin...,[/documentation/cloud/getting-started/]
6,cloud-premium.md,Premium Tier,https://qdrant.tech/documentation/cloud-premium/,[/documentation/cloud/premium/]
7,cloud-pricing-payments.md,Billing & Payments,https://qdrant.tech/documentation/cloud-pricin...,"[aws-marketplace, gcp-marketplace, azure-marke..."
8,cloud-quickstart.md,Cloud Quickstart,https://qdrant.tech/documentation/cloud-quicks...,"[../cloud-quick-start, cloud-quick-start, clou..."
9,cloud-rbac/_index.md,Cloud RBAC,https://qdrant.tech/documentation/cloud-rbac/,[]


In [17]:
slug_paths = {document["path"] for document in parsed_documents if document["metadata"].get("slug")}

display(pages_df[pages_df["path"].isin(slug_paths)])


,path,page_title,page_url,aliases
14,cloud-tab.md,Welcome to Qdrant Cloud,https://qdrant.tech/documentation/cloud-intro/,[]
45,deploy-tab.md,Deploy Qdrant,https://qdrant.tech/documentation/deploy-intro/,"[/documentation/deploy-intro, /documentation/c..."
58,ecosystem-tab.md,Explore the Qdrant Ecosystem,https://qdrant.tech/documentation/ecosystem/,[/documentation/build/]


In [18]:
page_url_by_path = dict(zip(pages_df["path"], pages_df["page_url"]))


def slugify_heading(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "-", text)

    return text.strip("-")


section_records = []

for path, group in heading_hierarchy_df.groupby("path", sort=False):
    page_url = page_url_by_path[path]
    anchor_counts = {}

    for row in group.itertuples(index=False):
        anchor = slugify_heading(row.heading)

        duplicate_index = anchor_counts.get(anchor, 0)
        anchor_counts[anchor] = duplicate_index + 1

        if duplicate_index:
            anchor = f"{anchor}-{duplicate_index}"

        breadcrumbs = [row.page_title]

        for heading in row.heading_path:
            if not breadcrumbs or heading != breadcrumbs[-1]:
                breadcrumbs.append(heading)

        section_records.append(
            {
                "path": row.path,
                "page_title": row.page_title,
                "heading_index": row.heading_index,
                "level": row.level,
                "section_title": row.heading,
                "breadcrumbs": breadcrumbs,
                "page_url": page_url,
                "section_url": f"{page_url}#{anchor}",
            }
        )

sections_df = pd.DataFrame(section_records)

print(f"Sections: {len(sections_df):,}")

display(sections_df.head(20))


Sections: 2,701


,path,page_title,heading_index,level,section_title,breadcrumbs,page_url,section_url
0,_index.md,Documentation,0,1,Qdrant Documentation,"[Documentation, Qdrant Documentation]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#qdrant-docu...
1,_index.md,Documentation,1,2,Getting Started,"[Documentation, Qdrant Documentation, Getting ...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#getting-sta...
2,_index.md,Documentation,2,2,Develop,"[Documentation, Qdrant Documentation, Develop]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#develop
3,_index.md,Documentation,3,2,Deploy,"[Documentation, Qdrant Documentation, Deploy]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#deploy
4,_index.md,Documentation,4,2,Ecosystem,"[Documentation, Qdrant Documentation, Ecosystem]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#ecosystem
5,_index.md,Documentation,5,2,Tutorials & Examples,"[Documentation, Qdrant Documentation, Tutorial...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#tutorials-e...
6,_index.md,Documentation,6,2,Learn,"[Documentation, Qdrant Documentation, Learn]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#learn
7,_index.md,Documentation,7,2,API Reference,"[Documentation, Qdrant Documentation, API Refe...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#api-reference
8,capacity-planning.md,Capacity Planning,0,1,Capacity Planning,[Capacity Planning],https://qdrant.tech/documentation/capacity-pla...,https://qdrant.tech/documentation/capacity-pla...
9,capacity-planning.md,Capacity Planning,1,2,Calculating RAM and Disk Size,"[Capacity Planning, Calculating RAM and Disk S...",https://qdrant.tech/documentation/capacity-pla...,https://qdrant.tech/documentation/capacity-pla...


In [19]:
display(
    sections_df[sections_df["path"] == "search/filtering.md"][
        ["level", "section_title", "breadcrumbs", "section_url"]
    ].head(20)
)


,level,section_title,breadcrumbs,section_url
2030,1,Filtering,[Filtering],https://qdrant.tech/documentation/search/filte...
2031,2,Filtering clauses,"[Filtering, Filtering clauses]",https://qdrant.tech/documentation/search/filte...
2032,3,Must,"[Filtering, Filtering clauses, Must]",https://qdrant.tech/documentation/search/filte...
2033,3,Should,"[Filtering, Filtering clauses, Should]",https://qdrant.tech/documentation/search/filte...
2034,3,Must Not,"[Filtering, Filtering clauses, Must Not]",https://qdrant.tech/documentation/search/filte...
2035,3,Clauses combination,"[Filtering, Filtering clauses, Clauses combina...",https://qdrant.tech/documentation/search/filte...
2036,2,Filtering conditions,"[Filtering, Filtering conditions]",https://qdrant.tech/documentation/search/filte...
2037,3,Match,"[Filtering, Filtering conditions, Match]",https://qdrant.tech/documentation/search/filte...
2038,3,Match Any,"[Filtering, Filtering conditions, Match Any]",https://qdrant.tech/documentation/search/filte...
2039,3,Match Except,"[Filtering, Filtering conditions, Match Except]",https://qdrant.tech/documentation/search/filte...


### 4.5 Parsed Section Schema

Each document is converted into structured sections while preserving the original Markdown content and the hierarchical metadata derived in the previous steps.

Section boundaries are defined by Markdown headings. Content before the first heading is preserved as a page-level section so that no source text is discarded.

In [20]:
section_metadata_lookup = {
    (row.path, row.heading_index): {
        "page_title": row.page_title,
        "section_title": row.section_title,
        "level": row.level,
        "breadcrumbs": row.breadcrumbs,
        "page_url": row.page_url,
        "section_url": row.section_url,
    }
    for row in sections_df.itertuples(index=False)
}

parsed_sections = []

for document in tqdm(parsed_documents, desc="Building parsed sections", unit="document"):
    path = document["path"]
    content = document["content"]
    lines = content.splitlines()

    tokens = document_tokens[path]

    heading_tokens = [token for token in tokens if token.type == "heading_open"]

    page_url = page_url_by_path[path]
    page_title = document["metadata"]["title"]

    # Preserve content before the first heading.
    first_heading_line = heading_tokens[0].map[0] if heading_tokens else len(lines)

    preamble = "\n".join(lines[:first_heading_line]).strip()

    if preamble or not heading_tokens:
        parsed_sections.append(
            {
                "path": path,
                "page_title": page_title,
                "section_index": 0,
                "level": 0,
                "section_title": page_title,
                "breadcrumbs": [page_title],
                "page_url": page_url,
                "section_url": page_url,
                "section_text": (preamble if preamble else content.strip()),
                "metadata": document["metadata"],
            }
        )

    section_offset = 1 if (preamble or not heading_tokens) else 0

    for heading_index, token in enumerate(heading_tokens):
        section_start = token.map[1]

        if heading_index + 1 < len(heading_tokens):
            section_end = heading_tokens[heading_index + 1].map[0]
        else:
            section_end = len(lines)

        section_text = "\n".join(lines[section_start:section_end]).strip()

        metadata = section_metadata_lookup[(path, heading_index)]

        parsed_sections.append(
            {
                "path": path,
                "page_title": metadata["page_title"],
                "section_index": heading_index + section_offset,
                "level": metadata["level"],
                "section_title": metadata["section_title"],
                "breadcrumbs": metadata["breadcrumbs"],
                "page_url": metadata["page_url"],
                "section_url": metadata["section_url"],
                "section_text": section_text,
                "metadata": document["metadata"],
            }
        )

parsed_sections_df = pd.DataFrame(parsed_sections)

print(f"Documents: {len(parsed_documents):,}\n" f"Parsed sections: {len(parsed_sections_df):,}")

display(parsed_sections_df.head(20))


Building parsed sections: 100%|██████████| 335/335 [00:00<00:00, 31818.92document/s]


Documents: 335
Parsed sections: 2,735


,path,page_title,section_index,level,section_title,breadcrumbs,page_url,section_url,section_text,metadata
0,_index.md,Documentation,0,1,Qdrant Documentation,"[Documentation, Qdrant Documentation]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#qdrant-docu...,Qdrant is an AI-native vector search engine fo...,"{'title': 'Documentation', 'short_description'..."
1,_index.md,Documentation,1,2,Getting Started,"[Documentation, Qdrant Documentation, Getting ...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#getting-sta...,- [Local Quickstart](/documentation/quickstart...,"{'title': 'Documentation', 'short_description'..."
2,_index.md,Documentation,2,2,Develop,"[Documentation, Qdrant Documentation, Develop]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#develop,- [Manage Data](/documentation/manage-data/ind...,"{'title': 'Documentation', 'short_description'..."
3,_index.md,Documentation,3,2,Deploy,"[Documentation, Qdrant Documentation, Deploy]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#deploy,- [Deploy Overview](/documentation/deploy-intr...,"{'title': 'Documentation', 'short_description'..."
4,_index.md,Documentation,4,2,Ecosystem,"[Documentation, Qdrant Documentation, Ecosystem]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#ecosystem,- [Frameworks](/documentation/frameworks/index...,"{'title': 'Documentation', 'short_description'..."
5,_index.md,Documentation,5,2,Tutorials & Examples,"[Documentation, Qdrant Documentation, Tutorial...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#tutorials-e...,- [Tutorials](/documentation/tutorials-lp-over...,"{'title': 'Documentation', 'short_description'..."
6,_index.md,Documentation,6,2,Learn,"[Documentation, Qdrant Documentation, Learn]",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#learn,- [Articles](/articles/index.md) — Long-form a...,"{'title': 'Documentation', 'short_description'..."
7,_index.md,Documentation,7,2,API Reference,"[Documentation, Qdrant Documentation, API Refe...",https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#api-reference,- [Qdrant API Reference](https://api.qdrant.te...,"{'title': 'Documentation', 'short_description'..."
8,capacity-planning.md,Capacity Planning,0,1,Capacity Planning,[Capacity Planning],https://qdrant.tech/documentation/capacity-pla...,https://qdrant.tech/documentation/capacity-pla...,Sizing a Qdrant cluster means estimating how m...,"{'title': 'Capacity Planning', 'short_descript..."
9,capacity-planning.md,Capacity Planning,1,2,Calculating RAM and Disk Size,"[Capacity Planning, Calculating RAM and Disk S...",https://qdrant.tech/documentation/capacity-pla...,https://qdrant.tech/documentation/capacity-pla...,Estimate how much RAM and disk each collection...,"{'title': 'Capacity Planning', 'short_descript..."


In [21]:
empty_sections = parsed_sections_df["section_text"].str.strip().eq("").sum()

print(
    f"Parsed sections: {len(parsed_sections_df):,}\n"
    f"Empty sections: {empty_sections:,}\n"
    f"Median section size: {parsed_sections_df['section_text'].str.len().median():,.0f} chars\n"
    f"Maximum section size: {parsed_sections_df['section_text'].str.len().max():,} chars"
)

display(
    parsed_sections_df[parsed_sections_df["path"] == "search/filtering.md"][
        ["section_index", "level", "section_title", "breadcrumbs", "section_text"]
    ].head(20)
)


Parsed sections: 2,735
Empty sections: 103
Median section size: 578 chars
Maximum section size: 31,375 chars


,section_index,level,section_title,breadcrumbs,section_text
2058,0,1,Filtering,[Filtering],"With Qdrant, you can set conditions when searc..."
2059,1,2,Filtering clauses,"[Filtering, Filtering clauses]",Qdrant allows you to combine conditions in cla...
2060,2,3,Must,"[Filtering, Filtering clauses, Must]","When using `must`, the clause becomes `true` o..."
2061,3,3,Should,"[Filtering, Filtering clauses, Should]","When using `should`, the clause becomes `true`..."
2062,4,3,Must Not,"[Filtering, Filtering clauses, Must Not]","When using `must_not`, the clause becomes `tru..."
2063,5,3,Clauses combination,"[Filtering, Filtering clauses, Clauses combina...",It is also possible to use several clauses sim...
2064,6,2,Filtering conditions,"[Filtering, Filtering conditions]",Different types of values in payload correspon...
2065,7,3,Match,"[Filtering, Filtering conditions, Match]","{{< code-snippet path=""/documentation/headless..."
2066,8,3,Match Any,"[Filtering, Filtering conditions, Match Any]",*Available as of v1.1.0*\n\nIn case you want t...
2067,9,3,Match Except,"[Filtering, Filtering conditions, Match Except]",*Available as of v1.2.0*\n\nIn case you want t...


In [22]:
shortcode_pattern = re.compile(r"\{\{<\s*([a-zA-Z0-9_-]+)(?:\s+[^>]*)?\s*>\}\}")

shortcode_records = []

for row in parsed_sections_df.itertuples(index=False):
    for match in shortcode_pattern.finditer(row.section_text):
        shortcode_records.append(
            {
                "path": row.path,
                "section_title": row.section_title,
                "shortcode": match.group(1),
                "raw": match.group(0),
            }
        )

shortcodes_df = pd.DataFrame(shortcode_records)

print(
    f"Shortcode occurrences: {len(shortcodes_df):,}\n"
    f"Sections with shortcodes: {shortcodes_df['path'].nunique():,}"
)

display(
    shortcodes_df["shortcode"]
    .value_counts()
    .rename_axis("shortcode")
    .reset_index(name="occurrences")
)

display(shortcodes_df.head(20))


Shortcode occurrences: 441
Sections with shortcodes: 51


,shortcode,occurrences
0,code-snippet,417
1,figure,23
2,accordion,1


,path,section_title,shortcode,raw
0,cloud/cluster-monitoring.md,Alerts,accordion,{{< accordion >}}
1,edge/edge-bm25.md,Configure a Sparse Vector,code-snippet,"{{< code-snippet path=""/documentation/headless..."
2,edge/edge-bm25.md,Create a BM25 Embedder,code-snippet,"{{< code-snippet path=""/documentation/headless..."
3,edge/edge-bm25.md,Embed and Upsert Documents,code-snippet,"{{< code-snippet path=""/documentation/headless..."
4,edge/edge-bm25.md,Query,code-snippet,"{{< code-snippet path=""/documentation/headless..."
5,edge/edge-data-synchronization-patterns.md,Initialize Edge Shard from Existing Qdrant Col...,code-snippet,"{{< code-snippet path=""/documentation/headless..."
6,edge/edge-data-synchronization-patterns.md,Update Qdrant Edge with Server-Side Changes,code-snippet,"{{< code-snippet path=""/documentation/headless..."
7,edge/edge-data-synchronization-patterns.md,Update a Server Collection from an Edge Shard,code-snippet,"{{< code-snippet path=""/documentation/headless..."
8,edge/edge-data-synchronization-patterns.md,Update a Server Collection from an Edge Shard,code-snippet,"{{< code-snippet path=""/documentation/headless..."
9,edge/edge-data-synchronization-patterns.md,Update a Server Collection from an Edge Shard,code-snippet,"{{< code-snippet path=""/documentation/headless..."


#### Shortcode Resolution

Code snippet shortcodes are resolved against the downloaded `headless/snippets` sources before chunking.

The resolver preserves the ordering rules used by the Qdrant documentation while replacing Hugo placeholders with the actual Markdown code blocks.

In [23]:
CODE_SNIPPET_PATTERN = re.compile(r"\{\{<\s*code-snippet\b(?P<attrs>.*?)>\}\}", re.DOTALL)

SHORTCODE_ATTR_PATTERN = re.compile(r'([\w-]+)="([^"]*)"')


def parse_shortcode_attributes(text: str) -> dict:
    return dict(SHORTCODE_ATTR_PATTERN.findall(text))


@lru_cache(maxsize=2048)
def get_snippet_order(snippet_dir: Path, explicit_order: str | None = None) -> list[str]:
    if explicit_order:
        return explicit_order.split()

    current_dir = snippet_dir

    while current_dir == QDRANT_DOCS_DIR or QDRANT_DOCS_DIR in current_dir.parents:
        index_path = current_dir / "_index.md"

        if index_path.is_file():
            raw_text = index_path.read_text(encoding="utf-8")
            metadata, _ = frontmatter.parse(raw_text)

            snippets_order = metadata.get("snippetsOrder")

            if snippets_order:
                if isinstance(snippets_order, str):
                    return snippets_order.split()

                return list(snippets_order)

        if current_dir == QDRANT_DOCS_DIR:
            break

        current_dir = current_dir.parent

    return []


def resolve_code_snippet(match: re.Match) -> str:
    return resolve_code_snippet_text(match.group(0))


@lru_cache(maxsize=2048)
def resolve_code_snippet_text(shortcode: str) -> str:
    match = CODE_SNIPPET_PATTERN.fullmatch(shortcode)
    if match is None:
        raise ValueError(f"Invalid code shortcode: {shortcode}")
    attributes = parse_shortcode_attributes(match.group("attrs"))

    shortcode_path = attributes.get("path")

    if not shortcode_path:
        raise ValueError("code-snippet shortcode has no path")

    relative_path = shortcode_path.strip("/")

    if relative_path.startswith("documentation/"):
        relative_path = relative_path.removeprefix("documentation/")

    snippet_dir = QDRANT_DOCS_DIR / relative_path

    block = attributes.get("block")

    directories = [
        snippet_dir,
        (snippet_dir / "generated" / block if block else snippet_dir / "generated"),
    ]
    snippet_files = {}

    for directory in directories:
        if not directory.is_dir():
            continue

        for path in sorted(directory.iterdir()):
            if path.is_file() and path.suffix.lower() == ".md" and not path.name.startswith("_"):
                snippet_files[path.stem] = path.read_text(encoding="utf-8").strip()

    if not snippet_files:
        raise FileNotFoundError(f"No Markdown snippets found for {shortcode_path}")

    order = get_snippet_order(snippet_dir, attributes.get("order"))

    ordered_names = [name for name in order if name in snippet_files]

    remaining_names = sorted(name for name in snippet_files if name not in ordered_names)

    names = ordered_names + remaining_names

    return "\n\n".join(snippet_files[name] for name in names)


def resolve_section_shortcodes(text: str) -> tuple[str, list[dict]]:
    section_errors = []

    def replace(match: re.Match) -> str:
        try:
            return resolve_code_snippet(match)

        except Exception as exc:
            section_errors.append({"shortcode": match.group(0), "error": str(exc)})

            return match.group(0)

    resolved_text = CODE_SNIPPET_PATTERN.sub(replace, text)

    return resolved_text, section_errors


resolved_sections_df = parsed_sections_df.copy()

resolved_sections_df["section_text_raw"] = resolved_sections_df["section_text"]


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    resolution_results = list(
        tqdm(
            executor.map(resolve_section_shortcodes, resolved_sections_df["section_text_raw"]),
            total=len(resolved_sections_df),
            desc="Resolving code snippets",
            unit="section",
        )
    )


resolved_sections_df["section_text"] = [resolved_text for resolved_text, _ in resolution_results]

resolution_errors = [error for _, errors in resolution_results for error in errors]


print(
    f"Workers: {MAX_WORKERS}\n"
    f"Sections: {len(resolved_sections_df):,}\n"
    f"Resolution errors: {len(resolution_errors):,}"
)


Resolving code snippets: 100%|██████████| 2735/2735 [00:01<00:00, 2522.53section/s]

Workers: 16
Sections: 2,735
Resolution errors: 0


#### Source-Only Markup Diagnostics


Before extracting visual assets, the resolved section text is inspected for source-only HTML comments.

HTML comments can contain large blocks of Markdown or HTML that are present in the repository but are not rendered on the published documentation page. They should therefore be removed before both multimodal asset extraction and retrieval-text preparation, preventing hidden source content from entering either corpus.


In [24]:
HTML_COMMENT_PATTERN = re.compile(r"<!--.*?-->", re.DOTALL)


html_comment_records = []

for row in resolved_sections_df.itertuples(index=False):
    matches = list(HTML_COMMENT_PATTERN.finditer(row.section_text or ""))

    if not matches:
        continue

    html_comment_records.append(
        {
            "path": row.path,
            "section_index": row.section_index,
            "page_title": row.page_title,
            "section_title": row.section_title,
            "comment_occurrences": len(matches),
            "comment_chars": sum(len(match.group(0)) for match in matches),
        }
    )


html_comments_df = pd.DataFrame(html_comment_records)

print("Sections with HTML comments: " f"{len(html_comments_df):,}")

if not html_comments_df.empty:
    print(
        f"HTML comment occurrences: {html_comments_df['comment_occurrences'].sum():,}\n"
        f"Source-only characters: {html_comments_df['comment_chars'].sum():,}"
    )

    display(html_comments_df.sort_values("comment_chars", ascending=False).head(20))


Sections with HTML comments: 16
HTML comment occurrences: 23
Source-only characters: 11,068


,path,section_index,page_title,section_title,comment_occurrences,comment_chars
14,tutorials-lp-overview.md,5,Overview,Migrate to Qdrant,2,8485
4,tutorials-build-essentials/_index.md,0,Essential Examples,Integration Examples,4,761
15,tutorials-operations/_index.md,0,Operations & Scale,Operations & Scale Tutorials,3,523
2,ops-monitoring/memory-usage.md,3,Memory Usage,API,1,351
5,tutorials-build-essentials/agentic-rag-camelai...,0,Discord RAG Bot,Discord RAG Bot,1,114
9,tutorials-build-essentials/data-ingestion-begi...,0,S3 Ingestion with LangChain,S3 Ingestion with LangChain,1,109
7,tutorials-build-essentials/agentic-rag-crewai-...,5,Agentic RAG with CrewAI,Getting Started,1,103
6,tutorials-build-essentials/agentic-rag-crewai-...,0,Agentic RAG with CrewAI,Agentic RAG with CrewAI,1,102
8,tutorials-build-essentials/agentic-rag-crewai-...,14,Agentic RAG with CrewAI,Conclusion,1,102
12,tutorials-build-essentials/rag-deepseek.md,0,5-Minute RAG with DeepSeek,5-Minute RAG with DeepSeek,1,82


The detected HTML comments are source-only content rather than user-visible documentation. They are removed at this point so all subsequent image extraction and retrieval preprocessing operate on renderable documentation content.


In [25]:
renderable_sections_df = resolved_sections_df.copy()

renderable_sections_df["section_text"] = (
    renderable_sections_df["section_text"]
    .fillna("")
    .map(lambda text: HTML_COMMENT_PATTERN.sub("", text))
    .str.strip()
)


remaining_html_comment_starts = (
    renderable_sections_df["section_text"].map(lambda text: "<!--" in text).sum()
)


print(
    f"Sections after source-markup cleanup: {len(renderable_sections_df):,}\n"
    f"Remaining HTML comment markers: {remaining_html_comment_starts:,}"
)

assert remaining_html_comment_starts == 0


Sections after source-markup cleanup: 2,735
Remaining HTML comment markers: 0


#### Figure Metadata Extraction

Figure shortcodes are extracted as structured metadata instead of being discarded.

The image source, alternative text, caption, and parent section metadata are preserved for a later multimodal retrieval extension.

In [26]:
FIGURE_PATTERN = re.compile(r"\{\{<\s*figure\b(?P<attrs>.*?)>\}\}", re.DOTALL)

FIGURE_ATTR_PATTERN = re.compile(r'([\w-]+)=(?:"([^"]*)"|\'([^\']*)\'|([^\s>]+))')


def parse_figure_attributes(text: str) -> dict:
    attributes = {}

    for match in FIGURE_ATTR_PATTERN.finditer(text):
        key = match.group(1)

        value = next(value for value in match.groups()[1:] if value is not None)

        attributes[key] = value

    return attributes


figure_records = []

for row in tqdm(
    renderable_sections_df.itertuples(index=False),
    total=len(renderable_sections_df),
    desc="Extracting figure metadata",
    unit="section",
):
    for match in FIGURE_PATTERN.finditer(row.section_text):
        attributes = parse_figure_attributes(match.group("attrs"))

        figure_records.append(
            {
                "path": row.path,
                "section_index": row.section_index,
                "page_title": row.page_title,
                "section_title": row.section_title,
                "page_url": row.page_url,
                "section_url": row.section_url,
                "src": attributes.get("src"),
                "alt": attributes.get("alt"),
                "caption": attributes.get("caption"),
                "raw_shortcode": match.group(0),
            }
        )

figures_df = pd.DataFrame(figure_records)

print(
    f"Figure shortcodes: {len(figures_df):,}\n"
    f"Figures with src: {figures_df['src'].notna().sum():,}\n"
    f"Figures with alt: {figures_df['alt'].notna().sum():,}\n"
    f"Figures with caption: {figures_df['caption'].notna().sum():,}"
)

display(figures_df.head(20))


Extracting figure metadata: 100%|██████████| 2735/2735 [00:00<00:00, 792827.52section/s]

Figure shortcodes: 23
Figures with src: 23
Figures with alt: 4
Figures with caption: 23


,path,section_index,page_title,section_title,page_url,section_url,src,alt,caption,raw_shortcode
0,manage-data/multitenancy.md,1,Multitenancy,Partition by Payload,https://qdrant.tech/documentation/manage-data/...,https://qdrant.tech/documentation/manage-data/...,/docs/defragmentation.png,Tenants defragmentation with is_tenant,"Grouping tenants together by tenant ID, if `is...","{{< figure src=""/docs/defragmentation.png"" alt..."
1,manage-data/multitenancy.md,6,Multitenancy,Tiered Multitenancy,https://qdrant.tech/documentation/manage-data/...,https://qdrant.tech/documentation/manage-data/...,/docs/tenant-promotion.png,Tiered multitenancy with tenant promotion,Tiered multitenancy with tenant promotion,"{{< figure src=""/docs/tenant-promotion.png"" al..."
2,manage-data/points.md,6,Points,Update Mode,https://qdrant.tech/documentation/manage-data/...,https://qdrant.tech/documentation/manage-data/...,/docs/embedding-model-migration.png,None,Embedding model migration in blue-green deploy...,"{{< figure src=""/docs/embedding-model-migratio..."
3,manage-data/quantization.md,8,Quantization,1.5-Bit and 2-Bit Quantization,https://qdrant.tech/documentation/manage-data/...,https://qdrant.tech/documentation/manage-data/...,/docs/2-bit-quantization.png,None,2-bit quantization,{{<figure src=/docs/2-bit-quantization.png cap...
4,manage-data/quantization.md,9,Quantization,Asymmetric Quantization,https://qdrant.tech/documentation/manage-data/...,https://qdrant.tech/documentation/manage-data/...,/docs/asymmetric-quantization.png,None,Asymmetric quantization,{{<figure src=/docs/asymmetric-quantization.pn...
5,overview/vector-search.md,1,Understanding Vector Search in Qdrant,A Brief History of Search,https://qdrant.tech/documentation/overview/vec...,https://qdrant.tech/documentation/overview/vec...,/docs/gettingstarted/inverted-index.png,None,A simplified version of the inverted index.,{{< figure src=/docs/gettingstarted/inverted-i...
6,overview/vector-search.md,1,Understanding Vector Search in Qdrant,A Brief History of Search,https://qdrant.tech/documentation/overview/vec...,https://qdrant.tech/documentation/overview/vec...,/docs/gettingstarted/tokenization.png,None,The process of tokenization with an additional...,{{< figure src=/docs/gettingstarted/tokenizati...
7,overview/vector-search.md,1,Understanding Vector Search in Qdrant,A Brief History of Search,https://qdrant.tech/documentation/overview/vec...,https://qdrant.tech/documentation/overview/vec...,/docs/gettingstarted/query.png,None,An example of a query vectorized to sparse for...,{{< figure src=/docs/gettingstarted/query.png ...
8,overview/vector-search.md,2,Understanding Vector Search in Qdrant,The Tower of Babel,https://qdrant.tech/documentation/overview/vec...,https://qdrant.tech/documentation/overview/vec...,/docs/gettingstarted/babel.jpg,None,"The Tower of Babel, Pieter Bruegel.",{{< figure src=/docs/gettingstarted/babel.jpg ...
9,overview/vector-search.md,3,Understanding Vector Search in Qdrant,The Representation Revolution,https://qdrant.tech/documentation/overview/vec...,https://qdrant.tech/documentation/overview/vec...,/docs/gettingstarted/input.png,None,"Input queries contain different words, but the...",{{< figure src=/docs/gettingstarted/input.png ...


#### Markdown Image Extraction

Standard Markdown images are extracted separately from Hugo figure shortcodes.

Their source paths and alternative text are preserved together with the parent page and section metadata for the multimodal retrieval pipeline.

In [27]:
def extract_markdown_images(row) -> list[dict]:
    parser = MarkdownIt("commonmark")
    tokens = parser.parse(row.section_text)

    records = []

    for token in tokens:
        if token.type != "inline" or not token.children:
            continue

        for child in token.children:
            if child.type != "image":
                continue

            records.append(
                {
                    "path": row.path,
                    "section_index": row.section_index,
                    "page_title": row.page_title,
                    "section_title": row.section_title,
                    "page_url": row.page_url,
                    "section_url": row.section_url,
                    "src": child.attrGet("src"),
                    "alt": child.content or None,
                    "caption": None,
                    "image_type": "markdown",
                }
            )

    return records


section_rows = list(renderable_sections_df.itertuples(index=False))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    markdown_image_results = list(
        tqdm(
            executor.map(extract_markdown_images, section_rows),
            total=len(section_rows),
            desc="Extracting Markdown images",
            unit="section",
        )
    )


markdown_image_records = [record for records in markdown_image_results for record in records]

markdown_images_df = pd.DataFrame(markdown_image_records)

print(f"Workers: {MAX_WORKERS}\n" f"Markdown images: {len(markdown_images_df):,}")

if not markdown_images_df.empty:
    print(
        f"Images with src: {markdown_images_df['src'].notna().sum():,}\n"
        f"Images with alt: {markdown_images_df['alt'].notna().sum():,}\n"
        f"Unique image sources: {markdown_images_df['src'].nunique():,}"
    )

    display(markdown_images_df.head(20))


Extracting Markdown images: 100%|██████████| 2735/2735 [00:00<00:00, 5299.39section/s]

Workers: 16
Markdown images: 296
Images with src: 296
Images with alt: 294
Unique image sources: 278


,path,section_index,page_title,section_title,page_url,section_url,src,alt,caption,image_type
0,cloud-account-setup.md,2,Account Setup,The Qdrant Cloud Console,https://qdrant.tech/documentation/cloud-accoun...,https://qdrant.tech/documentation/cloud-accoun...,/documentation/cloud/console-overview.png,Qdrant Cloud Console overview,None,markdown
1,cloud-account-setup.md,3,Account Setup,Switching Between Accounts,https://qdrant.tech/documentation/cloud-accoun...,https://qdrant.tech/documentation/cloud-accoun...,/documentation/cloud/account-switcher.png,Switching between accounts,None,markdown
2,cloud-account-setup.md,4,Account Setup,Creating Additional Accounts,https://qdrant.tech/documentation/cloud-accoun...,https://qdrant.tech/documentation/cloud-accoun...,/documentation/cloud/create-account-modal.png,Create a new account,None,markdown
3,cloud-account-setup.md,5,Account Setup,Managing Accounts,https://qdrant.tech/documentation/cloud-accoun...,https://qdrant.tech/documentation/cloud-accoun...,/documentation/cloud/accounts-list.png,Managing accounts,None,markdown
4,cloud-account-setup.md,6,Account Setup,Account Settings,https://qdrant.tech/documentation/cloud-accoun...,https://qdrant.tech/documentation/cloud-accoun...,/documentation/cloud/account-settings.png,Account settings,None,markdown
5,cloud-api.md,3,Qdrant Cloud API,Authentication,https://qdrant.tech/documentation/cloud-api/,https://qdrant.tech/documentation/cloud-api/#a...,/documentation/cloud/authentication.png,Authentication,None,markdown
6,cloud-pricing-payments.md,1,Billing & Payments,Billing,https://qdrant.tech/documentation/cloud-pricin...,https://qdrant.tech/documentation/cloud-pricin...,/documentation/cloud/payment-options.png,Payment Options,None,markdown
7,cloud-rbac/role-management.md,1,Role Management,Built-In Roles,https://qdrant.tech/documentation/cloud-rbac/r...,https://qdrant.tech/documentation/cloud-rbac/r...,/documentation/cloud/role-based-access-control...,image.png,None,markdown
8,cloud-rbac/role-management.md,2,Role Management,Custom Roles,https://qdrant.tech/documentation/cloud-rbac/r...,https://qdrant.tech/documentation/cloud-rbac/r...,/documentation/cloud/role-based-access-control...,image.png,None,markdown
9,cloud-rbac/role-management.md,3,Role Management,Creating a Custom Role,https://qdrant.tech/documentation/cloud-rbac/r...,https://qdrant.tech/documentation/cloud-rbac/r...,/documentation/cloud/role-based-access-control...,image.png,None,markdown


In [28]:
figure_images_df = figures_df[
    [
        "path",
        "section_index",
        "page_title",
        "section_title",
        "page_url",
        "section_url",
        "src",
        "alt",
        "caption",
    ]
].copy()

figure_images_df["image_type"] = "figure"


images_df = pd.concat([markdown_images_df, figure_images_df], ignore_index=True)


print(
    f"Image occurrences: {len(images_df):,}\n"
    f"Unique image sources: {images_df['src'].nunique():,}\n"
    f"Images with alt: {images_df['alt'].notna().sum():,}\n"
    f"Images with caption: {images_df['caption'].notna().sum():,}"
)

display(images_df[["path", "section_title", "image_type", "src", "alt", "caption"]].head(30))


Image occurrences: 319
Unique image sources: 300
Images with alt: 298
Images with caption: 23


,path,section_title,image_type,src,alt,caption
0,cloud-account-setup.md,The Qdrant Cloud Console,markdown,/documentation/cloud/console-overview.png,Qdrant Cloud Console overview,None
1,cloud-account-setup.md,Switching Between Accounts,markdown,/documentation/cloud/account-switcher.png,Switching between accounts,None
2,cloud-account-setup.md,Creating Additional Accounts,markdown,/documentation/cloud/create-account-modal.png,Create a new account,None
3,cloud-account-setup.md,Managing Accounts,markdown,/documentation/cloud/accounts-list.png,Managing accounts,None
4,cloud-account-setup.md,Account Settings,markdown,/documentation/cloud/account-settings.png,Account settings,None
5,cloud-api.md,Authentication,markdown,/documentation/cloud/authentication.png,Authentication,None
6,cloud-pricing-payments.md,Billing,markdown,/documentation/cloud/payment-options.png,Payment Options,None
7,cloud-rbac/role-management.md,Built-In Roles,markdown,/documentation/cloud/role-based-access-control...,image.png,None
8,cloud-rbac/role-management.md,Custom Roles,markdown,/documentation/cloud/role-based-access-control...,image.png,None
9,cloud-rbac/role-management.md,Creating a Custom Role,markdown,/documentation/cloud/role-based-access-control...,image.png,None


In [29]:
def classify_image_source(src: str) -> str:
    if src.startswith(("https://", "http://")):
        return "external_url"

    if src.startswith("/"):
        return "site_path"

    return "relative_path"


images_df["source_type"] = images_df["src"].fillna("").map(classify_image_source)

source_type_counts = (
    images_df["source_type"].value_counts().rename_axis("source_type").reset_index(name="images")
)

duplicate_sources = (
    images_df["src"]
    .value_counts()
    .loc[lambda counts: counts > 1]
    .rename_axis("src")
    .reset_index(name="occurrences")
)

display(source_type_counts)

print(f"Duplicate image sources: {len(duplicate_sources):,}")
display(duplicate_sources.head(20))


,source_type,images
0,site_path,302
1,external_url,17


Duplicate image sources: 5


,src,occurrences
0,https://colab.research.google.com/assets/colab...,15
1,/documentation/cloud/cloud-grafana-dashboard.png,3
2,/docs/fastapi_neural_search.png,2
3,/docs/workflow-neural-search.png,2
4,/docs/embedding-model-migration.png,2


In [30]:
external_images_df = (
    images_df[images_df["source_type"] == "external_url"][
        ["src", "path", "section_title", "alt", "caption"]
    ]
    .drop_duplicates(subset=["src"])
    .reset_index(drop=True)
)


def get_image_extension(src: str) -> str:
    path = urlparse(src).path
    return Path(path).suffix.lower() or "<none>"


unique_images_df = images_df.drop_duplicates(subset=["src"]).copy()

unique_images_df["extension"] = unique_images_df["src"].map(get_image_extension)

extension_counts = (
    unique_images_df["extension"].value_counts().rename_axis("extension").reset_index(name="images")
)


print(
    f"Unique images: {len(unique_images_df):,}\n"
    f"Unique external sources: {len(external_images_df):,}"
)

display(external_images_df)
display(extension_counts)


Unique images: 300
Unique external sources: 3


,src,path,section_title,alt,caption
0,https://colab.research.google.com/assets/colab...,embeddings/mistral.md,Mistral,Open In Colab,None
1,https://raw.githubusercontent.com/ramonpzg/mlo...,overview/what-is-qdrant.md,What Are Vector Databases?,dbs,None
2,https://raw.githubusercontent.com/ramonpzg/mlo...,overview/what-is-qdrant.md,High-Level Overview of Qdrant's Architecture,qdrant,None


,extension,images
0,.png,270
1,.jpg,18
2,.webp,10
3,.svg,2


In [31]:
RASTER_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

RAW_GITHUB_BASE_URL = (
    "https://raw.githubusercontent.com/qdrant/landing_page/"
    f"{QDRANT_DOCS_COMMIT}/qdrant-landing/static"
)


image_assets_df = unique_images_df.copy().reset_index(drop=True)


def build_image_download_url(row) -> str:
    if row["source_type"] == "external_url":
        return row["src"]

    return f"{RAW_GITHUB_BASE_URL}{row['src']}"


image_assets_df["download_url"] = image_assets_df.apply(build_image_download_url, axis=1)

image_assets_df["is_raster"] = image_assets_df["extension"].isin(RASTER_EXTENSIONS)

image_assets_df["is_colab_badge"] = image_assets_df["src"].str.contains(
    "colab.research.google.com", case=False, na=False
)

image_assets_df["use_for_multimodal"] = (
    image_assets_df["is_raster"] & ~image_assets_df["is_colab_badge"]
)


print(
    f"Unique image assets: {len(image_assets_df):,}\n"
    f"Raster images: {image_assets_df['is_raster'].sum():,}\n"
    f"Excluded Colab badges: {image_assets_df['is_colab_badge'].sum():,}\n"
    f"Multimodal candidates: {image_assets_df['use_for_multimodal'].sum():,}"
)

display(
    image_assets_df[["src", "source_type", "extension", "download_url", "use_for_multimodal"]].head(
        30
    )
)


Unique image assets: 300
Raster images: 298
Excluded Colab badges: 1
Multimodal candidates: 298


,src,source_type,extension,download_url,use_for_multimodal
0,/documentation/cloud/console-overview.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
1,/documentation/cloud/account-switcher.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
2,/documentation/cloud/create-account-modal.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
3,/documentation/cloud/accounts-list.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
4,/documentation/cloud/account-settings.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
5,/documentation/cloud/authentication.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
6,/documentation/cloud/payment-options.png,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
7,/documentation/cloud/role-based-access-control...,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
8,/documentation/cloud/role-based-access-control...,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True
9,/documentation/cloud/role-based-access-control...,site_path,.png,https://raw.githubusercontent.com/qdrant/landi...,True


In [32]:
DOWNLOAD_IMAGES = False  # Optional assets are not required for text retrieval.
IMAGES_DIR = RAW_DATA_DIR / "qdrant-images" / QDRANT_DOCS_COMMIT
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def build_local_image_path(row) -> Path:
    if row.source_type == "site_path":
        return IMAGES_DIR / row.src.lstrip("/")

    parsed_url = urlparse(row.src)
    filename = Path(parsed_url.path).name

    image_id = uuid.uuid5(uuid.NAMESPACE_URL, row.src).hex[:12]

    return IMAGES_DIR / "external" / f"{image_id}_{filename}"


def download_image(row) -> dict:
    if not DOWNLOAD_IMAGES:
        return {"src": row.src, "local_path": None, "status": "skipped", "error": None}
    target_path = build_local_image_path(row)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    if target_path.is_file() and target_path.stat().st_size > 0:
        return {
            "src": row.src,
            "local_path": str(target_path.relative_to(PROJECT_ROOT)),
            "status": "existing",
            "error": None,
        }

    try:
        request = Request(row.download_url, headers={"User-Agent": "qdrant-final-project"})

        with urlopen(request, timeout=60) as response:
            content = response.read()

        atomic_write(target_path, content)

        return {
            "src": row.src,
            "local_path": str(target_path.relative_to(PROJECT_ROOT)),
            "status": "downloaded",
            "error": None,
        }

    except Exception as exc:
        return {"src": row.src, "local_path": None, "status": "error", "error": str(exc)}


multimodal_images_df = (
    image_assets_df[image_assets_df["use_for_multimodal"]].copy().reset_index(drop=True)
)

image_rows = list(multimodal_images_df.itertuples(index=False))


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    download_results = list(
        tqdm(
            executor.map(download_image, image_rows),
            total=len(image_rows),
            desc="Downloading images",
            unit="image",
        )
    )


image_downloads_df = pd.DataFrame(download_results)

status_counts = (
    image_downloads_df["status"].value_counts().rename_axis("status").reset_index(name="images")
)

print(f"Workers: {MAX_WORKERS}\n" f"Images requested: {len(image_rows):,}")

display(status_counts)


Workers: 16
Images requested: 298


,status,images
0,skipped,298


In [33]:
def validate_image(row) -> dict:
    image_path = PROJECT_ROOT / row.local_path

    try:
        with Image.open(image_path) as image:
            image.verify()

        with Image.open(image_path) as image:
            width, height = image.size

            return {
                "src": row.src,
                "local_path": row.local_path,
                "valid": True,
                "format": image.format,
                "width": width,
                "height": height,
                "mode": image.mode,
                "size_bytes": image_path.stat().st_size,
                "error": None,
            }

    except Exception as exc:
        return {
            "src": row.src,
            "local_path": row.local_path,
            "valid": False,
            "format": None,
            "width": None,
            "height": None,
            "mode": None,
            "size_bytes": (image_path.stat().st_size if image_path.exists() else None),
            "error": str(exc),
        }


downloaded_image_rows = list(
    image_downloads_df[image_downloads_df["status"].isin(["downloaded", "existing"])].itertuples(
        index=False
    )
)


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    image_validation_results = list(
        tqdm(
            executor.map(validate_image, downloaded_image_rows),
            total=len(downloaded_image_rows),
            desc="Validating images",
            unit="image",
        )
    )


image_validation_df = pd.DataFrame(
    image_validation_results,
    columns=[
        "src",
        "local_path",
        "valid",
        "format",
        "width",
        "height",
        "mode",
        "size_bytes",
        "error",
    ],
)

print(
    f"Workers: {MAX_WORKERS}\n"
    f"Images checked: {len(image_validation_df):,}\n"
    f"Valid images: {image_validation_df['valid'].sum():,}\n"
    f"Invalid images: {(~image_validation_df['valid'].astype(bool)).sum():,}"
)

display(
    image_validation_df[
        ["local_path", "format", "width", "height", "mode", "size_bytes", "valid"]
    ].head(20)
)


Validating images: 0image [00:00, ?image/s]

Workers: 16
Images checked: 0
Valid images: 0
Invalid images: 0


,local_path,format,width,height,mode,size_bytes,valid


### 4.6 Retrieval Text Cleanup


Before chunking, the renderable documentation text is inspected for markup that should not remain in the retrieval corpus.

Source-only HTML comments were removed before visual asset extraction. Figure metadata and image assets have also already been preserved separately, so figure shortcodes can now be replaced with descriptive text. Accordion wrappers can be removed while retaining their textual content.


In [34]:
ACCORDION_OPEN_PATTERN = re.compile(r"\{\{<\s*accordion\s*>\}\}")

ACCORDION_CLOSE_PATTERN = re.compile(r"\{\{<\s*/accordion\s*>\}\}")


retrieval_text_diagnostics = pd.DataFrame(
    {
        "pattern": ["figure_shortcode", "accordion_open", "accordion_close", "html_comment_marker"],
        "sections": [
            renderable_sections_df["section_text"]
            .map(lambda text: bool(FIGURE_PATTERN.search(text or "")))
            .sum(),
            renderable_sections_df["section_text"]
            .map(lambda text: bool(ACCORDION_OPEN_PATTERN.search(text or "")))
            .sum(),
            renderable_sections_df["section_text"]
            .map(lambda text: bool(ACCORDION_CLOSE_PATTERN.search(text or "")))
            .sum(),
            renderable_sections_df["section_text"].map(lambda text: "<!--" in (text or "")).sum(),
        ],
    }
)

display(retrieval_text_diagnostics)


,pattern,sections
0,figure_shortcode,20
1,accordion_open,1
2,accordion_close,1
3,html_comment_marker,0


The remaining retrieval-specific markup can now be normalized without losing information needed by the multimodal extension. Figure shortcodes are replaced with their caption or alt text, and accordion wrappers are removed while their body content is preserved.


In [35]:
def replace_figure_with_text(match: re.Match) -> str:
    attributes = parse_figure_attributes(match.group("attrs"))

    caption = attributes.get("caption")
    alt = attributes.get("alt")

    description = caption or alt

    if not description:
        return ""

    return f"Image: {description}"


clean_sections_df = renderable_sections_df.copy()

clean_sections_df["section_text"] = (
    clean_sections_df["section_text"]
    .str.replace(FIGURE_PATTERN, replace_figure_with_text, regex=True)
    .str.replace(ACCORDION_OPEN_PATTERN, "", regex=True)
    .str.replace(ACCORDION_CLOSE_PATTERN, "", regex=True)
    .str.strip()
)


In [36]:
remaining_figures = (
    clean_sections_df["section_text"]
    .map(lambda text: bool(FIGURE_PATTERN.search(text or "")))
    .sum()
)

remaining_accordions = (
    clean_sections_df["section_text"]
    .map(
        lambda text: (
            bool(ACCORDION_OPEN_PATTERN.search(text or ""))
            or bool(ACCORDION_CLOSE_PATTERN.search(text or ""))
        )
    )
    .sum()
)

remaining_html_comments = (
    clean_sections_df["section_text"].map(lambda text: "<!--" in (text or "")).sum()
)


print(
    f"Sections: {len(clean_sections_df):,}\n"
    f"Unresolved figures: {remaining_figures:,}\n"
    f"Unresolved accordions: {remaining_accordions:,}\n"
    f"Remaining HTML comments: {remaining_html_comments:,}"
)

assert remaining_figures == 0
assert remaining_accordions == 0
assert remaining_html_comments == 0


Sections: 2,735
Unresolved figures: 0
Unresolved accordions: 0
Remaining HTML comments: 0


## 5. Structure-Aware Chunking

### 5.1 Chunking Strategy

The documentation has already been split along semantic Markdown section boundaries. These sections are treated as the primary retrieval units because they preserve the original document structure and topic hierarchy.

Before applying any additional splitting, section-size distributions are measured to characterize the corpus and identify sections that require further chunking.


In [37]:
section_size_df = clean_sections_df[
    ["path", "section_index", "level", "section_title", "section_text"]
].copy()

section_size_df["chars"] = section_size_df["section_text"].str.len()

section_size_df["words"] = section_size_df["section_text"].str.split().str.len()


print(
    f"Sections: {len(section_size_df):,}\n"
    f"Median chars: {section_size_df['chars'].median():,.0f}\n"
    f"P90 chars: {section_size_df['chars'].quantile(0.9):,.0f}\n"
    f"P95 chars: {section_size_df['chars'].quantile(0.95):,.0f}\n"
    f"P99 chars: {section_size_df['chars'].quantile(0.99):,.0f}\n"
    f"Maximum chars: {section_size_df['chars'].max():,}"
)

display(section_size_df.sort_values("chars", ascending=False).head(20))

fig = px.histogram(section_size_df, x="chars", nbins=100, title="Section Size Distribution")

fig.show()


Sections: 2,735
Median chars: 625
P90 chars: 2,803
P95 chars: 4,663
P99 chars: 10,814
Maximum chars: 31,375


,path,section_index,level,section_title,section_text,chars,words
75,cloud-quickstart.md,5,2,5. Populate the collection,"Next, we will populate the collection with men...",31375,3725
161,cloud/cluster-monitoring.md,8,3,Cluster System Metrics `/sys_metrics`,"In Qdrant Cloud, each Qdrant cluster will expo...",29086,1089
2613,tutorials-operations/time-based-sharding.md,3,2,Ingest Historical Data,"Time series data often arrives in streams, wit...",21939,2201
2287,tutorials-basics/search-beginners.md,5,2,4. Upload Data to the Cluster,The dataset consists of a list of science fict...,19950,2451
2133,search/search.md,16,3,Lookup in Groups,When the points in a group share large fields ...,18725,1659
1905,private-cloud/configuration.md,0,1,Private Cloud Configuration,The Qdrant Private Cloud helm chart has severa...,18700,2247
1558,ops-configuration/configuration.md,6,2,Configuration Options,The following YAML example describes the avail...,17447,2538
2158,search/text-search/text-filtering.md,6,2,Filter on Text Strings,"To filter on text values in a payload field, f...",16175,1217
1137,inference/matryoshka-models.md,0,1,Reduce Vector Dimensionality with Matryoshka M...,[Matryoshka Representation Learning](https://a...,15097,1037
1274,manage-data/points.md,20,2,Batch Update,_Available as of v1.5.0_\n\nYou can batch mult...,13842,667


Character counts are useful for corpus diagnostics, but the actual embedding limit is defined in tokens. The tokenizer for the same dense checkpoint used by retrieval, `BAAI/bge-small-en-v1.5`, is therefore used to measure section lengths and select an embedding-safe chunk budget rather than relying on an arbitrary character limit.

`AutoTokenizer` is used here only for exact token-budget measurement. Retrieval embeddings themselves are generated later with Qdrant's FastEmbed implementation of the course models.

The resulting strategy is:

1. Preserve complete Markdown sections whenever they fit within the target token budget.
2. Split only oversized sections.
3. Prefer semantic boundaries such as paragraphs, lists, tables, and fenced code blocks when additional splitting is required.
4. Preserve section hierarchy and neighboring-section metadata for every resulting chunk.


In [38]:
DENSE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
TOKENIZATION_BATCH_SIZE = 256

os.environ["TOKENIZERS_PARALLELISM"] = "true"

dense_tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_NAME, use_fast=True)

section_texts = clean_sections_df["section_text"].fillna("").tolist()

token_lengths = []

for batch_start in tqdm(
    range(0, len(section_texts), TOKENIZATION_BATCH_SIZE),
    desc="Counting dense-model tokens",
    unit="batch",
):
    batch_texts = section_texts[batch_start : batch_start + TOKENIZATION_BATCH_SIZE]

    encoded = dense_tokenizer(
        batch_texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_length=True,
        verbose=False,
    )

    token_lengths.extend(encoded["length"])


section_token_df = clean_sections_df[["path", "section_index", "level", "section_title"]].copy()

section_token_df["tokens"] = token_lengths


print(
    f"Model: {DENSE_MODEL_NAME}\n"
    f"Tokenizer max length: {dense_tokenizer.model_max_length:,}\n"
    f"Special-token overhead: {dense_tokenizer.num_special_tokens_to_add(pair=False)}\n"
    f"Sections: {len(section_token_df):,}\n"
    f"Median tokens: {section_token_df['tokens'].median():,.0f}\n"
    f"P90 tokens: {section_token_df['tokens'].quantile(0.9):,.0f}\n"
    f"P95 tokens: {section_token_df['tokens'].quantile(0.95):,.0f}\n"
    f"P99 tokens: {section_token_df['tokens'].quantile(0.99):,.0f}\n"
    f"Maximum tokens: {section_token_df['tokens'].max():,}"
)

display(section_token_df.sort_values("tokens", ascending=False).head(20))


Counting dense-model tokens: 100%|██████████| 11/11 [00:00<00:00, 55.17batch/s]


Model: BAAI/bge-small-en-v1.5
Tokenizer max length: 512
Special-token overhead: 2
Sections: 2,735
Median tokens: 173
P90 tokens: 867
P95 tokens: 1,378
P99 tokens: 3,258
Maximum tokens: 9,148


,path,section_index,level,section_title,tokens
75,cloud-quickstart.md,5,2,5. Populate the collection,9148
2613,tutorials-operations/time-based-sharding.md,3,2,Ingest Historical Data,6552
2133,search/search.md,16,3,Lookup in Groups,5762
2287,tutorials-basics/search-beginners.md,5,2,4. Upload Data to the Cluster,5472
2096,search/hybrid-queries.md,8,3,Re-Scoring Examples,4922
2158,search/text-search/text-filtering.md,6,2,Filter on Text Strings,4908
2461,tutorials-develop/code-search.md,7,2,Querying the codebase,4579
1309,manage-data/vectors.md,4,3,Multivectors,4243
1299,manage-data/quantization.md,23,3,Memory and Speed Tuning,4114
1137,inference/matryoshka-models.md,0,1,Reduce Vector Dimensionality with Matryoshka M...,4100


The token distribution is used to determine whether section boundaries alone are sufficient as final embedding units. Sections that exceed the dense model context cannot be embedded intact without truncation.

Therefore, short sections remain intact, while only sections exceeding the selected token budget are split further. This preserves the semantic structure of the documentation while preventing embedding-time truncation.


In [39]:
candidate_budgets = [256, 384, 448, 512]

budget_stats = pd.DataFrame(
    {
        "token_budget": candidate_budgets,
        "oversized_sections": [
            (section_token_df["tokens"] > budget).sum() for budget in candidate_budgets
        ],
    }
)

budget_stats["oversized_pct"] = budget_stats["oversized_sections"] / len(section_token_df) * 100

display(budget_stats)


,token_budget,oversized_sections,oversized_pct
0,256,976,35.685558
1,384,641,23.436929
2,448,538,19.670932
3,512,469,17.148080


A 448-token content budget is the leading candidate: it substantially reduces unnecessary splitting while leaving part of the model context window available for structural metadata.

Before fixing this threshold, the actual token overhead of the document hierarchy prefix is measured. The embedding input will include the section breadcrumbs in addition to the chunk body, while `chunk_text` itself remains unchanged in the Qdrant payload.

In [40]:
def build_embedding_prefix(row) -> str:
    page_title = str(row["page_title"]).strip()
    breadcrumbs = row["breadcrumbs"]
    hierarchy = (
        [str(item).strip() for item in breadcrumbs if str(item).strip()]
        if isinstance(breadcrumbs, (list, tuple))
        else [str(breadcrumbs).strip()]
    )
    section_path = hierarchy[1:] if hierarchy and hierarchy[0] == page_title else hierarchy
    prefix = [f"Document: {page_title}"]
    if section_path:
        prefix.append(f"Section: {' > '.join(section_path)}")
    return "\n".join(prefix) + "\n\n"


embedding_prefixes = clean_sections_df.apply(build_embedding_prefix, axis=1).tolist()

prefix_encoding = dense_tokenizer(
    embedding_prefixes,
    add_special_tokens=False,
    truncation=False,
    padding=False,
    return_length=True,
    verbose=False,
)

prefix_token_df = clean_sections_df[
    ["path", "section_index", "page_title", "section_title", "breadcrumbs"]
].copy()

prefix_token_df["prefix_tokens"] = prefix_encoding["length"]


print(
    f"Median prefix tokens: {prefix_token_df['prefix_tokens'].median():,.0f}\n"
    f"P90 prefix tokens: {prefix_token_df['prefix_tokens'].quantile(0.9):,.0f}\n"
    f"P95 prefix tokens: {prefix_token_df['prefix_tokens'].quantile(0.95):,.0f}\n"
    f"P99 prefix tokens: {prefix_token_df['prefix_tokens'].quantile(0.99):,.0f}\n"
    f"Maximum prefix tokens: {prefix_token_df['prefix_tokens'].max():,}"
)

display(prefix_token_df.sort_values("prefix_tokens", ascending=False).head(20))


Median prefix tokens: 18
P90 prefix tokens: 31
P95 prefix tokens: 36
P99 prefix tokens: 45
Maximum prefix tokens: 56


,path,section_index,page_title,section_title,breadcrumbs,prefix_tokens
2026,search-precision/reranking-semantic-search.md,13,Reranking for Better Search,There are three key parts to ingestion: Creati...,"[Reranking for Better Search, Implementing Vec...",56
2407,tutorials-build-essentials/video-anomaly-edge-...,14,"Video Anomaly Detection Part 1: Architecture, ...",Async Upload to VSS,"[Video Anomaly Detection Part 1: Architecture,...",53
2348,tutorials-build-essentials/data-ingestion-begi...,7,S3 Ingestion with LangChain,Example: Configuring LangChain to Load Files f...,"[S3 Ingestion with LangChain, S3 Ingestion wit...",53
2300,tutorials-build-essentials/agentic-rag-camelai...,8,Discord RAG Bot,Configure the QdrantStorage,"[Discord RAG Bot, Qdrant Agentic RAG Discord B...",52
421,edge/edge-synchronization-guide.md,3,Synchronize with a Server,2. Initialize an Immutable Edge Shard from a S...,"[Synchronize with a Server, Synchronize Qdrant...",51
2406,tutorials-build-essentials/video-anomaly-edge-...,13,"Video Anomaly Detection Part 1: Architecture, ...",Video Chunking for VSS,"[Video Anomaly Detection Part 1: Architecture,...",51
2408,tutorials-build-essentials/video-anomaly-edge-...,15,"Video Anomaly Detection Part 1: Architecture, ...",Docker Compose for VSS,"[Video Anomaly Detection Part 1: Architecture,...",51
2427,tutorials-build-essentials/video-anomaly-edge-...,16,Video Anomaly Detection Part 2: Edge-to-Cloud ...,Option B: Time-Windowed Eviction (`time_window`),[Video Anomaly Detection Part 2: Edge-to-Cloud...,49
2304,tutorials-build-essentials/agentic-rag-camelai...,12,Discord RAG Bot,Create a New Discord Bot,"[Discord RAG Bot, Qdrant Agentic RAG Discord B...",49
739,faq/qdrant-fundamentals.md,29,Qdrant Fundamentals,"If `limit` is higher than `hnsw_ef`, does Qdra...","[Qdrant Fundamentals, Frequently Asked Questio...",49


The measured structural-prefix overhead fits within the portion of the model context reserved for hierarchy metadata. Based on this measurement, the final chunking configuration uses a 448-token maximum body size.

Sections within this budget are preserved intact, while larger sections are split further along semantic Markdown boundaries.


In [41]:
CHUNK_TOKEN_BUDGET = 448

print(
    f"Chunk token budget: {CHUNK_TOKEN_BUDGET}\n"
    f"Dense model context: {dense_tokenizer.model_max_length}\n"
    f"Maximum observed prefix: {prefix_token_df['prefix_tokens'].max()}"
)


Chunk token budget: 448
Dense model context: 512
Maximum observed prefix: 56


#### Semantic Block Diagnostics

Oversized sections should be split along natural Markdown boundaries rather than at arbitrary token positions.

Before implementing the final splitter, oversized sections are decomposed into top-level Markdown blocks such as paragraphs, lists, blockquotes, and fenced code blocks. Their token sizes are measured to determine whether block-aware greedy packing is sufficient or whether an additional fallback is required for individually oversized blocks.

In [42]:
oversized_sections_df = clean_sections_df.merge(
    section_token_df[["path", "section_index", "tokens"]], on=["path", "section_index"], how="left"
)

oversized_sections_df = oversized_sections_df[
    oversized_sections_df["tokens"] > CHUNK_TOKEN_BUDGET
].reset_index(drop=True)


block_parser = MarkdownIt("commonmark")


def extract_top_level_blocks(row) -> list[dict]:
    text = row["section_text"] or ""
    lines = text.splitlines(keepends=True)

    tokens = block_parser.parse(text)

    records = []

    for token in tokens:
        if token.level != 0 or token.map is None:
            continue

        start_line, end_line = token.map

        if end_line <= start_line:
            continue

        block_text = "".join(lines[start_line:end_line]).strip()

        if not block_text:
            continue

        records.append(
            {
                "path": row["path"],
                "section_index": row["section_index"],
                "block_index": len(records),
                "block_type": token.type.removesuffix("_open"),
                "block_text": block_text,
            }
        )

    # Defensive fallback for valid text not represented
    # by a top-level Markdown token.
    if not records and text.strip():
        records.append(
            {
                "path": row["path"],
                "section_index": row["section_index"],
                "block_index": 0,
                "block_type": "fallback",
                "block_text": text.strip(),
            }
        )

    return records


semantic_block_records = []

for _, row in tqdm(
    oversized_sections_df.iterrows(),
    total=len(oversized_sections_df),
    desc="Extracting semantic blocks",
    unit="section",
):
    semantic_block_records.extend(extract_top_level_blocks(row))


semantic_blocks_df = pd.DataFrame(semantic_block_records)

block_texts = semantic_blocks_df["block_text"].tolist()

block_token_lengths = []

for batch_start in tqdm(
    range(0, len(block_texts), TOKENIZATION_BATCH_SIZE), desc="Counting block tokens", unit="batch"
):
    batch_texts = block_texts[batch_start : batch_start + TOKENIZATION_BATCH_SIZE]

    encoded = dense_tokenizer(
        batch_texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_length=True,
        verbose=False,
    )

    block_token_lengths.extend(encoded["length"])


semantic_blocks_df["tokens"] = block_token_lengths

oversized_blocks_df = semantic_blocks_df[
    semantic_blocks_df["tokens"] > CHUNK_TOKEN_BUDGET
].sort_values("tokens", ascending=False)


print(
    f"Oversized sections: {len(oversized_sections_df):,}\n"
    f"Semantic blocks: {len(semantic_blocks_df):,}\n"
    f"Blocks over budget: {len(oversized_blocks_df):,}\n"
    f"Maximum block tokens: {semantic_blocks_df['tokens'].max():,}"
)

display(oversized_blocks_df.head(20))


Counting block tokens: 100%|██████████| 24/24 [00:00<00:00, 144.54batch/s]

Oversized sections: 538
Semantic blocks: 5,907
Blocks over budget: 128
Maximum block tokens: 3,992


,path,section_index,block_index,block_type,block_text,tokens
2656,ops-configuration/configuration.md,6,2,fence,```yaml\nlog_level: INFO\n\n# Logging configur...,3992
2975,private-cloud/configuration.md,0,1,fence,```yaml\noperator:\n # Amount of replicas for...,3986
60,cloud/cluster-monitoring.md,8,2,paragraph,| Name ...,2737
57,cloud/cluster-monitoring.md,3,1,bullet_list,- title: Memory Overutilized\n content: |\n ...,2700
892,hybrid-cloud/operator-configuration.md,0,2,fence,```yaml\n# Additional pod annotations\npodAnno...,1969
2963,private-cloud/api-reference.md,53,3,paragraph,| Field | Description | Default | Validation |...,1931
5081,tutorials-develop/code-search.md,7,12,paragraph,| module | file_name ...,1831
739,frameworks/_index.md,0,0,paragraph,| Framework ...,1561
31,cloud-quickstart.md,5,2,fence,```rust\n// generate embeddings and prepare po...,1560
33,cloud-quickstart.md,5,4,fence,"```java\nString[][] menuItems = {\n {""Pad T...",1543


#### Oversized Block Splitting

Most Markdown blocks already fit within the 448-token chunk budget and can be packed without modification. Blocks that exceed the budget require additional structure-aware splitting.

Specialized splitting is applied before greedy chunk assembly so that tables, lists, and fenced code blocks retain as much of their original structure as possible. A token-based fallback is reserved for content that cannot be divided safely using Markdown structure.

In [43]:
@lru_cache(maxsize=8192)
def token_data(text: str) -> tuple:
    encoded = dense_tokenizer.backend_tokenizer.encode(text, add_special_tokens=False)
    return tuple(encoded.ids), tuple(encoded.offsets)


def encode_tokens(text: str) -> tuple[int, ...]:
    return token_data(text)[0]


def count_tokens(text: str) -> int:
    return len(encode_tokens(text))


def source_windows(
    text: str, token_budget: int, overlap_ratio: float = 0.0, prefix: str = "", suffix: str = ""
) -> list[dict]:
    """Slice original characters, never reconstruct source code with decode()."""
    if token_budget < 1 or not 0 <= overlap_ratio < 1:
        raise ValueError("Require a positive budget and 0 <= overlap < 1.")
    _, offsets = token_data(text)
    if not offsets:
        return []

    def wrap(body):
        return "\n".join(part for part in (prefix, body, suffix) if part)

    records, start = [], 0
    overlap = int(token_budget * overlap_ratio)
    while start < len(offsets):
        end = min(start + token_budget, len(offsets))
        left = 0 if start == 0 else offsets[start][0]
        while end > start:
            right = len(text) if end == len(offsets) else offsets[end][0]
            body = text[left:right]
            candidate = wrap(body)
            if right > left and count_tokens(candidate) <= token_budget:
                break
            end -= 1
        if end == start:
            raise ValueError("Token budget cannot hold the wrapper and source fragment.")
        records.append(
            {
                "chunk_index": len(records),
                "chunk_text": candidate,
                "tokens": count_tokens(candidate),
                "source_start": left,
                "source_end": right,
            }
        )
        if end == len(offsets):
            break
        start = max(start + 1, end - overlap)
    return records


def split_tokens(text: str, token_budget: int = CHUNK_TOKEN_BUDGET) -> list[str]:
    return [row["chunk_text"] for row in source_windows(text, token_budget)]


def pack_units(
    units, token_budget: int, separator: str = "\n", prefix: str = "", suffix: str = ""
) -> list[str]:
    """Pack semantic units; retokenize joins because token counts are not additive."""

    def wrap(body):
        return "\n".join(part for part in (prefix, body, suffix) if part)

    parts, current = [], []
    for unit in units:
        if count_tokens(wrap(separator.join([*current, unit]))) <= token_budget:
            current.append(unit)
            continue
        if current:
            parts.append(wrap(separator.join(current)))
            current = []
        if count_tokens(wrap(unit)) <= token_budget:
            current = [unit]
        else:
            parts.extend(
                row["chunk_text"]
                for row in source_windows(unit, token_budget, prefix=prefix, suffix=suffix)
            )
    if current:
        parts.append(wrap(separator.join(current)))
    return [part for part in parts if part.strip()]


def split_lines_greedily(lines, token_budget, prefix="", suffix=""):
    return pack_units(lines, token_budget, prefix=prefix, suffix=suffix)


# Regression checks execute only during a future approved notebook run.
splitter_fixture = 'QdrantClient(host="localhost")\n    must_not = True\n'
fixture_windows = source_windows(splitter_fixture, 8)
assert "".join(row["chunk_text"] for row in fixture_windows) == splitter_fixture
assert all(row["tokens"] <= 8 for row in fixture_windows)
print("Source-preserving token windows and shared packing utilities ready.")


Source-preserving token windows and shared packing utilities ready.


#### Fenced Code Block Splitting

Large fenced code blocks are split by source lines while preserving the original opening fence, language identifier, and closing fence in every resulting chunk.

Whole lines are kept whenever possible. Only an individual line that cannot fit within the token budget is split at the tokenizer level as a final fallback. Every produced code fragment is validated against the same 448-token body budget.

In [44]:
FENCE_OPEN_PATTERN = re.compile(r"^(?P<indent>[ \t]*)(?P<marker>`{3,}|~{3,})(?P<info>.*)$")


def is_matching_fence_close(line: str, marker: str) -> bool:
    stripped = line.strip()

    if not stripped:
        return False

    marker_char = re.escape(marker[0])

    return bool(re.fullmatch(rf"{marker_char}{{{len(marker)},}}", stripped))


def build_fenced_part(opening_line: str, body_lines: list[str], closing_line: str) -> str:
    components = [opening_line]

    components.extend(body_lines)
    components.append(closing_line)

    return "\n".join(components).strip()


def split_wrapped_text_by_tokens(
    text: str, prefix: str, suffix: str, token_budget: int
) -> list[str]:
    return [
        row["chunk_text"]
        for row in source_windows(text, token_budget, prefix=prefix, suffix=suffix)
    ]


@lru_cache(maxsize=1024)
def split_fenced_code_block(text: str, token_budget: int = CHUNK_TOKEN_BUDGET) -> list[str]:
    text = text.strip()

    if not text:
        return []

    if count_tokens(text) <= token_budget:
        return [text]

    lines = text.splitlines()

    if not lines:
        return []

    opening_match = FENCE_OPEN_PATTERN.match(lines[0])

    if opening_match is None:
        return split_lines_greedily(lines, token_budget=token_budget)

    marker = opening_match.group("marker")
    opening_line = lines[0]

    has_closing_fence = len(lines) > 1 and is_matching_fence_close(lines[-1], marker)

    if has_closing_fence:
        closing_line = lines[-1]
        body_lines = lines[1:-1]
    else:
        # A valid Markdown fence may remain open until EOF.
        # Each retrieval fragment is closed explicitly so that
        # the resulting chunk remains structurally valid Markdown.
        closing_line = marker
        body_lines = lines[1:]

    if not body_lines:
        candidate = build_fenced_part(opening_line, [], closing_line)

        if count_tokens(candidate) <= token_budget:
            return [candidate]

        raise ValueError("Fence wrapper exceeds token budget.")

    return pack_units(body_lines, token_budget, prefix=opening_line, suffix=closing_line)


oversized_fences_df = semantic_blocks_df[
    (semantic_blocks_df["block_type"] == "fence")
    & (semantic_blocks_df["tokens"] > CHUNK_TOKEN_BUDGET)
].copy()


fence_split_records = []

for _, row in tqdm(
    oversized_fences_df.iterrows(),
    total=len(oversized_fences_df),
    desc="Splitting oversized code blocks",
    unit="block",
):
    parts = split_fenced_code_block(row["block_text"])

    part_token_lengths = [count_tokens(part) for part in parts]

    fence_split_records.append(
        {
            "path": row["path"],
            "section_index": row["section_index"],
            "block_index": row["block_index"],
            "original_tokens": row["tokens"],
            "parts": len(parts),
            "max_part_tokens": max(part_token_lengths, default=0),
            "within_budget": all(tokens <= CHUNK_TOKEN_BUDGET for tokens in part_token_lengths),
        }
    )


fence_split_df = pd.DataFrame(fence_split_records)


print(
    f"Oversized fenced blocks: {len(oversized_fences_df):,}\n"
    f"Generated code fragments: {fence_split_df['parts'].sum():,}\n"
    f"Maximum fragment tokens: {fence_split_df['max_part_tokens'].max():,}\n"
    f"Budget violations: {(~fence_split_df['within_budget']).sum():,}"
)

display(fence_split_df.sort_values("original_tokens", ascending=False).head(20))


Splitting oversized code blocks: 100%|██████████| 88/88 [00:01<00:00, 68.32block/s]

Oversized fenced blocks: 88
Generated code fragments: 219
Maximum fragment tokens: 448
Budget violations: 0


,path,section_index,block_index,original_tokens,parts,max_part_tokens,within_budget
39,ops-configuration/configuration.md,6,2,3992,10,447,True
42,private-cloud/configuration.md,0,1,3986,10,447,True
31,hybrid-cloud/operator-configuration.md,0,2,1969,5,447,True
1,cloud-quickstart.md,5,2,1560,4,446,True
3,cloud-quickstart.md,5,4,1543,4,437,True
5,cloud-quickstart.md,5,6,1533,4,437,True
2,cloud-quickstart.md,5,3,1508,4,445,True
4,cloud-quickstart.md,5,5,1481,4,439,True
0,cloud-quickstart.md,5,1,1460,4,437,True
16,fastembed/fastembed-colbert.md,5,9,1373,4,431,True


#### Markdown Table Diagnostics

CommonMark does not interpret GitHub-style pipe tables as a dedicated table token, so large Markdown tables may appear as ordinary paragraph blocks.

Oversized paragraph blocks are therefore inspected for pipe-table separator rows used in the Qdrant documentation source, including compact single-dash delimiters such as `|-|-|-|`. Detected tables are then analyzed before applying row-aware splitting.


In [45]:
TABLE_SEPARATOR_PATTERN = re.compile(
    r"""
    ^\s*
    \|?
    \s*:?-{1,}:?\s*
    (
        \|
        \s*:?-{1,}:?\s*
    )+
    \|?
    \s*$
    """,
    re.VERBOSE,
)


def is_markdown_table(text: str) -> bool:
    lines = [line for line in text.splitlines() if line.strip()]

    if len(lines) < 2:
        return False

    header_line = lines[0]
    separator_line = lines[1]

    return "|" in header_line and bool(TABLE_SEPARATOR_PATTERN.fullmatch(separator_line))


oversized_non_fence_df = semantic_blocks_df[
    (semantic_blocks_df["tokens"] > CHUNK_TOKEN_BUDGET)
    & (semantic_blocks_df["block_type"] != "fence")
].copy()


oversized_non_fence_df["is_markdown_table"] = oversized_non_fence_df["block_text"].apply(
    is_markdown_table
)


oversized_tables_df = oversized_non_fence_df[oversized_non_fence_df["is_markdown_table"]].copy()


table_diagnostic_records = []

for _, row in oversized_tables_df.iterrows():
    lines = [line for line in row["block_text"].splitlines() if line.strip()]

    header_line = lines[0]
    separator_line = lines[1]
    data_rows = lines[2:]

    repeated_header = "\n".join([header_line, separator_line])

    header_tokens = count_tokens(repeated_header)

    row_token_lengths = [count_tokens(data_row) for data_row in data_rows]

    header_row_token_lengths = [
        count_tokens("\n".join([repeated_header, data_row])) for data_row in data_rows
    ]

    table_diagnostic_records.append(
        {
            "path": row["path"],
            "section_index": row["section_index"],
            "block_index": row["block_index"],
            "table_tokens": row["tokens"],
            "data_rows": len(data_rows),
            "header_tokens": header_tokens,
            "max_row_tokens": max(row_token_lengths, default=0),
            "max_header_row_tokens": max(header_row_token_lengths, default=header_tokens),
            "header_fits_budget": (header_tokens < CHUNK_TOKEN_BUDGET),
            "all_rows_fit_with_header": all(
                tokens <= CHUNK_TOKEN_BUDGET for tokens in header_row_token_lengths
            ),
        }
    )


table_diagnostics_df = pd.DataFrame(table_diagnostic_records)


type_counts = (
    oversized_non_fence_df["block_type"]
    .value_counts()
    .rename_axis("block_type")
    .reset_index(name="count")
)


problem_tables_df = table_diagnostics_df[
    ~table_diagnostics_df["all_rows_fit_with_header"]
].sort_values("max_header_row_tokens", ascending=False)


print(
    f"Oversized blocks after fences: {len(oversized_non_fence_df):,}\n"
    f"Detected oversized tables: {len(oversized_tables_df):,}\n"
    f"Maximum table tokens: {table_diagnostics_df['table_tokens'].max():,}\n"
    f"Maximum header tokens: {table_diagnostics_df['header_tokens'].max():,}\n"
    f"Maximum single row tokens: {table_diagnostics_df['max_row_tokens'].max():,}\n"
    f"Maximum header + row tokens: {table_diagnostics_df['max_header_row_tokens'].max():,}\n"
    f"Tables requiring special handling: {len(problem_tables_df):,}"
)


display(type_counts)

display(problem_tables_df.head(20))


Oversized blocks after fences: 40
Detected oversized tables: 29
Maximum table tokens: 2,737
Maximum header tokens: 389
Maximum single row tokens: 172
Maximum header + row tokens: 555
Tables requiring special handling: 7


,block_type,count
0,paragraph,29
1,bullet_list,7
2,ordered_list,4


,path,section_index,block_index,table_tokens,data_rows,header_tokens,max_row_tokens,max_header_row_tokens,header_fits_budget,all_rows_fit_with_header
24,tutorials-build-essentials/qdrant-n8n.md,9,20,625,2,389,166,555,True,False
7,examples/_index.md,2,1,1082,8,367,172,539,True,False
3,datasets.md,1,1,814,3,378,150,528,True,False
27,tutorials-develop/code-search.md,7,12,1831,10,359,156,515,True,False
25,tutorials-develop/code-search.md,7,4,1091,5,352,156,508,True,False
28,tutorials-develop/code-search.md,9,3,1077,5,340,150,490,True,False
26,tutorials-develop/code-search.md,7,8,1073,5,340,149,489,True,False


#### Markdown Table Splitting

Oversized Markdown tables are normally split by complete data rows while repeating the original header and separator in every fragment.

Some wide tables cannot fit a complete header together with an individual row within the 448-token budget. For these rare cases, the row is converted into explicit `column: value` pairs and packed into token-safe fragments. This preserves the relationship between column names and values without truncating or arbitrarily splitting the original table row.

In [46]:
def build_table_fragment(header_line: str, separator_line: str, data_rows: list[str]) -> str:
    return "\n".join([header_line, separator_line, *data_rows]).strip()


def parse_markdown_table_row(line: str) -> list[str]:
    line = line.strip()

    if line.startswith("|"):
        line = line[1:]

    if line.endswith("|"):
        line = line[:-1]

    cells = re.split(r"(?<!\\)\|", line)

    return [cell.replace(r"\|", "|").strip() for cell in cells]


def build_table_row_pairs(header_line: str, data_row: str) -> list[str]:
    headers = parse_markdown_table_row(header_line)

    values = parse_markdown_table_row(data_row)

    pair_count = max(len(headers), len(values))

    pairs = []

    for index in range(pair_count):
        header = headers[index] if index < len(headers) else f"column_{index + 1}"

        value = values[index] if index < len(values) else ""

        if not header and not value:
            continue

        pairs.append(f"- {header}: {value}")

    return pairs


def split_table_row_as_pairs(header_line: str, data_row: str, token_budget: int) -> list[str]:
    pair_lines = build_table_row_pairs(header_line, data_row)

    prefix = "Table row:"

    fragments = []
    current_lines = []

    for pair_line in pair_lines:
        candidate = "\n".join([prefix, *current_lines, pair_line]).strip()

        if count_tokens(candidate) <= token_budget:
            current_lines.append(pair_line)
            continue

        if current_lines:
            fragments.append("\n".join([prefix, *current_lines]).strip())

            current_lines = []

        single_pair = "\n".join([prefix, pair_line]).strip()

        if count_tokens(single_pair) <= token_budget:
            current_lines = [pair_line]
            continue

        prefix_tokens = count_tokens(prefix)

        available_budget = max(1, token_budget - prefix_tokens)

        pair_parts = split_tokens(pair_line, token_budget=available_budget)

        for pair_part in pair_parts:
            fragment = "\n".join([prefix, pair_part]).strip()

            if count_tokens(fragment) > token_budget:
                raise ValueError("Unable to split oversized " "table cell within token budget.")

            fragments.append(fragment)

    if current_lines:
        fragments.append("\n".join([prefix, *current_lines]).strip())

    return fragments


@lru_cache(maxsize=1024)
def split_markdown_table(text: str, token_budget: int = CHUNK_TOKEN_BUDGET) -> list[str]:
    lines = [line for line in text.splitlines() if line.strip()]

    if len(lines) < 2:
        return split_tokens(text, token_budget=token_budget)

    if not is_markdown_table(text):
        return split_tokens(text, token_budget=token_budget)

    if count_tokens(text) <= token_budget:
        return [text.strip()]

    header_line = lines[0]
    separator_line = lines[1]
    data_rows = lines[2:]

    header_fragment = build_table_fragment(header_line, separator_line, [])

    header_fits = count_tokens(header_fragment) < token_budget

    fragments = []
    current_rows = []

    for data_row in data_rows:
        single_row_fragment = build_table_fragment(header_line, separator_line, [data_row])

        row_fits_with_header = header_fits and count_tokens(single_row_fragment) <= token_budget

        if not row_fits_with_header:
            if current_rows:
                fragments.append(build_table_fragment(header_line, separator_line, current_rows))

                current_rows = []

            fragments.extend(split_table_row_as_pairs(header_line, data_row, token_budget))

            continue

        candidate = build_table_fragment(header_line, separator_line, current_rows + [data_row])

        if count_tokens(candidate) <= token_budget:
            current_rows.append(data_row)
            continue

        if current_rows:
            fragments.append(build_table_fragment(header_line, separator_line, current_rows))

        current_rows = [data_row]

    if current_rows:
        fragments.append(build_table_fragment(header_line, separator_line, current_rows))

    return [fragment for fragment in fragments if fragment.strip()]


table_split_records = []

for _, row in tqdm(
    oversized_tables_df.iterrows(),
    total=len(oversized_tables_df),
    desc="Splitting oversized tables",
    unit="table",
):
    fragments = split_markdown_table(row["block_text"])

    fragment_token_lengths = [count_tokens(fragment) for fragment in fragments]

    table_split_records.append(
        {
            "path": row["path"],
            "section_index": row["section_index"],
            "block_index": row["block_index"],
            "original_tokens": row["tokens"],
            "parts": len(fragments),
            "max_part_tokens": max(fragment_token_lengths, default=0),
            "within_budget": all(tokens <= CHUNK_TOKEN_BUDGET for tokens in fragment_token_lengths),
        }
    )


table_split_df = pd.DataFrame(table_split_records)


print(
    f"Oversized tables: {len(oversized_tables_df):,}\n"
    f"Generated table fragments: {table_split_df['parts'].sum():,}\n"
    f"Maximum fragment tokens: {table_split_df['max_part_tokens'].max():,}\n"
    f"Budget violations: {(~table_split_df['within_budget']).sum():,}"
)

display(table_split_df.sort_values("original_tokens", ascending=False).head(20))


Splitting oversized tables: 100%|██████████| 29/29 [00:00<00:00, 69.91table/s]

Oversized tables: 29
Generated table fragments: 103
Maximum fragment tokens: 448
Budget violations: 0


,path,section_index,block_index,original_tokens,parts,max_part_tokens,within_budget
0,cloud/cluster-monitoring.md,8,2,2737,12,440,True
18,private-cloud/api-reference.md,53,3,1931,5,439,True
27,tutorials-develop/code-search.md,7,12,1831,10,168,True
9,frameworks/_index.md,0,0,1561,6,442,True
25,tutorials-develop/code-search.md,7,4,1091,5,168,True
7,examples/_index.md,2,1,1082,8,448,True
28,tutorials-develop/code-search.md,9,3,1077,5,162,True
26,tutorials-develop/code-search.md,7,8,1073,5,161,True
20,security.md,10,3,892,3,446,True
6,examples/_index.md,1,1,846,3,424,True


#### Markdown List Diagnostics

The remaining oversized list blocks are inspected at the top-level list-item boundary.

Complete list items are preferred as atomic splitting units because they may contain nested paragraphs, code blocks, or sublists. Measuring their token sizes determines whether oversized lists can be split cleanly by item or whether a small number of individually oversized items require an additional fallback.

In [47]:
oversized_lists_df = (
    oversized_non_fence_df[
        oversized_non_fence_df["block_type"].isin(["bullet_list", "ordered_list"])
    ]
    .copy()
    .reset_index(drop=True)
)


def extract_top_level_list_items(text: str) -> list[str]:
    lines = text.splitlines(keepends=True)

    parser = MarkdownIt("commonmark")

    tokens = parser.parse(text)

    list_items = []

    for token in tokens:
        if token.type == "list_item_open" and token.level == 1 and token.map is not None:
            start_line, end_line = token.map

            item_text = "".join(lines[start_line:end_line]).strip()

            if item_text:
                list_items.append(item_text)

    return list_items


list_diagnostic_records = []

for _, row in tqdm(
    oversized_lists_df.iterrows(),
    total=len(oversized_lists_df),
    desc="Inspecting oversized lists",
    unit="list",
):
    list_items = extract_top_level_list_items(row["block_text"])

    item_token_lengths = [count_tokens(item) for item in list_items]

    list_diagnostic_records.append(
        {
            "path": row["path"],
            "section_index": (row["section_index"]),
            "block_index": (row["block_index"]),
            "block_type": (row["block_type"]),
            "list_tokens": (row["tokens"]),
            "top_level_items": len(list_items),
            "max_item_tokens": max(item_token_lengths, default=0),
            "oversized_items": sum(tokens > CHUNK_TOKEN_BUDGET for tokens in item_token_lengths),
        }
    )


list_diagnostics_df = pd.DataFrame(list_diagnostic_records)


print(
    f"Oversized lists: {len(oversized_lists_df):,}\n"
    f"Top-level list items: {list_diagnostics_df['top_level_items'].sum():,}\n"
    f"Maximum list-item tokens: {list_diagnostics_df['max_item_tokens'].max():,}\n"
    f"Individually oversized items: {list_diagnostics_df['oversized_items'].sum():,}"
)


display(list_diagnostics_df.sort_values("list_tokens", ascending=False))


Inspecting oversized lists: 100%|██████████| 11/11 [00:00<00:00, 406.61list/s]

Oversized lists: 11
Top-level list items: 80
Maximum list-item tokens: 400
Individually oversized items: 0


,path,section_index,block_index,block_type,list_tokens,top_level_items,max_item_tokens,oversized_items
1,cloud/cluster-monitoring.md,3,1,bullet_list,2700,11,400,0
9,security.md,17,1,bullet_list,935,3,399,0
6,production-checklist.md,5,1,bullet_list,578,7,115,0
8,security.md,9,1,bullet_list,573,3,300,0
0,_index.md,3,0,bullet_list,562,14,46,0
10,tutorials-build-essentials/qdrant-n8n.md,7,5,ordered_list,557,3,199,0
5,platforms/apify.md,1,1,ordered_list,539,4,266,0
7,search/search-relevance.md,1,18,bullet_list,537,18,62,0
2,hybrid-cloud/hybrid-cloud-setup.md,3,0,ordered_list,513,5,281,0
4,ops-configuration/configuration.md,1,1,ordered_list,512,4,195,0


#### Markdown List Splitting

Oversized Markdown lists are split at top-level list-item boundaries.

Complete items are packed whenever they fit. An item that exceeds the budget is split using source character offsets, with token-budget checks on the resulting fragments.

In [48]:
@lru_cache(maxsize=1024)
def split_markdown_list(text: str, token_budget: int = CHUNK_TOKEN_BUDGET) -> list[str]:
    if not text.strip():
        return []
    if count_tokens(text) <= token_budget:
        return [text]
    units = extract_top_level_list_items(text) or [text]
    # Very long individual items use the same source-preserving fallback.
    return pack_units(units, token_budget)


list_split_records = []

for _, row in tqdm(
    oversized_lists_df.iterrows(),
    total=len(oversized_lists_df),
    desc="Splitting oversized lists",
    unit="list",
):
    fragments = split_markdown_list(row["block_text"])

    fragment_token_lengths = [count_tokens(fragment) for fragment in fragments]

    list_split_records.append(
        {
            "path": row["path"],
            "section_index": row["section_index"],
            "block_index": row["block_index"],
            "block_type": row["block_type"],
            "original_tokens": row["tokens"],
            "parts": len(fragments),
            "max_part_tokens": max(fragment_token_lengths, default=0),
            "within_budget": all(tokens <= CHUNK_TOKEN_BUDGET for tokens in fragment_token_lengths),
        }
    )


list_split_df = pd.DataFrame(list_split_records)


print(
    f"Oversized lists: {len(oversized_lists_df):,}\n"
    f"Generated list fragments: {list_split_df['parts'].sum():,}\n"
    f"Maximum fragment tokens: {list_split_df['max_part_tokens'].max():,}\n"
    f"Budget violations: {(~list_split_df['within_budget']).sum():,}"
)

display(list_split_df.sort_values("original_tokens", ascending=False))


Splitting oversized lists: 100%|██████████| 11/11 [00:00<00:00, 219.89list/s]

Oversized lists: 11
Generated list fragments: 30
Maximum fragment tokens: 446
Budget violations: 0


,path,section_index,block_index,block_type,original_tokens,parts,max_part_tokens,within_budget
1,cloud/cluster-monitoring.md,3,1,bullet_list,2700,9,400,True
9,security.md,17,1,bullet_list,935,3,399,True
6,production-checklist.md,5,1,bullet_list,578,2,419,True
8,security.md,9,1,bullet_list,573,2,382,True
0,_index.md,3,0,bullet_list,562,2,446,True
10,tutorials-build-essentials/qdrant-n8n.md,7,5,ordered_list,557,2,362,True
5,platforms/apify.md,1,1,ordered_list,539,2,273,True
7,search/search-relevance.md,1,18,bullet_list,537,2,397,True
2,hybrid-cloud/hybrid-cloud-setup.md,3,0,ordered_list,513,2,306,True
4,ops-configuration/configuration.md,1,1,ordered_list,512,2,389,True


#### Remaining Oversized Block Diagnostics

After structure-aware handling of fenced code blocks, Markdown tables, and lists, any remaining oversized block is inspected individually before applying a generic fallback.

This avoids unnecessary token-level splitting when the remaining content has a recognizable structure that can still be preserved.

In [49]:
remaining_oversized_blocks_df = (
    oversized_non_fence_df[
        (~oversized_non_fence_df["is_markdown_table"])
        & (~oversized_non_fence_df["block_type"].isin(["bullet_list", "ordered_list"]))
    ]
    .copy()
    .reset_index(drop=True)
)


print(f"Remaining oversized blocks: " f"{len(remaining_oversized_blocks_df):,}")

display(
    remaining_oversized_blocks_df[["path", "section_index", "block_index", "block_type", "tokens"]]
)


for _, row in remaining_oversized_blocks_df.iterrows():
    print(
        f"{'=' * 100}\n"
        f"Path: {row['path']}\n"
        f"Section index: {row['section_index']}\n"
        f"Block index: {row['block_index']}\n"
        f"Block type: {row['block_type']}\n"
        f"Tokens: {row['tokens']:,}\n"
        f"{'=' * 100}\n\n"
        f"{row['block_text']}\n"
    )


Remaining oversized blocks: 0


,path,section_index,block_index,block_type,tokens


#### Normalized Semantic Blocks

All oversized semantic blocks can now be handled using structure-aware rules: fenced code blocks are split by source lines, Markdown tables by rows with header preservation or column-value fallback, and lists by complete top-level items.

The original block order is now converted into a normalized sequence of token-safe units. Blocks that already fit within the budget remain unchanged, while oversized blocks are replaced by their structure-aware fragments.

No generic token-level block fallback is required for the observed corpus.

Unexpected oversized prose and very long list items fall back to character slices derived from tokenizer offsets, without decoding or changing the original text.

In [50]:
def normalize_semantic_block(row, token_budget: int = CHUNK_TOKEN_BUDGET) -> tuple[list[str], str]:
    block_text = row["block_text"]
    block_type = row["block_type"]
    block_tokens = row["tokens"]

    if block_tokens <= token_budget:
        return [block_text], "unchanged"

    if block_type == "fence":
        return (split_fenced_code_block(block_text, token_budget=token_budget), "fence")

    if is_markdown_table(block_text):
        return (split_markdown_table(block_text, token_budget=token_budget), "table")

    if block_type in {"bullet_list", "ordered_list"}:
        return (split_markdown_list(block_text, token_budget=token_budget), "list")

    return split_tokens(block_text, token_budget), "source_window_fallback"


normalized_block_records = []

for _, row in tqdm(
    semantic_blocks_df.iterrows(),
    total=len(semantic_blocks_df),
    desc="Normalizing semantic blocks",
    unit="block",
):
    parts, split_strategy = normalize_semantic_block(row)

    for part_index, part_text in enumerate(parts):
        part_text = part_text.strip()

        if not part_text:
            continue

        normalized_block_records.append(
            {
                "path": row["path"],
                "section_index": row["section_index"],
                "block_index": row["block_index"],
                "part_index": part_index,
                "block_type": row["block_type"],
                "split_strategy": split_strategy,
                "block_text": part_text,
                "tokens": count_tokens(part_text),
            }
        )


normalized_blocks_df = (
    pd.DataFrame(normalized_block_records)
    .sort_values(["path", "section_index", "block_index", "part_index"])
    .reset_index(drop=True)
)


budget_violations = (normalized_blocks_df["tokens"] > CHUNK_TOKEN_BUDGET).sum()

empty_blocks = normalized_blocks_df["block_text"].str.strip().eq("").sum()

covered_blocks = (
    normalized_blocks_df[["path", "section_index", "block_index"]].drop_duplicates().shape[0]
)

original_blocks = (
    semantic_blocks_df[["path", "section_index", "block_index"]].drop_duplicates().shape[0]
)


print(
    f"Original semantic blocks: {original_blocks:,}\n"
    f"Normalized block units: {len(normalized_blocks_df):,}\n"
    f"Covered original blocks: {covered_blocks:,}\n"
    f"Maximum normalized unit tokens: {normalized_blocks_df['tokens'].max():,}\n"
    f"Budget violations: {budget_violations:,}\n"
    f"Empty normalized units: {empty_blocks:,}"
)

display(
    normalized_blocks_df["split_strategy"]
    .value_counts()
    .rename_axis("split_strategy")
    .reset_index(name="units")
)


Normalizing semantic blocks: 100%|██████████| 5907/5907 [00:01<00:00, 3352.90block/s]

Original semantic blocks: 5,907
Normalized block units: 6,131
Covered original blocks: 5,907
Maximum normalized unit tokens: 448
Budget violations: 0
Empty normalized units: 0


,split_strategy,units
0,unchanged,5779
1,fence,219
2,table,103
3,list,30


#### Greedy Chunk Assembly

The normalized semantic units are now greedily packed into final chunks within each original Markdown section.

Chunks never cross section boundaries. Units are appended in their original order while the combined text remains within the 448-token body budget. No overlap is introduced at this stage, providing a clean retrieval baseline without duplicated content.

Actual token counts are recalculated for every combined candidate rather than estimated from individual unit lengths.

In [51]:
def pack_normalized_section(
    section_df: pd.DataFrame, token_budget: int = CHUNK_TOKEN_BUDGET
) -> list[dict]:
    section_df = section_df.sort_values(["block_index", "part_index"])

    chunks = []

    current_texts = []
    current_units = []

    def flush_current_chunk():
        if not current_texts:
            return

        chunk_text = "\n\n".join(current_texts).strip()

        chunk_tokens = count_tokens(chunk_text)

        if chunk_tokens > token_budget:
            raise ValueError(
                "Greedy packing produced an oversized chunk: " f"{chunk_tokens} > {token_budget}"
            )

        chunks.append(
            {
                "chunk_text": chunk_text,
                "tokens": chunk_tokens,
                "source_units": len(current_units),
                "source_block_indices": sorted({unit["block_index"] for unit in current_units}),
                "split_strategies": sorted({unit["split_strategy"] for unit in current_units}),
            }
        )

    for row in section_df.itertuples(index=False):
        unit_text = row.block_text.strip()

        if not unit_text:
            continue

        unit_tokens = count_tokens(unit_text)

        if unit_tokens > token_budget:
            raise ValueError(
                "Normalized unit exceeds token budget: " f"{unit_tokens} > {token_budget}"
            )

        candidate_text = "\n\n".join([*current_texts, unit_text]).strip()

        if current_texts and count_tokens(candidate_text) > token_budget:
            flush_current_chunk()

            current_texts = [unit_text]

            current_units = [{"block_index": row.block_index, "split_strategy": row.split_strategy}]

            continue

        current_texts.append(unit_text)

        current_units.append({"block_index": row.block_index, "split_strategy": row.split_strategy})

    flush_current_chunk()

    for chunk_index, chunk in enumerate(chunks):
        chunk["chunk_index"] = chunk_index

    return chunks


oversized_chunk_records = []

section_groups = list(normalized_blocks_df.groupby(["path", "section_index"], sort=False))


for (path, section_index), section_df in tqdm(
    section_groups, total=len(section_groups), desc="Packing oversized sections", unit="section"
):
    section_chunks = pack_normalized_section(section_df)

    for chunk in section_chunks:
        oversized_chunk_records.append({"path": path, "section_index": section_index, **chunk})


oversized_chunks_df = (
    pd.DataFrame(oversized_chunk_records)
    .sort_values(["path", "section_index", "chunk_index"])
    .reset_index(drop=True)
)


covered_sections = oversized_chunks_df[["path", "section_index"]].drop_duplicates().shape[0]

expected_sections = normalized_blocks_df[["path", "section_index"]].drop_duplicates().shape[0]

budget_violations = (oversized_chunks_df["tokens"] > CHUNK_TOKEN_BUDGET).sum()

empty_chunks = oversized_chunks_df["chunk_text"].str.strip().eq("").sum()


print(
    f"Oversized sections: {expected_sections:,}\n"
    f"Covered oversized sections: {covered_sections:,}\n"
    f"Generated chunks: {len(oversized_chunks_df):,}\n"
    f"Maximum chunk tokens: {oversized_chunks_df['tokens'].max():,}\n"
    f"Budget violations: {budget_violations:,}\n"
    f"Empty chunks: {empty_chunks:,}"
)

display(oversized_chunks_df["tokens"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))


Packing oversized sections: 100%|██████████| 538/538 [00:02<00:00, 198.73section/s]

Oversized sections: 538
Covered oversized sections: 538
Generated chunks: 2,046
Maximum chunk tokens: 448
Budget violations: 0
Empty chunks: 0


count    2046.000000
mean      320.631476
std       113.449225
min        11.000000
50%       354.000000
90%       439.000000
95%       444.000000
99%       448.000000
max       448.000000
Name: tokens, dtype: float64

#### Full Corpus Chunk Assembly

Sections that already fit within the 448-token budget are preserved unchanged as single chunks. Oversized sections use the structure-aware chunks produced above.

Both paths are combined into one corpus-wide chunk table while retaining the original page and section metadata. Chunk indices remain local to each section and preserve document order.

In [52]:
section_metadata_columns = [
    "path",
    "section_index",
    "level",
    "page_title",
    "section_title",
    "breadcrumbs",
    "page_url",
    "section_url",
]


section_metadata_df = clean_sections_df[section_metadata_columns + ["section_text"]].merge(
    section_token_df[["path", "section_index", "tokens"]],
    on=["path", "section_index"],
    how="left",
    validate="one_to_one",
)


intact_sections_df = section_metadata_df[section_metadata_df["tokens"] <= CHUNK_TOKEN_BUDGET].copy()


intact_chunks_df = intact_sections_df[section_metadata_columns + ["section_text", "tokens"]].rename(
    columns={"section_text": "chunk_text"}
)

intact_chunks_df["chunk_index"] = 0
intact_chunks_df["source_units"] = 1
intact_chunks_df["source_block_indices"] = None
intact_chunks_df["split_strategies"] = [["section"] for _ in range(len(intact_chunks_df))]


oversized_chunks_with_metadata_df = oversized_chunks_df.merge(
    section_metadata_df[section_metadata_columns],
    on=["path", "section_index"],
    how="left",
    validate="many_to_one",
)


chunks_df = pd.concat(
    [
        intact_chunks_df[
            section_metadata_columns
            + [
                "chunk_index",
                "chunk_text",
                "tokens",
                "source_units",
                "source_block_indices",
                "split_strategies",
            ]
        ],
        oversized_chunks_with_metadata_df[
            section_metadata_columns
            + [
                "chunk_index",
                "chunk_text",
                "tokens",
                "source_units",
                "source_block_indices",
                "split_strategies",
            ]
        ],
    ],
    ignore_index=True,
)


chunks_df = chunks_df.sort_values(["path", "section_index", "chunk_index"]).reset_index(drop=True)


source_sections = clean_sections_df[["path", "section_index"]].drop_duplicates().shape[0]

covered_sections = chunks_df[["path", "section_index"]].drop_duplicates().shape[0]

budget_violations = (chunks_df["tokens"] > CHUNK_TOKEN_BUDGET).sum()

empty_chunks = chunks_df["chunk_text"].fillna("").str.strip().eq("").sum()

duplicate_chunk_keys = chunks_df.duplicated(subset=["path", "section_index", "chunk_index"]).sum()


print(
    f"Source sections: {source_sections:,}\n"
    f"Covered sections: {covered_sections:,}\n"
    f"Intact section chunks: {len(intact_chunks_df):,}\n"
    f"Oversized-section chunks: {len(oversized_chunks_df):,}\n"
    f"Total chunks: {len(chunks_df):,}\n"
    f"Maximum chunk tokens: {chunks_df['tokens'].max():,}\n"
    f"Budget violations: {budget_violations:,}\n"
    f"Empty chunks: {empty_chunks:,}\n"
    f"Duplicate chunk keys: {duplicate_chunk_keys:,}"
)


Source sections: 2,735
Covered sections: 2,735
Intact section chunks: 2,197
Oversized-section chunks: 2,046
Total chunks: 4,243
Maximum chunk tokens: 448
Budget violations: 0
Empty chunks: 109
Duplicate chunk keys: 0


#### Empty Section Diagnostics

Some parsed Markdown sections contain no direct body text. This can occur when a heading acts only as a structural parent for nested subsections.

Before excluding empty chunks from the retrieval corpus, these sections are inspected to verify that they represent navigation or hierarchy nodes rather than lost document content.

In [53]:
empty_chunks_df = (
    chunks_df[chunks_df["chunk_text"].fillna("").str.strip().eq("")].copy().reset_index(drop=True)
)


empty_section_keys = empty_chunks_df[["path", "section_index"]].drop_duplicates()


empty_sections_df = clean_sections_df.merge(
    empty_section_keys, on=["path", "section_index"], how="inner", validate="one_to_one"
)


def has_descendant_section(row, sections_df: pd.DataFrame) -> bool:
    same_page = sections_df[sections_df["path"] == row["path"]]

    later_sections = same_page[same_page["section_index"] > row["section_index"]]

    if later_sections.empty:
        return False

    current_level = row["level"]

    for descendant in later_sections.itertuples(index=False):
        if descendant.level <= current_level:
            break

        return True

    return False


empty_sections_df["has_descendant_section"] = empty_sections_df.apply(
    has_descendant_section, axis=1, sections_df=clean_sections_df
)


print(
    f"Empty chunks: {len(empty_chunks_df):,}\n"
    f"Empty source sections: {len(empty_sections_df):,}\n"
    f"With descendant sections: {empty_sections_df['has_descendant_section'].sum():,}\n"
    f"Without descendant sections: {(~empty_sections_df['has_descendant_section']).sum():,}"
)

display(
    empty_sections_df[
        [
            "path",
            "section_index",
            "level",
            "page_title",
            "section_title",
            "breadcrumbs",
            "has_descendant_section",
        ]
    ]
    .sort_values(["has_descendant_section", "path", "section_index"], ascending=[True, True, True])
    .head(50)
)


Empty chunks: 109
Empty source sections: 109
With descendant sections: 87
Without descendant sections: 22


,path,section_index,level,page_title,section_title,breadcrumbs,has_descendant_section
7,cloud-tab.md,0,0,Welcome to Qdrant Cloud,Welcome to Qdrant Cloud,[Welcome to Qdrant Cloud],False
14,dl-cloud-getting-started.md,0,0,Getting Started,Getting Started,[Getting Started],False
15,dl-cloud-interfaces.md,0,0,Interfaces & Tools,Interfaces & Tools,[Interfaces & Tools],False
16,dl-cloud-support.md,0,0,Support,Support,[Support],False
17,dl-getting-started.md,0,0,Getting Started,Getting Started,[Getting Started],False
18,dl-integration-examples.md,0,0,Ecosystem Guides,Ecosystem Guides,[Ecosystem Guides],False
19,dl-integrations.md,0,0,Integrations,Integrations,[Integrations],False
20,dl-managed-services.md,0,0,Cloud,Cloud,[Cloud],False
21,dl-migrate.md,0,0,Migrate to Qdrant,Migrate to Qdrant,[Migrate to Qdrant],False
22,dl-support.md,0,0,Support,Support,[Support],False


#### Empty Leaf Section Diagnostics

Empty sections without descendants require additional inspection before exclusion.

Unlike structural parent headings, these sections may represent routing pages, metadata-only documents, source-only content removed during cleanup, or genuinely empty documentation pages. Their original and cleaned text lengths together with front matter metadata are inspected before deciding how they should be handled.

In [54]:
empty_leaf_sections_df = (
    empty_sections_df[~empty_sections_df["has_descendant_section"]].copy().reset_index(drop=True)
)


def extract_metadata_summary(metadata) -> dict:
    if not isinstance(metadata, dict):
        return {}

    keys = [
        "title",
        "slug",
        "aliases",
        "redirect",
        "draft",
        "weight",
        "description",
        "hideInSidebar",
        "externalLink",
    ]

    return {key: metadata.get(key) for key in keys if key in metadata}


empty_leaf_sections_df["raw_chars"] = (
    empty_leaf_sections_df["section_text_raw"].fillna("").str.len()
)

empty_leaf_sections_df["resolved_chars"] = (
    empty_leaf_sections_df["section_text"].fillna("").str.len()
)

empty_leaf_sections_df["metadata_summary"] = empty_leaf_sections_df["metadata"].apply(
    extract_metadata_summary
)


print(
    f"Empty leaf sections: {len(empty_leaf_sections_df):,}\n"
    f"With non-empty raw source text: {(empty_leaf_sections_df['raw_chars'] > 0).sum():,}\n"
    f"Completely empty before cleanup: {(empty_leaf_sections_df['raw_chars'] == 0).sum():,}"
)


display(
    empty_leaf_sections_df[
        [
            "path",
            "section_index",
            "level",
            "page_title",
            "section_title",
            "page_url",
            "raw_chars",
            "resolved_chars",
            "metadata_summary",
        ]
    ].sort_values(["raw_chars", "path"], ascending=[False, True])
)


Empty leaf sections: 22
With non-empty raw source text: 0
Completely empty before cleanup: 22


,path,section_index,level,page_title,section_title,page_url,raw_chars,resolved_chars,metadata_summary
0,cloud-tab.md,0,0,Welcome to Qdrant Cloud,Welcome to Qdrant Cloud,https://qdrant.tech/documentation/cloud-intro/,0,0,"{'title': 'Welcome to Qdrant Cloud', 'slug': '..."
1,dl-cloud-getting-started.md,0,0,Getting Started,Getting Started,https://qdrant.tech/documentation/dl-cloud-get...,0,0,"{'title': 'Getting Started', 'weight': 1, 'des..."
2,dl-cloud-interfaces.md,0,0,Interfaces & Tools,Interfaces & Tools,https://qdrant.tech/documentation/dl-cloud-int...,0,0,"{'title': 'Interfaces & Tools', 'weight': 25, ..."
3,dl-cloud-support.md,0,0,Support,Support,https://qdrant.tech/documentation/dl-cloud-sup...,0,0,"{'title': 'Support', 'weight': 300, 'descripti..."
4,dl-getting-started.md,0,0,Getting Started,Getting Started,https://qdrant.tech/documentation/dl-getting-s...,0,0,"{'title': 'Getting Started', 'weight': 100, 'd..."
5,dl-integration-examples.md,0,0,Ecosystem Guides,Ecosystem Guides,https://qdrant.tech/documentation/dl-integrati...,0,0,"{'title': 'Ecosystem Guides', 'weight': 1100, ..."
6,dl-integrations.md,0,0,Integrations,Integrations,https://qdrant.tech/documentation/dl-integrati...,0,0,"{'title': 'Integrations', 'weight': 500, 'desc..."
7,dl-managed-services.md,0,0,Cloud,Cloud,https://qdrant.tech/documentation/dl-managed-s...,0,0,"{'title': 'Cloud', 'weight': 200, 'description..."
8,dl-migrate.md,0,0,Migrate to Qdrant,Migrate to Qdrant,https://qdrant.tech/documentation/dl-migrate/,0,0,"{'title': 'Migrate to Qdrant', 'weight': 100, ..."
9,dl-support.md,0,0,Support,Support,https://qdrant.tech/documentation/dl-support/,0,0,"{'title': 'Support', 'weight': 500, 'descripti..."


#### Retrieval Chunk Finalization

Empty sections are excluded from the vector retrieval corpus.

The diagnostics confirm that all empty leaf sections were already empty in the original parsed source, so their absence is not caused by preprocessing. Empty parent sections serve only as structural hierarchy nodes, while empty leaf sections contain no retrievable body content.

Their document and hierarchy metadata remain available in the parsed corpus, but no embedding is created for an empty retrieval unit.

In [55]:
pre_filter_chunk_count = len(chunks_df)

empty_chunk_count = chunks_df["chunk_text"].fillna("").str.strip().eq("").sum()


retrieval_chunks_df = (
    chunks_df[chunks_df["chunk_text"].fillna("").str.strip().ne("")].copy().reset_index(drop=True)
)


retrieval_section_count = retrieval_chunks_df[["path", "section_index"]].drop_duplicates().shape[0]

non_empty_source_sections = (
    clean_sections_df[clean_sections_df["section_text"].fillna("").str.strip().ne("")][
        ["path", "section_index"]
    ]
    .drop_duplicates()
    .shape[0]
)

budget_violations = (retrieval_chunks_df["tokens"] > CHUNK_TOKEN_BUDGET).sum()

empty_retrieval_chunks = retrieval_chunks_df["chunk_text"].fillna("").str.strip().eq("").sum()

duplicate_chunk_keys = retrieval_chunks_df.duplicated(
    subset=["path", "section_index", "chunk_index"]
).sum()


print(
    f"Chunks before filtering: {pre_filter_chunk_count:,}\n"
    f"Excluded empty chunks: {empty_chunk_count:,}\n"
    f"Final retrieval chunks: {len(retrieval_chunks_df):,}\n"
    f"Non-empty source sections: {non_empty_source_sections:,}\n"
    f"Covered retrieval sections: {retrieval_section_count:,}\n"
    f"Maximum chunk tokens: {retrieval_chunks_df['tokens'].max():,}\n"
    f"Budget violations: {budget_violations:,}\n"
    f"Empty retrieval chunks: {empty_retrieval_chunks:,}\n"
    f"Duplicate chunk keys: {duplicate_chunk_keys:,}"
)


Chunks before filtering: 4,243
Excluded empty chunks: 109
Final retrieval chunks: 4,134
Non-empty source sections: 2,626
Covered retrieval sections: 2,626
Maximum chunk tokens: 448
Budget violations: 0
Empty retrieval chunks: 0
Duplicate chunk keys: 0


### 5.2 Neighbor Sections

The final-project payload stores the text of the adjacent documentation sections for optional contextual expansion.

Neighbors are defined at the **source-section level**, not at the chunk level. Empty structural headings are skipped, so `prev_section_text` and `next_section_text` always refer to the nearest non-empty section on the same documentation page. Neighbor text is stored in the payload but is **not** included in the retrieval embedding.

In [56]:
non_empty_sections_df = (
    clean_sections_df[clean_sections_df["section_text"].fillna("").str.strip().ne("")]
    .copy()
    .sort_values(["path", "section_index"])
    .reset_index(drop=True)
)

non_empty_sections_df["prev_section_text"] = non_empty_sections_df.groupby("path", sort=False)[
    "section_text"
].shift(1)

non_empty_sections_df["next_section_text"] = non_empty_sections_df.groupby("path", sort=False)[
    "section_text"
].shift(-1)

section_neighbors_df = non_empty_sections_df[
    ["path", "section_index", "prev_section_text", "next_section_text"]
].copy()

retrieval_chunks_df = retrieval_chunks_df.drop(
    columns=["prev_section_text", "next_section_text"], errors="ignore"
).merge(section_neighbors_df, on=["path", "section_index"], how="left", validate="many_to_one")

print(
    f"Retrieval chunks: {len(retrieval_chunks_df):,}\n"
    f"Chunks with previous-section context: {retrieval_chunks_df['prev_section_text'].notna().sum():,}\n"
    f"Chunks with next-section context: {retrieval_chunks_df['next_section_text'].notna().sum():,}\n"
    f"Neighbor mapping rows: {len(section_neighbors_df):,}"
)


Retrieval chunks: 4,134
Chunks with previous-section context: 3,746
Chunks with next-section context: 3,653
Neighbor mapping rows: 2,626


### 5.3 Chunk Metadata

Each retrieval chunk receives a deterministic key and UUID, retrieval metadata, lightweight tags, and a separate embedding input.

`chunk_text` remains the clean retrieval body stored in Qdrant. `embedding_text` prepends the documentation hierarchy to the body so the embedding model can use page/section context without polluting the stored chunk text. Point IDs are derived deterministically from the source path, section index, and chunk index, making repeated ingestion idempotent.

In [57]:
def build_chunk_tags(row) -> list[str]:
    tags = []

    path_parts = Path(row["path"]).parts

    if path_parts:
        tags.append(path_parts[0].replace("_", "-").lower())

    for value in [row["page_title"], row["section_title"]]:
        if value is None or pd.isna(value):
            continue

        tag = slugify_heading(str(value))

        if tag:
            tags.append(tag)

    return list(dict.fromkeys(tags))


def build_chunk_key(row) -> str:
    return (
        f"{row['path']}::"
        f"section={int(row['section_index'])}::"
        f"chunk={int(row['chunk_index'])}"
    )


retrieval_chunks_df["chunk_key"] = retrieval_chunks_df.apply(build_chunk_key, axis=1)

retrieval_chunks_df["point_id"] = retrieval_chunks_df["chunk_key"].apply(
    lambda value: str(uuid.uuid5(uuid.NAMESPACE_URL, value))
)

retrieval_chunks_df["tags"] = retrieval_chunks_df.apply(build_chunk_tags, axis=1)

retrieval_chunks_df["embedding_prefix"] = retrieval_chunks_df.apply(build_embedding_prefix, axis=1)

retrieval_chunks_df["embedding_text"] = (
    retrieval_chunks_df["embedding_prefix"] + retrieval_chunks_df["chunk_text"]
)


print(
    f"Unique chunk keys: {retrieval_chunks_df['chunk_key'].nunique():,}\n"
    f"Unique point IDs: {retrieval_chunks_df['point_id'].nunique():,}\n"
    f"Chunks with tags: {retrieval_chunks_df['tags'].map(bool).sum():,}"
)

display(
    retrieval_chunks_df[
        ["point_id", "path", "page_title", "section_title", "chunk_index", "tokens", "tags"]
    ].head(10)
)


Unique chunk keys: 4,134
Unique point IDs: 4,134
Chunks with tags: 4,134


,point_id,path,page_title,section_title,chunk_index,tokens,tags
0,75b105ff-7837-5aba-afeb-d926c9add3c1,_index.md,Documentation,Qdrant Documentation,0,42,"[-index.md, documentation, qdrant-documentation]"
1,91b9feee-498b-5d8c-93e4-593b477aec7f,_index.md,Documentation,Getting Started,0,174,"[-index.md, documentation, getting-started]"
2,5f096909-55b9-5420-914c-7e44a9d7c191,_index.md,Documentation,Develop,0,173,"[-index.md, documentation, develop]"
3,dd19cbd1-b700-5263-b9a3-487281e89657,_index.md,Documentation,Deploy,0,446,"[-index.md, documentation, deploy]"
4,525179f9-8736-5528-b8ff-1a70178d8ce9,_index.md,Documentation,Deploy,1,116,"[-index.md, documentation, deploy]"
5,de2eda4c-4e90-5aae-b309-587fbfaa8a80,_index.md,Documentation,Ecosystem,0,150,"[-index.md, documentation, ecosystem]"
6,0e05f51f-4932-5567-bcfc-2f06638f7174,_index.md,Documentation,Tutorials & Examples,0,87,"[-index.md, documentation, tutorials-examples]"
7,911373ef-a27a-59cb-9995-a7f6ac855532,_index.md,Documentation,Learn,0,127,"[-index.md, documentation, learn]"
8,b32f9393-10c1-55da-89ab-a2a8bd931403,_index.md,Documentation,API Reference,0,52,"[-index.md, documentation, api-reference]"
9,2bf5ff8a-0a57-5942-a804-1791947f2757,capacity-planning.md,Capacity Planning,Capacity Planning,0,156,"[capacity-planning.md, capacity-planning]"


### 5.4 Chunk Statistics

The final chunk corpus is summarized after empty-section removal and neighbor/metadata enrichment. The distribution is used to confirm that the 448-token budget produces compact retrieval units without excessive fragmentation.

In [58]:
chunks_per_section_df = retrieval_chunks_df.groupby(["path", "section_index"], as_index=False).agg(
    chunks=("chunk_index", "count"), max_chunk_tokens=("tokens", "max")
)

print(
    f"Retrieval chunks: {len(retrieval_chunks_df):,}\n"
    f"Retrieval sections: {len(chunks_per_section_df):,}\n"
    f"Median chunk tokens: {retrieval_chunks_df['tokens'].median():,.0f}\n"
    f"P95 chunk tokens: {retrieval_chunks_df['tokens'].quantile(0.95):,.0f}\n"
    f"Maximum chunk tokens: {retrieval_chunks_df['tokens'].max():,}\n"
    f"Median chunks per section: {chunks_per_section_df['chunks'].median():,.0f}\n"
    f"Maximum chunks per section: {chunks_per_section_df['chunks'].max():,}"
)

fig = px.histogram(
    retrieval_chunks_df, x="tokens", nbins=60, title="Final Retrieval Chunk Token Distribution"
)

fig.show()


Retrieval chunks: 4,134
Retrieval sections: 2,626
Median chunk tokens: 242
P95 chunk tokens: 439
Maximum chunk tokens: 448
Median chunks per section: 1
Maximum chunks per section: 25


### 5.5 Chunking Validation

Final validation checks the actual dense-model input rather than relying only on body and prefix arithmetic.

Every `embedding_text` is tokenized exactly as it will be sent to `BAAI/bge-small-en-v1.5`, including model special tokens. The validation also checks identifiers, required metadata, section coverage, and retrieval-body constraints.

In [59]:
embedding_texts = retrieval_chunks_df["embedding_text"].fillna("").tolist()

embedding_token_lengths = []

for batch_start in tqdm(
    range(0, len(embedding_texts), TOKENIZATION_BATCH_SIZE),
    desc="Validating embedding inputs",
    unit="batch",
):
    batch_texts = embedding_texts[batch_start : batch_start + TOKENIZATION_BATCH_SIZE]

    encoded = dense_tokenizer(
        batch_texts,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True,
        verbose=False,
    )

    embedding_token_lengths.extend(encoded["length"])

retrieval_chunks_df["embedding_tokens"] = embedding_token_lengths

required_metadata_columns = [
    "point_id",
    "page_title",
    "section_title",
    "page_url",
    "section_url",
    "breadcrumbs",
    "chunk_text",
    "tags",
]

missing_required_values = {
    column: int(retrieval_chunks_df[column].isna().sum()) for column in required_metadata_columns
}

non_empty_source_sections = (
    clean_sections_df[clean_sections_df["section_text"].fillna("").str.strip().ne("")][
        ["path", "section_index"]
    ]
    .drop_duplicates()
    .shape[0]
)

covered_sections = retrieval_chunks_df[["path", "section_index"]].drop_duplicates().shape[0]

validation_summary = {
    "chunks": len(retrieval_chunks_df),
    "unique_point_ids": retrieval_chunks_df["point_id"].nunique(),
    "empty_chunks": int(retrieval_chunks_df["chunk_text"].fillna("").str.strip().eq("").sum()),
    "body_budget_violations": int((retrieval_chunks_df["tokens"] > CHUNK_TOKEN_BUDGET).sum()),
    "embedding_context_violations": int(
        (retrieval_chunks_df["embedding_tokens"] > dense_tokenizer.model_max_length).sum()
    ),
    "max_embedding_tokens": int(retrieval_chunks_df["embedding_tokens"].max()),
    "source_sections": non_empty_source_sections,
    "covered_sections": covered_sections,
}

validation_lines = [
    f"{key}: {value:,}" for key, value in validation_summary.items()
]
validation_lines.append("Required-field null counts:")
validation_lines.extend(
    f"  {column}: {missing:,}"
    for column, missing in missing_required_values.items()
)
print(*validation_lines, sep="\n")

assert validation_summary["chunks"] == validation_summary["unique_point_ids"]
assert validation_summary["empty_chunks"] == 0
assert validation_summary["body_budget_violations"] == 0
assert validation_summary["embedding_context_violations"] == 0
assert validation_summary["source_sections"] == validation_summary["covered_sections"]
assert all(missing == 0 for missing in missing_required_values.values())

print("Chunking validation: PASSED")


assert dense_tokenizer.is_fast, "Source offsets require a fast tokenizer."
assert not normalized_blocks_df["tokens"].gt(CHUNK_TOKEN_BUDGET).any()
RUN_MANIFEST["chunking"] = {
    "body_budget": CHUNK_TOKEN_BUDGET,
    "prefix": "explicit document title and section hierarchy",
    "strategy": "structure-aware/source-offset-fallback-v2",
    "chunks": len(retrieval_chunks_df),
}


Validating embedding inputs: 100%|██████████| 17/17 [00:00<00:00, 80.08batch/s]

chunks: 4,134
unique_point_ids: 4,134
empty_chunks: 0
body_budget_violations: 0
embedding_context_violations: 0
max_embedding_tokens: 495
source_sections: 2,626
covered_sections: 2,626
Required-field null counts:
  point_id: 0
  page_title: 0
  section_title: 0
  page_url: 0
  section_url: 0
  breadcrumbs: 0
  chunk_text: 0
  tags: 0
Chunking validation: PASSED


## 6. Embedding Models

This implementation deliberately follows the Qdrant Essentials model stack and FastEmbed workflow rather than introducing unrelated embedding libraries or checkpoints.

- **Dense:** `BAAI/bge-small-en-v1.5` - explicitly recommended by the Day 6 final project and also the default FastEmbed dense model introduced on Day 1.
- **Sparse:** `prithivida/Splade_PP_en_v1` - the SPLADE++ model used in the Day 3 sparse-retrieval lesson and the Day 5 Universal Query demo.
- **Late interaction:** `colbert-ir/colbertv2.0` - the 128-dimensional ColBERT model used in Day 5 and specified by the Day 6 final project.

All three retrieval representations are generated locally with Qdrant FastEmbed. Full-corpus vectors are produced in streaming batches during ingestion rather than materialized as one large in-memory matrix.

Course references:
- [Day 1 - Points, Vectors and Payloads](https://qdrant.tech/course/essentials/day-1/embedding-models/)
- [Day 3 - Sparse Retrieval Demo](https://qdrant.tech/course/essentials/day-3/sparse-retrieval-demo/)
- [Day 5 - Universal Query Demo](https://qdrant.tech/course/essentials/day-5/universal-query-demo/)
- [Day 6 - Final Project](https://qdrant.tech/course/essentials/day-6/final-project/)


### 6.1 Dense Embeddings

`BAAI/bge-small-en-v1.5` is used as the primary dense retriever. This is the 384-dimensional FastEmbed model recommended by the Day 6 final project for the speed-oriented dense baseline.

FastEmbed's retrieval-specific APIs are used consistently: documents are encoded with `passage_embed`, while user queries use `query_embed`. The expected output dimension is 384.


In [60]:
import onnxruntime as ort
import torch

available_onnx_providers = ort.get_available_providers()

use_cuda = (
    torch.cuda.is_available()
    and "CUDAExecutionProvider" in available_onnx_providers
)

if use_cuda:
    try:
        ort.preload_dlls(cuda=True, cudnn=True, directory="")
    except Exception as exc:
        print(f"CUDA initialization failed, falling back to CPU: {exc}")
        use_cuda = False

FASTEMBED_PROVIDERS = (
    ["CUDAExecutionProvider"]
    if use_cuda
    else ["CPUExecutionProvider"]
)

EMBEDDING_DEVICE = (
    torch.cuda.get_device_name(0)
    if use_cuda
    else "CPU"
)

RUN_MANIFEST["fastembed_providers"] = FASTEMBED_PROVIDERS
RUN_MANIFEST["embedding_device"] = EMBEDDING_DEVICE

print(
    f"ONNX Runtime: {ort.__version__}\n"
    f"Available providers: {available_onnx_providers}\n"
    f"FastEmbed providers: {FASTEMBED_PROVIDERS}\n"
    f"Embedding device: {EMBEDDING_DEVICE}"
)

ONNX Runtime: 1.23.2
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
FastEmbed providers: ['CPUExecutionProvider']
Embedding device: CPU


In [61]:
FASTEMBED_CACHE_DIR = RAW_DATA_DIR / "fastembed-cache"
FASTEMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DENSE_EMBEDDING_DIM = 384


dense_embedding_model = TextEmbedding(
    model_name=DENSE_MODEL_NAME, cache_dir=str(FASTEMBED_CACHE_DIR), providers=FASTEMBED_PROVIDERS
)

sample_embedding_text = retrieval_chunks_df.iloc[0]["embedding_text"]

sample_dense_vector = np.asarray(
    list(dense_embedding_model.passage_embed([sample_embedding_text]))[0]
)

print(f"Dense model: {DENSE_MODEL_NAME}\n" f"Dense vector shape: {sample_dense_vector.shape}")

assert sample_dense_vector.shape == (DENSE_EMBEDDING_DIM,)


Dense model: BAAI/bge-small-en-v1.5
Dense vector shape: (384,)


### 6.2 Sparse Embeddings

The sparse retrieval leg uses the same SPLADE++ checkpoint demonstrated in the course: `prithivida/Splade_PP_en_v1`.

Day 3 introduces this model for neural sparse retrieval and Day 5 uses it again in the dense + sparse + ColBERT Universal Query pipeline. Document vectors use `embed`, while query vectors use `query_embed`, matching the FastEmbed retrieval pattern.


In [62]:
SPARSE_MODEL_NAME = "prithivida/Splade_PP_en_v1"

sparse_embedding_model = SparseTextEmbedding(
    model_name=SPARSE_MODEL_NAME, cache_dir=str(FASTEMBED_CACHE_DIR), providers=FASTEMBED_PROVIDERS
)

sample_sparse_vector = list(sparse_embedding_model.embed([sample_embedding_text]))[0]

print(
    f"Sparse model: {SPARSE_MODEL_NAME}\n"
    f"Sample non-zero sparse dimensions: {len(sample_sparse_vector.indices):,}"
)

assert len(sample_sparse_vector.indices) == len(sample_sparse_vector.values)
assert len(sample_sparse_vector.indices) > 0


Sparse model: prithivida/Splade_PP_en_v1
Sample non-zero sparse dimensions: 91


### 6.3 ColBERT Multivectors

`colbert-ir/colbertv2.0` is the course ColBERT checkpoint. It produces one 128-dimensional vector per encoded token and is generated with `LateInteractionTextEmbedding`.

Qdrant stores the resulting matrix as a multivector and applies `MAX_SIM` during reranking. Following Day 5 and Day 6, ColBERT is used only after first-stage retrieval, and its HNSW index is disabled with `m=0`.


In [63]:
COLBERT_MODEL_NAME = "colbert-ir/colbertv2.0"
COLBERT_EMBEDDING_DIM = 128

colbert_embedding_model = LateInteractionTextEmbedding(
    model_name=COLBERT_MODEL_NAME, cache_dir=str(FASTEMBED_CACHE_DIR), providers=FASTEMBED_PROVIDERS
)

sample_colbert_vector = np.asarray(list(colbert_embedding_model.embed([sample_embedding_text]))[0])

print(
    f"ColBERT model: {COLBERT_MODEL_NAME}\n"
    f"Sample multivector shape: {sample_colbert_vector.shape}"
)

assert sample_colbert_vector.ndim == 2
assert sample_colbert_vector.shape[1] == (COLBERT_EMBEDDING_DIM)


ColBERT model: colbert-ir/colbertv2.0
Sample multivector shape: (43, 128)


### 6.4 Model Configuration

One compact configuration controls embedding and upload batches. A bounded in-memory LRU cache avoids repeated document encoding within the kernel; it is not persisted across kernel restarts. Cache keys include the complete input text and model identity, not a content hash. GPU batch sizes remain conservative until measured; compare ColBERT 8/16/32 and SPLADE 32/64 on representative batches after execution is approved.

In [64]:
@dataclass(frozen=True)
class IngestionConfig:
    pipeline_batch: int = 128
    upload_batch: int = 64
    upload_parallel: int = 1  # Benchmark 1/2/4; avoid process startup for small uploads.
    dense_batch: int = 128
    sparse_batch: int = 32
    colbert_batch: int = 8  # Try 16/32 only after checking peak GPU memory.
    cache_mib: int = 512


INGEST = IngestionConfig()
DENSE_BATCH_SIZE = INGEST.dense_batch
SPARSE_BATCH_SIZE = INGEST.sparse_batch
COLBERT_BATCH_SIZE = INGEST.colbert_batch
HYBRID_PREFETCH_LIMIT = 50
RERANK_CANDIDATE_LIMIT = 50
DEFAULT_SEARCH_LIMIT = 10

# Bounded, kernel-local cache: no cache files and no content hashes.
# Complete text + model identity + document/query role prevent stale reuse.
EMBEDDING_CACHE = globals().get("EMBEDDING_CACHE", OrderedDict())
EMBEDDING_CACHE_BYTES = globals().get("EMBEDDING_CACHE_BYTES", 0)
EMBEDDING_MODELS = {
    "dense": (DENSE_MODEL_NAME, dense_embedding_model.passage_embed, DENSE_BATCH_SIZE),
    "sparse": (SPARSE_MODEL_NAME, sparse_embedding_model.embed, SPARSE_BATCH_SIZE),
    "colbert": (COLBERT_MODEL_NAME, colbert_embedding_model.embed, COLBERT_BATCH_SIZE),
}


def embedding_nbytes(vector) -> int:
    if hasattr(vector, "indices") and hasattr(vector, "values"):
        return vector.indices.nbytes + vector.values.nbytes
    return np.asarray(vector).nbytes


def embed_documents(kind: str, texts: list[str], *, use_cache: bool = True):
    global EMBEDDING_CACHE_BYTES
    model_name, embed_fn, batch_size = EMBEDDING_MODELS[kind]
    keys = [
        (
            QDRANT_DOCS_COMMIT,
            model_name,
            tuple(FASTEMBED_PROVIDERS),
            version("fastembed-gpu"),
            "passage",
            text,
        )
        for text in texts
    ]
    missing = list(
        dict.fromkeys(key for key in keys if not use_cache or key not in EMBEDDING_CACHE)
    )
    fresh = (
        dict(zip(missing, embed_fn([key[-1] for key in missing], batch_size=batch_size)))
        if missing
        else {}
    )
    if len(fresh) != len(missing):
        raise RuntimeError(f"Incomplete {kind} embedding batch.")
    # Resolve the current batch before LRU eviction can remove any needed entry.
    result = [fresh[key] if key in fresh else EMBEDDING_CACHE[key] for key in keys]
    if use_cache:
        for key, vector in zip(keys, result):
            old = EMBEDDING_CACHE.pop(key, None)
            if old is not None:
                EMBEDDING_CACHE_BYTES -= embedding_nbytes(old)
            EMBEDDING_CACHE[key] = vector
            EMBEDDING_CACHE_BYTES += embedding_nbytes(vector)
        while EMBEDDING_CACHE and EMBEDDING_CACHE_BYTES > INGEST.cache_mib * 1024**2:
            _, evicted = EMBEDDING_CACHE.popitem(last=False)
            EMBEDDING_CACHE_BYTES -= embedding_nbytes(evicted)
    return result


@contextmanager
def record_seconds(timings: dict, stage: str):
    started = time.perf_counter()
    try:
        yield
    finally:
        timings[stage] = timings.get(stage, 0.0) + time.perf_counter() - started


RUN_MANIFEST["models"] = {kind: spec[0] for kind, spec in EMBEDDING_MODELS.items()}
RUN_MANIFEST["ingestion"] = vars(INGEST)
display(
    pd.DataFrame(
        [
            {"representation": kind, "model": model, "batch_size": batch}
            for kind, (model, _, batch) in EMBEDDING_MODELS.items()
        ]
    )
)


,representation,model,batch_size
0,dense,BAAI/bge-small-en-v1.5,128
1,sparse,prithivida/Splade_PP_en_v1,32
2,colbert,colbert-ir/colbertv2.0,8


## 7. Qdrant Collection Design

A single collection stores all three retrieval representations for each chunk. Dense and sparse vectors are searchable across the collection; the ColBERT multivector is configured for candidate reranking only. Payload fields preserve attribution, hierarchy, neighboring source sections, and lightweight filterable metadata.

### 7.1 Vector Configuration

In [65]:
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "docs_search")

DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"
COLBERT_VECTOR_NAME = "colbert"

vectors_config = {
    DENSE_VECTOR_NAME: models.VectorParams(
        size=DENSE_EMBEDDING_DIM, distance=models.Distance.COSINE
    ),
    COLBERT_VECTOR_NAME: models.VectorParams(
        size=COLBERT_EMBEDDING_DIM,
        distance=models.Distance.COSINE,
        multivector_config=(
            models.MultiVectorConfig(comparator=(models.MultiVectorComparator.MAX_SIM))
        ),
        hnsw_config=models.HnswConfigDiff(m=0),
    ),
}

sparse_vectors_config = {SPARSE_VECTOR_NAME: (models.SparseVectorParams())}

print(
    f"Collection: {COLLECTION_NAME}\n"
    f"Dense dimension: {DENSE_EMBEDDING_DIM}\n"
    f"ColBERT dimension: {COLBERT_EMBEDDING_DIM}\n"
    "ColBERT HNSW: disabled (reranking only)"
)


Collection: docs_search
Dense dimension: 384
ColBERT dimension: 128
ColBERT HNSW: disabled (reranking only)


### 7.2 Payload Schema

The payload follows the final-project schema and adds stable source identifiers useful for evaluation and filtering. `embedding_text` is deliberately not stored in Qdrant because it can always be reconstructed from breadcrumbs and `chunk_text`.

In [66]:
def nullable_text(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return str(value)


def build_qdrant_payload(row) -> dict:
    breadcrumbs = row["breadcrumbs"]
    if not isinstance(breadcrumbs, (list, tuple)):
        breadcrumbs = [str(breadcrumbs)]

    tags = row["tags"]
    if not isinstance(tags, (list, tuple)):
        tags = [str(tags)]

    return {
        "path": str(row["path"]),
        "source_commit": QDRANT_DOCS_COMMIT,
        "section_index": int(row["section_index"]),
        "chunk_index": int(row["chunk_index"]),
        "page_title": str(row["page_title"]),
        "section_title": str(row["section_title"]),
        "page_url": str(row["page_url"]),
        "section_url": str(row["section_url"]),
        "breadcrumbs": [str(value) for value in breadcrumbs],
        "chunk_text": str(row["chunk_text"]),
        "prev_section_text": nullable_text(row["prev_section_text"]),
        "next_section_text": nullable_text(row["next_section_text"]),
        "tags": [str(value) for value in tags],
        "chunk_tokens": int(row["tokens"]),
    }


sample_payload = build_qdrant_payload(retrieval_chunks_df.iloc[0])

print(f"Payload fields: {sorted(sample_payload)}")
display(pd.DataFrame([sample_payload]))


Payload fields: ['breadcrumbs', 'chunk_index', 'chunk_text', 'chunk_tokens', 'next_section_text', 'page_title', 'page_url', 'path', 'prev_section_text', 'section_index', 'section_title', 'section_url', 'source_commit', 'tags']


,path,source_commit,section_index,chunk_index,page_title,section_title,page_url,section_url,breadcrumbs,chunk_text,prev_section_text,next_section_text,tags,chunk_tokens
0,_index.md,46e80312568d1e4505917b94c1ef78f4000330aa,0,0,Documentation,Qdrant Documentation,https://qdrant.tech/documentation/,https://qdrant.tech/documentation/#qdrant-docu...,"[Documentation, Qdrant Documentation]",Qdrant is an AI-native vector search engine fo...,None,- [Local Quickstart](/documentation/quickstart...,"[-index.md, documentation, qdrant-documentation]",42


### 7.3 Collection Creation

For reproducible notebook runs the primary collection is recreated from scratch. Payload indexes are created before ingestion for fields used for exact lookup or filtering, following the indexing lessons from Day 2.

The dense vector keeps Qdrant's normal HNSW configuration. The ColBERT multivector deliberately uses `m=0` because the course treats it as a reranker rather than a whole-collection ANN retriever.

The primary `docs_search` collection remains unquantized so that Sections 8-9 provide a clean full-precision retrieval baseline. Collection optimization is evaluated later in isolated dense-only benchmark collections. This prevents quantization or storage-tier changes from silently altering the final-project baseline metrics.


In [67]:
RECREATE_COLLECTION = True  # Explicitly destructive only when this cell is executed.
DEFER_PRIMARY_INDEXING = True
PRIMARY_INDEXING_THRESHOLD_KB = 10000

if qdrant_client.collection_exists(COLLECTION_NAME):
    if RECREATE_COLLECTION:
        qdrant_client.delete_collection(COLLECTION_NAME)
    else:
        raise RuntimeError(
            f"Collection {COLLECTION_NAME!r} already exists. "
            "Set RECREATE_COLLECTION=True to rebuild it."
        )

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=vectors_config,
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=0 if DEFER_PRIMARY_INDEXING else PRIMARY_INDEXING_THRESHOLD_KB
    ),
    sparse_vectors_config=(sparse_vectors_config),
)

payload_indexes = {
    "path": models.PayloadSchemaType.KEYWORD,
    "section_url": models.PayloadSchemaType.KEYWORD,
    "tags": models.PayloadSchemaType.KEYWORD,
}

for field_name, field_schema in payload_indexes.items():
    qdrant_client.create_payload_index(
        collection_name=COLLECTION_NAME, field_name=field_name, field_schema=field_schema, wait=True
    )

collection_info = qdrant_client.get_collection(COLLECTION_NAME)

print(
    f"Collection status: {collection_info.status}\n"
    f"Payload indexes created: {list(payload_indexes)}"
)


Collection status: green
Payload indexes created: ['path', 'section_url', 'tags']


### 7.4 Data Ingestion

A single upload call consumes a bounded generator for the entire corpus; it does not recreate a worker pool per embedding batch. Dense, sparse, ColBERT, and PointStruct preparation have separate timers. The residual wall time includes transport, client overhead, and synchronization: with parallel workers it is **not** pure network time.

All writes use `wait=True`. A separate readiness barrier follows index re-enablement. On this small primary collection the normal planner may legitimately use full scan; forced-HNSW comparisons belong to Section 10.

In [68]:
def wait_for_collection(
    collection_name: str,
    *,
    require_hnsw: bool = False,
    timeout_seconds: float = 300,
    poll_interval_seconds: float = 0.2,
):
    """Readiness latency, not an exact measurement of internal index build time."""
    deadline = time.monotonic() + timeout_seconds
    stable = 0
    previous_state = None
    while time.monotonic() < deadline:
        info = qdrant_client.get_collection(collection_name)
        indexed = int(info.indexed_vectors_count or 0)
        points = int(info.points_count or 0)
        state = (str(info.status), points, indexed)
        ready = (
            info.status == models.CollectionStatus.GREEN
            and str(info.optimizer_status).lower() == "ok"
        )
        if require_hnsw:
            ready = ready and points > 0 and indexed >= 0.95 * points
        stable = stable + 1 if ready and state == previous_state else 0
        if stable >= 2:
            return info
        previous_state = state
        time.sleep(poll_interval_seconds)
    raise TimeoutError(f"Collection not ready: {collection_name}; last state: {previous_state}")


def build_qdrant_point(row, dense_vector, sparse_vector, colbert_vector):
    return models.PointStruct(
        id=row.point_id,
        vector={
            DENSE_VECTOR_NAME: np.asarray(dense_vector, dtype=np.float32).tolist(),
            SPARSE_VECTOR_NAME: models.SparseVector(
                indices=sparse_vector.indices.astype(np.uint32).tolist(),
                values=sparse_vector.values.astype(np.float32).tolist(),
            ),
            COLBERT_VECTOR_NAME: np.asarray(colbert_vector, dtype=np.float32).tolist(),
        },
        payload=build_qdrant_payload(row._asdict()),
    )


def iter_embedded_points(frame: pd.DataFrame, timings: dict):
    for start in tqdm(
        range(0, len(frame), INGEST.pipeline_batch), desc="Embedding chunks", unit="batch"
    ):
        batch = frame.iloc[start : start + INGEST.pipeline_batch]
        vectors = {}
        for kind in EMBEDDING_MODELS:
            with record_seconds(timings, kind):
                vectors[kind] = embed_documents(kind, batch["embedding_text"].tolist())
        assert all(len(v) == len(batch) for v in vectors.values())
        with record_seconds(timings, "point_preparation"):
            points = [
                build_qdrant_point(row, dense, sparse, colbert)
                for row, dense, sparse, colbert in zip(
                    batch.itertuples(index=False),
                    vectors["dense"],
                    vectors["sparse"],
                    vectors["colbert"],
                )
            ]
        yield from points


ingestion_timings = {}
ingestion_start = time.perf_counter()
qdrant_client.upload_points(
    collection_name=COLLECTION_NAME,
    points=iter_embedded_points(retrieval_chunks_df, ingestion_timings),
    batch_size=INGEST.upload_batch,
    parallel=INGEST.upload_parallel,
    max_retries=3,
    wait=True,
)
ingestion_seconds = time.perf_counter() - ingestion_start
ingestion_timings["non_generator_wall"] = max(
    0.0, ingestion_seconds - sum(ingestion_timings.values())
)
with record_seconds(ingestion_timings, "readiness"):
    if DEFER_PRIMARY_INDEXING:
        qdrant_client.update_collection(
            collection_name=COLLECTION_NAME,
            optimizers_config=models.OptimizersConfigDiff(
                indexing_threshold=PRIMARY_INDEXING_THRESHOLD_KB
            ),
        )
    wait_for_collection(COLLECTION_NAME)

stored_points = qdrant_client.count(COLLECTION_NAME, exact=True).count
assert stored_points == len(retrieval_chunks_df)
print(
    f"Stored: {stored_points:,}; embedding + upload: {ingestion_seconds:.2f}s; "
    f"throughput: {stored_points / ingestion_seconds:.1f} points/s"
)
display(pd.DataFrame(ingestion_timings.items(), columns=["stage", "seconds"]))
RUN_MANIFEST["ingestion_seconds"] = ingestion_timings


Embedding chunks: 100%|██████████| 33/33 [32:42<00:00, 59.48s/batch]


Stored: 4,134; embedding + upload: 1963.36s; throughput: 2.1 points/s


,stage,seconds
0,dense,472.175555
1,sparse,505.983682
2,colbert,943.775557
3,point_preparation,9.560163
4,non_generator_wall,31.869323
5,readiness,19.976389


## 8. Retrieval Experiments

Retrieval experiments isolate each search signal and then combine them using Qdrant's Query API. The same user query is used below for qualitative inspection; the formal comparison is performed on the evaluation set in Section 9.

Search functions return raw chunk-level results. Evaluation later collapses duplicate chunks from the same `section_url` so metrics reflect whether the correct documentation **section** was found.

In [69]:
DEMO_QUERY = "how to configure HNSW parameters for better recall"

# Transparent, domain-level aliases improve lexical candidate recall without replacing the query.
TECHNICAL_QUERY_EXPANSIONS = (
    (r"\b(?:one[- ]bit|1[- ]bit)\b", "binary quantization"),
    (r"\b(?:eight[- ]bit|8[- ]bit|int8)\b", "scalar quantization"),
    (r"\bnearest[- ]neighbou?r accuracy\b", "HNSW hnsw_ef recall"),
    (r"\b(?:no vector|without (?:a )?vector|all stored records)\b", "scroll points"),
    (r"\btoken embeddings?\b", "multivectors late interaction max similarity"),
    (r"\bmany zeros\b", "sparse vectors indices values"),
    (r"\bindexing threshold\b", "indexing optimizer HNSW"),
)


@lru_cache(maxsize=512)
def prepare_retrieval_query(query: str) -> str:
    normalized = re.sub(r"\s+", " ", query).strip()
    expansions = [
        term for pattern, term in TECHNICAL_QUERY_EXPANSIONS if re.search(pattern, normalized, re.I)
    ]
    return (
        normalized
        if not expansions
        else f"{normalized}\nTechnical terms: {'; '.join(dict.fromkeys(expansions))}"
    )


def format_search_results(points, snippet_chars: int = 240) -> pd.DataFrame:
    records = []

    for rank, point in enumerate(points, start=1):
        payload = point.payload or {}
        snippet = re.sub(r"\s+", " ", str(payload.get("chunk_text", ""))).strip()

        records.append(
            {
                "rank": rank,
                "score": float(point.score),
                "page_title": payload.get("page_title"),
                "section_title": payload.get("section_title"),
                "section_url": payload.get("section_url"),
                "snippet": snippet[:snippet_chars],
            }
        )

    return pd.DataFrame(records)


SEARCH_PAYLOAD = ["page_title", "section_title", "section_url", "chunk_text"]


def query_primary(*, limit: int, payload=SEARCH_PAYLOAD, **kwargs):
    return qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        limit=limit,
        with_payload=payload,
        with_vectors=False,
        **kwargs,
    ).points


### 8.1 Dense Retrieval

Dense retrieval provides the semantic baseline. The query is encoded with the BGE query encoder and searched against the named `dense` vector.

In [70]:
def embed_dense_query(query: str) -> list[float]:
    prepared_query = prepare_retrieval_query(query)
    vector = np.asarray(
        list(dense_embedding_model.query_embed([prepared_query]))[0], dtype=np.float32
    )
    return vector.tolist()


PRIMARY_DENSE_SEARCH_PARAMS = None  # Frozen after research, before held-out validation.


def search_dense(query: str, limit: int = DEFAULT_SEARCH_LIMIT, *, payload=SEARCH_PAYLOAD):
    return query_primary(
        query=embed_dense_query(query),
        using=DENSE_VECTOR_NAME,
        search_params=PRIMARY_DENSE_SEARCH_PARAMS,
        limit=limit,
        payload=payload,
    )


dense_demo_points = search_dense(DEMO_QUERY, limit=5)
display(format_search_results(dense_demo_points))


,rank,score,page_title,section_title,section_url,snippet
0,1,0.788133,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
1,2,0.757666,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
2,3,0.752222,Optimize Performance,Improving Precision,https://qdrant.tech/documentation/ops-optimiza...,Increase the `ef` and `m` parameters of the HN...
3,4,0.737701,Memory Tiers,HNSW Vector Index,https://qdrant.tech/documentation/ops-configur...,"The legacy parameter is `on_disk`, set in `hns..."
4,5,0.727837,Configuration,Configuration Options,https://qdrant.tech/documentation/ops-configur...,```yaml # When the maximum estimated amount of...


### 8.2 Sparse Retrieval

Sparse retrieval isolates the SPLADE++ signal using the same neural sparse model as the Day 3 and Day 5 course examples. Documents were encoded with `embed`; queries use `query_embed` so the sparse representation follows the model's retrieval-specific query path.


In [71]:
def embed_sparse_query(query: str) -> models.SparseVector:
    sparse_vector = next(sparse_embedding_model.query_embed([prepare_retrieval_query(query)]))

    return models.SparseVector(
        indices=sparse_vector.indices.astype(np.uint32).tolist(),
        values=sparse_vector.values.astype(np.float32).tolist(),
    )


def search_sparse(query: str, limit: int = DEFAULT_SEARCH_LIMIT, *, payload=SEARCH_PAYLOAD):
    return query_primary(
        query=embed_sparse_query(query), using=SPARSE_VECTOR_NAME, limit=limit, payload=payload
    )


sparse_demo_points = search_sparse(DEMO_QUERY, limit=5)
display(format_search_results(sparse_demo_points))


,rank,score,page_title,section_title,section_url,snippet
0,1,21.222157,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
1,2,19.475574,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
2,3,17.632473,Storage,Configuring Memmap storage,https://qdrant.tech/documentation/manage-data/...,"In addition, you can configure a [memory tier]..."
3,4,17.017151,Search Quality,Why Recall Won't Be 1.0 (And That's OK),https://qdrant.tech/documentation/migration-gu...,Even a correct migration will often show recal...
4,5,15.357033,Diagnosing Discrepancies,HNSW Index Not Built,https://qdrant.tech/documentation/migration-gu...,"On a freshly migrated collection, the HNSW ind..."


### 8.3 Hybrid Retrieval

Hybrid retrieval prefetches dense and sparse candidates in the same server-side Query API request. Fusion is applied by Qdrant rather than by client-side score arithmetic, avoiding direct comparison of incompatible dense and sparse score scales.

In [72]:
def build_hybrid_prefetches(
    query: str, prefetch_limit: int = HYBRID_PREFETCH_LIMIT
) -> list[models.Prefetch]:
    return [
        models.Prefetch(
            query=embed_dense_query(query),
            using=DENSE_VECTOR_NAME,
            params=PRIMARY_DENSE_SEARCH_PARAMS,
            limit=prefetch_limit,
        ),
        models.Prefetch(
            query=embed_sparse_query(query), using=SPARSE_VECTOR_NAME, limit=prefetch_limit
        ),
    ]


def get_fusion_query(fusion: str) -> models.FusionQuery:
    fusion = fusion.lower()

    if fusion == "rrf":
        return models.FusionQuery(fusion=models.Fusion.RRF)

    if fusion == "dbsf":
        return models.FusionQuery(fusion=models.Fusion.DBSF)

    raise ValueError(f"Unsupported fusion: {fusion}")


### 8.4 RRF Fusion

Reciprocal Rank Fusion combines the dense and sparse ranked lists by rank position. It is the primary hybrid baseline because it does not assume that raw scores from the two retrievers are calibrated to the same scale.

In [73]:
def search_hybrid(
    query: str,
    limit: int = DEFAULT_SEARCH_LIMIT,
    *,
    fusion: str = "rrf",
    prefetch_limit: int = HYBRID_PREFETCH_LIMIT,
    payload=SEARCH_PAYLOAD
):
    return query_primary(
        prefetch=build_hybrid_prefetches(query, prefetch_limit),
        query=get_fusion_query(fusion),
        limit=limit,
        payload=payload,
    )


search_hybrid_rrf = partial(search_hybrid, fusion="rrf")
rrf_demo_points = search_hybrid_rrf(DEMO_QUERY, limit=5)
display(format_search_results(rrf_demo_points))


,rank,score,page_title,section_title,section_url,snippet
0,1,0.833333,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
1,2,0.833333,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
2,3,0.342857,Search Quality,Why Recall Won't Be 1.0 (And That's OK),https://qdrant.tech/documentation/migration-gu...,Even a correct migration will often show recal...
3,4,0.333333,Storage,Configuring Memmap storage,https://qdrant.tech/documentation/manage-data/...,"In addition, you can configure a [memory tier]..."
4,5,0.316667,Optimize Performance,Improving Precision,https://qdrant.tech/documentation/ops-optimiza...,Increase the `ef` and `m` parameters of the HN...


### 8.5 DBSF Fusion

Distribution-Based Score Fusion normalizes the score distribution produced by each retrieval leg before combining them. It provides a server-side alternative to RRF and is evaluated on the same ground-truth set rather than selected by assumption.

In [74]:
search_hybrid_dbsf = partial(search_hybrid, fusion="dbsf")
dbsf_demo_points = search_hybrid_dbsf(DEMO_QUERY, limit=5)
display(format_search_results(dbsf_demo_points))


,rank,score,page_title,section_title,section_url,snippet
0,1,2.113393,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
1,2,2.012527,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
2,3,1.489518,Optimize Performance,Improving Precision,https://qdrant.tech/documentation/ops-optimiza...,Increase the `ef` and `m` parameters of the HN...
3,4,1.487988,Search Quality,Why Recall Won't Be 1.0 (And That's OK),https://qdrant.tech/documentation/migration-gu...,Even a correct migration will often show recal...
4,5,1.413365,Memory Tiers,HNSW Vector Index,https://qdrant.tech/documentation/ops-configur...,"The legacy parameter is `on_disk`, set in `hns..."


### 8.6 ColBERT Reranking

The final multistage pipeline mirrors the Day 5 Universal Query demo:

1. Dense and SPLADE prefetches execute in parallel.
2. Qdrant fuses the two ranked lists server-side with `FusionQuery` using RRF or DBSF.
3. The fused candidate list becomes a nested `Prefetch`.
4. A ColBERT query multivector reranks only those candidates with `MAX_SIM`.

This keeps the entire hybrid + reranking pipeline declarative and server-side instead of reproducing rank fusion or ColBERT scoring in Python.


In [75]:
def embed_colbert_query(query: str) -> list[list[float]]:
    multivector = np.asarray(
        list(colbert_embedding_model.query_embed([prepare_retrieval_query(query)]))[0],
        dtype=np.float32,
    )
    return multivector.tolist()


def search_hybrid_colbert(
    query: str,
    fusion: str = "rrf",
    limit: int = DEFAULT_SEARCH_LIMIT,
    candidate_limit: int | None = None,
    prefetch_limit: int = HYBRID_PREFETCH_LIMIT,
    *,
    payload=SEARCH_PAYLOAD,
):
    candidate_limit = RERANK_CANDIDATE_LIMIT if candidate_limit is None else candidate_limit
    prefetch_limit = max(prefetch_limit, candidate_limit)

    hybrid_candidate_prefetch = models.Prefetch(
        prefetch=build_hybrid_prefetches(query, prefetch_limit=prefetch_limit),
        query=get_fusion_query(fusion),
        limit=candidate_limit,
    )

    return query_primary(
        prefetch=hybrid_candidate_prefetch,
        query=embed_colbert_query(query),
        using=COLBERT_VECTOR_NAME,
        limit=limit,
        payload=payload,
    )


colbert_demo_points = search_hybrid_colbert(DEMO_QUERY, fusion="rrf", limit=5)
display(format_search_results(colbert_demo_points))


,rank,score,page_title,section_title,section_url,snippet
0,1,22.404793,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
1,2,19.722095,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
2,3,18.573849,Indexing,Vector Index,https://qdrant.tech/documentation/manage-data/...,```yaml storage: # Default parameters of HNSW ...
3,4,18.274092,Storage,Configuring Memmap storage,https://qdrant.tech/documentation/manage-data/...,"In addition, you can configure a [memory tier]..."
4,5,18.165890,Search Quality,Why Recall Won't Be 1.0 (And That's OK),https://qdrant.tech/documentation/migration-gu...,Even a correct migration will often show recal...


## 9. Retrieval Evaluation

The final project is evaluated at the **documentation-section level** because the ground truth identifies sections/anchors, while some long sections produce multiple retrieval chunks.

For each pipeline, chunk results are collapsed by `section_url` before evaluation, preserving the rank of the first occurrence. The complete returned section ranking (up to 50 unique sections) is then converted to a `ranx.Run`. `ranx` computes Recall@10 and MRR over the full returned ranking (up to 50 unique sections) from section-level qrels; the benchmark harness computes P50/P95 latency separately.


### 9.1 Evaluation Dataset

The 25 original development queries and the 20 previously observed held-out queries are now one 45-query **research set**. The earlier held-out results informed this revision, so those questions can no longer support an independent quality claim. `split` is the only routing field; `source_cohort` records historical provenance only and is never used for selection or evaluation routing.

Relevance is curated from the pinned documentation's section bodies, using explicit anchors rather than title regexes. These are agent-curated, non-exhaustive judgments, not an independently human-annotated gold standard. The final untouched holdout is materialized only after the restricted confirmation-cycle choice and `FINAL_CONFIG` are frozen in Sections 11.9–12.

In [76]:
# Explicit relevance judgments, reviewed against section bodies in the pinned corpus.
EVALUATION_SPLIT_ORDER = ("research", "confirmation", "final_holdout")
ALLOWED_EVALUATION_SPLITS = frozenset(EVALUATION_SPLIT_ORDER)

EVALUATION_SPECS = [
    {
        "query_id": "q01",
        "query": "how do I tune HNSW parameters to improve search recall?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["manage-data/indexing/#vector-index"],
    },
    {
        "query_id": "q02",
        "query": "how can scalar quantization reduce vector memory usage?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["manage-data/quantization/#scalar-quantization"],
    },
    {
        "query_id": "q03",
        "query": "when should I use product quantization in Qdrant?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["manage-data/quantization/#product-quantization"],
    },
    {
        "query_id": "q04",
        "query": "how do I enable binary quantization for a collection?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["manage-data/quantization/#setting-up-binary-quantization"],
    },
    {
        "query_id": "q05",
        "query": "how do must, should, and must_not filtering clauses work?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": [
            "search/filtering/#must",
            "search/filtering/#should",
            "search/filtering/#must-not",
        ],
    },
    {
        "query_id": "q06",
        "query": "how do I filter points inside a geographic bounding box?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["search/filtering/#geo-bounding-box"],
    },
    {
        "query_id": "q07",
        "query": "how do I create a payload index for faster filtered search?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["manage-data/indexing/#create-a-payload-index"],
    },
    {
        "query_id": "q08",
        "query": "how do I configure a full text index in Qdrant?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["manage-data/indexing/#full-text-index"],
    },
    {
        "query_id": "q09",
        "query": "what is the recommended way to upload points in batches?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["manage-data/points/#upload-points"],
    },
    {
        "query_id": "q10",
        "query": "how can I iterate over all points without similarity search?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["manage-data/points/#scroll-points"],
    },
    {
        "query_id": "q11",
        "query": "what identifier formats can Qdrant point IDs use?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["manage-data/points/#point-ids"],
    },
    {
        "query_id": "q12",
        "query": "how can I switch collections atomically with an alias?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["manage-data/collections/#collection-aliases"],
    },
    {
        "query_id": "q13",
        "query": "how do I configure the replication factor for a collection?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["scaling/distributed_deployment/#replication-factor"],
    },
    {
        "query_id": "q14",
        "query": "what does write consistency factor control in a cluster?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["scaling/consistency-guarantees/#write-consistency-factor"],
    },
    {
        "query_id": "q15",
        "query": "how do I group search results by a payload field?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["search/search/#grouping-api"],
    },
    {
        "query_id": "q16",
        "query": "how do I combine dense and sparse retrieval in one query?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["search/hybrid-queries/#hybrid-search"],
    },
    {
        "query_id": "q17",
        "query": "how are ColBERT token vectors stored and compared as multivectors?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["manage-data/vectors/#multivectors"],
    },
    {
        "query_id": "q18",
        "query": "what are sparse vectors and when should I use them?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["manage-data/vectors/#sparse-vectors"],
    },
    {
        "query_id": "q19",
        "query": "how does the recommendation API use positive and negative examples?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["search/explore/#recommendation-api"],
    },
    {
        "query_id": "q20",
        "query": "how do I create a collection snapshot for backup or migration?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": ["snapshots/#create-snapshot"],
    },
    {
        "query_id": "q21",
        "query": "how can I route points using a custom shard key?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": ["scaling/distributed_deployment/#multitenancy"],
    },
    {
        "query_id": "q22",
        "query": "when does the indexing optimizer build vector indexes?",
        "query_type": "concept",
        "source_cohort": "dev",
        "anchors": ["ops-optimization/optimizer/#indexing-optimizer"],
    },
    {
        "query_id": "q23",
        "query": "how should I estimate RAM and disk capacity for a Qdrant collection?",
        "query_type": "how-to",
        "source_cohort": "dev",
        "anchors": [
            "capacity-planning/#calculating-ram-and-disk-size",
            "capacity-planning/#original-vectors",
            "capacity-planning/#putting-it-together",
        ],
    },
    {
        "query_id": "q24",
        "query": "which endpoints expose Qdrant metrics and telemetry?",
        "query_type": "api-usage",
        "source_cohort": "dev",
        "anchors": [
            "ops-monitoring/monitoring/#metrics",
            "ops-monitoring/monitoring/#telemetry-endpoint",
        ],
    },
    {
        "query_id": "q25",
        "query": "Qdrant reports an incompatible file system - what does that mean?",
        "query_type": "troubleshooting",
        "source_cohort": "dev",
        "anchors": ["common-errors/#incompatible-file-system"],
    },
    {
        "query_id": "t01",
        "query": "I want greater nearest-neighbor accuracy without rebuilding my graph. Which query-time parameter should I adjust?",
        "query_type": "how-to",
        "source_cohort": "test",
        "anchors": ["manage-data/indexing/#vector-index"],
    },
    {
        "query_id": "t02",
        "query": "Why does converting float vectors to eight-bit components save about three quarters of their vector storage?",
        "query_type": "concept",
        "source_cohort": "test",
        "anchors": ["manage-data/quantization/#scalar-quantization"],
    },
    {
        "query_id": "t03",
        "query": "Is the strongest vector compression always the fastest option for distance calculations?",
        "query_type": "concept",
        "source_cohort": "test",
        "anchors": ["manage-data/quantization/#product-quantization"],
    },
    {
        "query_id": "t04",
        "query": "Can I turn on one-bit vector compression after the collection already exists?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["manage-data/quantization/#setting-up-binary-quantization"],
    },
    {
        "query_id": "t05",
        "query": "Exclude a point when either city is London or color is red. Which Boolean clause expresses that?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["search/filtering/#must-not"],
    },
    {
        "query_id": "t06",
        "query": "Return a point when at least one of two metadata conditions matches, even if the other does not.",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["search/filtering/#should"],
    },
    {
        "query_id": "t07",
        "query": "Which two opposite corners do I supply to restrict locations to a rectangle?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["search/filtering/#geo-bounding-box"],
    },
    {
        "query_id": "t08",
        "query": "How do I index a nested payload field using its dotted path?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["manage-data/indexing/#create-a-payload-index"],
    },
    {
        "query_id": "t09",
        "query": "I need word tokenization that ignores case and drops tokens outside a chosen length range. How is the index configured?",
        "query_type": "how-to",
        "source_cohort": "test",
        "anchors": ["manage-data/indexing/#full-text-index"],
    },
    {
        "query_id": "t10",
        "query": "I have no vector to search with and need every stored record matching a metadata condition. Which operation should I use?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["manage-data/points/#scroll-points"],
    },
    {
        "query_id": "t11",
        "query": "Can the identifier 550e8400-e29b-41d4-a716-446655440000 be used instead of an integer?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["manage-data/points/#point-ids"],
    },
    {
        "query_id": "t12",
        "query": "My documents have many chunks and occupy most of the first results. How can I return results grouped by document_id?",
        "query_type": "troubleshooting",
        "source_cohort": "test",
        "anchors": ["search/search/#grouping-api"],
    },
    {
        "query_id": "t13",
        "query": "Can one point contain a different number of token embeddings than another point, and how is one similarity score produced?",
        "query_type": "concept",
        "source_cohort": "test",
        "anchors": ["manage-data/vectors/#multivectors"],
    },
    {
        "query_id": "t14",
        "query": "For an embedding with many zeros, must I allocate and send the entire dense array?",
        "query_type": "concept",
        "source_cohort": "test",
        "anchors": ["manage-data/vectors/#sparse-vectors"],
    },
    {
        "query_id": "t15",
        "query": "Can I supply raw vectors as negative examples to the recommendation query?",
        "query_type": "api-usage",
        "source_cohort": "test",
        "anchors": ["search/explore/#recommendation-api"],
    },
    {
        "query_id": "t16",
        "query": "Does a collection snapshot taken on one cluster node back up the data on every other node too?",
        "query_type": "troubleshooting",
        "source_cohort": "test",
        "anchors": ["snapshots/#create-snapshot"],
    },
    {
        "query_id": "t17",
        "query": "Why can setting the indexing threshold to zero leave newly loaded vectors without an HNSW graph?",
        "query_type": "troubleshooting",
        "source_cohort": "test",
        "anchors": ["ops-optimization/optimizer/#indexing-optimizer"],
    },
    {
        "query_id": "t18",
        "query": "Does Prometheus need to scrape every peer separately when Qdrant is behind a load balancer?",
        "query_type": "how-to",
        "source_cohort": "test",
        "anchors": ["ops-monitoring/monitoring/#metrics"],
    },
    {
        "query_id": "t19",
        "query": "A startup check reports a FUSE filesystem and warns about possible corruption. Is it safe to keep using this storage?",
        "query_type": "troubleshooting",
        "source_cohort": "test",
        "anchors": ["common-errors/#incompatible-file-system"],
    },
    {
        "query_id": "t20",
        "query": "Which structures besides original vectors consume memory and disk, and should I allow operational headroom?",
        "query_type": "how-to",
        "source_cohort": "test",
        "anchors": ["capacity-planning/#putting-it-together"],
    },
]

section_catalog_df = (
    retrieval_chunks_df[["path", "page_title", "section_title", "section_url", "breadcrumbs"]]
    .drop_duplicates("section_url")
    .reset_index(drop=True)
)
known_section_urls = set(section_catalog_df["section_url"])


def materialize_evaluation_specs(specs: list[dict], *, split: str) -> pd.DataFrame:
    if split not in ALLOWED_EVALUATION_SPLITS:
        raise ValueError(
            f"Invalid evaluation split {split!r}; allowed={sorted(ALLOWED_EVALUATION_SPLITS)}"
        )
    records = []
    for spec in specs:
        expected_urls = [
            "https://qdrant.tech/documentation/" + anchor for anchor in spec["anchors"]
        ]
        missing = set(expected_urls) - known_section_urls
        if missing:
            raise ValueError(f"Missing explicit qrels for {spec['query_id']}: {sorted(missing)}")
        records.append(
            {
                **spec,
                "source_cohort": spec.get("source_cohort"),
                "split": split,
                "expected_urls": expected_urls,
                "expected_section_count": len(expected_urls),
            }
        )
    frame = pd.DataFrame(records)
    assert frame["query_id"].is_unique and frame["query"].is_unique
    return frame


evaluation_df = materialize_evaluation_specs(EVALUATION_SPECS, split="research")
all_evaluation_df = evaluation_df.copy()
display(
    evaluation_df[
        ["query_id", "split", "source_cohort", "query_type", "query", "expected_urls"]
    ]
)


,query_id,split,source_cohort,query_type,query,expected_urls
0,q01,research,dev,how-to,how do I tune HNSW parameters to improve searc...,[https://qdrant.tech/documentation/manage-data...
1,q02,research,dev,concept,how can scalar quantization reduce vector memo...,[https://qdrant.tech/documentation/manage-data...
2,q03,research,dev,concept,when should I use product quantization in Qdrant?,[https://qdrant.tech/documentation/manage-data...
3,q04,research,dev,how-to,how do I enable binary quantization for a coll...,[https://qdrant.tech/documentation/manage-data...
4,q05,research,dev,concept,"how do must, should, and must_not filtering cl...",[https://qdrant.tech/documentation/search/filt...
5,q06,research,dev,api-usage,how do I filter points inside a geographic bou...,[https://qdrant.tech/documentation/search/filt...
6,q07,research,dev,how-to,how do I create a payload index for faster fil...,[https://qdrant.tech/documentation/manage-data...
7,q08,research,dev,how-to,how do I configure a full text index in Qdrant?,[https://qdrant.tech/documentation/manage-data...
8,q09,research,dev,api-usage,what is the recommended way to upload points i...,[https://qdrant.tech/documentation/manage-data...
9,q10,research,dev,api-usage,how can I iterate over all points without simi...,[https://qdrant.tech/documentation/manage-data...


### 9.2 Qrels

Explicit qrels are stored at section-URL level inside the notebook. Multiple relevant anchors are allowed only when their bodies contain useful answer evidence. `recall_at_10` is the fraction of all judged relevant section URLs retrieved in the first 10 unique sections. Recall is measured against the judged qrels and therefore does not claim exhaustive relevance annotation.


In [77]:
def build_qrels_from_queries(queries: pd.DataFrame) -> pd.DataFrame:
    """Derive qrels metadata from the canonical query dataset."""
    return pd.DataFrame(
        [
            {
                "query_id": row.query_id,
                "split": row.split,
                "section_url": url,
                "relevance": 1,
            }
            for row in queries.itertuples(index=False)
            for url in row.expected_urls
        ],
        columns=["query_id", "split", "section_url", "relevance"],
    )


def validate_evaluation_consistency(
    queries: pd.DataFrame, qrels: pd.DataFrame
) -> pd.DataFrame:
    """Validate query/qrels routing without repairing or mutating either input."""
    query_columns = {"query_id", "split", "expected_urls"}
    qrel_columns = {"query_id", "split", "section_url", "relevance"}
    missing_query_columns = sorted(query_columns - set(queries.columns))
    missing_qrel_columns = sorted(qrel_columns - set(qrels.columns))
    if missing_query_columns or missing_qrel_columns:
        raise ValueError(
            "Missing evaluation columns: "
            f"queries={missing_query_columns}, qrels={missing_qrel_columns}"
        )

    stale_split_columns = sorted(
        column
        for column in [*queries.columns, *qrels.columns]
        if column in {"split_x", "split_y"}
    )
    if stale_split_columns:
        raise ValueError(f"Stale split columns found: {stale_split_columns}")

    duplicate_query_ids = sorted(
        queries.loc[queries["query_id"].duplicated(keep=False), "query_id"]
        .astype(str)
        .unique()
    )
    if duplicate_query_ids:
        raise ValueError(f"Duplicate query_id values: {duplicate_query_ids}")

    missing_split_ids = sorted(
        queries.loc[
            queries["split"].isna() | queries["split"].astype(str).str.strip().eq(""),
            "query_id",
        ].astype(str)
    )
    if missing_split_ids:
        raise ValueError(f"Queries without a split: {missing_split_ids}")

    invalid_split_rows = queries.loc[
        ~queries["split"].isin(ALLOWED_EVALUATION_SPLITS), ["query_id", "split"]
    ]
    if not invalid_split_rows.empty:
        raise ValueError(
            "Invalid query split assignments: "
            f"{invalid_split_rows.to_dict(orient='records')}"
        )

    multi_split_query_ids = sorted(
        queries.groupby("query_id")["split"].nunique().loc[lambda values: values.ne(1)].index
    )
    if multi_split_query_ids:
        raise ValueError(f"Queries assigned to multiple splits: {multi_split_query_ids}")

    duplicate_qrels = qrels.loc[
        qrels.duplicated(["query_id", "section_url"], keep=False),
        ["query_id", "section_url"],
    ]
    if not duplicate_qrels.empty:
        raise ValueError(
            "Duplicate qrels: "
            f"{duplicate_qrels.to_dict(orient='records')}"
        )

    query_ids = set(queries["query_id"].astype(str))
    qrel_query_ids = set(qrels["query_id"].astype(str))
    missing_qrels = sorted(query_ids - qrel_query_ids)
    unknown_qrels = sorted(qrel_query_ids - query_ids)
    if missing_qrels:
        raise ValueError(f"Queries without qrels: {missing_qrels}")
    if unknown_qrels:
        raise ValueError(f"Qrels reference unknown query_id values: {unknown_qrels}")

    multi_split_qrels = sorted(
        qrels.groupby("query_id")["split"].nunique().loc[lambda values: values.ne(1)].index
    )
    if multi_split_qrels:
        raise ValueError(f"Qrels span multiple splits: {multi_split_qrels}")

    split_check = qrels[["query_id", "split"]].merge(
        queries[["query_id", "split"]],
        on="query_id",
        how="left",
        validate="many_to_one",
        suffixes=("_qrel", "_query"),
    )
    split_mismatch = split_check.loc[
        split_check["split_qrel"].ne(split_check["split_query"])
    ]
    if not split_mismatch.empty:
        raise ValueError(
            "Query/qrel split mismatch: "
            f"{split_mismatch.drop_duplicates().to_dict(orient='records')}"
        )

    expected_pairs = {
        (str(row.query_id), url)
        for row in queries.itertuples(index=False)
        for url in row.expected_urls
    }
    observed_pairs = set(
        qrels[["query_id", "section_url"]]
        .astype(str)
        .itertuples(index=False, name=None)
    )
    missing_pairs = sorted(expected_pairs - observed_pairs)
    unexpected_pairs = sorted(observed_pairs - expected_pairs)
    if missing_pairs or unexpected_pairs:
        affected_query_ids = sorted(
            {query_id for query_id, _ in [*missing_pairs, *unexpected_pairs]}
        )
        raise ValueError(
            "Qrels do not match expected_urls for query_id values "
            f"{affected_query_ids}: missing={missing_pairs}, unexpected={unexpected_pairs}"
        )

    if not qrels["relevance"].eq(1).all():
        affected_query_ids = sorted(
            qrels.loc[qrels["relevance"].ne(1), "query_id"].astype(str).unique()
        )
        raise ValueError(f"Unexpected relevance values for: {affected_query_ids}")

    split_counts = (
        queries["split"]
        .value_counts()
        .reindex(EVALUATION_SPLIT_ORDER, fill_value=0)
    )
    return pd.DataFrame(
        {
            "split": split_counts.index,
            "queries": split_counts.astype(int).values,
            "qrels": [
                int(qrels.loc[qrels["split"].eq(split)].shape[0])
                for split in split_counts.index
            ],
        }
    )


def append_evaluation_cohort(
    queries: pd.DataFrame,
    qrels: pd.DataFrame,
    cohort: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Atomically stage query rows and qrels derived from their canonical splits."""
    validate_evaluation_consistency(queries, qrels)
    combined_queries = pd.concat([queries, cohort], ignore_index=True)
    combined_qrels = build_qrels_from_queries(combined_queries)

    old_judgments = set(
        qrels[["query_id", "section_url", "relevance"]]
        .astype({"query_id": str, "section_url": str})
        .itertuples(index=False, name=None)
    )
    preserved_judgments = set(
        combined_qrels.loc[
            combined_qrels["query_id"].astype(str).isin(queries["query_id"].astype(str)),
            ["query_id", "section_url", "relevance"],
        ]
        .astype({"query_id": str, "section_url": str})
        .itertuples(index=False, name=None)
    )
    if old_judgments != preserved_judgments:
        raise ValueError("Appending a cohort would change existing relevance judgments.")

    validate_evaluation_consistency(combined_queries, combined_qrels)
    return combined_queries, combined_qrels


def synchronize_evaluation_metadata(
    queries: pd.DataFrame,
    qrels: pd.DataFrame,
    **metadata,
) -> pd.DataFrame:
    """Validate state, then refresh manifest and in-notebook audit data from it."""
    summary = validate_evaluation_consistency(queries, qrels)
    RUN_MANIFEST["evaluation"] = {
        "labels": "explicit-agent-curated-anchors-v3",
        "source_of_truth": "all_evaluation_df.split",
        "allowed_splits": list(EVALUATION_SPLIT_ORDER),
        "query_counts": {
            row.split: int(row.queries)
            for row in summary.itertuples(index=False)
        },
        "qrel_counts": {
            row.split: int(row.qrels)
            for row in summary.itertuples(index=False)
        },
        **metadata,
    }
    remember_result("qrels", qrels)
    return summary


qrels_df = build_qrels_from_queries(all_evaluation_df)
evaluation_consistency_summary_df = synchronize_evaluation_metadata(
    all_evaluation_df,
    qrels_df,
    final_holdout_materialized=False,
)
print(f"Explicit relevance judgments: {len(qrels_df)}; queries: {qrels_df.query_id.nunique()}")
display(qrels_df)


Explicit relevance judgments: 50; queries: 45


,query_id,split,section_url,relevance
0,q01,research,https://qdrant.tech/documentation/manage-data/...,1
1,q02,research,https://qdrant.tech/documentation/manage-data/...,1
2,q03,research,https://qdrant.tech/documentation/manage-data/...,1
3,q04,research,https://qdrant.tech/documentation/manage-data/...,1
4,q05,research,https://qdrant.tech/documentation/search/filte...,1
5,q05,research,https://qdrant.tech/documentation/search/filte...,1
6,q05,research,https://qdrant.tech/documentation/search/filte...,1
7,q06,research,https://qdrant.tech/documentation/search/filte...,1
8,q07,research,https://qdrant.tech/documentation/manage-data/...,1
9,q08,research,https://qdrant.tech/documentation/manage-data/...,1


### 9.3 Recall@10

Recall@10 is computed only by `ranx` as `recall@10`: for each query it is the fraction of judged relevant section URLs found among the first ten unique retrieved sections, averaged across queries.

The evaluation run below executes all retrieval variants after a short warm-up. It stores the full deduplicated section ranking needed for `ranx` plus end-to-end latency observations used for P50/P95.


In [78]:
EVAL_TOP_K = 10
EVAL_RAW_RESULT_LIMIT = 50
BENCHMARK_REPEATS = 5
BENCHMARK_WARMUP_ROUNDS = 2
RANX_QUALITY_METRICS = ["recall@10", "mrr"]
assert EVAL_RAW_RESULT_LIMIT == HYBRID_PREFETCH_LIMIT == RERANK_CANDIDATE_LIMIT == 50


def collapse_points_to_sections(points, max_sections: int = EVAL_TOP_K) -> list[dict]:
    records, seen = [], set()
    for point in points:
        url = (point.payload or {}).get("section_url")
        if url and url not in seen:
            seen.add(url)
            records.append(
                {"rank": len(records) + 1, "section_url": url, "score": float(point.score)}
            )
        if len(records) >= max_sections:
            break
    return records


def build_ranx_inputs(quality_runs: pd.DataFrame) -> tuple[Qrels, Run]:
    """Build section-level ranx inputs while preserving the retrieval ranking."""
    if quality_runs.empty or not quality_runs["query_id"].is_unique:
        raise ValueError("ranx evaluation requires exactly one quality row per query.")

    qrels, run = {}, {}
    for row in quality_runs.itertuples(index=False):
        query_id = str(row.query_id)
        expected_urls = list(dict.fromkeys(row.expected_urls))
        ranked_urls = list(dict.fromkeys(row.retrieved_section_urls))
        if not expected_urls:
            raise ValueError(f"Recall@10 requires qrels for {query_id}.")

        qrels[query_id] = {url: 1 for url in expected_urls}
        # ranx orders documents by score. Strictly descending synthetic scores encode
        # the already produced pipeline order without changing or truncating it.
        run[query_id] = {
            url: float(len(ranked_urls) - rank)
            for rank, url in enumerate(ranked_urls)
        }

    return Qrels(qrels=qrels), Run(run=run)


def evaluate_quality_with_ranx(
    quality_runs: pd.DataFrame, *, return_mean: bool = True
):
    qrels, run = build_ranx_inputs(quality_runs)
    scores = evaluate(
        qrels,
        run,
        RANX_QUALITY_METRICS,
        return_mean=return_mean,
    )
    if return_mean:
        return {
            "recall_at_10": float(scores["recall@10"]),
            "mrr": float(scores["mrr"]),
        }
    return pd.DataFrame(
        {
            "query_id": qrels.get_query_ids(),
            "recall_at_10": np.asarray(scores["recall@10"], dtype=float),
            "mrr": np.asarray(scores["mrr"], dtype=float),
        }
    )


def run_benchmark(
    name: str,
    functions: dict,
    queries: pd.DataFrame,
    *,
    repeats: int = BENCHMARK_REPEATS,
) -> pd.DataFrame:
    """One randomized, fully warmed protocol shared by Sections 9–11.

    ranx consumes one full section ranking per query/configuration for quality.
    The harness retains repeated latency observations for P50/P95 separately.
    """
    rng = random.Random(SEED)
    rows = list(queries.itertuples(index=False))
    records, first_pass = [], []
    phases = (
        [("first_pass", -1)]
        + [("warmup", i) for i in range(BENCHMARK_WARMUP_ROUNDS)]
        + [("warm", i) for i in range(repeats)]
    )
    for phase, repeat in phases:
        tasks = [(key, row) for key in functions for row in rows]
        rng.shuffle(tasks)
        for key, row in tasks:
            started = time.perf_counter()
            points = functions[key](row)
            sections = collapse_points_to_sections(
                points, max_sections=EVAL_RAW_RESULT_LIMIT
            )
            elapsed = (time.perf_counter() - started) * 1000.0
            if phase == "warmup":
                continue
            record = {
                "setting": key,
                "query_id": row.query_id,
                "split": row.split,
                "query": row.query,
                "query_type": row.query_type,
                "repeat": repeat,
                "phase": phase,
                "quality_run": repeat == 0,
                "latency_ms": elapsed,
                "expected_urls": list(row.expected_urls),
                "retrieved_section_urls": [
                    section["section_url"] for section in sections
                ],
                "retrieved_point_ids": [str(point.id) for point in points],
            }
            (first_pass if phase == "first_pass" else records).append(record)
    remember_result(name + "_first_pass", pd.DataFrame(first_pass))
    return remember_result(name, pd.DataFrame(records))


def summarize_benchmark(runs: pd.DataFrame, group: str = "setting") -> pd.DataFrame:
    quality_rows = []
    quality_runs = runs[runs["quality_run"]]
    for group_value, group_runs in quality_runs.groupby(group, sort=False):
        quality_rows.append(
            {group: group_value, **evaluate_quality_with_ranx(group_runs)}
        )
    quality = pd.DataFrame(quality_rows).set_index(group)
    latency = runs.groupby(group)["latency_ms"].agg(
        p50_ms=lambda values: float(np.percentile(values, 50)),
        p95_ms=lambda values: float(np.percentile(values, 95)),
    )
    return quality.join(latency).reset_index()


def rank_configurations(summary: pd.DataFrame) -> pd.DataFrame:
    return summary.sort_values(
        ["recall_at_10", "mrr", "p95_ms", "p50_ms"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)


@lru_cache(maxsize=256)
def research_dense_query(query: str) -> tuple:
    # Search-only experiments use this cache; end-to-end Section 9 does not.
    return tuple(embed_dense_query(query))


def prepare_dense_queries(frame: pd.DataFrame) -> dict:
    return {
        row.query_id: list(research_dense_query(row.query)) for row in frame.itertuples(index=False)
    }


def search_dense_collection(
    collection_name,
    query_vector,
    *,
    params=None,
    limit=EVAL_RAW_RESULT_LIMIT,
    query_filter=None,
    payload=("section_url",)
):
    return qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        using=DENSE_VECTOR_NAME,
        search_params=params,
        query_filter=query_filter,
        limit=limit,
        with_payload=list(payload) if payload is not False else False,
        with_vectors=False,
    ).points


PIPELINE_FUNCTIONS = {
    "dense": search_dense,
    "sparse": search_sparse,
    "hybrid_rrf": search_hybrid_rrf,
    "hybrid_dbsf": search_hybrid_dbsf,
    "rrf_colbert": partial(search_hybrid_colbert, fusion="rrf"),
    "dbsf_colbert": partial(search_hybrid_colbert, fusion="dbsf"),
}
evaluation_runs_df = run_benchmark(
    "retrieval_research",
    {
        key: (
            lambda row, fn=fn: fn(row.query, limit=EVAL_RAW_RESULT_LIMIT, payload=["section_url"])
        )
        for key, fn in PIPELINE_FUNCTIONS.items()
    },
    evaluation_df,
).rename(columns={"setting": "pipeline"})
retrieval_summary_df = summarize_benchmark(evaluation_runs_df, "pipeline")
recall_summary_df = retrieval_summary_df[["pipeline", "recall_at_10"]]
display(recall_summary_df)


,pipeline,recall_at_10
0,sparse,0.740741
1,hybrid_rrf,0.837037
2,dbsf_colbert,0.837037
3,dense,0.814815
4,hybrid_dbsf,0.829630
5,rrf_colbert,0.837037


### 9.4 MRR

Mean Reciprocal Rank measures how early the first relevant section appears. It is computed only by `ranx` as `mrr` over the complete returned section ranking, without truncating the run to ten results. A correct rank-1 result contributes `1.0`, rank 2 contributes `0.5`, and a query with no retrieved relevant section contributes `0`.


In [79]:
mrr_summary_df = retrieval_summary_df[["pipeline", "mrr"]]
display(mrr_summary_df)


,pipeline,mrr
0,sparse,0.506599
1,hybrid_rrf,0.607669
2,dbsf_colbert,0.661683
3,dense,0.599128
4,hybrid_dbsf,0.600105
5,rrf_colbert,0.661683


### 9.5 P50 / P95 Latency

End-to-end measurements include uncached query embedding, Qdrant retrieval of 50 points with section URLs, and deduplication to up to 50 unique sections. Every query/configuration is warmed, then measured repeatedly in randomized order. The benchmark harness reports only P50 and P95 latency; first-pass observations remain available separately and are not called cold-cache results. These local percentiles are not a production SLA.


In [80]:
latency_summary_df = retrieval_summary_df[
    ["pipeline", "p50_ms", "p95_ms"]
].sort_values("p95_ms")
display(latency_summary_df)


,pipeline,p50_ms,p95_ms
3,dense,20.535886,87.190650
0,sparse,106.611390,195.619895
4,hybrid_dbsf,123.592247,217.837308
1,hybrid_rrf,160.463609,274.675649
2,dbsf_colbert,285.749293,394.762451
5,rrf_colbert,284.575133,397.313477


### 9.6 Fusion Strategy Benchmark

RRF and DBSF are compared on the research queries, before and after ColBERT. All pipelines already use the same fixed candidate pool and enter the joint selection in Section 9.8. The selected retrieval pipeline may also be dense-only or sparse-only.

In [81]:
fusion_benchmark_df = retrieval_summary_df[
    retrieval_summary_df["pipeline"].isin(
        ["hybrid_rrf", "hybrid_dbsf", "rrf_colbert", "dbsf_colbert"]
    )
].copy()
fusion_benchmark_df["stage"] = np.where(
    fusion_benchmark_df["pipeline"].str.endswith("colbert"), "colbert", "hybrid"
)
fusion_benchmark_df["fusion"] = np.where(
    fusion_benchmark_df["pipeline"].str.contains("rrf"), "rrf", "dbsf"
)
display(rank_configurations(fusion_benchmark_df))
print("Fusion is not frozen yet: all pipelines enter one fixed-pool joint selection.")


,pipeline,recall_at_10,mrr,p50_ms,p95_ms,stage,fusion
0,dbsf_colbert,0.837037,0.661683,285.749293,394.762451,colbert,dbsf
1,rrf_colbert,0.837037,0.661683,284.575133,397.313477,colbert,rrf
2,hybrid_rrf,0.837037,0.607669,160.463609,274.675649,hybrid,rrf
3,hybrid_dbsf,0.829630,0.600105,123.592247,217.837308,hybrid,dbsf


Fusion is not frozen yet: all pipelines enter one fixed-pool joint selection.


### 9.7 Retrieval Ablation Study

The retrieval ablation compares single-signal retrieval, two server-side fusion strategies, and both fusion strategies with ColBERT reranking. This makes the contribution of dense semantics, sparse lexical matching, fusion, and late interaction measurable on the same queries.

The primary course success criterion is Recall@10 ≥ 0.8. Configurations are ordered by Recall@10, MRR, P95 latency, and P50 latency; all quality values come from the same `ranx` evaluation.


In [82]:
retrieval_ablation_df = rank_configurations(retrieval_summary_df)
retrieval_ablation_df["meets_recall_target"] = retrieval_ablation_df["recall_at_10"] >= 0.80
remember_result("retrieval_ablation", retrieval_ablation_df)
display(retrieval_ablation_df)
px.scatter(
    retrieval_ablation_df,
    x="p95_ms",
    y="recall_at_10",
    text="pipeline",
    hover_data=["mrr", "p50_ms"],
    title="Research retrieval ablation",
).show()


,pipeline,recall_at_10,mrr,p50_ms,p95_ms,meets_recall_target
0,dbsf_colbert,0.837037,0.661683,285.749293,394.762451,True
1,rrf_colbert,0.837037,0.661683,284.575133,397.313477,True
2,hybrid_rrf,0.837037,0.607669,160.463609,274.675649,True
3,hybrid_dbsf,0.829630,0.600105,123.592247,217.837308,True
4,dense,0.814815,0.599128,20.535886,87.190650,True
5,sparse,0.740741,0.506599,106.611390,195.619895,False


### 9.8 Joint Fusion Selection with a Fixed Candidate Pool

The existing dense, sparse, hybrid and ColBERT pipelines share one fixed 50-candidate pool. The earlier 100/200-candidate sweep was removed because it did not improve Recall@10 and only increased latency. Selection reuses the joint randomized benchmark from Section 9.3 instead of executing the same pipelines again. Recall@10 and MRR are computed by `ranx`; P50/P95 come from the benchmark harness. The common decision order is Recall@10, MRR, P95, then P50.


In [83]:
candidate_configs = {
    "dense": {"pipeline": "dense", "fusion": None, "candidate_limit": None},
    "sparse": {"pipeline": "sparse", "fusion": None, "candidate_limit": None},
    "hybrid_rrf": {"pipeline": "hybrid_rrf", "fusion": "rrf", "candidate_limit": None},
    "hybrid_dbsf": {"pipeline": "hybrid_dbsf", "fusion": "dbsf", "candidate_limit": None},
    "rrf_colbert": {"pipeline": "rrf_colbert", "fusion": "rrf", "candidate_limit": RERANK_CANDIDATE_LIMIT},
    "dbsf_colbert": {"pipeline": "dbsf_colbert", "fusion": "dbsf", "candidate_limit": RERANK_CANDIDATE_LIMIT},
}
finalist_functions = {
    key: (
        lambda row, fn=fn: fn(
            row.query, limit=EVAL_RAW_RESULT_LIMIT, payload=["section_url"]
        )
    )
    for key, fn in PIPELINE_FUNCTIONS.items()
}
finalist_summary_df = retrieval_ablation_df.rename(columns={"pipeline": "setting"}).copy()
selected_key = str(finalist_summary_df.iloc[0]["setting"])
FROZEN_HELDOUT_PIPELINES = tuple(finalist_summary_df["setting"])
assert set(FROZEN_HELDOUT_PIPELINES) == set(finalist_functions)
FINAL_RETRIEVAL_CONFIG = candidate_configs[selected_key].copy()
FINAL_RETRIEVAL_PIPELINE = FINAL_RETRIEVAL_CONFIG["pipeline"]
FINAL_FUSION = FINAL_RETRIEVAL_CONFIG["fusion"]


def search_selected_points(query: str, *, payload=SEARCH_PAYLOAD):
    """Run the frozen pipeline with the same 50-candidate contract as evaluation."""
    if FINAL_RETRIEVAL_CONFIG["candidate_limit"] is not None:
        return search_hybrid_colbert(
            query,
            fusion=FINAL_FUSION,
            candidate_limit=FINAL_RETRIEVAL_CONFIG["candidate_limit"],
            prefetch_limit=FINAL_RETRIEVAL_CONFIG["candidate_limit"],
            limit=EVAL_RAW_RESULT_LIMIT,
            payload=payload,
        )
    return PIPELINE_FUNCTIONS[FINAL_RETRIEVAL_PIPELINE](
        query, limit=EVAL_RAW_RESULT_LIMIT, payload=payload
    )


def search_final_sections(query: str, top_k: int = EVAL_TOP_K):
    """Serve the same overfetch/dedup contract that the evaluation measures."""
    if not 1 <= top_k <= EVAL_TOP_K:
        raise ValueError(f"The validated endpoint supports 1..{EVAL_TOP_K} sections.")
    points = search_selected_points(query, payload=["section_url"])
    return collapse_points_to_sections(points, max_sections=top_k)


RUN_MANIFEST["selected_retrieval"] = FINAL_RETRIEVAL_CONFIG
RUN_MANIFEST["heldout_comparison_pipelines"] = list(FROZEN_HELDOUT_PIPELINES)
display(finalist_summary_df)
print("Frozen retrieval configuration:", FINAL_RETRIEVAL_CONFIG)
# Held-out retrieval and optimization are evaluated only at the end of Section 11.


,setting,recall_at_10,mrr,p50_ms,p95_ms,meets_recall_target
0,dbsf_colbert,0.837037,0.661683,285.749293,394.762451,True
1,rrf_colbert,0.837037,0.661683,284.575133,397.313477,True
2,hybrid_rrf,0.837037,0.607669,160.463609,274.675649,True
3,hybrid_dbsf,0.829630,0.600105,123.592247,217.837308,True
4,dense,0.814815,0.599128,20.535886,87.190650,True
5,sparse,0.740741,0.506599,106.611390,195.619895,False


Frozen retrieval configuration: {'pipeline': 'dbsf_colbert', 'fusion': 'dbsf', 'candidate_limit': 50}


The retrieval configuration is selected using the 45-query research set. Hybrid and ColBERT remain implemented regardless of the winner: ColBERT is excluded from serving only when the measured quality/latency ranking does not select it. No improvement, target attainment or held-out quality is claimed until the revised notebook is run. The selected endpoint uses the same 50-point overfetch and section-level deduplication contract as evaluation.

## 10. Retrieval System Tuning

This section applies the remaining Qdrant Essentials experiments that are directly relevant to documentation retrieval. The goal is not to choose the final architecture yet, but to measure three independent design dimensions before the final collection is assembled:

1. chunking strategy;
2. HNSW index/search parameters;
3. payload indexing for filtered retrieval.

Each experiment isolates one variable as much as possible and persists its results for the final architecture decision. Dynamic oversampling is intentionally deferred until these experiments and the quantization study are complete.

### 10.1 Chunking Strategy Ablation

Day 1 emphasizes that chunking should be chosen empirically because structure preservation, overlap, storage cost, and retrieval quality trade off against one another.

The production chunker is compared against two controlled section-bounded baselines using the same BGE model and the same qrels:

- `structure_aware`: the current Markdown-aware chunker;
- `fixed_no_overlap`: fixed token windows within each source section;
- `fixed_overlap_15pct`: the same fixed windows with 15% overlap.

Exact dense search is used here so HNSW approximation does not confound the chunking comparison. The experiment measures Recall@10, MRR, P50/P95, number of chunks, token volume, and preparation time. The structure-aware baseline reuses its already stored dense vectors, so preparation times are not cold embedding-speed comparisons. No winner is selected in this section.

The following cell defines shared dense snapshot, collection-building and cleanup helpers used throughout Sections 10–11; all implementations remain here in the notebook.

In [84]:
CHUNKING_ABLATION_COLLECTION_PREFIX = f"{COLLECTION_NAME}_chunking"
CHUNKING_ABLATION_TOKEN_BUDGET = CHUNK_TOKEN_BUDGET
CHUNKING_ABLATION_OVERLAP_RATIO = 0.15
KEEP_RESEARCH_COLLECTIONS = False
RESEARCH_UPLOAD_BATCH = 256
RESEARCH_UPLOAD_PARALLEL = INGEST.upload_parallel
RESEARCH_INDEXING_THRESHOLD_KB = 10
RESEARCH_FULL_SCAN_THRESHOLD_KB = 10  # Server minimum; forces ANN for larger benchmark segments.
CHUNKING_ABLATION_COLLECTIONS = {
    "structure_aware": f"{CHUNKING_ABLATION_COLLECTION_PREFIX}_structure",
    "fixed_no_overlap": f"{CHUNKING_ABLATION_COLLECTION_PREFIX}_fixed",
    "fixed_overlap_15pct": f"{CHUNKING_ABLATION_COLLECTION_PREFIX}_overlap15",
}


def snapshot_dense_points(collection_name: str) -> list[dict]:
    snapshot, offset = [], None
    while True:
        records, offset = qdrant_client.scroll(
            collection_name=collection_name,
            limit=256,
            offset=offset,
            with_payload=["section_url", "tags", "path"],
            with_vectors=[DENSE_VECTOR_NAME],
        )
        for record in records:
            vector = np.asarray(record.vector[DENSE_VECTOR_NAME], dtype=np.float32)
            if vector.shape != (DENSE_EMBEDDING_DIM,):
                raise ValueError(f"Invalid dense vector: {record.id}: {vector.shape}")
            snapshot.append(
                {"id": record.id, "vector": vector.tolist(), "payload": record.payload or {}}
            )
        if offset is None:
            break
    return snapshot


def snapshot_points(snapshot):
    return (
        models.PointStruct(
            id=item["id"], vector={DENSE_VECTOR_NAME: item["vector"]}, payload=item["payload"]
        )
        for item in snapshot
    )


def rebuild_dense_collection(
    collection_name: str,
    points,
    expected_points: int,
    *,
    m: int = 16,
    ef_construct: int = 100,
    quantization=None,
    memory=models.Memory.CACHED,
    indexed_payload_fields=(),
) -> dict:
    """Shared deferred-ingestion path for all dense-only experiments."""
    if collection_name == COLLECTION_NAME:
        raise ValueError("Research helpers must not replace the primary collection.")
    if qdrant_client.collection_exists(collection_name):
        qdrant_client.delete_collection(collection_name)
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config={
            DENSE_VECTOR_NAME: models.VectorParams(
                size=DENSE_EMBEDDING_DIM, distance=models.Distance.COSINE, memory=memory
            )
        },
        hnsw_config=models.HnswConfigDiff(
            m=m, ef_construct=ef_construct, full_scan_threshold=RESEARCH_FULL_SCAN_THRESHOLD_KB
        ),
        quantization_config=quantization,
        optimizers_config=models.OptimizersConfigDiff(indexing_threshold=0),
    )
    for field in indexed_payload_fields:
        qdrant_client.create_payload_index(
            collection_name=collection_name,
            field_name=field,
            field_schema=models.PayloadSchemaType.KEYWORD,
            wait=True,
        )
    started = time.perf_counter()
    qdrant_client.upload_points(
        collection_name=collection_name,
        points=points,
        batch_size=RESEARCH_UPLOAD_BATCH,
        parallel=RESEARCH_UPLOAD_PARALLEL,
        max_retries=3,
        wait=True,
    )
    upload_seconds = time.perf_counter() - started
    stored = qdrant_client.count(collection_name, exact=True).count
    if stored != expected_points:
        raise ValueError(f"{collection_name}: {stored} != {expected_points}")
    started = time.perf_counter()
    qdrant_client.update_collection(
        collection_name=collection_name,
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=RESEARCH_INDEXING_THRESHOLD_KB
        ),
    )
    info = wait_for_collection(collection_name, require_hnsw=m > 0)
    return {
        "collection": collection_name,
        "points": stored,
        "upload_seconds": upload_seconds,
        "readiness_seconds": time.perf_counter() - started,
        "indexed_vectors_count": int(info.indexed_vectors_count or 0),
    }


def cleanup_research_collections(collections, keep: bool):
    for name in collections:
        if name == COLLECTION_NAME:
            raise ValueError("Never clean up the primary collection.")
        if not keep and qdrant_client.collection_exists(name):
            qdrant_client.delete_collection(name)
    print("Research collections retained." if keep else "Temporary research collections removed.")


dense_snapshot = snapshot_dense_points(COLLECTION_NAME)
assert len(dense_snapshot) == len(retrieval_chunks_df)
dense_vectors_by_id = {str(item["id"]): item["vector"] for item in dense_snapshot}
research_query_vectors = prepare_dense_queries(evaluation_df)


In [85]:
def split_token_windows(text: str, *, token_budget: int, overlap_ratio: float) -> list[dict]:
    return source_windows(text, token_budget, overlap_ratio)


def build_fixed_chunking_variant(overlap_ratio: float) -> pd.DataFrame:
    records = []

    source_df = (
        clean_sections_df[clean_sections_df["section_text"].fillna("").str.strip().ne("")]
        .sort_values(["path", "section_index"])
        .reset_index(drop=True)
    )

    for row in tqdm(
        list(source_df.itertuples(index=False)),
        desc=f"Fixed chunking overlap={overlap_ratio:.0%}",
        unit="section",
    ):
        row_dict = row._asdict()

        windows = split_token_windows(
            row.section_text,
            token_budget=CHUNKING_ABLATION_TOKEN_BUDGET,
            overlap_ratio=overlap_ratio,
        )

        for window in windows:
            chunk_key = (
                f"chunking-ablation::{overlap_ratio:.4f}::"
                f"{row.path}::section={int(row.section_index)}::"
                f"chunk={int(window['chunk_index'])}"
            )

            prefix = build_embedding_prefix(row_dict)

            records.append(
                {
                    "point_id": str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_key)),
                    "path": row.path,
                    "section_index": int(row.section_index),
                    "chunk_index": int(window["chunk_index"]),
                    "section_url": row.section_url,
                    "chunk_text": window["chunk_text"],
                    "tokens": int(window["tokens"]),
                    "embedding_text": prefix + window["chunk_text"],
                }
            )

    return pd.DataFrame(records)


structure_aware_ablation_df = retrieval_chunks_df[
    [
        "point_id",
        "path",
        "section_index",
        "chunk_index",
        "section_url",
        "chunk_text",
        "tokens",
        "embedding_text",
    ]
].copy()

fixed_no_overlap_df = build_fixed_chunking_variant(0.0)
fixed_overlap_15_df = build_fixed_chunking_variant(CHUNKING_ABLATION_OVERLAP_RATIO)

chunking_ablation_datasets = {
    "structure_aware": structure_aware_ablation_df,
    "fixed_no_overlap": fixed_no_overlap_df,
    "fixed_overlap_15pct": fixed_overlap_15_df,
}

chunking_corpus_summary_df = pd.DataFrame(
    [
        {
            "strategy": strategy,
            "chunks": len(dataset_df),
            "total_body_tokens": int(dataset_df["tokens"].sum()),
            "median_chunk_tokens": float(dataset_df["tokens"].median()),
            "p95_chunk_tokens": float(dataset_df["tokens"].quantile(0.95)),
            "max_chunk_tokens": int(dataset_df["tokens"].max()),
        }
        for strategy, dataset_df in chunking_ablation_datasets.items()
    ]
)

display(chunking_corpus_summary_df)

assert all(
    dataset_df["tokens"].max() <= CHUNKING_ABLATION_TOKEN_BUDGET
    for dataset_df in chunking_ablation_datasets.values()
)


Fixed chunking overlap=15%: 100%|██████████| 2626/2626 [00:00<00:00, 5185.00section/s] 


,strategy,chunks,total_body_tokens,median_chunk_tokens,p95_chunk_tokens,max_chunk_tokens
0,structure_aware,4134,1004866,242.0,439.0,448
1,fixed_no_overlap,3855,997620,236.0,448.0,448
2,fixed_overlap_15pct,4004,1089958,259.0,448.0,448


In [86]:
def ingest_chunking_variant(strategy: str, dataset_df: pd.DataFrame) -> dict:
    timings = {}

    def points():
        for start in range(0, len(dataset_df), INGEST.pipeline_batch):
            batch = dataset_df.iloc[start : start + INGEST.pipeline_batch]
            with record_seconds(timings, "dense"):
                vectors = (
                    [
                        dense_vectors_by_id[str(row.point_id)]
                        for row in batch.itertuples(index=False)
                    ]
                    if strategy == "structure_aware"
                    else embed_documents("dense", batch["embedding_text"].tolist())
                )
            for row, vector in zip(batch.itertuples(index=False), vectors):
                yield models.PointStruct(
                    id=row.point_id,
                    vector={DENSE_VECTOR_NAME: np.asarray(vector, dtype=np.float32).tolist()},
                    payload={"section_url": row.section_url, "path": row.path},
                )

    started = time.perf_counter()
    result = rebuild_dense_collection(
        CHUNKING_ABLATION_COLLECTIONS[strategy], points(), len(dataset_df), m=0
    )
    return {
        "strategy": strategy,
        **result,
        "embedding_and_ingest_seconds": time.perf_counter() - started,
        "dense_stage_seconds": timings.get("dense", 0.0),
        "dense_vectors_reused": strategy == "structure_aware",
    }


# Check full inputs, not only the fixed-window body budget.
for strategy, frame in chunking_ablation_datasets.items():
    lengths = [
        count_tokens(text) + dense_tokenizer.num_special_tokens_to_add(pair=False)
        for text in frame["embedding_text"]
    ]
    assert max(lengths) <= dense_tokenizer.model_max_length, strategy

chunking_ingestion_df = pd.DataFrame(
    [
        ingest_chunking_variant(strategy, frame)
        for strategy, frame in chunking_ablation_datasets.items()
    ]
)
display(chunking_ingestion_df)
print("Baseline vectors are reused; ingestion timings are NOT comparable cold embedding timings.")


,strategy,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count,embedding_and_ingest_seconds,dense_stage_seconds,dense_vectors_reused
0,structure_aware,docs_search_chunking_structure,4134,0.625831,1.219897,4134,2.017776,0.015281,True
1,fixed_no_overlap,docs_search_chunking_fixed,3855,214.329018,1.087727,3855,215.595926,212.787746,False
2,fixed_overlap_15pct,docs_search_chunking_overlap15,4004,172.812984,1.231481,4004,174.221952,170.754704,False


Baseline vectors are reused; ingestion timings are NOT comparable cold embedding timings.


In [87]:
chunking_eval_runs_df = run_benchmark(
    "chunking_research",
    {
        strategy: (
            lambda row, name=name: search_dense_collection(
                name, research_query_vectors[row.query_id], params=models.SearchParams(exact=True)
            )
        )
        for strategy, name in CHUNKING_ABLATION_COLLECTIONS.items()
    },
    evaluation_df,
)
chunking_ablation_summary_df = (
    summarize_benchmark(chunking_eval_runs_df)
    .rename(columns={"setting": "strategy"})
    .merge(chunking_corpus_summary_df, on="strategy", validate="one_to_one")
    .merge(chunking_ingestion_df, on="strategy", validate="one_to_one")
)
remember_result("chunking_summary", chunking_ablation_summary_df)
display(rank_configurations(chunking_ablation_summary_df))
px.scatter(
    chunking_ablation_summary_df,
    x="p95_ms",
    y="recall_at_10",
    size="chunks",
    hover_name="strategy",
    hover_data=["mrr", "total_body_tokens"],
    title="Research chunking ablation: exact dense search",
).show()
print("Exploratory ablation; the primary corpus is not silently replaced.")


,strategy,recall_at_10,mrr,p50_ms,p95_ms,chunks,total_body_tokens,median_chunk_tokens,p95_chunk_tokens,max_chunk_tokens,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count,embedding_and_ingest_seconds,dense_stage_seconds,dense_vectors_reused
0,structure_aware,0.814815,0.599128,1.500229,1.840684,4134,1004866,242.0,439.0,448,docs_search_chunking_structure,4134,0.625831,1.219897,4134,2.017776,0.015281,True
1,fixed_overlap_15pct,0.814815,0.591726,1.540651,1.810302,4004,1089958,259.0,448.0,448,docs_search_chunking_overlap15,4004,172.812984,1.231481,4004,174.221952,170.754704,False
2,fixed_no_overlap,0.792593,0.578659,1.474711,1.783363,3855,997620,236.0,448.0,448,docs_search_chunking_fixed,3855,214.329018,1.087727,3855,215.595926,212.787746,False


Exploratory ablation; the primary corpus is not silently replaced.


In [88]:
cleanup_research_collections(CHUNKING_ABLATION_COLLECTIONS.values(), KEEP_RESEARCH_COLLECTIONS)

Temporary research collections removed.


### 10.2 HNSW Index and Search Tuning

The full-precision dense grid tests `m=8/16/32`, `ef_construct=100/200/400` and `hnsw_ef=64/128/256`. One `m=0` exact-scan baseline replaces the redundant no-graph combinations.

On this small corpus, `indexing_threshold=10` KB enables graph construction and `full_scan_threshold=10` KB—the Qdrant 1.19 minimum—forces ANN for benchmark segments larger than that threshold. Every upload uses `wait=True`. Dense-only graph readiness requires green status, a healthy optimizer and stable counts with at least 95% indexed; counters are approximate. Readiness includes scheduling and polling, not just internal build time.

All settings use shared query vectors and the randomized, fully warmed benchmark helper. They are compared only by `ranx` Recall@10/MRR and harness P50/P95, alongside technical build diagnostics. This is a research ablation, not a production-scale capacity claim.


In [89]:
HNSW_M_VALUES = [0, 8, 16, 32]
HNSW_EF_CONSTRUCT_VALUES = [100, 200, 400]
HNSW_SEARCH_EF_VALUES = [64, 128, 256]
HNSW_RESEARCH_COLLECTIONS = {
    (m, efc): f"{COLLECTION_NAME}_hnsw_m{m}_efc{efc}"
    for m in HNSW_M_VALUES
    for efc in ([100] if m == 0 else HNSW_EF_CONSTRUCT_VALUES)
}
print(
    f"HNSW collections: {len(HNSW_RESEARCH_COLLECTIONS)}; one m=0 baseline.\n"
    f"Full-scan threshold: {RESEARCH_FULL_SCAN_THRESHOLD_KB} KB; "
    f"indexing threshold: {RESEARCH_INDEXING_THRESHOLD_KB} KB."
)


HNSW collections: 10; one m=0 baseline.
Full-scan threshold: 10 KB; indexing threshold: 10 KB.


In [90]:
research_dense_snapshot = dense_snapshot  # Reuse the one validated snapshot from Section 10.1.
print(f"Shared dense snapshot: {len(research_dense_snapshot):,} points.")


Shared dense snapshot: 4,134 points.


In [91]:
hnsw_build_df = pd.DataFrame(
    [
        {
            "m": m,
            "ef_construct": efc,
            **rebuild_dense_collection(
                name,
                snapshot_points(research_dense_snapshot),
                len(research_dense_snapshot),
                m=m,
                ef_construct=efc,
            ),
        }
        for (m, efc), name in HNSW_RESEARCH_COLLECTIONS.items()
    ]
)
display(hnsw_build_df)
print("readiness_seconds includes optimizer scheduling and polling, not just index construction.")


,m,ef_construct,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count
0,0,100,docs_search_hnsw_m0_efc100,4134,0.669246,1.219466,4134
1,8,100,docs_search_hnsw_m8_efc100,4134,0.618625,1.424296,4134
2,8,200,docs_search_hnsw_m8_efc200,4134,0.627929,1.421649,4134
3,8,400,docs_search_hnsw_m8_efc400,4134,0.626364,1.420484,4134
4,16,100,docs_search_hnsw_m16_efc100,4134,0.630192,1.420705,4134
5,16,200,docs_search_hnsw_m16_efc200,4134,0.647646,1.421171,4134
6,16,400,docs_search_hnsw_m16_efc400,4134,0.637890,1.420457,4134
7,32,100,docs_search_hnsw_m32_efc100,4134,0.613915,1.219089,4134
8,32,200,docs_search_hnsw_m32_efc200,4134,0.626027,1.421499,4134
9,32,400,docs_search_hnsw_m32_efc400,4134,0.604122,1.622717,4134


readiness_seconds includes optimizer scheduling and polling, not just index construction.


In [92]:
hnsw_settings, hnsw_functions = {}, {}
for (m, efc), name in HNSW_RESEARCH_COLLECTIONS.items():
    for ef in ([0] if m == 0 else HNSW_SEARCH_EF_VALUES):
        label = f"m{m}_efc{efc}_ef{ef}"
        params = models.SearchParams(exact=True) if m == 0 else models.SearchParams(hnsw_ef=ef)
        hnsw_settings[label] = {"m": m, "ef_construct": efc, "hnsw_ef": ef}
        hnsw_functions[label] = lambda row, name=name, params=params: search_dense_collection(
            name, research_query_vectors[row.query_id], params=params
        )
hnsw_eval_runs_df = run_benchmark(
    "hnsw_research", hnsw_functions, evaluation_df
)


In [93]:
hnsw_benchmark_df = (
    summarize_benchmark(hnsw_eval_runs_df)
    .merge(
        pd.DataFrame.from_dict(hnsw_settings, orient="index").rename_axis("setting").reset_index(),
        on="setting",
        validate="one_to_one",
    )
    .merge(hnsw_build_df, on=["m", "ef_construct"], validate="many_to_one")
)
remember_result("hnsw_summary", hnsw_benchmark_df)
display(rank_configurations(hnsw_benchmark_df))
px.scatter(
    hnsw_benchmark_df,
    x="p95_ms",
    y="recall_at_10",
    symbol="m",
    hover_data=["hnsw_ef", "ef_construct", "mrr", "p50_ms", "readiness_seconds"],
    title="Research HNSW benchmark",
).show()


,setting,recall_at_10,mrr,p50_ms,p95_ms,m,ef_construct,hnsw_ef,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count
0,m8_efc400_ef64,0.814815,0.599128,1.363150,1.548426,8,400,64,docs_search_hnsw_m8_efc400,4134,0.626364,1.420484,4134
1,m8_efc100_ef64,0.814815,0.599128,1.359440,1.556215,8,100,64,docs_search_hnsw_m8_efc100,4134,0.618625,1.424296,4134
2,m8_efc100_ef128,0.814815,0.599128,1.406927,1.592843,8,100,128,docs_search_hnsw_m8_efc100,4134,0.618625,1.424296,4134
3,m32_efc100_ef64,0.814815,0.599128,1.369596,1.601632,32,100,64,docs_search_hnsw_m32_efc100,4134,0.613915,1.219089,4134
4,m8_efc200_ef64,0.814815,0.599128,1.365972,1.605104,8,200,64,docs_search_hnsw_m8_efc200,4134,0.627929,1.421649,4134
5,m16_efc200_ef64,0.814815,0.599128,1.376877,1.617984,16,200,64,docs_search_hnsw_m16_efc200,4134,0.647646,1.421171,4134
6,m8_efc400_ef128,0.814815,0.599128,1.427753,1.642531,8,400,128,docs_search_hnsw_m8_efc400,4134,0.626364,1.420484,4134
7,m8_efc200_ef128,0.814815,0.599128,1.447554,1.644550,8,200,128,docs_search_hnsw_m8_efc200,4134,0.627929,1.421649,4134
8,m32_efc400_ef64,0.814815,0.599128,1.398985,1.656950,32,400,64,docs_search_hnsw_m32_efc400,4134,0.604122,1.622717,4134
9,m16_efc100_ef128,0.814815,0.599128,1.452996,1.662703,16,100,128,docs_search_hnsw_m16_efc100,4134,0.630192,1.420705,4134


In [94]:
cleanup_research_collections(HNSW_RESEARCH_COLLECTIONS.values(), KEEP_RESEARCH_COLLECTIONS)

Temporary research collections removed.


### 10.3 Payload Index and Filter Benchmark

Day 2 explains that payload fields are not indexed automatically and recommends creating indexes before HNSW construction for fields used in filtering.

To measure the effect on this corpus, two otherwise identical dense collections are built:

- one without a payload index;
- one with a keyword index on `tags` created before HNSW is enabled.

Representative tags are selected from judged relevant sections and spread across observed corpus selectivities. Each filtered query keeps only qrels whose sections actually contain its tag, so both variants can be compared with valid `ranx` Recall@10/MRR and harness P50/P95 latency. Derived `speedup` summarizes the P95 efficiency change. No final storage configuration is selected here.

With a 10 KB full-scan threshold, filtering can legitimately select an index-driven enumeration plan; this tests the complete filtered-search planner, not HNSW traversal in isolation.


In [95]:
FILTER_BENCHMARK_COLLECTIONS = {
    "unindexed": f"{COLLECTION_NAME}_filter_unindexed",
    "indexed": f"{COLLECTION_NAME}_filter_indexed",
}
FILTER_BENCHMARK_HNSW_EF = 128
FILTER_BENCHMARK_TAG_COUNT = 5

tag_counts, section_tags_by_url = {}, {}
for item in research_dense_snapshot:
    payload = item["payload"]
    section_url = payload.get("section_url")
    tags = set(payload.get("tags", []))
    section_tags_by_url.setdefault(section_url, set()).update(tags)
    for tag in tags:
        tag_counts[tag] = tag_counts.get(tag, 0) + 1

filter_case_records = []
for row in evaluation_df.itertuples(index=False):
    relevant_tags = set().union(
        *(section_tags_by_url.get(url, set()) for url in row.expected_urls)
    )
    for tag in sorted(relevant_tags):
        filtered_expected = [
            url for url in row.expected_urls if tag in section_tags_by_url.get(url, set())
        ]
        filter_case_records.append(
            {
                **row._asdict(),
                "base_query_id": row.query_id,
                "query_id": f"{row.query_id}::{tag}",
                "tag": tag,
                "expected_urls": filtered_expected,
                "expected_section_count": len(filtered_expected),
            }
        )
all_filter_cases_df = pd.DataFrame(filter_case_records)
filter_tag_candidates_df = (
    all_filter_cases_df.groupby("tag")
    .agg(
        query_cases=("query_id", "nunique"),
        judged_relevant_sections=("expected_section_count", "sum"),
    )
    .reset_index()
    .assign(points=lambda frame: frame["tag"].map(tag_counts))
    .query("points >= 20 and query_cases >= 2")
    .sort_values(["points", "tag"])
    .reset_index(drop=True)
)
if len(filter_tag_candidates_df) < 2:
    raise ValueError("Need at least two tags with valid filtered qrels.")
positions = np.unique(
    np.linspace(0, len(filter_tag_candidates_df) - 1, FILTER_BENCHMARK_TAG_COUNT, dtype=int)
)
filter_tag_candidates_df = filter_tag_candidates_df.iloc[positions].copy()
filter_tag_candidates_df["matched_fraction"] = filter_tag_candidates_df["points"] / len(
    research_dense_snapshot
)
FILTER_BENCHMARK_TAGS = filter_tag_candidates_df["tag"].tolist()
filter_cases_df = all_filter_cases_df[
    all_filter_cases_df["tag"].isin(FILTER_BENCHMARK_TAGS)
].reset_index(drop=True)
assert len(filter_cases_df) >= 10
display(filter_tag_candidates_df)
display(filter_cases_df[["query_id", "tag", "expected_urls"]])


,tag,query_cases,judged_relevant_sections,points,matched_fraction
0,monitoring-telemetry,2,3,24,0.005806
3,snapshots,2,2,43,0.010402
7,quantization,6,6,71,0.017175
10,ops-optimization,2,2,79,0.019110
14,manage-data,22,22,464,0.112240


,query_id,tag,expected_urls
0,q01::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
1,q02::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
2,q02::quantization,quantization,[https://qdrant.tech/documentation/manage-data...
3,q03::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
4,q03::quantization,quantization,[https://qdrant.tech/documentation/manage-data...
5,q04::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
6,q04::quantization,quantization,[https://qdrant.tech/documentation/manage-data...
7,q07::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
8,q08::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...
9,q09::manage-data,manage-data,[https://qdrant.tech/documentation/manage-data...


In [96]:
filter_collection_build_df = pd.DataFrame(
    [
        {
            "variant": variant,
            **rebuild_dense_collection(
                name,
                snapshot_points(research_dense_snapshot),
                len(research_dense_snapshot),
                m=16,
                ef_construct=200,
                indexed_payload_fields=("tags",) if variant == "indexed" else (),
            ),
        }
        for variant, name in FILTER_BENCHMARK_COLLECTIONS.items()
    ]
)
display(filter_collection_build_df)


,variant,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count
0,unindexed,docs_search_filter_unindexed,4134,0.642527,1.420954,4134
1,indexed,docs_search_filter_indexed,4134,0.622469,1.825242,4134


In [97]:
def search_with_tag_filter(collection_name, query_vector, tag, *, exact=False):
    return search_dense_collection(
        collection_name,
        query_vector,
        query_filter=models.Filter(
            must=[models.FieldCondition(key="tags", match=models.MatchValue(value=tag))]
        ),
        params=models.SearchParams(exact=exact, hnsw_ef=FILTER_BENCHMARK_HNSW_EF),
        limit=EVAL_RAW_RESULT_LIMIT,
        payload=("section_url",),
    )


filter_benchmark_runs_df = run_benchmark(
    "filters_research",
    {
        variant: (
            lambda row, name=name: search_with_tag_filter(
                name, research_query_vectors[row.base_query_id], row.tag
            )
        )
        for variant, name in FILTER_BENCHMARK_COLLECTIONS.items()
    },
    filter_cases_df,
)
filter_benchmark_runs_df["tag"] = filter_benchmark_runs_df["query_id"].map(
    filter_cases_df.set_index("query_id")["tag"]
)
filter_benchmark_summary_df = summarize_benchmark(filter_benchmark_runs_df).rename(
    columns={"setting": "variant"}
)
baseline_p95 = filter_benchmark_summary_df.set_index("variant").loc["unindexed", "p95_ms"]
filter_benchmark_summary_df["speedup"] = (
    baseline_p95 / filter_benchmark_summary_df["p95_ms"]
)
filter_tag_summary_df = pd.concat(
    [
        summarize_benchmark(tag_runs).assign(tag=tag)
        for tag, tag_runs in filter_benchmark_runs_df.groupby("tag", sort=True)
    ],
    ignore_index=True,
).rename(columns={"setting": "variant"})
remember_result("filter_summary", filter_benchmark_summary_df)
display(filter_benchmark_summary_df)
display(
    filter_tag_summary_df[
        ["variant", "tag", "recall_at_10", "mrr", "p50_ms", "p95_ms"]
    ]
)
print("Recall@10/MRR use tag-valid qrels through ranx; latency uses the benchmark harness.")


,variant,recall_at_10,mrr,p50_ms,p95_ms,speedup
0,indexed,0.882353,0.660254,1.457020,1.848422,2.825202
1,unindexed,0.882353,0.660254,4.640523,5.222164,1.000000


,variant,tag,recall_at_10,mrr,p50_ms,p95_ms
0,indexed,manage-data,0.909091,0.732278,1.486745,1.866847
1,unindexed,manage-data,0.909091,0.732278,4.741162,5.293002
2,indexed,monitoring-telemetry,1.000000,0.750000,1.124376,2.019497
3,unindexed,monitoring-telemetry,1.000000,0.750000,3.922180,4.404568
4,unindexed,ops-optimization,1.000000,0.600000,4.251368,4.931619
5,indexed,ops-optimization,1.000000,0.600000,1.560595,1.844557
6,indexed,quantization,0.666667,0.499278,1.417080,1.751660
7,unindexed,quantization,0.666667,0.499278,4.365866,4.957981
8,indexed,snapshots,1.000000,0.321429,1.303387,1.584408
9,unindexed,snapshots,1.000000,0.321429,4.218114,4.740163


Recall@10/MRR use tag-valid qrels through ranx; latency uses the benchmark harness.


In [98]:
cleanup_research_collections(FILTER_BENCHMARK_COLLECTIONS.values(), KEEP_RESEARCH_COLLECTIONS)

Temporary research collections removed.


## 11. Optimization & Scale


This section tests scalar INT8 and binary 1-bit quantization on the same dense vectors without modifying the primary three-representation collection.

Original vectors use the same `cached` tier in every variant; compressed copies are `pinned`. HNSW construction is deferred during ingestion. Baseline, raw quantization and all rescoring factors are measured together in randomized order after full-query warmup. Quality is measured only with `ranx` Recall@10 and MRR; latency is measured as P50/P95 by the benchmark harness.

Vector-byte estimates are separated from actual collection memory. Neither a 32× binary-copy compression ratio nor one favorable P95 observation is treated as a measured reduction in total RAM or a guaranteed speedup.

Course references:

- [Vector Quantization Methods](https://qdrant.tech/course/essentials/day-4/what-is-quantization/)
- [Accuracy Recovery with Rescoring](https://qdrant.tech/course/essentials/day-4/rescoring-oversampling-indexing/)
- [Large-Scale Data Ingestion](https://qdrant.tech/course/essentials/day-4/large-scale-ingestion/)
- [Quantization Performance Optimization](https://qdrant.tech/course/essentials/day-4/pitstop-project/)


### 11.1 Experiment Configuration


Three dense-only collections use identical vectors, payloads, HNSW settings and original-vector memory tiers. Only quantization changes. All original vectors remain `cached`; quantized representations are `pinned`. This controls a previous confound between quantization and memory placement. A `cold`-original deployment requires a separate benchmark and is not silently inferred from these results.

In [99]:
OPTIMIZATION_COLLECTIONS = {
    method: f"{COLLECTION_NAME}_optimization_{method}"
    for method in ("baseline", "scalar", "binary")
}
OPTIMIZATION_HNSW_EF = 128
OPTIMIZATION_SEARCH_LIMIT = EVAL_RAW_RESULT_LIMIT
OPTIMIZATION_OVERSAMPLING_FACTORS = [2.0, 3.0, 5.0, 8.0, 10.0]
# Hold original-vector placement constant to isolate quantization/rescoring.
OPTIMIZATION_ORIGINAL_MEMORY = models.Memory.CACHED
KEEP_BENCHMARK_COLLECTIONS = False
RUN_MANIFEST["optimization_protocol"] = {
    "hnsw_ef": OPTIMIZATION_HNSW_EF,
    "search_limit": OPTIMIZATION_SEARCH_LIMIT,
    "original_memory": OPTIMIZATION_ORIGINAL_MEMORY.value,
}
print("No optimization winner is assumed before measurements.")


No optimization winner is assumed before measurements.


### 11.2 Dense Vector Snapshot


The benchmark reuses the single dense snapshot from Section 10.1, including only the payload fields required by the experiments. It does not copy entire chunks and neighboring sections into memory again.

In [100]:
# The same validated snapshot is reused throughout Section 10 and Section 11.
assert len(dense_snapshot) == len(retrieval_chunks_df)
print(f"Reused {len(dense_snapshot):,} dense vectors; no additional source scroll.")


Reused 4,134 dense vectors; no additional source scroll.


### 11.3 Deferred HNSW Ingestion


All dense experiments share deferred indexing and `wait=True` upload. The size-based full-scan fallback is disabled for ANN comparisons. Readiness is polled separately and reported as `readiness_seconds`, not as exact internal index build time. Unused payload indexes are not created in the unfiltered optimization experiment.

In [101]:
def build_quantization_config(method: str):
    if method == "baseline":
        return None
    if method == "scalar":
        return models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type=models.ScalarType.INT8, quantile=0.99, memory=models.Memory.PINNED
            )
        )
    if method == "binary":
        return models.BinaryQuantization(
            binary=models.BinaryQuantizationConfig(
                encoding=models.BinaryQuantizationEncoding.ONE_BIT, memory=models.Memory.PINNED
            )
        )
    raise ValueError(f"Unknown quantization method: {method}")


optimization_ingestion_df = pd.DataFrame(
    [
        {
            "method": method,
            **rebuild_dense_collection(
                name,
                snapshot_points(dense_snapshot),
                len(dense_snapshot),
                m=16,
                ef_construct=100,
                quantization=build_quantization_config(method),
                memory=OPTIMIZATION_ORIGINAL_MEMORY,
            ),
        }
        for method, name in OPTIMIZATION_COLLECTIONS.items()
    ]
)
display(optimization_ingestion_df)
RUN_MANIFEST["research_collections"] = {
    "full_scan_threshold_kb": RESEARCH_FULL_SCAN_THRESHOLD_KB,
    "indexing_threshold_kb": RESEARCH_INDEXING_THRESHOLD_KB,
    "readiness": "green + stable counts; >=95% indexed for dense-only HNSW tests",
}


,method,collection,points,upload_seconds,readiness_seconds,indexed_vectors_count
0,baseline,docs_search_optimization_baseline,4134,0.650403,1.421006,4134
1,scalar,docs_search_optimization_scalar,4134,0.617358,1.420966,4134
2,binary,docs_search_optimization_binary,4134,0.686590,1.421512,4134


### 11.4 Search and Quality Measurement


Query embeddings are reused from Section 10, outside the timed region. Each query returns 50 points and is collapsed to a section-level ranking before `ranx` evaluation. Consequently, oversampling applies to a 50-point search limit; the chosen factor cannot be transferred without testing to another limit or a hybrid prefetch stage.


In [102]:
optimization_query_vectors = research_query_vectors


def search_optimized_dense(
    collection_name,
    query_vector,
    *,
    use_quantization: bool,
    rescore: bool = False,
    oversampling=None,
    exact: bool = False
):
    quantization = (
        models.QuantizationSearchParams(ignore=True)
        if exact
        else (
            models.QuantizationSearchParams(
                ignore=False, rescore=rescore, oversampling=oversampling
            )
            if use_quantization
            else None
        )
    )
    return search_dense_collection(
        collection_name,
        query_vector,
        params=models.SearchParams(
            exact=exact, hnsw_ef=OPTIMIZATION_HNSW_EF, quantization=quantization
        ),
        limit=OPTIMIZATION_SEARCH_LIMIT,
    )


def optimization_search(method, vector, *, rescore=False, oversampling=None, exact=False):
    return search_optimized_dense(
        OPTIMIZATION_COLLECTIONS[method],
        vector,
        use_quantization=method != "baseline",
        rescore=rescore,
        oversampling=oversampling,
        exact=exact,
    )


### 11.5 Baseline and Raw Quantization


Baseline, raw quantization and all oversampling settings are measured together using the same randomized, fully warmed protocol. This subsection displays the non-rescored rows from that shared run; the following subsection displays rescoring rows. First-pass and per-repeat observations are retained in the notebook audit bundle.

In [103]:
optimization_settings = {
    "baseline_full_precision": {"method": "baseline", "rescore": False, "oversampling": None},
    "scalar_no_rescore": {"method": "scalar", "rescore": False, "oversampling": None},
    "binary_no_rescore": {"method": "binary", "rescore": False, "oversampling": None},
}
for method in ("scalar", "binary"):
    for factor in OPTIMIZATION_OVERSAMPLING_FACTORS:
        optimization_settings[f"{method}_rescore_{factor:g}x"] = {
            "method": method,
            "rescore": True,
            "oversampling": factor,
        }


# Measure baseline, raw quantization and all rescoring settings in ONE randomized run.
optimization_runs_df = run_benchmark(
    "quantization_research",
    {
        label: (
            lambda row, config=config: optimization_search(
                vector=optimization_query_vectors[row.query_id], **config
            )
        )
        for label, config in optimization_settings.items()
    },
    evaluation_df,
)
optimization_benchmark_df = (
    summarize_benchmark(optimization_runs_df)
    .rename(columns={"setting": "label"})
    .merge(
        pd.DataFrame.from_dict(optimization_settings, orient="index")
        .rename_axis("label")
        .reset_index(),
        on="label",
        validate="one_to_one",
    )
)
baseline_p95 = optimization_benchmark_df.set_index("label").loc["baseline_full_precision", "p95_ms"]
optimization_benchmark_df["speedup"] = (
    baseline_p95 / optimization_benchmark_df["p95_ms"]
)
raw_quantization_df = optimization_benchmark_df[~optimization_benchmark_df["rescore"]]
display(raw_quantization_df)


,label,recall_at_10,mrr,p50_ms,p95_ms,method,rescore,oversampling,speedup
8,binary_no_rescore,0.718519,0.453078,1.321839,1.525496,binary,False,NaN,1.114867
9,baseline_full_precision,0.814815,0.599128,1.444433,1.700725,baseline,False,NaN,1.000000
10,scalar_no_rescore,0.792593,0.587769,1.324581,1.544303,scalar,False,NaN,1.101290


### 11.6 Oversampling and Rescoring Sweep


Quantization accuracy is recovered by retrieving an enlarged candidate pool from the compressed representation and then rescoring those candidates with the original float32 vectors.

The course oversampling factors `2x`, `3x`, `5x`, `8x`, and `10x` are evaluated for both scalar and binary quantization. This makes the latency-quality trade-off explicit instead of choosing an oversampling factor in advance.


In [104]:
rescoring_df = optimization_benchmark_df[optimization_benchmark_df["rescore"]].copy()
remember_result("quantization_summary", optimization_benchmark_df)
display(rank_configurations(optimization_benchmark_df))
px.line(
    rescoring_df.sort_values(["method", "oversampling"]),
    x="oversampling",
    y="recall_at_10",
    line_dash="method",
    markers=True,
    hover_data=["mrr", "p50_ms", "p95_ms", "speedup"],
    title="Research quantization and rescoring",
).show()
display(
    optimization_runs_df.groupby(["setting", "repeat"])["latency_ms"]
    .quantile(0.95)
    .rename("p95_ms")
    .reset_index()
)


,label,recall_at_10,mrr,p50_ms,p95_ms,method,rescore,oversampling,speedup
0,binary_rescore_2x,0.814815,0.600112,1.333768,1.587563,binary,True,2.0,1.071280
1,binary_rescore_3x,0.814815,0.599216,1.341363,1.612495,binary,True,3.0,1.054716
2,scalar_rescore_3x,0.814815,0.599128,1.356427,1.562760,scalar,True,3.0,1.088283
3,scalar_rescore_2x,0.814815,0.599128,1.333436,1.568371,scalar,True,2.0,1.084389
4,scalar_rescore_5x,0.814815,0.599128,1.398720,1.618232,scalar,True,5.0,1.050977
5,binary_rescore_5x,0.814815,0.599128,1.401994,1.626184,binary,True,5.0,1.045838
6,baseline_full_precision,0.814815,0.599128,1.444433,1.700725,baseline,False,NaN,1.000000
7,binary_rescore_10x,0.814815,0.599128,1.505356,1.727017,binary,True,10.0,0.984776
8,binary_rescore_8x,0.814815,0.599128,1.478151,1.731605,binary,True,8.0,0.982167
9,scalar_rescore_8x,0.814815,0.599128,1.485166,1.735722,scalar,True,8.0,0.979837


,setting,repeat,p95_ms
0,baseline_full_precision,0,1.651713
1,baseline_full_precision,1,1.770835
2,baseline_full_precision,2,1.682994
3,baseline_full_precision,3,1.667294
4,baseline_full_precision,4,1.715434
...,...,...,...
60,scalar_rescore_8x,0,1.805364
61,scalar_rescore_8x,1,1.662720
62,scalar_rescore_8x,2,1.731188
63,scalar_rescore_8x,3,1.710807


### 11.7 Dense-Vector Memory Economics


Theoretical vector bytes are reported separately for originals and quantized copies. Because this controlled experiment caches the originals in every variant, adding quantization does not demonstrate a 4× or 32× reduction in total resident memory. Payload, HNSW, sparse vectors, ColBERT, allocator overhead and OS cache are outside these estimates; actual process RSS is not inferred from vector arithmetic.

In [105]:
optimization_point_count = len(dense_snapshot)
float32_bytes = optimization_point_count * DENSE_EMBEDDING_DIM * 4
compressed_bytes = {
    "baseline": 0,
    "scalar": optimization_point_count * DENSE_EMBEDDING_DIM,
    "binary": (optimization_point_count * DENSE_EMBEDDING_DIM + 7) // 8,
}
optimization_memory_df = pd.DataFrame(
    [
        {
            "method": method,
            "original_vector_mib": float32_bytes / 1024**2,
            "quantized_vector_mib": size / 1024**2,
            "cached_original_plus_quantized_mib": (float32_bytes + size) / 1024**2,
            "compressed_copy_ratio": float32_bytes / size if size else 1.0,
        }
        for method, size in compressed_bytes.items()
    ]
)
display(optimization_memory_df)
print(
    "Vector-byte estimates only: originals remain present. "
    "32x applies to the binary copy, not collection RAM, disk use or process RSS."
)


,method,original_vector_mib,quantized_vector_mib,cached_original_plus_quantized_mib,compressed_copy_ratio
0,baseline,6.055664,0.000000,6.055664,1.0
1,scalar,6.055664,1.513916,7.569580,4.0
2,binary,6.055664,0.189240,6.244904,32.0


Vector-byte estimates only: originals remain present. 32x applies to the binary copy, not collection RAM, disk use or process RSS.


### 11.8 Initial Selection and Diagnostic Validation

Apply the common decision order Recall@10 → MRR → P95 → P50 to dense optimization and HNSW research results. After the initial retrieval choice is frozen, materialize a diagnostic validation set and compare the already defined pipelines once. Because those outcomes become visible here, Section 11.9 promotes this diagnostic set into the next research cycle and performs one restricted confirmation comparison. Section 12 freezes the complete architecture and builds the final collection; the untouched holdout remains isolated in Section 13. Only `ranx` Recall@10/MRR and harness P50/P95 participate in selection; `speedup` remains descriptive.


In [106]:
# One decision rule for every component: Recall@10 → MRR → P95 → P50.
selected_optimization_row = rank_configurations(optimization_benchmark_df).iloc[0]
final_optimization_label = str(selected_optimization_row["label"])
FINAL_OPTIMIZATION_CONFIG = optimization_settings[final_optimization_label].copy()
FINAL_OPTIMIZATION_METHOD = FINAL_OPTIMIZATION_CONFIG["method"]
FINAL_OPTIMIZATION_OVERSAMPLING = FINAL_OPTIMIZATION_CONFIG["oversampling"]
FINAL_OPTIMIZATION_COLLECTION = OPTIMIZATION_COLLECTIONS[FINAL_OPTIMIZATION_METHOD]
RUN_MANIFEST["selected_dense_optimization"] = FINAL_OPTIMIZATION_CONFIG
print("Frozen dense-only optimization:", final_optimization_label)


# Record the timing-dependent HNSW winner observed in the current environment.
experimental_hnsw_row = rank_configurations(
    hnsw_benchmark_df[hnsw_benchmark_df["m"].gt(0)]
).iloc[0]
EXPERIMENTAL_HNSW_CONFIG = {
    "setting": str(experimental_hnsw_row["setting"]),
    "m": int(experimental_hnsw_row["m"]),
    "ef_construct": int(experimental_hnsw_row["ef_construct"]),
    "hnsw_ef": int(experimental_hnsw_row["hnsw_ef"]),
}
PRIMARY_DENSE_SEARCH_PARAMS = models.SearchParams(
    hnsw_ef=EXPERIMENTAL_HNSW_CONFIG["hnsw_ef"]
)
qdrant_client.update_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        DENSE_VECTOR_NAME: models.VectorParamsDiff(
            hnsw_config=models.HnswConfigDiff(
                m=EXPERIMENTAL_HNSW_CONFIG["m"],
                ef_construct=EXPERIMENTAL_HNSW_CONFIG["ef_construct"],
            )
        )
    },
)
wait_for_collection(COLLECTION_NAME, require_hnsw=True)
RUN_MANIFEST["observed_hnsw_current_run"] = EXPERIMENTAL_HNSW_CONFIG
print("Observed HNSW winner in the current environment:", EXPERIMENTAL_HNSW_CONFIG)

# Production HNSW is an immutable retrieval-v1 decision, not a replay result.
FROZEN_RETRIEVAL_V1_HNSW_CONFIG = {
    "setting": "m8_efc400_ef64",
    "m": 8,
    "ef_construct": 400,
    "hnsw_ef": 64,
}
FINAL_HNSW_CONFIG = FROZEN_RETRIEVAL_V1_HNSW_CONFIG.copy()
RUN_MANIFEST["selected_hnsw"] = FINAL_HNSW_CONFIG
RUN_MANIFEST["frozen_retrieval_v1_hnsw"] = FINAL_HNSW_CONFIG
print("Canonical production HNSW (retrieval-v1):", FINAL_HNSW_CONFIG)


# Fresh holdout: new section URLs, materialized only after both selections are frozen.
HELDOUT_EVALUATION_SPECS = [
    {"query_id": "h01", "query": "Which vector size and distance function must I declare when creating a collection?", "query_type": "how-to", "anchors": ["manage-data/collections/#create-a-collection"]},
    {"query_id": "h02", "query": "Can one collection store multiple named embeddings with different dimensions and metrics?", "query_type": "concept", "anchors": ["manage-data/collections/#collection-with-multiple-vectors"]},
    {"query_id": "h03", "query": "How can I add or remove a named vector from an existing collection schema?", "query_type": "api-usage", "anchors": ["manage-data/vectors/#adding-and-removing-named-vectors"]},
    {"query_id": "h04", "query": "How do I erase selected vector values while keeping their points and payloads?", "query_type": "api-usage", "anchors": ["manage-data/points/#delete-vectors"]},
    {"query_id": "h05", "query": "Which payload operation replaces all existing metadata instead of merging new fields?", "query_type": "api-usage", "anchors": ["manage-data/payload/#overwrite-payload"]},
    {"query_id": "h06", "query": "How can I remove only particular payload keys from selected points?", "query_type": "api-usage", "anchors": ["manage-data/payload/#delete-payload-keys"]},
    {"query_id": "h07", "query": "Which filter matches a keyword field against any value in an allowed list?", "query_type": "how-to", "anchors": ["search/filtering/#match-any"]},
    {"query_id": "h08", "query": "Which matching condition excludes records whose field equals any value from a list?", "query_type": "how-to", "anchors": ["search/filtering/#match-except"]},
    {"query_id": "h09", "query": "How do I ensure two conditions match fields of the same object inside a nested array?", "query_type": "concept", "anchors": ["search/filtering/#nested-object-filter"]},
    {"query_id": "h10", "query": "How can I discard search results whose similarity score is below a minimum value?", "query_type": "api-usage", "anchors": ["search/search/#filtering-results-by-score"]},
    {"query_id": "h11", "query": "Can I use an already stored point ID as the similarity query instead of sending its vector?", "query_type": "api-usage", "anchors": ["search/search/#query-by-id"]},
    {"query_id": "h12", "query": "What ordering requirement prevents duplicate or skipped records during paginated search?", "query_type": "concept", "anchors": ["search/search/#stable-ordering"]},
    {"query_id": "h13", "query": "How can I change the rank constant used by reciprocal rank fusion?", "query_type": "how-to", "anchors": ["search/hybrid-queries/#setting-rrf-constant-k"]},
    {"query_id": "h14", "query": "How can dense and sparse branches receive unequal importance in reciprocal rank fusion?", "query_type": "how-to", "anchors": ["search/hybrid-queries/#weighted-rrf"]},
    {"query_id": "h15", "query": "How do I retrieve cheaply with one representation and refine candidates with another representation?", "query_type": "concept", "anchors": ["search/hybrid-queries/#multi-stage-queries"]},
    {"query_id": "h16", "query": "How can a collection snapshot be recovered from a URL or a file already on the server?", "query_type": "api-usage", "anchors": ["snapshots/#recover-from-a-url-or-local-file"]},
    {"query_id": "h17", "query": "Which operation relocates an existing shard from one cluster peer to another?", "query_type": "api-usage", "anchors": ["scaling/distributed_deployment/#moving-shards"]},
    {"query_id": "h18", "query": "Where are majority, quorum and all-replica consistency choices configured for reads?", "query_type": "concept", "anchors": ["scaling/consistency-guarantees/#read-consistency"]},
    {"query_id": "h19", "query": "Which optimizer removes deleted points when a segment has accumulated enough stale records?", "query_type": "concept", "anchors": ["ops-optimization/optimizer/#vacuum-optimizer"]},
    {"query_id": "h20", "query": "What does HTTP 507 mean when an upload is rejected and how should storage be checked?", "query_type": "troubleshooting", "anchors": ["common-errors/#insufficient-storage-http-507"]},
]
heldout_evaluation_df = materialize_evaluation_specs(
    HELDOUT_EVALUATION_SPECS, split="confirmation"
)
assert set(evaluation_df["query_id"]).isdisjoint(heldout_evaluation_df["query_id"])
assert set().union(*evaluation_df["expected_urls"]).isdisjoint(
    set().union(*heldout_evaluation_df["expected_urls"])
)
all_evaluation_df, qrels_df = append_evaluation_cohort(
    all_evaluation_df, qrels_df, heldout_evaluation_df
)
evaluation_consistency_summary_df = synchronize_evaluation_metadata(
    all_evaluation_df,
    qrels_df,
    confirmation_section_disjoint=True,
    final_holdout_materialized=False,
)

heldout_retrieval_runs_df = run_benchmark(
    "retrieval_heldout_frozen_comparison",
    {key: finalist_functions[key] for key in FROZEN_HELDOUT_PIPELINES},
    heldout_evaluation_df,
)
heldout_retrieval_summary_df = rank_configurations(
    summarize_benchmark(heldout_retrieval_runs_df)
)
heldout_retrieval_summary_df.insert(
    1, "heldout_rank", np.arange(1, len(heldout_retrieval_summary_df) + 1)
)
research_comparison_df = finalist_summary_df[
    ["setting", "recall_at_10", "mrr", "p50_ms", "p95_ms"]
].rename(
    columns={
        "recall_at_10": "research_recall_at_10",
        "mrr": "research_mrr",
        "p50_ms": "research_p50_ms",
        "p95_ms": "research_p95_ms",
    }
)
heldout_retrieval_summary_df = heldout_retrieval_summary_df.merge(
    research_comparison_df, on="setting", how="left", validate="one_to_one"
)
heldout_retrieval_summary_df["meets_course_target"] = (
    heldout_retrieval_summary_df["recall_at_10"] >= 0.80
)
heldout_retrieval_summary_df["selected_on_research"] = (
    heldout_retrieval_summary_df["setting"] == selected_key
)
heldout_best_key = str(heldout_retrieval_summary_df.iloc[0]["setting"])
research_winner_confirmed = heldout_best_key == selected_key
RUN_MANIFEST["confirmation_comparison"] = {
    "heldout_best": heldout_best_key,
    "research_winner_confirmed": research_winner_confirmed,
}
print("Confirmation-split comparison of all frozen retrieval pipelines:")
display(heldout_retrieval_summary_df)
print(
    f"Research winner retained: {selected_key}; "
    f"confirmation winner: {heldout_best_key}; confirmed: {research_winner_confirmed}"
)

heldout_vectors = prepare_dense_queries(heldout_evaluation_df)
heldout_optimization_configs = {
    "baseline_full_precision": optimization_settings["baseline_full_precision"],
    final_optimization_label: FINAL_OPTIMIZATION_CONFIG,
}
heldout_optimization_runs_df = run_benchmark(
    "quantization_heldout",
    {
        label: (
            lambda row, config=config: optimization_search(
                vector=heldout_vectors[row.query_id], **config
            )
        )
        for label, config in heldout_optimization_configs.items()
    },
    heldout_evaluation_df,
)
heldout_optimization_summary_df = summarize_benchmark(heldout_optimization_runs_df)
print("Confirmation-split dense-only optimization — search-only latency:")
display(heldout_optimization_summary_df)
print("Confirmation results are research evidence; the final holdout remains untouched until after the final freeze.")


# Persist compact decision evidence INSIDE the notebook output, without extra files.
import base64
import gzip


def frame_records(frame: pd.DataFrame) -> list[dict]:
    return json.loads(frame.to_json(orient="records"))


def compact_quality_evidence(frame: pd.DataFrame) -> list[dict]:
    setting_column = "pipeline" if "pipeline" in frame.columns else "setting"
    quality = frame.loc[frame["quality_run"]].copy()
    evidence = []
    for setting, setting_runs in quality.groupby(setting_column, sort=False):
        ranx_scores = evaluate_quality_with_ranx(
            setting_runs, return_mean=False
        )
        metadata = setting_runs[
            ["query_id", "split", "query_type"]
        ].drop_duplicates("query_id")
        scored = metadata.merge(
            ranx_scores, on="query_id", validate="one_to_one"
        )
        scored.insert(0, "setting", setting)
        evidence.append(scored)
    return frame_records(pd.concat(evidence, ignore_index=True))


def repeat_p95_evidence(frame: pd.DataFrame) -> list[dict]:
    setting_column = "pipeline" if "pipeline" in frame.columns else "setting"
    repeat_p95 = (
        frame.groupby([setting_column, "repeat"])["latency_ms"]
        .quantile(0.95)
        .rename("p95_ms")
        .reset_index()
        .rename(columns={setting_column: "setting"})
    )
    return frame_records(repeat_p95)


def make_notebook_audit_bundle():
    # Validate immediately before serialization so audit metadata cannot be stale.
    consistency_summary = validate_evaluation_consistency(
        all_evaluation_df, qrels_df
    )
    # Keep decision-relevant evidence; omit repeated top-50 URLs and point IDs.
    payload = {
        "evaluation_consistency": frame_records(consistency_summary),
        "manifest": RUN_MANIFEST,
        "final_config": FINAL_CONFIG,
        "queries": frame_records(all_evaluation_df),
        "qrels": frame_records(qrels_df),
        "summaries": {
            "retrieval_research": frame_records(finalist_summary_df),
            "retrieval_diagnostic": frame_records(heldout_retrieval_summary_df),
            "retrieval_confirmation_research": frame_records(confirmation_research_summary_df),
            "retrieval_final_heldout": frame_records(final_heldout_retrieval_summary_df),
            "chunking": frame_records(chunking_ablation_summary_df),
            "hnsw": frame_records(hnsw_benchmark_df),
            "payload_filter": frame_records(filter_benchmark_summary_df),
            "quantization_research": frame_records(optimization_benchmark_df),
            "quantization_heldout": frame_records(heldout_optimization_summary_df),
            "vector_memory": frame_records(optimization_memory_df),
            "research_conclusions": frame_records(research_conclusions_df),
        },
        "per_query_quality_by_stage": {
            "initial_research": compact_quality_evidence(evaluation_runs_df),
            "confirmation_diagnostic": compact_quality_evidence(heldout_retrieval_runs_df),
            "confirmation_selection": compact_quality_evidence(confirmation_research_runs_df),
            "final_holdout": compact_quality_evidence(final_heldout_retrieval_runs_df),
        },
        "repeat_p95_by_stage": {
            "initial_research": repeat_p95_evidence(evaluation_runs_df),
            "confirmation_diagnostic": repeat_p95_evidence(heldout_retrieval_runs_df),
            "confirmation_selection": repeat_p95_evidence(confirmation_research_runs_df),
            "final_holdout": repeat_p95_evidence(final_heldout_retrieval_runs_df),
        },
        "benchmark_run_names": sorted(BENCHMARK_RUNS),
    }
    serialized = json.dumps(payload, separators=(",", ":")).encode("utf-8")
    encoded = base64.b64encode(gzip.compress(serialized, mtime=0)).decode("ascii")
    return {
        "encoding": "gzip+base64/json",
        "scope": "compact-decision-evidence-v1",
        "raw_runs_embedded": False,
        "uncompressed_bytes": len(serialized),
        "payload": encoded,
    }


def decode_notebook_audit_bundle(bundle: dict):
    return json.loads(gzip.decompress(base64.b64decode(bundle["payload"])))


# The bundle is serialized in Section 12 after the final decision is recorded.


Frozen dense-only optimization: binary_rescore_2x
Observed HNSW winner in the current environment: {'setting': 'm8_efc400_ef64', 'm': 8, 'ef_construct': 400, 'hnsw_ef': 64}
Canonical production HNSW (retrieval-v1): {'setting': 'm8_efc400_ef64', 'm': 8, 'ef_construct': 400, 'hnsw_ef': 64}
Confirmation-split comparison of all frozen retrieval pipelines:


,setting,heldout_rank,recall_at_10,mrr,p50_ms,p95_ms,research_recall_at_10,research_mrr,research_p50_ms,research_p95_ms,meets_course_target,selected_on_research
0,hybrid_rrf,1,1.00,0.752976,125.780118,202.728563,0.837037,0.607669,160.463609,274.675649,True,False
1,hybrid_dbsf,2,1.00,0.729167,157.519634,217.250122,0.829630,0.600105,123.592247,217.837308,True,False
2,dense,3,0.95,0.734524,22.930800,87.081384,0.814815,0.599128,20.535886,87.190650,True,False
3,dbsf_colbert,4,0.95,0.614083,287.927319,384.326213,0.837037,0.661683,285.749293,394.762451,True,True
4,rrf_colbert,5,0.95,0.609917,279.858259,382.627738,0.837037,0.661683,284.575133,397.313477,True,False
5,sparse,6,0.90,0.595473,105.496780,183.513053,0.740741,0.506599,106.611390,195.619895,True,False


Research winner retained: dbsf_colbert; confirmation winner: hybrid_rrf; confirmed: False
Confirmation-split dense-only optimization — search-only latency:


,setting,recall_at_10,mrr,p50_ms,p95_ms
0,baseline_full_precision,0.95,0.734524,1.332777,1.657184
1,binary_rescore_2x,0.95,0.739881,1.275606,1.521595


Confirmation results are research evidence; the final holdout remains untouched until after the final freeze.


### 11.9 Restricted Confirmation Cycle and Final Freeze


Section 11.8 reports the timing- and environment-dependent experimental HNSW winner observed in the current execution; it can differ between execution environments and repeated Run All executions. The benchmark measures search over already computed query vectors, so BGE/SPLADE inference is outside its search-only latency. `EXPERIMENTAL_HNSW_CONFIG` preserves that current-run result for research and reporting only. Separately, the historical CPU portability replay remains fixed at `m=8, ef_construct=100, hnsw_ef=64`. Neither the historical replay nor any current or future experimental winner supersedes the canonical `retrieval-v1` production configuration, `m=8, ef_construct=400, hnsw_ef=64`; `FROZEN_RETRIEVAL_V1_HNSW_CONFIG` remains the independent source used by the final collection.

The diagnostic validation in Section 11.8 did not confirm the initial ColBERT winner. Those 20 observed queries form the canonical `confirmation` split; the original 45 queries remain in `research`. The restricted selection pool therefore contains 65 queries while preserving each query's single source-of-truth split. To avoid research-for-research's-sake, exactly two predeclared finalists are compared once: `hybrid_rrf`, which won the confirmation diagnostic, and `rrf_colbert`, which won the initial research set. The final choice uses only Recall@10, MRR, P95 and P50, in that order. It is then frozen before a new section-disjoint 20-query `final_holdout` is materialized. The final holdout evaluates only the frozen winner and never re-selects it.


In [107]:
CONFIRMATION_PIPELINES = ("hybrid_rrf", "rrf_colbert")
confirmation_research_df = all_evaluation_df.loc[
    all_evaluation_df["split"].isin({"research", "confirmation"})
].copy()
expected_selection_query_ids = set(evaluation_df["query_id"]) | set(
    heldout_evaluation_df["query_id"]
)
if set(confirmation_research_df["query_id"]) != expected_selection_query_ids:
    raise ValueError("Confirmation selection pool does not match research + confirmation.")
confirmation_qrels_df = qrels_df.loc[
    qrels_df["query_id"].isin(confirmation_research_df["query_id"])
].copy()
validate_evaluation_consistency(confirmation_research_df, confirmation_qrels_df)
assert set(CONFIRMATION_PIPELINES).issubset(finalist_functions)

confirmation_research_runs_df = run_benchmark(
    "retrieval_confirmation_research",
    {key: finalist_functions[key] for key in CONFIRMATION_PIPELINES},
    confirmation_research_df,
)
confirmation_research_summary_df = rank_configurations(
    summarize_benchmark(confirmation_research_runs_df)
)
confirmation_research_summary_df.insert(
    1, "confirmation_rank", np.arange(1, len(confirmation_research_summary_df) + 1)
)

selected_key = str(confirmation_research_summary_df.iloc[0]["setting"])
FINAL_RETRIEVAL_CONFIG = candidate_configs[selected_key].copy()
FINAL_RETRIEVAL_PIPELINE = FINAL_RETRIEVAL_CONFIG["pipeline"]
FINAL_FUSION = FINAL_RETRIEVAL_CONFIG["fusion"]
FINAL_RETRIEVAL_FREEZE = {
    "selection_rule": ["recall_at_10_desc", "mrr_desc", "p95_ms_asc", "p50_ms_asc"],
    "selection_query_ids": confirmation_research_df["query_id"].tolist(),
    "selection_split_counts": {
        split: int(count)
        for split, count in confirmation_research_df["split"].value_counts().items()
    },
    "candidates": list(CONFIRMATION_PIPELINES),
    "winner": selected_key,
    "configuration": FINAL_RETRIEVAL_CONFIG,
}
RUN_MANIFEST["final_retrieval_freeze"] = FINAL_RETRIEVAL_FREEZE
display(confirmation_research_summary_df)
print("Final retrieval configuration frozen before final holdout:", FINAL_RETRIEVAL_CONFIG)


,setting,confirmation_rank,recall_at_10,mrr,p50_ms,p95_ms
0,hybrid_rrf,1,0.887179,0.652379,166.935920,212.161524
1,rrf_colbert,2,0.871795,0.645755,289.399008,400.730196


Final retrieval configuration frozen before final holdout: {'pipeline': 'hybrid_rrf', 'fusion': 'rrf', 'candidate_limit': None}


## 12. Final Vector Store Configuration


### 12.1 Architecture Freeze


The final architecture below is derived only from the already completed research and confirmation DataFrames. No final-holdout row is available or referenced at this freeze point. All component candidates and their Recall@10, MRR, P50 and P95 values remain visible in one evidence table. For HNSW, the timing-dependent experimental winner from the current execution remains observational research evidence, while the historical CPU portability replay (`m=8`, `ef_construct=100`, `hnsw_ef=64`) is preserved as a separate immutable record. `FINAL_CONFIG` deliberately takes production settings from `FROZEN_RETRIEVAL_V1_HNSW_CONFIG` (`m=8`, `ef_construct=400`, query-time `hnsw_ef=64`). The following cell reports these records separately; neither experimental evidence source replaces the canonical production source of truth.


In [108]:
# Freeze every final component before materializing final_holdout.
# The common ranking rule is Recall@10, MRR, P95, then P50.
selected_chunking_row = rank_configurations(chunking_ablation_summary_df).iloc[0]
FINAL_CHUNKING_STRATEGY = str(selected_chunking_row["strategy"])
if FINAL_CHUNKING_STRATEGY != "structure_aware":
    raise RuntimeError(
        "The measured chunking winner is not represented by retrieval_chunks_df; "
        "a final collection cannot be built without changing the frozen corpus."
    )

selected_filter_row = rank_configurations(filter_benchmark_summary_df).iloc[0]
FINAL_PAYLOAD_VARIANT = str(selected_filter_row["variant"])

# Confirmation compared the research-selected quantization against full precision.
selected_optimization_row = rank_configurations(
    heldout_optimization_summary_df
).iloc[0]
final_optimization_label = str(selected_optimization_row["setting"])
FINAL_OPTIMIZATION_CONFIG = optimization_settings[final_optimization_label].copy()
FINAL_OPTIMIZATION_METHOD = FINAL_OPTIMIZATION_CONFIG["method"]
FINAL_OPTIMIZATION_OVERSAMPLING = FINAL_OPTIMIZATION_CONFIG["oversampling"]

selected_retrieval_row = confirmation_research_summary_df.set_index("setting").loc[
    selected_key
]
selected_hnsw_row = hnsw_benchmark_df.set_index("setting").loc[
    FINAL_HNSW_CONFIG["setting"]
]
selected_filter_row = filter_benchmark_summary_df.set_index("variant").loc[
    FINAL_PAYLOAD_VARIANT
]
selected_quantization_row = heldout_optimization_summary_df.set_index("setting").loc[
    final_optimization_label
]


def architecture_evidence_rows(
    frame: pd.DataFrame,
    *,
    component: str,
    configuration_column: str,
    selected: str,
) -> pd.DataFrame:
    evidence = frame[
        [configuration_column, "recall_at_10", "mrr", "p50_ms", "p95_ms"]
    ].copy()
    evidence.insert(0, "component", component)
    evidence = evidence.rename(columns={configuration_column: "configuration"})
    evidence["selected"] = evidence["configuration"].astype(str).eq(str(selected))
    return evidence


final_architecture_evidence_df = pd.concat(
    [
        architecture_evidence_rows(
            confirmation_research_summary_df,
            component="retrieval / fusion / ColBERT",
            configuration_column="setting",
            selected=selected_key,
        ),
        architecture_evidence_rows(
            chunking_ablation_summary_df,
            component="chunking",
            configuration_column="strategy",
            selected=FINAL_CHUNKING_STRATEGY,
        ),
        architecture_evidence_rows(
            hnsw_benchmark_df,
            component="HNSW",
            configuration_column="setting",
            selected=FINAL_HNSW_CONFIG["setting"],
        ),
        architecture_evidence_rows(
            filter_benchmark_summary_df,
            component="payload indexing",
            configuration_column="variant",
            selected=FINAL_PAYLOAD_VARIANT,
        ),
        architecture_evidence_rows(
            optimization_benchmark_df,
            component="quantization research",
            configuration_column="label",
            selected=final_optimization_label,
        ),
        architecture_evidence_rows(
            heldout_optimization_summary_df,
            component="quantization confirmation",
            configuration_column="setting",
            selected=final_optimization_label,
        ),
    ],
    ignore_index=True,
)
remember_result("final_architecture_evidence", final_architecture_evidence_df)
display(final_architecture_evidence_df.round(4))

colbert_enabled = selected_key.endswith("_colbert")
dense_enabled = selected_key != "sparse"
sparse_enabled = selected_key in {
    "sparse",
    "hybrid_rrf",
    "hybrid_dbsf",
    "rrf_colbert",
    "dbsf_colbert",
}
quantization_method = (
    "full_precision" if FINAL_OPTIMIZATION_METHOD == "baseline" else FINAL_OPTIMIZATION_METHOD
)
payload_indexes = (
    {"tags": models.PayloadSchemaType.KEYWORD.value}
    if FINAL_PAYLOAD_VARIANT == "indexed"
    else {}
)

FINAL_CONFIG = {
    "collection_name": "docs_search_final",
    "build_source_collection": COLLECTION_NAME,
    "selection": {
        "rule": ["recall_at_10_desc", "mrr_desc", "p95_ms_asc", "p50_ms_asc"],
        "query_splits": ["research", "confirmation"],
        "final_holdout_used": False,
    },
    "chunking": {
        "strategy": FINAL_CHUNKING_STRATEGY,
        "token_budget": int(CHUNK_TOKEN_BUDGET),
        "fixed_overlap_ratio": None,
    },
    "representations": {
        "dense": dense_enabled,
        "sparse": sparse_enabled,
        "colbert": colbert_enabled,
    },
    "vector_names": {
        "dense": DENSE_VECTOR_NAME,
        "sparse": SPARSE_VECTOR_NAME,
        "colbert": COLBERT_VECTOR_NAME,
    },
    "dense": {
        "model": DENSE_MODEL_NAME,
        "size": int(DENSE_EMBEDDING_DIM),
        "distance": models.Distance.COSINE.value,
    },
    "sparse": {
        "model": SPARSE_MODEL_NAME if sparse_enabled else None,
    },
    "colbert": {
        "enabled": colbert_enabled,
        "model": COLBERT_MODEL_NAME if colbert_enabled else None,
        "size": int(COLBERT_EMBEDDING_DIM) if colbert_enabled else None,
        "distance": models.Distance.COSINE.value if colbert_enabled else None,
        "comparator": models.MultiVectorComparator.MAX_SIM.value if colbert_enabled else None,
        "hnsw_m": 0 if colbert_enabled else None,
    },
    "retrieval": {
        "pipeline": selected_key,
        "fusion": FINAL_FUSION,
        "prefetch_limit": int(HYBRID_PREFETCH_LIMIT),
        "candidate_limit": (
            int(RERANK_CANDIDATE_LIMIT) if colbert_enabled else None
        ),
        "raw_result_limit": int(EVAL_RAW_RESULT_LIMIT),
        "top_k": int(EVAL_TOP_K),
    },
    "hnsw": {
        "m": int(FINAL_HNSW_CONFIG["m"]),
        "ef_construct": int(FINAL_HNSW_CONFIG["ef_construct"]),
        "hnsw_ef": int(FINAL_HNSW_CONFIG["hnsw_ef"]),
        "full_scan_threshold_kb": int(RESEARCH_FULL_SCAN_THRESHOLD_KB),
    },
    "quantization": {
        "label": final_optimization_label,
        "method": quantization_method,
        "scalar_type": (
            models.ScalarType.INT8.value if quantization_method == "scalar" else None
        ),
        "scalar_quantile": 0.99 if quantization_method == "scalar" else None,
        "binary_encoding": (
            models.BinaryQuantizationEncoding.ONE_BIT.value
            if quantization_method == "binary"
            else None
        ),
        "memory": (
            models.Memory.PINNED.value
            if quantization_method != "full_precision"
            else None
        ),
        "rescore": bool(FINAL_OPTIMIZATION_CONFIG["rescore"]),
        "oversampling": (
            float(FINAL_OPTIMIZATION_OVERSAMPLING)
            if FINAL_OPTIMIZATION_OVERSAMPLING is not None
            else None
        ),
    },
    "payload_indexes": payload_indexes,
    "memory": {
        "dense_original": OPTIMIZATION_ORIGINAL_MEMORY.value,
        "dense_quantized": "pinned" if quantization_method != "full_precision" else None,
    },
    "storage": {
        "payload_on_disk": True,
        "deferred_hnsw": True,
        "indexing_threshold_kb": int(RESEARCH_INDEXING_THRESHOLD_KB),
    },
    "ingestion": {
        "upload_batch": int(INGEST.upload_batch),
        "upload_parallel": int(INGEST.upload_parallel),
        "max_retries": 3,
        "wait": True,
    },
}

if FINAL_CONFIG["representations"]["colbert"] != (
    FINAL_CONFIG["retrieval"]["candidate_limit"] is not None
):
    raise ValueError("ColBERT storage and reranking configuration disagree.")
if FINAL_CONFIG["quantization"]["method"] != "full_precision" and not dense_enabled:
    raise ValueError("Dense quantization cannot be selected without a dense representation.")

RUN_MANIFEST["final_config"] = FINAL_CONFIG
display(pd.json_normalize(FINAL_CONFIG, sep="."))
display(
    Markdown(
        f"""
### Frozen decisions from existing results

- **Retrieval / fusion / ColBERT — `{selected_key}`:** Recall@10={selected_retrieval_row['recall_at_10']:.3f}, MRR={selected_retrieval_row['mrr']:.3f}, P50={selected_retrieval_row['p50_ms']:.2f} ms, P95={selected_retrieval_row['p95_ms']:.2f} ms on the existing research + confirmation selection pool. Fusion=`{FINAL_FUSION or 'none'}`; ColBERT={'enabled' if colbert_enabled else 'disabled'} exactly as selected by this pipeline.
- **Chunking — `{FINAL_CHUNKING_STRATEGY}`:** Recall@10={selected_chunking_row['recall_at_10']:.3f}, MRR={selected_chunking_row['mrr']:.3f}, P50={selected_chunking_row['p50_ms']:.2f} ms, P95={selected_chunking_row['p95_ms']:.2f} ms across the existing chunking ablation.
- **HNSW — `{FINAL_HNSW_CONFIG['setting']}`:** Recall@10={selected_hnsw_row['recall_at_10']:.3f}, MRR={selected_hnsw_row['mrr']:.3f}, P50={selected_hnsw_row['p50_ms']:.2f} ms, P95={selected_hnsw_row['p95_ms']:.2f} ms across the existing HNSW grid.
- **Payload indexing — `{FINAL_PAYLOAD_VARIANT}`:** Recall@10={selected_filter_row['recall_at_10']:.3f}, MRR={selected_filter_row['mrr']:.3f}, P50={selected_filter_row['p50_ms']:.2f} ms, P95={selected_filter_row['p95_ms']:.2f} ms in the existing filtered benchmark.
- **Quantization — `{final_optimization_label}`:** Recall@10={selected_quantization_row['recall_at_10']:.3f}, MRR={selected_quantization_row['mrr']:.3f}, P50={selected_quantization_row['p50_ms']:.2f} ms, P95={selected_quantization_row['p95_ms']:.2f} ms in the existing confirmation comparison. Oversampling and rescoring remain query-time settings.

No final-holdout result participates in these decisions. The controlled dense-memory tier and deferred-HNSW ingestion protocol are retained from the existing quantization and ingestion experiments rather than re-tuned.
"""
    )
)


,component,configuration,recall_at_10,mrr,p50_ms,p95_ms,selected
0,retrieval / fusion / ColBERT,hybrid_rrf,0.8872,0.6524,166.9359,212.1615,True
1,retrieval / fusion / ColBERT,rrf_colbert,0.8718,0.6458,289.3990,400.7302,False
2,chunking,structure_aware,0.8148,0.5991,1.5002,1.8407,True
3,chunking,fixed_overlap_15pct,0.8148,0.5917,1.5407,1.8103,False
4,chunking,fixed_no_overlap,0.7926,0.5787,1.4747,1.7834,False
5,HNSW,m16_efc100_ef256,0.8148,0.5991,1.5610,1.8043,False
6,HNSW,m8_efc400_ef64,0.8148,0.5991,1.3631,1.5484,True
7,HNSW,m8_efc200_ef64,0.8148,0.5991,1.3660,1.6051,False
8,HNSW,m32_efc100_ef128,0.8148,0.5991,1.4473,1.7176,False
9,HNSW,m8_efc400_ef256,0.8148,0.5991,1.5444,1.7341,False


,collection_name,build_source_collection,selection.rule,selection.query_splits,selection.final_holdout_used,chunking.strategy,chunking.token_budget,chunking.fixed_overlap_ratio,representations.dense,representations.sparse,...,payload_indexes.tags,memory.dense_original,memory.dense_quantized,storage.payload_on_disk,storage.deferred_hnsw,storage.indexing_threshold_kb,ingestion.upload_batch,ingestion.upload_parallel,ingestion.max_retries,ingestion.wait
0,docs_search_final,docs_search,"[recall_at_10_desc, mrr_desc, p95_ms_asc, p50_...","[research, confirmation]",False,structure_aware,448,None,True,True,...,keyword,cached,pinned,True,True,10,64,1,3,True



### Frozen decisions from existing results

- **Retrieval / fusion / ColBERT — `hybrid_rrf`:** Recall@10=0.887, MRR=0.652, P50=166.94 ms, P95=212.16 ms on the existing research + confirmation selection pool. Fusion=`rrf`; ColBERT=disabled exactly as selected by this pipeline.
- **Chunking — `structure_aware`:** Recall@10=0.815, MRR=0.599, P50=1.50 ms, P95=1.84 ms across the existing chunking ablation.
- **HNSW — `m8_efc400_ef64`:** Recall@10=0.815, MRR=0.599, P50=1.36 ms, P95=1.55 ms across the existing HNSW grid.
- **Payload indexing — `indexed`:** Recall@10=0.882, MRR=0.660, P50=1.46 ms, P95=1.85 ms in the existing filtered benchmark.
- **Quantization — `binary_rescore_2x`:** Recall@10=0.950, MRR=0.740, P50=1.28 ms, P95=1.52 ms in the existing confirmation comparison. Oversampling and rescoring remain query-time settings.

No final-holdout result participates in these decisions. The controlled dense-memory tier and deferred-HNSW ingestion protocol are retained from the existing quantization and ingestion experiments rather than re-tuned.


### 12.2 Build and Validate the Final Collection


A new stable `docs_search_final` collection is created from the validated vectors already stored in the research source collection. Only representations selected by `FINAL_CONFIG` are copied. Vector indexing remains disabled during upload; selected payload indexes are created before HNSW is enabled. The source and benchmark collections are removed only after exact-count and structural validation pass.


In [109]:
def selected_final_vector_names(config: dict) -> list[str]:
    return [
        config["vector_names"][kind]
        for kind in ("dense", "sparse", "colbert")
        if config["representations"][kind]
    ]


def final_quantization_config(config: dict):
    quantization = config["quantization"]
    method = quantization["method"]
    if method == "full_precision":
        return None
    if method == "scalar":
        return models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type=models.ScalarType(quantization["scalar_type"]),
                quantile=quantization["scalar_quantile"],
                memory=models.Memory(quantization["memory"]),
            )
        )
    if method == "binary":
        return models.BinaryQuantization(
            binary=models.BinaryQuantizationConfig(
                encoding=models.BinaryQuantizationEncoding(
                    quantization["binary_encoding"]
                ),
                memory=models.Memory(quantization["memory"]),
            )
        )
    raise ValueError(f"Unsupported final quantization method: {method}")


def build_final_vector_schema(config: dict) -> tuple[dict, dict]:
    vectors, sparse_vectors = {}, {}
    if config["representations"]["dense"]:
        vectors[config["vector_names"]["dense"]] = models.VectorParams(
            size=config["dense"]["size"],
            distance=models.Distance(config["dense"]["distance"]),
            memory=models.Memory(config["memory"]["dense_original"]),
            hnsw_config=models.HnswConfigDiff(
                m=config["hnsw"]["m"],
                ef_construct=config["hnsw"]["ef_construct"],
                full_scan_threshold=config["hnsw"]["full_scan_threshold_kb"],
            ),
            quantization_config=final_quantization_config(config),
        )
    if config["representations"]["colbert"]:
        vectors[config["vector_names"]["colbert"]] = models.VectorParams(
            size=config["colbert"]["size"],
            distance=models.Distance(config["colbert"]["distance"]),
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator(config["colbert"]["comparator"])
            ),
            hnsw_config=models.HnswConfigDiff(m=config["colbert"]["hnsw_m"]),
        )
    if config["representations"]["sparse"]:
        sparse_vectors[config["vector_names"]["sparse"]] = models.SparseVectorParams()
    return vectors, sparse_vectors


def iter_final_points(config: dict, scroll_batch: int = 256):
    vector_names = selected_final_vector_names(config)
    offset = None
    while True:
        records, offset = qdrant_client.scroll(
            collection_name=config["build_source_collection"],
            limit=scroll_batch,
            offset=offset,
            with_payload=True,
            with_vectors=vector_names,
        )
        for record in records:
            if not isinstance(record.vector, dict):
                raise ValueError(f"Expected named vectors for point {record.id}.")
            missing_vectors = sorted(set(vector_names) - set(record.vector))
            if missing_vectors:
                raise ValueError(
                    f"Point {record.id} is missing selected vectors: {missing_vectors}"
                )
            yield models.PointStruct(
                id=record.id,
                vector={name: record.vector[name] for name in vector_names},
                payload=record.payload or {},
            )
        if offset is None:
            break


def validate_final_collection(config: dict) -> pd.DataFrame:
    collection_name = config["collection_name"]
    info = qdrant_client.get_collection(collection_name)
    exact_points = qdrant_client.count(collection_name, exact=True).count
    if exact_points != len(retrieval_chunks_df):
        raise ValueError(
            f"Final exact point count mismatch: {exact_points} != {len(retrieval_chunks_df)}"
        )
    if info.status != models.CollectionStatus.GREEN:
        raise ValueError(f"Final collection is not GREEN: {info.status}")

    vector_schema = info.config.params.vectors
    if not isinstance(vector_schema, dict):
        raise ValueError("Final collection must use named vectors.")
    sparse_schema = info.config.params.sparse_vectors or {}
    expected_dense_names = {
        config["vector_names"][kind]
        for kind in ("dense", "colbert")
        if config["representations"][kind]
    }
    expected_sparse_names = (
        {config["vector_names"]["sparse"]}
        if config["representations"]["sparse"]
        else set()
    )
    if set(vector_schema) != expected_dense_names:
        raise ValueError(
            f"Final dense/multivector names mismatch: {set(vector_schema)} != {expected_dense_names}"
        )
    if set(sparse_schema) != expected_sparse_names:
        raise ValueError(
            f"Final sparse-vector names mismatch: {set(sparse_schema)} != {expected_sparse_names}"
        )

    if config["representations"]["dense"]:
        dense = vector_schema[config["vector_names"]["dense"]]
        if dense.size != config["dense"]["size"]:
            raise ValueError(f"Final dense dimension mismatch: {dense.size}")
        if dense.distance.value != config["dense"]["distance"]:
            raise ValueError(f"Final dense distance mismatch: {dense.distance}")
        if dense.hnsw_config is None:
            raise ValueError("Final dense HNSW configuration is missing.")
        actual_hnsw = (
            dense.hnsw_config.m,
            dense.hnsw_config.ef_construct,
            dense.hnsw_config.full_scan_threshold,
        )
        expected_hnsw = (
            config["hnsw"]["m"],
            config["hnsw"]["ef_construct"],
            config["hnsw"]["full_scan_threshold_kb"],
        )
        if actual_hnsw != expected_hnsw:
            raise ValueError(f"Final HNSW mismatch: {actual_hnsw} != {expected_hnsw}")
        if dense.memory is None or dense.memory.value != config["memory"]["dense_original"]:
            raise ValueError(f"Final dense memory tier mismatch: {dense.memory}")
        method = config["quantization"]["method"]
        actual_quantization = dense.quantization_config
        expected_type = {
            "full_precision": type(None),
            "scalar": models.ScalarQuantization,
            "binary": models.BinaryQuantization,
        }[method]
        if not isinstance(actual_quantization, expected_type):
            raise ValueError(
                f"Final quantization mismatch for {method}: {actual_quantization}"
            )
        if method == "scalar":
            scalar = actual_quantization.scalar
            if (
                scalar.type.value != config["quantization"]["scalar_type"]
                or not np.isclose(
                    float(scalar.quantile),
                    float(config["quantization"]["scalar_quantile"]),
                    rtol=1e-6,
                    atol=1e-8,
                )
                or scalar.memory.value != config["quantization"]["memory"]
            ):
                raise ValueError(f"Final scalar quantization details mismatch: {scalar}")
        if method == "binary":
            binary = actual_quantization.binary
            if (
                binary.encoding.value != config["quantization"]["binary_encoding"]
                or binary.memory.value != config["quantization"]["memory"]
            ):
                raise ValueError(f"Final binary quantization details mismatch: {binary}")

    if config["representations"]["colbert"]:
        colbert = vector_schema[config["vector_names"]["colbert"]]
        if colbert.size != config["colbert"]["size"]:
            raise ValueError(f"Final ColBERT dimension mismatch: {colbert.size}")
        if colbert.distance.value != config["colbert"]["distance"]:
            raise ValueError(f"Final ColBERT distance mismatch: {colbert.distance}")
        if colbert.multivector_config is None:
            raise ValueError("Final ColBERT multivector configuration is missing.")
        if (
            colbert.multivector_config.comparator.value
            != config["colbert"]["comparator"]
        ):
            raise ValueError("Final ColBERT comparator does not match FINAL_CONFIG.")
        if colbert.hnsw_config is None or colbert.hnsw_config.m != 0:
            raise ValueError("Final ColBERT HNSW must be disabled with m=0.")

    actual_payload_indexes = set(info.payload_schema)
    expected_payload_indexes = set(config["payload_indexes"])
    if actual_payload_indexes != expected_payload_indexes:
        raise ValueError(
            f"Final payload indexes mismatch: {actual_payload_indexes} != {expected_payload_indexes}"
        )
    for field_name, schema_name in config["payload_indexes"].items():
        actual_type = info.payload_schema[field_name].data_type.value
        if actual_type != schema_name:
            raise ValueError(
                f"Payload index type mismatch for {field_name}: {actual_type} != {schema_name}"
            )

    if info.config.params.on_disk_payload != config["storage"]["payload_on_disk"]:
        raise ValueError("Final payload storage does not match FINAL_CONFIG.")
    if (
        info.config.optimizer_config.indexing_threshold
        != config["storage"]["indexing_threshold_kb"]
    ):
        raise ValueError("Final indexing threshold does not match FINAL_CONFIG.")

    return pd.DataFrame(
        [
            {"check": "exact_point_count", "value": int(exact_points), "status": "PASS"},
            {"check": "collection_status", "value": info.status.value, "status": "PASS"},
            {"check": "vector_names", "value": sorted(expected_dense_names | expected_sparse_names), "status": "PASS"},
            {"check": "payload_indexes", "value": sorted(expected_payload_indexes), "status": "PASS"},
            {"check": "hnsw", "value": config["hnsw"], "status": "PASS"},
            {"check": "quantization", "value": config["quantization"], "status": "PASS"},
            {"check": "storage", "value": config["storage"], "status": "PASS"},
        ]
    )


final_vectors_config, final_sparse_vectors_config = build_final_vector_schema(FINAL_CONFIG)
source_point_count = qdrant_client.count(
    FINAL_CONFIG["build_source_collection"], exact=True
).count
if source_point_count != len(retrieval_chunks_df):
    raise ValueError(
        f"Build source count mismatch: {source_point_count} != {len(retrieval_chunks_df)}"
    )

if qdrant_client.collection_exists(FINAL_CONFIG["collection_name"]):
    qdrant_client.delete_collection(FINAL_CONFIG["collection_name"])
qdrant_client.create_collection(
    collection_name=FINAL_CONFIG["collection_name"],
    vectors_config=final_vectors_config,
    sparse_vectors_config=final_sparse_vectors_config or None,
    hnsw_config=models.HnswConfigDiff(
        m=FINAL_CONFIG["hnsw"]["m"],
        ef_construct=FINAL_CONFIG["hnsw"]["ef_construct"],
        full_scan_threshold=FINAL_CONFIG["hnsw"]["full_scan_threshold_kb"],
    ),
    optimizers_config=models.OptimizersConfigDiff(indexing_threshold=0),
    on_disk_payload=FINAL_CONFIG["storage"]["payload_on_disk"],
)

final_ingestion_started = time.perf_counter()
qdrant_client.upload_points(
    collection_name=FINAL_CONFIG["collection_name"],
    points=iter_final_points(FINAL_CONFIG),
    batch_size=FINAL_CONFIG["ingestion"]["upload_batch"],
    parallel=FINAL_CONFIG["ingestion"]["upload_parallel"],
    max_retries=FINAL_CONFIG["ingestion"]["max_retries"],
    wait=FINAL_CONFIG["ingestion"]["wait"],
)
final_ingestion_seconds = time.perf_counter() - final_ingestion_started

# Payload indexes are created while vector indexing remains deferred.
for field_name, schema_name in FINAL_CONFIG["payload_indexes"].items():
    qdrant_client.create_payload_index(
        collection_name=FINAL_CONFIG["collection_name"],
        field_name=field_name,
        field_schema=models.PayloadSchemaType(schema_name),
        wait=True,
    )
qdrant_client.update_collection(
    collection_name=FINAL_CONFIG["collection_name"],
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=FINAL_CONFIG["storage"]["indexing_threshold_kb"]
    ),
)
wait_for_collection(
    FINAL_CONFIG["collection_name"],
    require_hnsw=FINAL_CONFIG["representations"]["dense"],
)
final_collection_validation_df = validate_final_collection(FINAL_CONFIG)
display(final_collection_validation_df)

# Delete benchmark and build-source collections only after the final collection passes.
collection_names_before_final_cleanup = [
    item.name for item in qdrant_client.get_collections().collections
]
for collection_name in collection_names_before_final_cleanup:
    if collection_name != FINAL_CONFIG["collection_name"]:
        qdrant_client.delete_collection(collection_name)
remaining_collections = sorted(
    item.name for item in qdrant_client.get_collections().collections
)
if remaining_collections != [FINAL_CONFIG["collection_name"]]:
    raise ValueError(f"Unexpected collections after final cleanup: {remaining_collections}")
final_point_count = qdrant_client.count(
    FINAL_CONFIG["collection_name"], exact=True
).count


def _final_query_filter(tag: str | None):
    if tag is None:
        return None
    if "tags" not in FINAL_CONFIG["payload_indexes"]:
        raise ValueError("FINAL_CONFIG does not include a tags payload index.")
    return models.Filter(
        must=[models.FieldCondition(key="tags", match=models.MatchValue(value=tag))]
    )


def _final_dense_search_params() -> models.SearchParams:
    quantization = FINAL_CONFIG["quantization"]
    quantization_params = (
        None
        if quantization["method"] == "full_precision"
        else models.QuantizationSearchParams(
            ignore=False,
            rescore=quantization["rescore"],
            oversampling=quantization["oversampling"],
        )
    )
    return models.SearchParams(
        hnsw_ef=FINAL_CONFIG["hnsw"]["hnsw_ef"],
        quantization=quantization_params,
    )


def _query_final_points(
    query: str,
    *,
    client: QdrantClient = qdrant_client,
    collection_name: str | None = None,
    tag: str | None = None,
    payload=SEARCH_PAYLOAD,
):
    config = FINAL_CONFIG
    collection_name = config["collection_name"] if collection_name is None else collection_name
    query_filter = _final_query_filter(tag)
    dense_params = _final_dense_search_params()
    raw_limit = config["retrieval"]["raw_result_limit"]

    if config["retrieval"]["pipeline"] == "dense":
        return client.query_points(
            collection_name=collection_name,
            query=embed_dense_query(query),
            using=config["vector_names"]["dense"],
            search_params=dense_params,
            query_filter=query_filter,
            limit=raw_limit,
            with_payload=payload,
            with_vectors=False,
        ).points
    if config["retrieval"]["pipeline"] == "sparse":
        return client.query_points(
            collection_name=collection_name,
            query=embed_sparse_query(query),
            using=config["vector_names"]["sparse"],
            query_filter=query_filter,
            limit=raw_limit,
            with_payload=payload,
            with_vectors=False,
        ).points

    prefetches = [
        models.Prefetch(
            query=embed_dense_query(query),
            using=config["vector_names"]["dense"],
            params=dense_params,
            filter=query_filter,
            limit=config["retrieval"]["prefetch_limit"],
        ),
        models.Prefetch(
            query=embed_sparse_query(query),
            using=config["vector_names"]["sparse"],
            filter=query_filter,
            limit=config["retrieval"]["prefetch_limit"],
        ),
    ]
    fusion_query = get_fusion_query(config["retrieval"]["fusion"])
    if config["representations"]["colbert"]:
        fused_candidates = models.Prefetch(
            prefetch=prefetches,
            query=fusion_query,
            filter=query_filter,
            limit=config["retrieval"]["candidate_limit"],
        )
        return client.query_points(
            collection_name=collection_name,
            prefetch=fused_candidates,
            query=embed_colbert_query(query),
            using=config["vector_names"]["colbert"],
            query_filter=query_filter,
            limit=raw_limit,
            with_payload=payload,
            with_vectors=False,
        ).points
    return client.query_points(
        collection_name=collection_name,
        prefetch=prefetches,
        query=fusion_query,
        query_filter=query_filter,
        limit=raw_limit,
        with_payload=payload,
        with_vectors=False,
    ).points


def retrieve_final_sections(
    query: str,
    *,
    client: QdrantClient = qdrant_client,
    collection_name: str | None = None,
    top_k: int | None = None,
    tag: str | None = None,
) -> list[dict]:
    """Single production-like retrieval entry point driven only by FINAL_CONFIG."""
    top_k = FINAL_CONFIG["retrieval"]["top_k"] if top_k is None else top_k
    if not 1 <= top_k <= FINAL_CONFIG["retrieval"]["top_k"]:
        raise ValueError(
            f"top_k must be in 1..{FINAL_CONFIG['retrieval']['top_k']}"
        )
    points = _query_final_points(
        query,
        client=client,
        collection_name=collection_name,
        tag=tag,
        payload=SEARCH_PAYLOAD,
    )
    return collapse_points_to_sections(points, max_sections=top_k)


RUN_MANIFEST["final_architecture"] = FINAL_CONFIG
RUN_MANIFEST["final_collection_validation"] = frame_records(
    final_collection_validation_df
)
RUN_MANIFEST["final_ingestion_seconds"] = float(final_ingestion_seconds)
print(
    f"Final collection: {remaining_collections}; points={final_point_count:,}; "
    f"copied in {final_ingestion_seconds:.2f}s.\n"
    "STOP: final_holdout has not been materialized or executed."
)


,check,value,status
0,exact_point_count,4134,PASS
1,collection_status,green,PASS
2,vector_names,"[dense, sparse]",PASS
3,payload_indexes,[tags],PASS
4,hnsw,"{'m': 8, 'ef_construct': 400, 'hnsw_ef': 64, '...",PASS
5,quantization,"{'label': 'binary_rescore_2x', 'method': 'bina...",PASS
6,storage,"{'payload_on_disk': True, 'deferred_hnsw': Tru...",PASS


Final collection: ['docs_search_final']; points=4,134; copied in 3.68s.
STOP: final_holdout has not been materialized or executed.


**Stop point.** The final architecture is frozen and `docs_search_final` is built and validated. The following final-holdout section is intentionally separate and must not be used to alter `FINAL_CONFIG`.


## 13. Final Holdout Evaluation


This section is downstream of the architecture freeze and final-store build. It evaluates only the frozen production entry point and never changes `FINAL_CONFIG`. Do not execute it as part of the architecture-selection and collection-build stage.


In [110]:
FINAL_HELDOUT_EVALUATION_SPECS = [
    {"query_id": "f01", "query": "Which two changes improve a large bulk upload before ingestion starts and while data is being sent concurrently?", "query_type": "how-to", "anchors": ["manage-data/bulk-upload/#create-payload-indexes-before-ingesting-data", "manage-data/bulk-upload/#parallelize-across-multiple-threads"]},
    {"query_id": "f02", "query": "Where can I update both the parameters and the schema of a named vector in an existing collection?", "query_type": "api-usage", "anchors": ["manage-data/collections/#update-vector-parameters", "manage-data/collections/#update-vector-schema"]},
    {"query_id": "f03", "query": "Which graph-search algorithm improves filtered ANN traversal by evaluating filter conditions during expansion?", "query_type": "concept", "anchors": ["manage-data/indexing/#the-acorn-search-algorithm"]},
    {"query_id": "f04", "query": "How can sparse IDF statistics be calculated independently for each tenant?", "query_type": "how-to", "anchors": ["manage-data/multitenancy/#per-tenant-idf-statistics"]},
    {"query_id": "f05", "query": "Which payload API returns counts for each distinct value of a field?", "query_type": "api-usage", "anchors": ["manage-data/payload/#facet-counts"]},
    {"query_id": "f06", "query": "How can I update points only when their existing payload satisfies a condition?", "query_type": "api-usage", "anchors": ["manage-data/points/#conditional-updates"]},
    {"query_id": "f07", "query": "How do I turn off quantization for a collection that already has it configured?", "query_type": "how-to", "anchors": ["manage-data/quantization/#disabling-quantization"]},
    {"query_id": "f08", "query": "How do I configure vector storage to use memory-mapped files instead of keeping everything in RAM?", "query_type": "how-to", "anchors": ["manage-data/storage/#configuring-memmap-storage"]},
    {"query_id": "f09", "query": "How can a read request prefer replicas located on the same machine or deployment zone?", "query_type": "concept", "anchors": ["scaling/consistency-guarantees/#read-affinity"]},
    {"query_id": "f10", "query": "Which shard transfer methods can be selected when moving data between cluster peers?", "query_type": "concept", "anchors": ["scaling/distributed_deployment/#shard-transfer-method"]},
    {"query_id": "f11", "query": "How does consensus checkpointing help a failed Qdrant node rejoin the cluster?", "query_type": "troubleshooting", "anchors": ["scaling/node-failure-recovery/#consensus-checkpointing"]},
    {"query_id": "f12", "query": "Which search mode uses positive and negative context pairs to describe a target region?", "query_type": "concept", "anchors": ["search/explore/#context-search"]},
    {"query_id": "f13", "query": "How can I filter geographic points using an arbitrary polygon rather than a rectangle or radius?", "query_type": "api-usage", "anchors": ["search/filtering/#geo-polygon"]},
    {"query_id": "f14", "query": "How should I choose between reciprocal-rank and distribution-based fusion for a hybrid query?", "query_type": "concept", "anchors": ["search/hybrid-queries/#choosing-a-fusion-method"]},
    {"query_id": "f15", "query": "Which search parameter prevents requests from scanning data that has not been indexed yet?", "query_type": "how-to", "anchors": ["search/low-latency-search/#indexed-only-search-parameter"]},
    {"query_id": "f16", "query": "Which relevance method balances similarity with diversity to reduce redundant results?", "query_type": "concept", "anchors": ["search/search-relevance/#maximal-marginal-relevance-mmr"]},
    {"query_id": "f17", "query": "How can several independent similarity searches be submitted in one request?", "query_type": "api-usage", "anchors": ["search/search/#batch-search-api"]},
    {"query_id": "f18", "query": "Where do I tune the BM25 k and document-length normalization parameters?", "query_type": "how-to", "anchors": ["search/text-search/full-text-search/#configuring-bm25-parameters"]},
    {"query_id": "f19", "query": "Where are JSON Web Token validation and claims configured for granular Qdrant access?", "query_type": "how-to", "anchors": ["security/#jwt-configuration"]},
    {"query_id": "f20", "query": "What procedure rotates an administrator API key without leaving the old key active?", "query_type": "how-to", "anchors": ["security/#rotate-an-admin-api-key"]},
]
final_heldout_evaluation_df = materialize_evaluation_specs(
    FINAL_HELDOUT_EVALUATION_SPECS, split="final_holdout"
)
assert set(confirmation_research_df["query_id"]).isdisjoint(
    final_heldout_evaluation_df["query_id"]
)
assert set().union(*confirmation_research_df["expected_urls"]).isdisjoint(
    set().union(*final_heldout_evaluation_df["expected_urls"])
)

# Add the final cohort only after the final retrieval freeze, then rebuild qrels
# from the canonical query splits before any final-holdout evaluation.
all_evaluation_df, qrels_df = append_evaluation_cohort(
    all_evaluation_df, qrels_df, final_heldout_evaluation_df
)
evaluation_consistency_summary_df = synchronize_evaluation_metadata(
    all_evaluation_df,
    qrels_df,
    confirmation_section_disjoint=True,
    final_holdout_materialized=True,
    final_holdout_section_disjoint=True,
)

final_holdout_ids = set(final_heldout_evaluation_df["query_id"])
tuning_routes = {
    "initial_research": evaluation_df,
    "confirmation_diagnostic": heldout_evaluation_df,
    "confirmation_selection": confirmation_research_df,
}
for route_name, route_frame in tuning_routes.items():
    leaked_ids = sorted(final_holdout_ids.intersection(route_frame["query_id"]))
    if leaked_ids:
        raise ValueError(f"final_holdout leaked into {route_name}: {leaked_ids}")

final_heldout_retrieval_runs_df = run_benchmark(
    "retrieval_final_heldout",
    {selected_key: (lambda row: _query_final_points(row.query, payload=["section_url"]))},
    final_heldout_evaluation_df,
)
final_heldout_retrieval_summary_df = summarize_benchmark(
    final_heldout_retrieval_runs_df
)
final_heldout_pass = bool(
    final_heldout_retrieval_summary_df.iloc[0]["recall_at_10"] >= 0.80
)

RUN_MANIFEST["final_heldout"] = {
    "evaluated_pipeline": selected_key,
    "query_ids": final_heldout_evaluation_df["query_id"].tolist(),
    "reselected": False,
    "recall_target": 0.80,
    "pass": final_heldout_pass,
}
display(final_heldout_retrieval_summary_df)
print("Final untouched holdout verdict:", "PASS" if final_heldout_pass else "FAIL")
display(evaluation_consistency_summary_df)
print(
    "Evaluation consistency: PASS | "
    + " | ".join(
        f"{row.split} queries: {row.queries}"
        for row in evaluation_consistency_summary_df.itertuples(index=False)
    )
    + f" | qrels: {len(qrels_df)}"
)


,setting,recall_at_10,mrr,p50_ms,p95_ms
0,hybrid_rrf,1.0,0.704167,101.990022,189.519


Final untouched holdout verdict: PASS


,split,queries,qrels
0,research,45,50
1,confirmation,20,20
2,final_holdout,20,22


Evaluation consistency: PASS | research queries: 45 | confirmation queries: 20 | final_holdout queries: 20 | qrels: 92


## 14. Retrieval Research Conclusions


This section keeps two evaluation records explicit and separate. The immutable historical CPU portability replay retains its observed `m=8, ef_construct=100, hnsw_ef=64` configuration and historical metrics; it is environment-specific portability evidence and is not relabeled or replaced. The current-run final holdout is generated from the `docs_search_final` collection actually built by that execution, using the canonical `retrieval-v1` production configuration (`m=8`, `ef_construct=400`, query-time `hnsw_ef=64`). During the targeted schema restoration the holdout was intentionally not run because restoration was not a tuning round; a subsequent full Run All legitimately rebuilds the canonical collection and evaluates that actual artifact while leaving the historical replay unchanged.


In [111]:
def frame_column(frame: pd.DataFrame, name: str | None) -> pd.Series:
    if name and name in frame.columns:
        return frame[name]
    return pd.Series(np.nan, index=frame.index)


def experiment_rows(
    frame: pd.DataFrame,
    *,
    experiment: str,
    evaluation_stage: str,
    configuration_col: str,
    selected: str,
) -> pd.DataFrame:
    configurations = frame[configuration_col].astype(str)
    return pd.DataFrame(
        {
            "experiment": experiment,
            "evaluation_stage": evaluation_stage,
            "configuration": configurations,
            "decision": np.where(
                configurations.eq(str(selected)), "selected", "candidate"
            ),
            "recall_at_10": frame_column(frame, "recall_at_10"),
            "mrr": frame_column(frame, "mrr"),
            "p50_ms": frame_column(frame, "p50_ms"),
            "p95_ms": frame_column(frame, "p95_ms"),
        }
    )


# All component decisions were frozen in FINAL_CONFIG before final_holdout.
research_conclusions_df = pd.concat(
    [
        experiment_rows(
            finalist_summary_df, experiment="retrieval pipeline", evaluation_stage="initial_research",
            configuration_col="setting", selected=selected_key,
        ),
        experiment_rows(
            heldout_retrieval_summary_df, experiment="retrieval pipeline", evaluation_stage="confirmation_diagnostic",
            configuration_col="setting", selected=selected_key,
        ),
        experiment_rows(
            confirmation_research_summary_df, experiment="retrieval pipeline",
            evaluation_stage="confirmation_selection", configuration_col="setting",
            selected=selected_key,
        ),
        experiment_rows(
            final_heldout_retrieval_summary_df, experiment="retrieval pipeline",
            evaluation_stage="final_holdout", configuration_col="setting",
            selected=selected_key,
        ),
        experiment_rows(
            chunking_ablation_summary_df, experiment="chunking", evaluation_stage="research",
            configuration_col="strategy", selected=FINAL_CHUNKING_STRATEGY,
        ),
        experiment_rows(
            hnsw_benchmark_df, experiment="HNSW", evaluation_stage="research",
            configuration_col="setting", selected=FINAL_HNSW_CONFIG["setting"],
        ),
        experiment_rows(
            filter_benchmark_summary_df, experiment="payload filter", evaluation_stage="research",
            configuration_col="variant", selected=FINAL_PAYLOAD_VARIANT,
        ),
        experiment_rows(
            optimization_benchmark_df, experiment="quantization", evaluation_stage="research",
            configuration_col="label", selected=final_optimization_label,
        ),
        experiment_rows(
            heldout_optimization_summary_df, experiment="quantization", evaluation_stage="confirmation_diagnostic",
            configuration_col="setting", selected=final_optimization_label,
        ),
    ],
    ignore_index=True,
)
research_conclusions_df = research_conclusions_df.sort_values(
    ["experiment", "evaluation_stage", "decision", "recall_at_10", "mrr", "p95_ms", "p50_ms"],
    ascending=[True, True, False, False, False, True, True],
).reset_index(drop=True)
remember_result("research_conclusions", research_conclusions_df)


selected_research_row = confirmation_research_summary_df.set_index("setting").loc[
    selected_key
]

# Immutable-style snapshot of the historical CPU portability replay.
# It never reads current metrics, current winners or live collection metadata.
CPU_PORTABILITY_REPLAY_RECORD = {
    "purpose": "CPU portability validation",
    "status": "historical replay evidence",
    "production_authority": False,
    "evaluated_before_canonical_collection_restore": True,
    "config": {"m": 8, "ef_construct": 100, "hnsw_ef": 64},
    "metrics": {
        "recall_at_10": 1.0,
        "mrr": 0.7291666667,
        "p50_ms": 108.6274744994,
        "p95_ms": 198.0850929998,
    },
}
CANONICAL_RETRIEVAL_V1 = {
    "tag": "retrieval-v1",
    "hnsw": FROZEN_RETRIEVAL_V1_HNSW_CONFIG.copy(),
    "authoritative_for_production": True,
}
# Current-run values follow the collection actually built by this execution.
current_run_metrics_df = final_heldout_retrieval_summary_df.loc[
    final_heldout_retrieval_summary_df["setting"].eq(
        FINAL_CONFIG["retrieval"]["pipeline"]
    ),
    ["setting", "recall_at_10", "mrr", "p50_ms", "p95_ms"],
].copy()
if len(current_run_metrics_df) != 1:
    raise ValueError(
        "Expected exactly one current-run holdout row for the frozen pipeline."
    )
current_run_holdout_manifest = RUN_MANIFEST.get("final_heldout", {})
if (
    current_run_holdout_manifest.get("evaluated_pipeline")
    != FINAL_CONFIG["retrieval"]["pipeline"]
    or current_run_holdout_manifest.get("reselected") is not False
):
    raise ValueError("Current-run holdout does not match the frozen pipeline.")
current_run_row = current_run_metrics_df.iloc[0]
current_run_results_df = current_run_metrics_df.rename(
    columns={
        "setting": "Configuration",
        "recall_at_10": "Recall@10",
        "mrr": "MRR",
        "p50_ms": "P50 ms",
        "p95_ms": "P95 ms",
    }
)
display(current_run_results_df.round(4))


def qdrant_model_metadata(value):
    return None if value is None else value.model_dump(mode="json", exclude_none=True)


def capture_current_run_collection_metadata(config: dict) -> dict:
    # Current-run metadata must match the canonical collection built by this execution.
    validate_final_collection(config)
    collection_name = config["collection_name"]
    info = qdrant_client.get_collection(collection_name)
    exact_point_count = qdrant_client.count(collection_name, exact=True).count
    dense_vectors = info.config.params.vectors
    sparse_vectors = info.config.params.sparse_vectors or {}
    if not isinstance(dense_vectors, dict):
        raise ValueError("Final collection must use named vectors.")
    return {
        "collection_name": collection_name,
        "exact_point_count": int(exact_point_count),
        "status": info.status.value,
        "vector_names": sorted([*dense_vectors, *sparse_vectors]),
        "vector_dimensions": {
            **{name: int(params.size) for name, params in dense_vectors.items()},
            **{name: None for name in sparse_vectors},
        },
        "vector_schema": {
            "dense_or_multivector": {
                name: qdrant_model_metadata(params)
                for name, params in dense_vectors.items()
            },
            "sparse": {
                name: qdrant_model_metadata(params)
                for name, params in sparse_vectors.items()
            },
        },
        "hnsw_config": qdrant_model_metadata(info.config.hnsw_config),
        "vector_hnsw_config": {
            name: qdrant_model_metadata(params.hnsw_config)
            for name, params in dense_vectors.items()
        },
        "quantization_config": {
            "collection": qdrant_model_metadata(info.config.quantization_config),
            "vectors": {
                name: qdrant_model_metadata(params.quantization_config)
                for name, params in dense_vectors.items()
            },
        },
        "payload_indexes": {
            field_name: {
                "data_type": schema.data_type.value,
                "params": qdrant_model_metadata(schema.params),
            }
            for field_name, schema in info.payload_schema.items()
        },
        "storage": {
            "on_disk_payload": bool(info.config.params.on_disk_payload),
            "optimizer_indexing_threshold_kb": int(
                info.config.optimizer_config.indexing_threshold
            ),
        },
        "validated_against_current_run_config": True,
    }


# Revalidate the canonical query/qrels state immediately before audit capture.
final_consistency_summary_df = validate_evaluation_consistency(
    all_evaluation_df, qrels_df
)
current_run_collection_metadata = capture_current_run_collection_metadata(
    FINAL_CONFIG
)
current_run_metrics_record = frame_records(current_run_metrics_df)[0]
dataset_split_manifest = {
    "summary": frame_records(final_consistency_summary_df),
    "query_ids_by_split": {
        split: sorted(
            all_evaluation_df.loc[
                all_evaluation_df["split"].eq(split), "query_id"
            ].astype(str)
        )
        for split in EVALUATION_SPLIT_ORDER
    },
}
qrels_metadata = {
    "total": int(len(qrels_df)),
    "counts_by_split": {
        row.split: int(row.qrels)
        for row in final_consistency_summary_df.itertuples(index=False)
    },
    "records_key": "qrels",
}
model_versions = {
    "dense": {
        "enabled": FINAL_CONFIG["representations"]["dense"],
        "model": FINAL_CONFIG["dense"]["model"],
    },
    "sparse": {
        "enabled": FINAL_CONFIG["representations"]["sparse"],
        "model": FINAL_CONFIG["sparse"]["model"],
    },
    "colbert": {
        "enabled": FINAL_CONFIG["representations"]["colbert"],
        "model": FINAL_CONFIG["colbert"]["model"],
    },
}
qdrant_versions = {
    "server": qdrant_client.info().version,
    "client": version("qdrant-client"),
}
reproducibility_metadata = {
    "python": RUN_MANIFEST["python"],
    "packages": {
        **RUN_MANIFEST["packages"],
        "transformers": RUN_MANIFEST["transformers"],
        "torch": RUN_MANIFEST["torch"],
    },
}

RUN_MANIFEST["historical_cpu_portability_replay"] = (
    CPU_PORTABILITY_REPLAY_RECORD
)
RUN_MANIFEST["current_run"] = {
    "config": FINAL_CONFIG,
    "metrics": current_run_metrics_record,
    "collection_metadata": current_run_collection_metadata,
}
RUN_MANIFEST["canonical_retrieval_v1"] = CANONICAL_RETRIEVAL_V1
RUN_MANIFEST["retrieval_phase_status"] = {
    "retrieval_architecture_frozen": True,
    "historical_cpu_replay_preserved": True,
    "current_run_recorded_separately": True,
    "canonical_retrieval_v1_recorded_separately": True,
    "cpu_replay_supersedes_production": False,
    "retrieval_phase_complete": True,
}

selection_pipeline_rows = confirmation_research_summary_df.set_index("setting")
hybrid_rrf_row = selection_pipeline_rows.loc["hybrid_rrf"]
rrf_colbert_row = selection_pipeline_rows.loc["rrf_colbert"]
reranker_decision = (
    "ColBERT MaxSim was evaluated with the same 50-candidate pool, but "
    "`rrf_colbert` ranked below `hybrid_rrf` without reranking under the frozen "
    "Recall@10 → MRR → P95 → P50 rule: "
    f"hybrid_rrf=({hybrid_rrf_row['recall_at_10']:.3f}, {hybrid_rrf_row['mrr']:.3f}, "
    f"{hybrid_rrf_row['p50_ms']:.2f} ms, {hybrid_rrf_row['p95_ms']:.2f} ms) vs "
    f"rrf_colbert=({rrf_colbert_row['recall_at_10']:.3f}, {rrf_colbert_row['mrr']:.3f}, "
    f"{rrf_colbert_row['p50_ms']:.2f} ms, {rrf_colbert_row['p95_ms']:.2f} ms). "
    "It was therefore excluded from the final serving pipeline."
)

current_run_verdict = "PASS" if final_heldout_pass else "FAIL"
display(
    Markdown(
        f"""
### Canonical production, historical replay and current run

1. **Canonical production architecture (`retrieval-v1`):** structure-aware chunks; dense `{FINAL_CONFIG['dense']['model']}` + SPLADE sparse retrieval; `{FINAL_CONFIG['retrieval']['fusion'].upper()}` fusion; ColBERT {'enabled' if FINAL_CONFIG['representations']['colbert'] else 'disabled'}; HNSW m={CANONICAL_RETRIEVAL_V1['hnsw']['m']}, ef_construct={CANONICAL_RETRIEVAL_V1['hnsw']['ef_construct']}, hnsw_ef={CANONICAL_RETRIEVAL_V1['hnsw']['hnsw_ef']}; `{FINAL_CONFIG['quantization']['label']}`; payload index on `{', '.join(FINAL_CONFIG['payload_indexes'])}`.
2. **Why selected:** `{selected_key}` ranked first under the predeclared Recall@10 → MRR → P95 → P50 rule on research + confirmation (Recall@10={selected_research_row['recall_at_10']:.3f}, MRR={selected_research_row['mrr']:.3f}, P50={selected_research_row['p50_ms']:.2f} ms, P95={selected_research_row['p95_ms']:.2f} ms). The architecture was frozen before final holdout.
3. **Reranker decision:** {reranker_decision}
4. **Historical CPU portability replay (`ef_construct=100`):** Recall@10={CPU_PORTABILITY_REPLAY_RECORD['metrics']['recall_at_10']:.3f}, MRR={CPU_PORTABILITY_REPLAY_RECORD['metrics']['mrr']:.3f}, P50={CPU_PORTABILITY_REPLAY_RECORD['metrics']['p50_ms']:.2f} ms, P95={CPU_PORTABILITY_REPLAY_RECORD['metrics']['p95_ms']:.2f} ms. This immutable record does not evaluate or supersede canonical `retrieval-v1`.
5. **Current run evaluation (`ef_construct={FINAL_CONFIG['hnsw']['ef_construct']}`):** Recall@10={current_run_row['recall_at_10']:.3f}, MRR={current_run_row['mrr']:.3f}, P50={current_run_row['p50_ms']:.2f} ms, P95={current_run_row['p95_ms']:.2f} ms; verdict={current_run_verdict}.
"""
    )
)


def course_top_three(query: str) -> list[dict]:
    rows, seen = [], set()
    for point in _query_final_points(query, payload=SEARCH_PAYLOAD):
        payload = point.payload or {}
        section_url = payload.get("section_url")
        if not section_url or section_url in seen:
            continue
        seen.add(section_url)
        rows.append(
            {
                "section_title": payload.get("section_title") or "Untitled section",
                "section_url": section_url,
                "score": float(point.score),
            }
        )
        if len(rows) == 3:
            break
    return rows


course_example_queries = confirmation_research_df.head(2)["query"].tolist()
course_example_blocks = []
for query_number, query in enumerate(course_example_queries, start=1):
    result_lines = [f"{query_number}) **{query}**  ", "Top 3:"]
    result_lines.extend(
        f"   {rank}) {item['section_title']} → {item['section_url']} → {item['score']:.4f}"
        for rank, item in enumerate(course_top_three(query), start=1)
    )
    course_example_blocks.append("\n".join(result_lines))
course_examples_markdown = "\n\n".join(course_example_blocks)
query_type_counts = confirmation_research_df["query_type"].value_counts().to_dict()
reranker_setting = (
    f"ColBERT (MaxSim), top-k={RERANK_CANDIDATE_LIMIT}"
    if selected_key.endswith("colbert")
    else (
        f"ColBERT (MaxSim), top-k={RERANK_CANDIDATE_LIMIT} — tested, not selected; "
        f"rrf_colbert ranked below hybrid_rrf without reranking under "
        f"Recall@10 → MRR → P95 → P50: "
        f"{rrf_colbert_row['recall_at_10']:.3f}/{rrf_colbert_row['mrr']:.3f}/"
        f"{rrf_colbert_row['p50_ms']:.2f}/{rrf_colbert_row['p95_ms']:.2f} ms vs "
        f"{hybrid_rrf_row['recall_at_10']:.3f}/{hybrid_rrf_row['mrr']:.3f}/"
        f"{hybrid_rrf_row['p50_ms']:.2f}/{hybrid_rrf_row['p95_ms']:.2f} ms"
    )
)
surprise = (
    "The restricted confirmation cycle selected hybrid fusion without ColBERT reranking."
    if not selected_key.endswith("colbert")
    else "ColBERT retained its quality advantage after the restricted confirmation cycle."
)

course_result_markdown = f"""
**[Day 6] Final Project: Production-Ready Documentation Search Engine**

**High-Level Summary**
- **Domain:** "Documentation search for Qdrant"
- **Current Run Result:** "Hybrid RRF reached Recall@10={current_run_row['recall_at_10']:.3f} with P95={current_run_row['p95_ms']:.2f} ms using the collection built by this execution."

**Reproducibility**
- **Notebook/App:** <link>
- **Repo (optional):** https://github.com/artyomboyko/Qdrant_Final_Project/
- **Models:** dense={DENSE_MODEL_NAME}, sparse={SPARSE_MODEL_NAME}, colbert={COLBERT_MODEL_NAME} (tested, not selected)
- **Current-run collection:** {FINAL_CONFIG['collection_name']} (Cosine), points={final_point_count}
- **Dataset:** {len(clean_sections_df)} sections from Qdrant documentation (snapshot: YYYY-MM-DD)
- **Ground truth:** {len(all_evaluation_df)} queries (how-to / concept / api / troubleshooting)

**Current Run Settings**
- **Chunking:** structure-aware section chunks (oversized sections split to the token budget)
- **Payload fields:** page_title, section_title, section_url, breadcrumbs, tags, prev_section_text, next_section_text
- **Fusion:** {FINAL_CONFIG['retrieval']['fusion'].upper()}, k_dense={FINAL_CONFIG['retrieval']['prefetch_limit']}, k_sparse={FINAL_CONFIG['retrieval']['prefetch_limit']}
- **Reranker:** {reranker_setting}
- **Index/Search params:** hnsw_ef={FINAL_CONFIG['hnsw']['hnsw_ef']}, m={FINAL_CONFIG['hnsw']['m']}, ef_construct={FINAL_CONFIG['hnsw']['ef_construct']}

**Canonical Production Settings (`retrieval-v1`)**
- **Index/Search params:** hnsw_ef={CANONICAL_RETRIEVAL_V1['hnsw']['hnsw_ef']}, m={CANONICAL_RETRIEVAL_V1['hnsw']['m']}, ef_construct={CANONICAL_RETRIEVAL_V1['hnsw']['ef_construct']}
- **Authority:** production source of truth; the CPU replay does not supersede it

**Historical CPU Portability Replay**
- **Index/Search params:** hnsw_ef={CPU_PORTABILITY_REPLAY_RECORD['config']['hnsw_ef']}, m={CPU_PORTABILITY_REPLAY_RECORD['config']['m']}, ef_construct={CPU_PORTABILITY_REPLAY_RECORD['config']['ef_construct']}
- **Evaluation:** Recall@10={CPU_PORTABILITY_REPLAY_RECORD['metrics']['recall_at_10']:.3f} | MRR={CPU_PORTABILITY_REPLAY_RECORD['metrics']['mrr']:.3f} | P50={CPU_PORTABILITY_REPLAY_RECORD['metrics']['p50_ms']:.2f} ms | P95={CPU_PORTABILITY_REPLAY_RECORD['metrics']['p95_ms']:.2f} ms
- **Authority:** historical portability evidence only

**Queries (examples)**
{course_examples_markdown}

**Current Run Evaluation**
- Recall@10: {current_run_row['recall_at_10']:.3f} | MRR: {current_run_row['mrr']:.3f} | P50: {current_run_row['p50_ms']:.2f} ms | P95: {current_run_row['p95_ms']:.2f} ms

**Why these matched**
- Dense retrieval captured semantic intent, sparse retrieval preserved exact technical terms, and RRF combined both rankings.

**Surprise**
- "{surprise}"

**Next step**
- "Continue with the separate portfolio/RAG extension while keeping retrieval frozen."
"""
print(course_result_markdown.strip())

# Keep canonical, historical and current-run evidence in separate branches.
retrieval_audit_payload = decode_notebook_audit_bundle(
    make_notebook_audit_bundle()
)
for ambiguous_key in ("final_config", "final_metrics", "collection_metadata"):
    retrieval_audit_payload.pop(ambiguous_key, None)
retrieval_audit_payload.update(
    {
        "canonical_retrieval_v1": CANONICAL_RETRIEVAL_V1,
        "historical_cpu_portability_replay": CPU_PORTABILITY_REPLAY_RECORD,
        "current_run": {
            "config": FINAL_CONFIG,
            "metrics": current_run_metrics_record,
            "collection_metadata": current_run_collection_metadata,
        },
        "dataset_split_manifest": dataset_split_manifest,
        "qrels_metadata": qrels_metadata,
        "model_versions": model_versions,
        "qdrant_versions": qdrant_versions,
        "source_commit": QDRANT_DOCS_COMMIT,
        "reproducibility_metadata": reproducibility_metadata,
        "manifest": RUN_MANIFEST,
    }
)
serialized_retrieval_audit = json.dumps(
    retrieval_audit_payload, separators=(",", ":")
).encode("utf-8")
retrieval_audit_bundle = {
    "encoding": "gzip+base64/json",
    "scope": "retrieval-audit-canonical-history-current-v1",
    "raw_runs_embedded": False,
    "uncompressed_bytes": len(serialized_retrieval_audit),
    "payload": base64.b64encode(
        gzip.compress(serialized_retrieval_audit, mtime=0)
    ).decode("ascii"),
}
display({"application/json": retrieval_audit_bundle}, raw=True)
display(
    Markdown(
        """
### Retrieval audit separation status

- canonical `retrieval-v1` recorded independently — **PASS**
- historical CPU portability replay is immutable-style evidence — **PASS**
- current-run evaluation is recorded separately — **PASS**
- neither experiment nor history can supersede production configuration — **PASS**
"""
    )
)


,Configuration,Recall@10,MRR,P50 ms,P95 ms
0,hybrid_rrf,1.0,0.7042,101.99,189.519



### Canonical production, historical replay and current run

1. **Canonical production architecture (`retrieval-v1`):** structure-aware chunks; dense `BAAI/bge-small-en-v1.5` + SPLADE sparse retrieval; `RRF` fusion; ColBERT disabled; HNSW m=8, ef_construct=400, hnsw_ef=64; `binary_rescore_2x`; payload index on `tags`.
2. **Why selected:** `hybrid_rrf` ranked first under the predeclared Recall@10 → MRR → P95 → P50 rule on research + confirmation (Recall@10=0.887, MRR=0.652, P50=166.94 ms, P95=212.16 ms). The architecture was frozen before final holdout.
3. **Reranker decision:** ColBERT MaxSim was evaluated with the same 50-candidate pool, but `rrf_colbert` ranked below `hybrid_rrf` without reranking under the frozen Recall@10 → MRR → P95 → P50 rule: hybrid_rrf=(0.887, 0.652, 166.94 ms, 212.16 ms) vs rrf_colbert=(0.872, 0.646, 289.40 ms, 400.73 ms). It was therefore excluded from the final serving pipeline.
4. **Historical CPU portability replay (`ef_construct=100`):** Recall@10=1.000, MRR=0.729, P50=108.63 ms, P95=198.09 ms. This immutable record does not evaluate or supersede canonical `retrieval-v1`.
5. **Current run evaluation (`ef_construct=400`):** Recall@10=1.000, MRR=0.704, P50=101.99 ms, P95=189.52 ms; verdict=PASS.


**[Day 6] Final Project: Production-Ready Documentation Search Engine**

**High-Level Summary**
- **Domain:** "Documentation search for Qdrant"
- **Current Run Result:** "Hybrid RRF reached Recall@10=1.000 with P95=189.52 ms using the collection built by this execution."

**Reproducibility**
- **Notebook/App:** <link>
- **Repo (optional):** https://github.com/artyomboyko/Qdrant_Final_Project/
- **Models:** dense=BAAI/bge-small-en-v1.5, sparse=prithivida/Splade_PP_en_v1, colbert=colbert-ir/colbertv2.0 (tested, not selected)
- **Current-run collection:** docs_search_final (Cosine), points=4134
- **Dataset:** 2735 sections from Qdrant documentation (snapshot: YYYY-MM-DD)
- **Ground truth:** 85 queries (how-to / concept / api / troubleshooting)

**Current Run Settings**
- **Chunking:** structure-aware section chunks (oversized sections split to the token budget)
- **Payload fields:** page_title, section_title, section_url, breadcrumbs, tags, prev_section_text, next_section_text
- **Fusion:


### Retrieval audit separation status

- canonical `retrieval-v1` recorded independently — **PASS**
- historical CPU portability replay is immutable-style evidence — **PASS**
- current-run evaluation is recorded separately — **PASS**
- neither experiment nor history can supersede production configuration — **PASS**


The retrieval decision remains frozen by the historical `retrieval-v1` baseline. The historical CPU replay holdout is immutable portability evidence collected with `ef_construct=100`, not a replacement for evaluation of the canonical `ef_construct=400` production collection. The current-run holdout is generated dynamically from the canonical final collection built by that execution and validates the frozen decision without reopening tuning. Section 15 deploys and validates matching local and Cloud artifacts against the canonical `retrieval-v1` schema; Sections 16–19 remain a separate portfolio/RAG extension and do not alter the retrieval decision.


## 15. Qdrant Cloud Deployment


This operational stage copies the frozen local `docs_search_final` artifact to Qdrant Cloud without recomputing embeddings or changing `FINAL_CONFIG`. Both local and Cloud runtime collections are created from the canonical `retrieval-v1` configuration (`m=8`, `ef_construct=400`, query-time `hnsw_ef=64`), never from a transient timing-dependent experimental HNSW winner. The local collection remains the reproducible production source of truth; the validated Cloud collection becomes the runtime backend for the later portfolio extension. Section 15 outputs reflect the actual execution of its deployment cells, while local and Cloud schema validation requires both production collections to match the canonical final configuration.


### 15.1 Cloud Connection


Cloud credentials are loaded only from the ignored project `.env` file. The API key is kept in kernel memory and is never printed. A separate gRPC-preferred client remains available to downstream notebook sections.


In [112]:
load_dotenv(PROJECT_ROOT / ".env", override=False)

QDRANT_CLOUD_URL = os.getenv("QDRANT_CLOUD_URL")
QDRANT_CLOUD_API_KEY = os.getenv("QDRANT_CLOUD_API_KEY")
missing_cloud_settings = [
    name
    for name, value in {
        "QDRANT_CLOUD_URL": QDRANT_CLOUD_URL,
        "QDRANT_CLOUD_API_KEY": QDRANT_CLOUD_API_KEY,
    }.items()
    if not value
]
if missing_cloud_settings:
    raise EnvironmentError(
        "Missing required Cloud settings in .env: "
        + ", ".join(missing_cloud_settings)
    )
if not QDRANT_CLOUD_URL.lower().startswith("https://"):
    raise ValueError("QDRANT_CLOUD_URL must use HTTPS.")

cloud_client = QdrantClient(
    url=QDRANT_CLOUD_URL,
    api_key=QDRANT_CLOUD_API_KEY,
    prefer_grpc=True,
    timeout=QDRANT_TIMEOUT_SECONDS,
)

cloud_collections = cloud_client.get_collections().collections

print(
    f"Qdrant Cloud connection: OK; server={cloud_client.info().version}; "
    f"collections={len(cloud_collections)}"
)


Qdrant Cloud connection: OK; server=1.19.1; collections=1


### 15.2 Recreate Final Collection in Cloud


The current Docker volume is backed by a Windows bind mount. A verified local snapshot was found to contain a zero-filled `wal/first-index`, and Qdrant Cloud therefore rejected it during restore. To keep deployment reproducible, this step recreates the Cloud schema from the live frozen local collection; the next step streams the already stored vectors and payloads without recomputing embeddings. No retrieval setting or `FINAL_CONFIG` value is changed.


In [113]:
CLOUD_COLLECTION_NAME = str(FINAL_CONFIG["collection_name"])
CLOUD_SCROLL_BATCH_SIZE = 256
CLOUD_UPLOAD_BATCH_SIZE = 128
CLOUD_UPLOAD_PARALLEL = min(MAX_WORKERS, 4)
CLOUD_TRANSFER_TIMEOUT_SECONDS = 3600
CLOUD_RECREATE_COLLECTION = True
CLOUD_DEPLOYMENT_METHOD = "point_streaming"
CLOUD_DEPLOYMENT_NOTE = (
    "Collection snapshots from the current Windows bind-backed local storage "
    "contain an invalid WAL first-index file, so the verified stored points "
    "are streamed without recomputing embeddings."
)
CLOUD_CONFIG_ADAPTATIONS = []

previous_transfer_client = globals().get("cloud_transfer_client")
if previous_transfer_client is not None:
    try:
        previous_transfer_client.close()
    except Exception:
        pass
cloud_transfer_client = QdrantClient(
    url=QDRANT_CLOUD_URL,
    api_key=QDRANT_CLOUD_API_KEY,
    prefer_grpc=False,
    timeout=CLOUD_TRANSFER_TIMEOUT_SECONDS,
)

# Refuse deployment unless the local artifact matches frozen retrieval-v1.
validate_final_collection(FINAL_CONFIG)
local_final_info = qdrant_client.get_collection(FINAL_CONFIG["collection_name"])
local_final_point_count = int(
    qdrant_client.count(FINAL_CONFIG["collection_name"], exact=True).count
)
if local_final_info.status != models.CollectionStatus.GREEN:
    raise RuntimeError(
        f"Local final collection is not GREEN: {local_final_info.status}"
    )
if local_final_point_count <= 0:
    raise RuntimeError("Local final collection is empty; deployment stopped.")

local_dense_vectors = local_final_info.config.params.vectors
local_sparse_vectors = local_final_info.config.params.sparse_vectors or {}
if not isinstance(local_dense_vectors, dict):
    raise ValueError("The local final collection must use named vectors.")

expected_dense_names = {
    FINAL_CONFIG["vector_names"][kind]
    for kind in ("dense", "colbert")
    if FINAL_CONFIG["representations"][kind]
}
expected_sparse_names = (
    {FINAL_CONFIG["vector_names"]["sparse"]}
    if FINAL_CONFIG["representations"]["sparse"]
    else set()
)
if set(local_dense_vectors) != expected_dense_names:
    raise ValueError(
        "Local dense/multivector schema disagrees with FINAL_CONFIG: "
        f"{sorted(local_dense_vectors)} != {sorted(expected_dense_names)}"
    )
if set(local_sparse_vectors) != expected_sparse_names:
    raise ValueError(
        "Local sparse schema disagrees with FINAL_CONFIG: "
        f"{sorted(local_sparse_vectors)} != {sorted(expected_sparse_names)}"
    )

cloud_vectors_config = {
    name: params.model_copy(deep=True)
    for name, params in local_dense_vectors.items()
}
cloud_sparse_vectors_config = {
    name: params.model_copy(deep=True)
    for name, params in local_sparse_vectors.items()
}
cloud_hnsw_config = models.HnswConfigDiff.model_validate(
    local_final_info.config.hnsw_config.model_dump(
        mode="python", exclude_none=True
    )
)

if cloud_transfer_client.collection_exists(CLOUD_COLLECTION_NAME):
    if not CLOUD_RECREATE_COLLECTION:
        raise RuntimeError(
            f"Cloud collection {CLOUD_COLLECTION_NAME!r} already exists."
        )
    cloud_transfer_client.delete_collection(CLOUD_COLLECTION_NAME)
if cloud_transfer_client.collection_exists(CLOUD_COLLECTION_NAME):
    raise RuntimeError(
        f"Cloud collection {CLOUD_COLLECTION_NAME!r} could not be removed."
    )

cloud_transfer_client.create_collection(
    collection_name=CLOUD_COLLECTION_NAME,
    vectors_config=cloud_vectors_config,
    sparse_vectors_config=cloud_sparse_vectors_config or None,
    on_disk_payload=bool(local_final_info.config.params.on_disk_payload),
    hnsw_config=cloud_hnsw_config,
    optimizers_config=models.OptimizersConfigDiff(indexing_threshold=0),
)
for field_name, schema in local_final_info.payload_schema.items():
    field_schema = (
        schema.params.model_copy(deep=True)
        if schema.params is not None
        else schema.data_type
    )
    cloud_transfer_client.create_payload_index(
        collection_name=CLOUD_COLLECTION_NAME,
        field_name=field_name,
        field_schema=field_schema,
        wait=True,
    )

print(
    f"Cloud collection recreated: {CLOUD_COLLECTION_NAME}; "
    f"dense/multivectors={sorted(cloud_vectors_config)}; "
    f"sparse={sorted(cloud_sparse_vectors_config)}; "
    f"payload indexes={sorted(local_final_info.payload_schema)}.\n"
    f"Deployment method: {CLOUD_DEPLOYMENT_METHOD}. {CLOUD_DEPLOYMENT_NOTE}"
)


Cloud collection recreated: docs_search_final; dense/multivectors=['dense']; sparse=['sparse']; payload indexes=['tags'].
Deployment method: point_streaming. Collection snapshots from the current Windows bind-backed local storage contain an invalid WAL first-index file, so the verified stored points are streamed without recomputing embeddings.


### 15.3 Upload Final Points


The corrupted-WAL snapshot path is not retried. Points are read from the frozen local `docs_search_final` with paginated `scroll` and uploaded through the Qdrant client's bounded parallel `upload_points` pipeline. Dense and sparse vectors, point IDs and payloads are copied exactly as stored; embeddings are not recomputed. Cloud builds its own HNSW/quantization artifacts from the selected final configuration, after which Section 15.4 validates the resulting collection.


In [114]:
def iter_local_final_points_for_cloud(progress):
    expected_vector_names = set(selected_final_vector_names(FINAL_CONFIG))
    offset = None
    transferred = 0
    while True:
        records, next_offset = qdrant_client.scroll(
            collection_name=FINAL_CONFIG["collection_name"],
            limit=CLOUD_SCROLL_BATCH_SIZE,
            offset=offset,
            with_payload=True,
            with_vectors=sorted(expected_vector_names),
        )
        if not records and next_offset is not None:
            raise RuntimeError("Local scroll returned an empty non-terminal page.")
        for record in records:
            if not isinstance(record.vector, dict):
                raise ValueError(f"Expected named vectors for point {record.id}.")
            actual_vector_names = set(record.vector)
            if actual_vector_names != expected_vector_names:
                raise ValueError(
                    f"Point {record.id} vector schema mismatch: "
                    f"{sorted(actual_vector_names)} != {sorted(expected_vector_names)}"
                )
            yield models.PointStruct(
                id=record.id,
                vector={
                    name: record.vector[name]
                    for name in sorted(expected_vector_names)
                },
                payload=dict(record.payload or {}),
            )
            transferred += 1
            progress.update(1)
        if next_offset is None:
            break
        offset = next_offset
    if transferred != local_final_point_count:
        raise ValueError(
            f"Local scroll transferred {transferred} points; "
            f"expected {local_final_point_count}."
        )


cloud_upload_started = time.perf_counter()
try:
    with tqdm(
        total=local_final_point_count,
        desc="Uploading stored points to Qdrant Cloud",
        unit="point",
    ) as cloud_upload_progress:
        cloud_transfer_client.upload_points(
            collection_name=CLOUD_COLLECTION_NAME,
            points=iter_local_final_points_for_cloud(cloud_upload_progress),
            batch_size=CLOUD_UPLOAD_BATCH_SIZE,
            parallel=CLOUD_UPLOAD_PARALLEL,
            max_retries=3,
            wait=True,
        )
    cloud_transfer_client.update_collection(
        collection_name=CLOUD_COLLECTION_NAME,
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=int(
                local_final_info.config.optimizer_config.indexing_threshold
            )
        ),
    )
    cloud_upload_seconds = time.perf_counter() - cloud_upload_started
finally:
    cloud_transfer_client.close()

print(
    f"Cloud upload completed: {local_final_point_count:,} points in "
    f"{cloud_upload_seconds:.2f}s; batch={CLOUD_UPLOAD_BATCH_SIZE}; "
    f"parallel={CLOUD_UPLOAD_PARALLEL}."
)


Uploading stored points to Qdrant Cloud: 100%|██████████| 4134/4134 [00:08<00:00, 501.14point/s] 


Cloud upload completed: 4,134 points in 8.62s; batch=128; parallel=4.


### 15.4 Validate Cloud Collection


Validation waits for Cloud indexing to settle, then compares exact point counts and a canonical deployment signature covering vector names and dimensions, sparse vectors, frozen `retrieval-v1` HNSW, quantization, memory/storage settings and payload indexes. Approximate counters are not used as the final count; `hnsw_ef=64` remains a query-time parameter in `FINAL_CONFIG`, not collection metadata.


In [115]:
def wait_for_backend_collection(
    client: QdrantClient,
    collection_name: str,
    *,
    require_hnsw: bool,
    timeout_seconds: float = 600,
    poll_interval_seconds: float = 1.0,
):
    deadline = time.monotonic() + timeout_seconds
    previous_elapsed = 0.0
    last_state = None
    with tqdm(
        total=timeout_seconds,
        desc="Waiting for Cloud collection",
        unit="s",
        leave=False,
    ) as progress:
        while time.monotonic() < deadline:
            info = client.get_collection(collection_name)
            points = int(info.points_count or 0)
            indexed = int(info.indexed_vectors_count or 0)
            last_state = (str(info.status), str(info.optimizer_status), points, indexed)
            ready = (
                info.status == models.CollectionStatus.GREEN
                and str(info.optimizer_status).lower() == "ok"
            )
            if require_hnsw:
                ready = ready and points > 0 and indexed >= 0.95 * points
            if ready:
                return info
            time.sleep(poll_interval_seconds)
            elapsed = timeout_seconds - max(0.0, deadline - time.monotonic())
            progress.update(max(0.0, elapsed - previous_elapsed))
            previous_elapsed = elapsed
    raise TimeoutError(
        f"Cloud collection did not become ready: {collection_name}; "
        f"last state={last_state}"
    )


def deployment_model_dump(value):
    if value is None:
        return None
    return value.model_dump(mode="json", exclude_none=True)


def deployment_collection_signature(info) -> dict:
    dense_vectors = info.config.params.vectors
    sparse_vectors = info.config.params.sparse_vectors or {}
    if not isinstance(dense_vectors, dict):
        raise ValueError("Deployment validation requires named vectors.")
    return {
        "vectors": {
            name: {
                "size": int(params.size),
                "distance": params.distance.value,
                "memory": (
                    params.memory.value if params.memory is not None else None
                ),
                "hnsw_config": deployment_model_dump(params.hnsw_config),
                "quantization_config": deployment_model_dump(
                    params.quantization_config
                ),
                "multivector_config": deployment_model_dump(
                    params.multivector_config
                ),
            }
            for name, params in sorted(dense_vectors.items())
        },
        "sparse_vectors": {
            name: deployment_model_dump(params)
            for name, params in sorted(sparse_vectors.items())
        },
        "collection_hnsw": deployment_model_dump(info.config.hnsw_config),
        "collection_quantization": deployment_model_dump(
            info.config.quantization_config
        ),
        "on_disk_payload": bool(info.config.params.on_disk_payload),
        "optimizer_indexing_threshold_kb": int(
            info.config.optimizer_config.indexing_threshold
        ),
        "payload_indexes": {
            name: {
                "data_type": schema.data_type.value,
                "params": deployment_model_dump(schema.params),
            }
            for name, schema in sorted(info.payload_schema.items())
        },
    }


cloud_final_info = wait_for_backend_collection(
    cloud_client,
    CLOUD_COLLECTION_NAME,
    require_hnsw=FINAL_CONFIG["representations"]["dense"],
)
cloud_final_point_count = int(
    cloud_client.count(CLOUD_COLLECTION_NAME, exact=True).count
)
if cloud_final_info.status != models.CollectionStatus.GREEN:
    raise AssertionError(f"Cloud status is not GREEN: {cloud_final_info.status}")
if cloud_final_point_count != local_final_point_count:
    raise AssertionError(
        f"Cloud/local exact count mismatch: "
        f"{cloud_final_point_count} != {local_final_point_count}"
    )

local_deployment_signature = deployment_collection_signature(local_final_info)
cloud_deployment_signature = deployment_collection_signature(cloud_final_info)
if cloud_deployment_signature != local_deployment_signature:
    raise AssertionError(
        "Cloud schema/config differs from the local frozen artifact. "
        "No silent adaptation is allowed.\nlocal="
        + json.dumps(local_deployment_signature, indent=2, sort_keys=True)
        + "\ncloud="
        + json.dumps(cloud_deployment_signature, indent=2, sort_keys=True)
    )

cloud_validation_df = pd.DataFrame(
    [
        {"check": "collection status GREEN", "passed": True},
        {"check": "exact point count", "passed": True},
        {"check": "dense/multivector schema", "passed": True},
        {"check": "sparse vector schema", "passed": True},
        {"check": "HNSW configuration", "passed": True},
        {"check": "quantization configuration", "passed": True},
        {"check": "memory/storage settings", "passed": True},
        {"check": "payload indexes", "passed": True},
    ]
)
CLOUD_DEPLOYMENT_MANIFEST = {
    "collection_name": CLOUD_COLLECTION_NAME,
    "source_collection": FINAL_CONFIG["collection_name"],
    "transfer_method": CLOUD_DEPLOYMENT_METHOD,
    "deployment_note": CLOUD_DEPLOYMENT_NOTE,
    "source_server_version": str(qdrant_client.info().version),
    "source_exact_point_count": local_final_point_count,
    "cloud_exact_point_count": cloud_final_point_count,
    "cloud_server_version": str(cloud_client.info().version),
    "schema_matches_local": True,
    "config_adaptations": CLOUD_CONFIG_ADAPTATIONS,
}
display(cloud_validation_df)
print(
    f"Cloud validation: PASS; exact points={cloud_final_point_count:,}; "
    "schema/config matches local docs_search_final."
)


,check,passed
0,collection status GREEN,True
1,exact point count,True
2,dense/multivector schema,True
3,sparse vector schema,True
4,HNSW configuration,True
5,quantization configuration,True
6,memory/storage settings,True
7,payload indexes,True


Cloud validation: PASS; exact points=4,134; schema/config matches local docs_search_final.


### 15.5 Cloud Smoke Test


One hybrid query exercises the frozen production retrieval path against the Cloud backend. It is a connectivity and ranking smoke test only: it does not run a benchmark, compute metrics, revisit the final holdout or alter the selected architecture.


In [116]:
CLOUD_SMOKE_TEST_QUERY = DEMO_QUERY
cloud_smoke_points = _query_final_points(
    CLOUD_SMOKE_TEST_QUERY,
    client=cloud_client,
    collection_name=CLOUD_COLLECTION_NAME,
    payload=SEARCH_PAYLOAD,
)
cloud_smoke_sections = collapse_points_to_sections(
    cloud_smoke_points, max_sections=3
)
if not cloud_smoke_sections:
    raise AssertionError("Cloud smoke test returned no ranked sections.")
if any(not row.get("section_url") for row in cloud_smoke_sections):
    raise AssertionError("Cloud smoke test returned a result without section_url.")
smoke_scores = [float(point.score) for point in cloud_smoke_points]
if any(left < right for left, right in zip(smoke_scores, smoke_scores[1:])):
    raise AssertionError("Cloud smoke-test results are not ranked by score.")

portfolio_qdrant_client = cloud_client
portfolio_collection_name = CLOUD_COLLECTION_NAME
CLOUD_DEPLOYMENT_MANIFEST["smoke_test"] = {
    "query": CLOUD_SMOKE_TEST_QUERY,
    "returned_sections": len(cloud_smoke_sections),
    "passed": True,
}
display(format_search_results(cloud_smoke_points[:3]))
print(
    "Cloud smoke test: PASS; cloud_client and portfolio_qdrant_client "
    "are ready for Section 16."
)


,rank,score,page_title,section_title,section_url,snippet
0,1,0.833333,Administration,Maximum HNSW ef Parameter,https://qdrant.tech/documentation/ops-configur...,A high HNSW `ef` value increases recall but al...
1,2,0.833333,Measuring ANN Recall,Tuning Search Recall,https://qdrant.tech/documentation/tutorials-se...,Toggle **advanced mode** in the ANN Recall tab...
2,3,0.342857,Search Quality,Why Recall Won't Be 1.0 (And That's OK),https://qdrant.tech/documentation/migration-gu...,Even a correct migration will often show recal...


Cloud smoke test: PASS; cloud_client and portfolio_qdrant_client are ready for Section 16.


### 15.6 Read-Only Frozen Collection Validation

This validation reads collection metadata and exact counts only. It performs no collection mutation, vector upload, embedding inference, benchmark, or holdout evaluation. Collection-time HNSW settings are checked separately from query-time `hnsw_ef`.


In [117]:
READ_ONLY_EXPECTED_FINAL_POINTS = 4134


def validate_frozen_collection_read_only(client: QdrantClient, label: str) -> dict:
    collection_name = FINAL_CONFIG["collection_name"]
    if not client.collection_exists(collection_name):
        raise AssertionError(f"{label}: {collection_name} does not exist.")

    info = client.get_collection(collection_name)
    exact_points = int(client.count(collection_name, exact=True).count)
    vectors = info.config.params.vectors
    sparse_vectors = info.config.params.sparse_vectors or {}
    if not isinstance(vectors, dict):
        raise AssertionError(f"{label}: named dense vectors are required.")

    dense_name = FINAL_CONFIG["vector_names"]["dense"]
    sparse_name = FINAL_CONFIG["vector_names"]["sparse"]
    expected_dense_names = {dense_name}
    expected_sparse_names = {sparse_name}
    if set(vectors) != expected_dense_names:
        raise AssertionError(f"{label}: dense vector names differ from frozen config.")
    if set(sparse_vectors) != expected_sparse_names:
        raise AssertionError(f"{label}: sparse vector names differ from frozen config.")

    dense = vectors[dense_name]
    expected_hnsw = FINAL_CONFIG["hnsw"]
    quantization = FINAL_CONFIG["quantization"]
    binary = dense.quantization_config
    checks = {
        "exact point count": exact_points == READ_ONLY_EXPECTED_FINAL_POINTS,
        "status GREEN": info.status == models.CollectionStatus.GREEN,
        "dense dimension": dense.size == FINAL_CONFIG["dense"]["size"],
        "dense distance": dense.distance.value == FINAL_CONFIG["dense"]["distance"],
        "dense HNSW": (
            dense.hnsw_config is not None
            and dense.hnsw_config.m == expected_hnsw["m"]
            and dense.hnsw_config.ef_construct == expected_hnsw["ef_construct"]
            and dense.hnsw_config.full_scan_threshold
            == expected_hnsw["full_scan_threshold_kb"]
        ),
        "collection HNSW": (
            info.config.hnsw_config.m == expected_hnsw["m"]
            and info.config.hnsw_config.ef_construct == expected_hnsw["ef_construct"]
            and info.config.hnsw_config.full_scan_threshold
            == expected_hnsw["full_scan_threshold_kb"]
        ),
        "binary quantization": (
            quantization["method"] == "binary"
            and isinstance(binary, models.BinaryQuantization)
            and binary.binary.encoding.value == quantization["binary_encoding"]
            and binary.binary.memory.value == quantization["memory"]
            and quantization["rescore"] is True
            and quantization["oversampling"] == 2.0
        ),
        "dense memory": dense.memory.value == FINAL_CONFIG["memory"]["dense_original"],
        "payload on disk": (
            info.config.params.on_disk_payload
            == FINAL_CONFIG["storage"]["payload_on_disk"]
        ),
        "indexing threshold": (
            info.config.optimizer_config.indexing_threshold
            == FINAL_CONFIG["storage"]["indexing_threshold_kb"]
        ),
        "tags keyword index": (
            set(info.payload_schema) == set(FINAL_CONFIG["payload_indexes"])
            and info.payload_schema["tags"].data_type.value
            == FINAL_CONFIG["payload_indexes"]["tags"]
        ),
        "ColBERT absent": FINAL_CONFIG["vector_names"]["colbert"] not in vectors,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise AssertionError(f"{label} frozen-schema validation failed: {failed}")
    return {
        "label": label, "collection": collection_name, "points": exact_points,
        "m": dense.hnsw_config.m,
        "ef_construct": dense.hnsw_config.ef_construct,
        "dense_size": dense.size, "distance": dense.distance.value,
        "sparse": "present", "tags_index": "keyword",
        "status": info.status.value,
    }


if FINAL_CONFIG["hnsw"]["hnsw_ef"] != 64:
    raise AssertionError("Frozen query-time hnsw_ef must remain 64.")
read_only_collection_summaries = [
    validate_frozen_collection_read_only(qdrant_client, "Local"),
    validate_frozen_collection_read_only(cloud_client, "Cloud"),
]
read_only_summary_blocks = [
    (
        f"{summary['label']} {summary['collection']}\n"
        f"points: {summary['points']}\n"
        f"HNSW: m={summary['m']}, ef_construct={summary['ef_construct']}\n"
        f"dense: {summary['dense_size']} / {summary['distance']}\n"
        f"sparse: {summary['sparse']}\n"
        f"tags index: {summary['tags_index']}\n"
        f"status: {summary['status']}"
    )
    for summary in read_only_collection_summaries
]
read_only_summary_blocks.append(
    f"final hnsw_ef: {FINAL_CONFIG['hnsw']['hnsw_ef']}\n"
    "embeddings recomputed: NO\n"
    "final_holdout rerun: NO\n"
    "collections recreated: NO"
)
print(*read_only_summary_blocks, sep="\n\n")


Local docs_search_final
points: 4134
HNSW: m=8, ef_construct=400
dense: 384 / Cosine
sparse: present
tags index: keyword
status: green

Cloud docs_search_final
points: 4134
HNSW: m=8, ef_construct=400
dense: 384 / Cosine
sparse: present
tags index: keyword
status: green

final hnsw_ef: 64
embeddings recomputed: NO
final_holdout rerun: NO
collections recreated: NO


**Stop point.** The frozen local artifact has been deployed and validated in Qdrant Cloud. Context Engineering and the remaining portfolio/RAG sections are intentionally not implemented here.


## 16. Context Engineering


Context construction starts after the frozen Cloud retrieval pipeline. All strategies share the same ranked input. This section prepares evidence and citations; it does not run generation or select a winning context strategy.


### 16.1 Configuration & Gemma Tokenizer


Gemma's tokenizer counts the **entire rendered evidence**, including source labels and metadata. Its immutable Hugging Face revision is pinned so token counts are reproducible. The 4,096-token evidence budget is distinct from the planned 16,384-token serving context window. The future model is `google/gemma-4-12B-it-qat-w4a16-ct`, served through a vLLM OpenAI-compatible endpoint on a 16 GB GPU target; weights and serving are outside this section. Fixed heuristics below are not tuned.


In [118]:
from dataclasses import asdict, replace

CONTEXT_MODEL_ID = "google/gemma-4-12B-it-qat-w4a16-ct"
CONTEXT_TOKENIZER_REVISION = "1d2c2d7f2466070e69d6fb3fd5ce9a7d75f2f6ee"
CONTEXT_TOKEN_BUDGET = 4096
NEIGHBOR_RELATIVE_RELEVANCE = 0.85
REDUNDANCY_WEIGHT = 0.30
DIVERSITY_BONUS = 0.05
ANCHOR_BONUS = 0.03
CONTEXT_EMBED_BATCH_SIZE = 32
CONTEXT_SANITY_QUERIES = 3
CONTEXT_PREVIEW_CHARS = 160
CONTEXT_STRATEGIES = (
    "retrieved_chunks",
    "adaptive_section_expansion",
    "evidence_aware_packing",
)
CONTEXT_PAYLOAD = (
    "page_title", "section_title", "page_url", "section_url", "breadcrumbs",
    "chunk_text", "prev_section_text", "next_section_text", "path", "section_index",
)

context_tokenizer = AutoTokenizer.from_pretrained(
    CONTEXT_MODEL_ID, revision=CONTEXT_TOKENIZER_REVISION,
)


def count_context_tokens(text: str) -> int:
    return len(context_tokenizer.encode(text, add_special_tokens=False))


### 16.2 Candidate Preparation


The existing `retrieve_final_sections()` keeps only URL/rank/score. For this section the caller uses `_query_final_points(..., client=cloud_client, payload=CONTEXT_PAYLOAD)`, preserving the frozen search while retaining its payload. The builder accepts these ranked points or flat payload-bearing dictionaries. URL-only records fail explicitly instead of triggering hidden retrieval.

Deduplication keeps the highest-ranked anchor, using section URL, then path/index, then point ID or exact text. Neighbor evidence is globally deduplicated by exact text after whitespace normalization: text already present as any retrieved anchor is excluded, and a repeated neighbor can expand at most one anchor. Payload prev/next fields have **no independent provenance**: expansions remain part of the anchor evidence, with its original page/section URL. No neighbor URL is inferred.


In [119]:
@dataclass(frozen=True)
class ContextCandidate:
    identity: tuple
    point_id: str | int | None
    retrieval_rank: int
    retrieval_score: float | None
    page_title: str | None
    section_title: str | None
    page_url: str | None
    section_url: str | None
    breadcrumbs: tuple
    chunk_text: str
    prev_section_text: str | None
    next_section_text: str | None
    path: str | None
    section_index: int | None

    @property
    def page_identity(self) -> str | tuple:
        # Used for diversity only; never emitted as a fabricated citation URL.
        return self.page_url or self.path or (
            self.section_url.split("#", 1)[0] if self.section_url else self.identity
        )


def normalize_context_candidates(retrieved_sections) -> list[ContextCandidate]:
    candidates = []
    for position, item in enumerate(retrieved_sections, 1):
        if isinstance(item, ContextCandidate):
            candidates.append(item)
            continue
        is_mapping = isinstance(item, dict)
        payload = item.get("payload", item) if is_mapping else (item.payload or {})
        point_id = item.get("point_id", item.get("id")) if is_mapping else item.id
        rank = item.get("retrieval_rank", item.get("rank", position)) if is_mapping else position
        score = item.get("retrieval_score", item.get("score")) if is_mapping else item.score
        text = payload.get("chunk_text")
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"Context input at rank {rank} needs nonempty chunk_text.")
        url, path, index = (payload.get(k) for k in ("section_url", "path", "section_index"))
        identity = (
            ("section", url) if url else
            ("structure", path, index) if path is not None and index is not None else
            ("point", str(point_id)) if point_id is not None else ("text", text)
        )
        breadcrumbs = payload.get("breadcrumbs") or ()
        candidates.append(ContextCandidate(
            identity=identity, point_id=str(point_id) if point_id is not None else None,
            retrieval_rank=int(rank), retrieval_score=float(score) if score is not None else None,
            page_title=payload.get("page_title"), section_title=payload.get("section_title"),
            page_url=payload.get("page_url"), section_url=url,
            breadcrumbs=(breadcrumbs,) if isinstance(breadcrumbs, str) else tuple(breadcrumbs),
            chunk_text=text, prev_section_text=payload.get("prev_section_text"),
            next_section_text=payload.get("next_section_text"), path=path, section_index=index,
        ))
    unique = {}
    for candidate in sorted(candidates, key=lambda c: c.retrieval_rank):
        unique.setdefault(candidate.identity, candidate)
    return list(unique.values())


def context_text_key(text: str) -> str:
    """Deterministic exact-content key with normalized whitespace."""
    return " ".join(text.split())


### 16.3 S0 — Retrieved Chunks Baseline


S0 packs whole anchor chunks in retrieval order, without scoring or expansion. If a block does not fit, it is skipped and later blocks can still fit. All strategies use the same renderer and count the full joined context on every admission. No truncation is applied. Empty input or a budget too small for any whole block produces an explicit empty ContextPack for downstream abstention.


In [120]:
@dataclass(frozen=True)
class ContextOption:
    anchor: ContextCandidate
    content: str
    expanded_with: str | None = None
    relevance: float | None = None


def render_context_option(option: ContextOption, source_id: str) -> str:
    anchor = option.anchor
    lines = [f"[{source_id}]"]
    for label, value in (
        ("Page", anchor.page_title), ("Section", anchor.section_title),
        ("URL", anchor.section_url or anchor.page_url),
    ):
        if value:
            lines.append(f"{label}: {value}")
    return "\n".join(lines + ["Content:", option.content])


def render_context_options(options: list[ContextOption]) -> str:
    return "\n\n".join(
        render_context_option(option, f"S{i}") for i, option in enumerate(options, 1)
    )


def context_options_fit(options: list[ContextOption], token_budget: int) -> bool:
    return count_context_tokens(render_context_options(options)) <= token_budget


def pack_in_retrieval_order(candidates, token_budget, expansions=None):
    selected = []
    for i, candidate in enumerate(candidates):
        anchor = ContextOption(candidate, candidate.chunk_text)
        if not context_options_fit(selected + [anchor], token_budget):
            continue
        expanded = expansions.get(i) if expansions else None
        selected.append(
            expanded if expanded and context_options_fit(selected + [expanded], token_budget)
            else anchor
        )
    return selected


### 16.4 S1 — Adaptive Section Expansion


S1 scores anchors and globally distinct nonempty neighbors in a batch. At most one neighbor is eligible per anchor: its cosine similarity must be positive and reach both 0.85 × anchor similarity and the median similarity of all retrieved anchors for that query. This query-adaptive floor prevents a weak anchor from admitting weak adjacent context. Anchor text always survives; an expansion that exceeds the remaining budget falls back to the anchor alone.

Scoring reuses the existing BGE model via `query_embed` on the original query and `passage_embed` in batches. Its existing input-length limit still applies to long passages; scores are a heuristic, not a full-text quality guarantee. Gemma token counting always covers the full evidence text.


In [121]:
def context_embedding_matrix(texts: list[str]) -> np.ndarray:
    vectors = np.asarray(list(dense_embedding_model.passage_embed(
        texts, batch_size=CONTEXT_EMBED_BATCH_SIZE,
    )), dtype=np.float32)
    if vectors.ndim != 2 or not np.isfinite(vectors).all():
        raise ValueError("Context embeddings must be a finite matrix.")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Context embedding has zero norm.")
    return vectors / norms


def neighbor_relevance_threshold(
    anchor_relevance: float, anchor_relevance_floor: float,
) -> float:
    return max(
        anchor_relevance * NEIGHBOR_RELATIVE_RELEVANCE,
        anchor_relevance_floor,
    )


def score_context_expansions(query, candidates):
    query_vector = np.asarray(
        list(dense_embedding_model.query_embed([query]))[0], dtype=np.float32,
    )
    norm = np.linalg.norm(query_vector)
    if not np.isfinite(query_vector).all() or norm == 0:
        raise ValueError("Context query embedding must be finite and nonzero.")
    query_vector = query_vector / norm
    texts = [c.chunk_text for c in candidates]
    anchor_text_keys = {context_text_key(text) for text in texts}
    neighbor_text_indices, neighbors_by_anchor = {}, {}
    for i, anchor in enumerate(candidates):
        seen = set()
        for side, text in (("prev", anchor.prev_section_text), ("next", anchor.next_section_text)):
            if not isinstance(text, str):
                continue
            key = context_text_key(text)
            if not key or key in anchor_text_keys or key in seen:
                continue
            seen.add(key)
            if key not in neighbor_text_indices:
                neighbor_text_indices[key] = len(texts)
                texts.append(text)
            neighbors_by_anchor.setdefault(i, []).append(
                (side, neighbor_text_indices[key], key)
            )
    matrix = context_embedding_matrix(texts)
    relevance = matrix @ query_vector
    anchor_relevance = relevance[:len(candidates)]
    anchor_relevance_floor = float(np.median(anchor_relevance))
    best, used_neighbor_keys = {}, set()
    anchor_order = sorted(
        range(len(candidates)), key=lambda i: (candidates[i].retrieval_rank, i)
    )
    for i in anchor_order:
        required_relevance = neighbor_relevance_threshold(
            float(anchor_relevance[i]), anchor_relevance_floor,
        )
        eligible = [
            neighbor for neighbor in neighbors_by_anchor.get(i, [])
            if neighbor[2] not in used_neighbor_keys
            and relevance[neighbor[1]] > 0
            and relevance[neighbor[1]] >= required_relevance
        ]
        if eligible:
            side, j, key = max(eligible, key=lambda neighbor: relevance[neighbor[1]])
            best[i] = (side, j)
            used_neighbor_keys.add(key)
    expansions = {
        i: ContextOption(
            candidates[i],
            f"Anchor:\n{candidates[i].chunk_text}\n\nRelevant adjacent context ({side}):\n{texts[j]}",
            expanded_with=side,
        )
        for i, (side, j) in best.items()
    }
    return expansions, matrix[:len(candidates)], query_vector


### 16.5 S2 — Evidence-Aware Budgeted Packing


S2 considers each anchor and its single eligible expanded variant. Utility is query cosine − 0.30 × maximum nonnegative cosine to selected evidence + 0.05 for a new page + 0.03 for an unexpanded anchor. Token cost is a hard admission constraint, not a score divisor. Selecting a variant removes both variants of that section.

Relevance is computed by matrix multiplication. Redundancy is updated for all candidates with one matrix–vector product per greedy step; no full pairwise matrix is needed. Ties retain retrieval order. This is context selection after retrieval, not a new retrieval reranker.


In [122]:
def pack_evidence_aware(candidates, expansions, anchor_matrix, query_vector, token_budget):
    options = [ContextOption(c, c.chunk_text) for c in candidates] + list(expansions.values())
    matrix = anchor_matrix
    if expansions:
        expanded_matrix = context_embedding_matrix([o.content for o in expansions.values()])
        matrix = np.vstack((matrix, expanded_matrix))
    relevance = matrix @ query_vector
    options = [replace(o, relevance=float(r)) for o, r in zip(options, relevance)]
    available = np.ones(len(options), dtype=bool)
    redundancy = np.zeros(len(options), dtype=np.float32)
    anchor_bonus = np.array([o.expanded_with is None for o in options]) * ANCHOR_BONUS
    selected, pages = [], set()
    while available.any():
        fits = np.array([
            available[i] and context_options_fit(selected + [option], token_budget)
            for i, option in enumerate(options)
        ])
        if not fits.any():
            break
        new_page = np.array([o.anchor.page_identity not in pages for o in options])
        utility = relevance - REDUNDANCY_WEIGHT * redundancy + DIVERSITY_BONUS * new_page + anchor_bonus
        chosen = int(np.argmax(np.where(fits, utility, -np.inf)))
        option = options[chosen]
        selected.append(option)
        pages.add(option.anchor.page_identity)
        available &= np.array([o.anchor.identity != option.anchor.identity for o in options])
        redundancy = np.maximum(redundancy, matrix @ matrix[chosen])
    return selected


### 16.6 Structured ContextPack & Provenance


Lightweight frozen dataclasses separate candidate metadata, selected evidence and the final pack. Source IDs S1, S2, … are assigned in final selection order. `sources` maps each ID to the original section URL (page URL fallback); missing URLs stay missing. Expanded evidence carries anchor provenance and an explicit prev/next flag. Breadcrumbs are preserved in the object but omitted from the prompt to avoid repeating titles.

`context_text` is ready for any future LLM adapter. Individual block token counts are informative; the authoritative `used_tokens` is counted on the complete joined string, because tokenizer boundaries are not necessarily additive. `asdict(pack)` provides a serializable representation.


In [123]:
@dataclass(frozen=True)
class ContextBlock:
    source_id: str
    kind: str
    point_id: str | int | None
    retrieval_rank: int
    page_title: str | None
    section_title: str | None
    page_url: str | None
    section_url: str | None
    content: str
    token_count: int
    retrieval_score: float | None
    relevance: float | None
    expanded_with: str | None
    breadcrumbs: tuple


@dataclass(frozen=True)
class ContextPack:
    query: str
    strategy: str
    token_budget: int
    used_tokens: int
    evidence_blocks: tuple[ContextBlock, ...]
    sources: dict[str, str | None]
    context_text: str


def make_context_pack(query, strategy, options, token_budget) -> ContextPack:
    blocks = []
    for i, option in enumerate(options, 1):
        anchor, source_id = option.anchor, f"S{i}"
        blocks.append(ContextBlock(
            source_id=source_id, kind="expanded_anchor" if option.expanded_with else "anchor",
            point_id=anchor.point_id, retrieval_rank=anchor.retrieval_rank,
            page_title=anchor.page_title, section_title=anchor.section_title,
            page_url=anchor.page_url, section_url=anchor.section_url,
            content=option.content,
            token_count=count_context_tokens(render_context_option(option, source_id)),
            retrieval_score=anchor.retrieval_score, relevance=option.relevance,
            expanded_with=option.expanded_with, breadcrumbs=anchor.breadcrumbs,
        ))
    text = render_context_options(options)
    used_tokens = count_context_tokens(text)
    if used_tokens > token_budget:
        raise AssertionError(f"Rendered context exceeds budget: {used_tokens} > {token_budget}")
    return ContextPack(
        query, strategy, token_budget, used_tokens, tuple(blocks),
        {b.source_id: b.section_url or b.page_url for b in blocks}, text,
    )


### 16.7 Unified Context Builder


In [124]:
def build_context(
    query: str,
    retrieved_sections,
    strategy: str = "retrieved_chunks",
    token_budget: int = CONTEXT_TOKEN_BUDGET,
) -> ContextPack:
    """Construct evidence from ranked input; never perform retrieval or generation."""
    if strategy not in CONTEXT_STRATEGIES:
        raise ValueError(f"Unknown context strategy: {strategy}")
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a nonempty string.")
    if isinstance(token_budget, bool) or not isinstance(token_budget, int) or token_budget < 0:
        raise ValueError("token_budget must be a nonnegative integer.")
    candidates = normalize_context_candidates(retrieved_sections)
    options = []
    if candidates and token_budget:
        if strategy == "retrieved_chunks":
            options = pack_in_retrieval_order(candidates, token_budget)
        else:
            expansions, matrix, query_vector = score_context_expansions(query, candidates)
            options = (
                pack_in_retrieval_order(candidates, token_budget, expansions)
                if strategy == "adaptive_section_expansion"
                else pack_evidence_aware(candidates, expansions, matrix, query_vector, token_budget)
            )
    return make_context_pack(query, strategy, options, token_budget)


### 16.8 Sanity Check


Three deterministic research/confirmation queries are selected by query ID. Each makes exactly one Cloud search. The same top section anchors, with their original point payloads, feed S0/S1/S2. Assertions verify exact content/provenance preservation against that input, without requesting URLs or consulting final holdout. Diagnostics describe context structure only; they do not measure answer quality.

Small synthetic behavioral cases additionally verify branches that need not activate for these real queries: S1 can accept a clearly relevant neighbor, and S2 can change selection when the evidence budget is binding. They exercise the production builder and existing BGE model without claiming a quality improvement.


In [125]:
context_sanity_queries = (
    all_evaluation_df.loc[
        all_evaluation_df["split"].isin(["research", "confirmation"]),
        ["query_id", "query", "split"],
    ]
    .sort_values("query_id")
    .head(CONTEXT_SANITY_QUERIES)
    .copy()
)
assert len(context_sanity_queries) == CONTEXT_SANITY_QUERIES
assert context_sanity_queries["query_id"].is_unique
assert context_sanity_queries["split"].isin(["research", "confirmation"]).all()

context_retrieval_inputs, context_packs, context_diagnostics = {}, {}, []
for row in tqdm(context_sanity_queries.itertuples(index=False),
                total=len(context_sanity_queries), desc="Context sanity", unit="query"):
    points = _query_final_points(
        row.query, client=cloud_client, collection_name=FINAL_CONFIG["collection_name"],
        payload=list(CONTEXT_PAYLOAD),
    )
    candidates = normalize_context_candidates(points)[:FINAL_CONFIG["retrieval"]["top_k"]]
    context_retrieval_inputs[row.query_id] = candidates
    original_by_rank = {c.retrieval_rank: c for c in candidates}
    for strategy in CONTEXT_STRATEGIES:
        pack = build_context(row.query, candidates, strategy)
        context_packs[row.query_id, strategy] = pack
        assert pack.context_text and pack.evidence_blocks, (row.query_id, strategy)
        assert pack.used_tokens == count_context_tokens(pack.context_text) <= pack.token_budget
        assert list(pack.sources) == [f"S{i}" for i in range(1, len(pack.evidence_blocks) + 1)]
        identities, pages, expanded_text_keys = [], set(), []
        anchor_text_keys = {context_text_key(c.chunk_text) for c in candidates}
        for block in pack.evidence_blocks:
            anchor = original_by_rank[block.retrieval_rank]
            identities.append(anchor.identity)
            pages.add(anchor.page_identity)
            assert block.point_id == anchor.point_id
            assert (block.page_title, block.section_title, block.page_url, block.section_url) == (
                anchor.page_title, anchor.section_title, anchor.page_url, anchor.section_url,
            )
            assert block.breadcrumbs == anchor.breadcrumbs
            assert pack.sources[block.source_id] == (anchor.section_url or anchor.page_url)
            url = pack.sources[block.source_id]
            assert url and urlparse(url).scheme in ("http", "https") and urlparse(url).netloc
            assert block.expanded_with in (None, "prev", "next")
            expected_content = anchor.chunk_text
            if block.expanded_with:
                neighbor = getattr(anchor, f"{block.expanded_with}_section_text")
                assert neighbor
                neighbor_key = context_text_key(neighbor)
                assert neighbor_key not in anchor_text_keys
                expanded_text_keys.append(neighbor_key)
                expected_content = (
                    f"Anchor:\n{anchor.chunk_text}\n\n"
                    f"Relevant adjacent context ({block.expanded_with}):\n{neighbor}"
                )
            assert block.content == expected_content
        assert len(identities) == len(set(identities))
        assert len(expanded_text_keys) == len(set(expanded_text_keys))
        if strategy == "retrieved_chunks":
            assert all(b.expanded_with is None for b in pack.evidence_blocks)
            assert [b.retrieval_rank for b in pack.evidence_blocks] == sorted(
                b.retrieval_rank for b in pack.evidence_blocks
            )
        expanded = sum(b.expanded_with is not None for b in pack.evidence_blocks)
        context_diagnostics.append({
            "query_id": row.query_id, "strategy": strategy,
            "used_tokens": pack.used_tokens, "evidence_blocks": len(pack.evidence_blocks),
            "anchor_blocks": len(pack.evidence_blocks) - expanded,
            "expanded_blocks": expanded, "unique_pages": len(pages),
            "budget_utilization": pack.used_tokens / pack.token_budget,
        })

context_diagnostics_df = pd.DataFrame(context_diagnostics)
display(context_diagnostics_df.round(3))
print("Context sanity: PASS. All rendered contexts fit the 4096-token evidence budget.")


Context sanity: 100%|██████████| 3/3 [00:25<00:00,  8.59s/query]


,query_id,strategy,used_tokens,evidence_blocks,anchor_blocks,expanded_blocks,unique_pages,budget_utilization
0,h01,retrieved_chunks,2695,10,10,0,8,0.658
1,h01,adaptive_section_expansion,2695,10,10,0,8,0.658
2,h01,evidence_aware_packing,2695,10,10,0,8,0.658
3,h02,retrieved_chunks,3603,10,10,0,9,0.880
4,h02,adaptive_section_expansion,3603,10,10,0,9,0.880
5,h02,evidence_aware_packing,3603,10,10,0,9,0.880
6,h03,retrieved_chunks,3433,10,10,0,6,0.838
7,h03,adaptive_section_expansion,3433,10,10,0,6,0.838
8,h03,evidence_aware_packing,3433,10,10,0,6,0.838


Context sanity: PASS. All rendered contexts fit the 4096-token evidence budget.


In [126]:
# Display previews only; the stored ContextPacks retain full evidence.
context_example_id = context_sanity_queries.iloc[0]["query_id"]
print(context_sanity_queries.iloc[0]["query"])
for strategy in CONTEXT_STRATEGIES:
    pack = context_packs[context_example_id, strategy]
    diagnostic = context_diagnostics_df.loc[
        (context_diagnostics_df["query_id"] == context_example_id)
        & (context_diagnostics_df["strategy"] == strategy)
    ].iloc[0]
    print(
        f"\n{strategy}: {pack.used_tokens}/{pack.token_budget} tokens; "
        f"{len(pack.evidence_blocks)} blocks "
        f"({diagnostic.anchor_blocks} anchor, {diagnostic.expanded_blocks} expanded); "
        f"{diagnostic.unique_pages} pages"
    )
    display(pd.DataFrame([
        {
            "source_id": b.source_id, "page": b.page_title, "section": b.section_title,
            "url": pack.sources[b.source_id],
            "preview": re.sub(r"\s+", " ", b.content)[:CONTEXT_PREVIEW_CHARS],
        }
        for b in pack.evidence_blocks
    ]))


Which vector size and distance function must I declare when creating a collection?

retrieved_chunks: 2695/4096 tokens; 10 blocks (10 anchor, 0 expanded); 8 pages


,source_id,page,section,url,preview
0,S1,Reranking for Better Search,Creating a Collection,https://qdrant.tech/documentation/search-preci...,A collection is basically a named group of poi...
1,S2,Collections,Collection with Multiple Vectors,https://qdrant.tech/documentation/manage-data/...,*Available as of v0.10.0* It is possible to ha...
2,S3,Semantic Search 101,3. Create a Collection,https://qdrant.tech/documentation/tutorials-ba...,"```go collectionName := ""my_books"" client.Crea..."
3,S4,Upstage,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/u...,```python from qdrant_client.models import Vec...
4,S5,Collections,Create a Collection,https://qdrant.tech/documentation/manage-data/...,```java import io.qdrant.client.QdrantClient; ...
5,S6,Nvidia,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/n...,```python from qdrant_client.models import Vec...
6,S7,Collections,Collection with Sparse Vectors,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...
7,S8,OpenAI,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/o...,```python from qdrant_client.models import Vec...
8,S9,Vectors,Float16,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...
9,S10,Mistral,Create a collection and Insert the documents,https://qdrant.tech/documentation/embeddings/m...,```python client.create_collection(collection_...



adaptive_section_expansion: 2695/4096 tokens; 10 blocks (10 anchor, 0 expanded); 8 pages


,source_id,page,section,url,preview
0,S1,Reranking for Better Search,Creating a Collection,https://qdrant.tech/documentation/search-preci...,A collection is basically a named group of poi...
1,S2,Collections,Collection with Multiple Vectors,https://qdrant.tech/documentation/manage-data/...,*Available as of v0.10.0* It is possible to ha...
2,S3,Semantic Search 101,3. Create a Collection,https://qdrant.tech/documentation/tutorials-ba...,"```go collectionName := ""my_books"" client.Crea..."
3,S4,Upstage,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/u...,```python from qdrant_client.models import Vec...
4,S5,Collections,Create a Collection,https://qdrant.tech/documentation/manage-data/...,```java import io.qdrant.client.QdrantClient; ...
5,S6,Nvidia,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/n...,```python from qdrant_client.models import Vec...
6,S7,Collections,Collection with Sparse Vectors,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...
7,S8,OpenAI,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/o...,```python from qdrant_client.models import Vec...
8,S9,Vectors,Float16,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...
9,S10,Mistral,Create a collection and Insert the documents,https://qdrant.tech/documentation/embeddings/m...,```python client.create_collection(collection_...



evidence_aware_packing: 2695/4096 tokens; 10 blocks (10 anchor, 0 expanded); 8 pages


,source_id,page,section,url,preview
0,S1,Semantic Search 101,3. Create a Collection,https://qdrant.tech/documentation/tutorials-ba...,"```go collectionName := ""my_books"" client.Crea..."
1,S2,Reranking for Better Search,Creating a Collection,https://qdrant.tech/documentation/search-preci...,A collection is basically a named group of poi...
2,S3,Mistral,Create a collection and Insert the documents,https://qdrant.tech/documentation/embeddings/m...,```python client.create_collection(collection_...
3,S4,Collections,Create a Collection,https://qdrant.tech/documentation/manage-data/...,```java import io.qdrant.client.QdrantClient; ...
4,S5,Nvidia,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/n...,```python from qdrant_client.models import Vec...
5,S6,OpenAI,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/o...,```python from qdrant_client.models import Vec...
6,S7,Vectors,Float16,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...
7,S8,Upstage,Creating a collection to insert the documents,https://qdrant.tech/documentation/embeddings/u...,```python from qdrant_client.models import Vec...
8,S9,Collections,Collection with Multiple Vectors,https://qdrant.tech/documentation/manage-data/...,*Available as of v0.10.0* It is possible to ha...
9,S10,Collections,Collection with Sparse Vectors,https://qdrant.tech/documentation/manage-data/...,```csharp using Qdrant.Client; using Qdrant.Cl...


In [127]:
# Boundary checks use existing retrieved payloads; no additional search or dataset.
_context_query = context_sanity_queries.iloc[0]["query"]
_context_anchor = context_retrieval_inputs[context_example_id][0]
_context_option = ContextOption(_context_anchor, _context_anchor.chunk_text)
_context_exact_budget = count_context_tokens(render_context_options([_context_option]))

for strategy in CONTEXT_STRATEGIES:
    assert build_context(_context_query, [], strategy).context_text == ""
    assert build_context(_context_query, [_context_anchor], strategy, token_budget=0).used_tokens == 0
assert not build_context(_context_query, [_context_anchor], token_budget=1).evidence_blocks
assert build_context(
    _context_query, [_context_anchor], token_budget=_context_exact_budget
).used_tokens == _context_exact_budget
assert not build_context(
    _context_query, [_context_anchor], token_budget=_context_exact_budget - 1
).evidence_blocks
assert normalize_context_candidates([
    replace(_context_anchor, retrieval_rank=99), _context_anchor,
]) == [_context_anchor]
assert len(normalize_context_candidates([
    {"chunk_text": "same text", "path": "example.md", "section_index": 0},
    {"chunk_text": "other chunk", "path": "example.md", "section_index": 0},
])) == 1
assert len(normalize_context_candidates([
    {"chunk_text": "same text"}, {"chunk_text": "same text"},
])) == 1

_context_expansion = ContextOption(
    _context_anchor, _context_anchor.chunk_text + "\n" + _context_anchor.chunk_text,
    expanded_with="next",
)
assert not context_options_fit([_context_expansion], _context_exact_budget)
assert pack_in_retrieval_order(
    [_context_anchor], _context_exact_budget, {0: _context_expansion},
) == [_context_option]

# A neighbor already present as a retrieved anchor must not be expanded.
_retrieved_text = "Configure HNSW search parameters for higher recall."
_dedup_anchor_a = replace(
    _context_anchor, identity=("regression", "anchor-a"), point_id="anchor-a",
    retrieval_rank=1, chunk_text="Account and billing information.",
    prev_section_text=None, next_section_text=_retrieved_text,
)
_dedup_anchor_b = replace(
    _context_anchor, identity=("regression", "anchor-b"), point_id="anchor-b",
    retrieval_rank=2, chunk_text=f"  {_retrieved_text}\n",
    prev_section_text=None, next_section_text=None,
)
_dedup_expansions, _, _ = score_context_expansions(
    _retrieved_text, [_dedup_anchor_a, _dedup_anchor_b],
)
assert 0 not in _dedup_expansions

# The same normalized neighbor may expand at most one anchor globally.
_shared_neighbor = "Scalar quantization stores vectors using int8 values."
_shared_anchor_a = replace(
    _dedup_anchor_a, chunk_text="Invoices can be paid by credit card.",
    next_section_text=_shared_neighbor,
)
_shared_anchor_b = replace(
    _dedup_anchor_b, chunk_text="Pasta cooks in salted water.",
    prev_section_text=f"\n{_shared_neighbor}  ",
)
_shared_expansions, _, _ = score_context_expansions(
    _shared_neighbor, [_shared_anchor_a, _shared_anchor_b],
)
_shared_keys = [
    context_text_key(getattr(option.anchor, f"{option.expanded_with}_section_text"))
    for option in _shared_expansions.values()
]
assert len(_shared_keys) == len(set(_shared_keys)) == 1

# The median anchor floor rejects a neighbor accepted by the old relative-only rule.
assert 0.30 >= 0.20 * NEIGHBOR_RELATIVE_RELEVANCE
assert neighbor_relevance_threshold(0.20, 0.50) == 0.50 > 0.30

for (query_id, strategy), pack in context_packs.items():
    anchors = {c.retrieval_rank: c for c in context_retrieval_inputs[query_id]}
    options = [
        ContextOption(
            anchors[b.retrieval_rank],
            b.content, b.expanded_with, b.relevance,
        )
        for b in pack.evidence_blocks
    ]
    assert render_context_options(options) == pack.context_text
    assert [b.source_id for b in pack.evidence_blocks] == list(pack.sources)
    assert all(
        b.token_count == count_context_tokens(render_context_option(o, b.source_id))
        for b, o in zip(pack.evidence_blocks, options)
    )
print("Boundary checks: PASS (budgets, dedup, relevance floor, expansion fallback, rendering).")


Boundary checks: PASS (budgets, dedup, relevance floor, expansion fallback, rendering).


In [128]:
# Controlled behavioral cases use the real BGE model and production builder.
_behavior_query = "How does scalar quantization reduce vector memory usage?"
_positive_neighbor = (
    "Scalar quantization represents vector components with lower-precision integer "
    "values, substantially reducing vector memory usage."
)
_positive_candidates = [
    replace(
        _context_anchor, identity=("behavior", "quantization"),
        point_id="behavior-quantization", retrieval_rank=1,
        chunk_text=(
            "Quantization is a technique used to reduce the memory required to "
            "represent vectors."
        ),
        prev_section_text=None, next_section_text=_positive_neighbor,
    ),
    replace(
        _context_anchor, identity=("behavior", "billing"),
        point_id="behavior-billing", retrieval_rank=2,
        chunk_text="Invoices list payment dates and customer billing addresses.",
        prev_section_text=None, next_section_text=None,
    ),
    replace(
        _context_anchor, identity=("behavior", "cooking"),
        point_id="behavior-cooking", retrieval_rank=3,
        chunk_text="Fresh pasta cooks in salted water before sauce is added.",
        prev_section_text=None, next_section_text=None,
    ),
]
_positive_expansions, _, _ = score_context_expansions(
    _behavior_query, _positive_candidates,
)
assert 0 in _positive_expansions
_positive_pack = build_context(
    _behavior_query, _positive_candidates,
    strategy="adaptive_section_expansion",
)
_expanded_blocks = [
    block for block in _positive_pack.evidence_blocks if block.expanded_with
]
assert len(_expanded_blocks) == 1
_expanded_block = _expanded_blocks[0]
assert _expanded_block.point_id == "behavior-quantization"
assert _expanded_block.expanded_with == "next"
assert _positive_neighbor in _expanded_block.content
assert _positive_pack.used_tokens <= _positive_pack.token_budget
assert (_expanded_block.page_url, _expanded_block.section_url) == (
    _context_anchor.page_url, _context_anchor.section_url,
)
assert _positive_pack.sources[_expanded_block.source_id] == (
    _context_anchor.section_url or _context_anchor.page_url
)

_limited_candidates = [
    replace(
        _context_anchor, identity=("limited", "billing"),
        point_id="limited-billing", retrieval_rank=1,
        chunk_text="Invoices record customer payments and billing addresses.",
        prev_section_text=None, next_section_text=None,
    ),
    replace(
        _context_anchor, identity=("limited", "cooking"),
        point_id="limited-cooking", retrieval_rank=2,
        chunk_text="Pasta cooks in salted water before tomato sauce is added.",
        prev_section_text=None, next_section_text=None,
    ),
    replace(
        _context_anchor, identity=("limited", "relevant"),
        point_id="limited-relevant", retrieval_rank=3,
        chunk_text=(
            "Scalar quantization reduces vector memory by representing floating-point "
            "vector components with lower-precision integer values while retaining "
            "useful approximation accuracy for similarity search."
        ),
        prev_section_text=None, next_section_text=None,
    ),
]
_relevant_option = ContextOption(
    _limited_candidates[2], _limited_candidates[2].chunk_text,
)
_limited_budget = count_context_tokens(render_context_options([_relevant_option]))
_s0_limited = build_context(
    _behavior_query, _limited_candidates, strategy="retrieved_chunks",
    token_budget=_limited_budget,
)
_s2_limited = build_context(
    _behavior_query, _limited_candidates, strategy="evidence_aware_packing",
    token_budget=_limited_budget,
)
assert len(_s0_limited.evidence_blocks) == 1
assert 0 < len(_s2_limited.evidence_blocks) < len(_limited_candidates)
assert len(_s2_limited.evidence_blocks) == 1
assert _s2_limited.used_tokens <= _limited_budget
assert _s2_limited.evidence_blocks[0].point_id == "limited-relevant"
assert _s2_limited.evidence_blocks[0].retrieval_rank == 3
assert _s0_limited.evidence_blocks[0].point_id != (
    _s2_limited.evidence_blocks[0].point_id
)

print(
    "Behavioral checks: PASS\n"
    f"- S1 expanded rank={_expanded_block.retrieval_rank}, "
    f"side={_expanded_block.expanded_with}.\n"
    f"- Limited budget={_limited_budget} tokens; "
    f"S0 ranks={[b.retrieval_rank for b in _s0_limited.evidence_blocks]}; "
    f"S2 ranks={[b.retrieval_rank for b in _s2_limited.evidence_blocks]}."
)


Behavioral checks: PASS
- S1 expanded rank=1, side=next.
- Limited budget=79 tokens; S0 ranks=[1]; S2 ranks=[3].


### 16.9 Context Engineering Conclusions


S0 is the retrieved-only baseline. S1 adds at most one query-relevant neighbor to each anchor when the budget permits. S2 selects evidence using relevance, redundancy, page diversity and a hard token budget.

All three return the same framework-agnostic ContextPack. Gemma 4 tokenization at a pinned Hugging Face revision measures the full rendered evidence against a 4,096-token budget. Neighbor evidence is exact-deduplicated globally, while citation IDs and original URLs are established before generation; adjacent text remains attributed to its anchor, with no invented neighbor provenance.

The sanity check validates implementation, budget behavior and provenance only. **No strategy is declared the winner.** S0/S1/S2 will be compared through end-to-end RAG evaluation after Gemma 4 generation is connected. Generation, vLLM deployment and agent frameworks are not implemented here.


## 17. RAG Generation with Gemma 4


### 17.1 vLLM Runtime and Connection


Gemma 4 is served by a dedicated vLLM container and reached from this `dev_host` notebook through vLLM's OpenAI-compatible HTTP API at the Docker hostname `vllm-gemma4-12b`. This subsection verifies runtime connectivity only; the ContextPack integration, generation adapter and grounded RAG prompting are implemented in the following subsections.


In [131]:
import json
import os
import time
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

VLLM_ROOT_URL = os.getenv("VLLM_ROOT_URL", "http://vllm-gemma4-12b:8000").rstrip("/")
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", f"{VLLM_ROOT_URL}/v1").rstrip("/")
VLLM_MODEL = os.getenv("VLLM_MODEL", "gemma-4-12b-it")

def _vllm_request(url, *, method="GET", payload=None, timeout=30, expect_json=True):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"} if data is not None else {}
    request = Request(url, data=data, headers=headers, method=method)
    try:
        with urlopen(request, timeout=timeout) as response:
            status = response.status
            body = response.read()
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace").strip()
        raise RuntimeError(f"{method} {url} failed with HTTP {exc.code}: {detail[:300]}") from exc
    except URLError as exc:
        raise RuntimeError(f"{method} {url} failed: {exc.reason}") from exc
    if not 200 <= status < 300:
        raise RuntimeError(f"{method} {url} returned unexpected HTTP {status}")
    return json.loads(body) if expect_json else status

health_status = _vllm_request(f"{VLLM_ROOT_URL}/health", timeout=10, expect_json=False)
models_response = _vllm_request(f"{VLLM_BASE_URL}/models", timeout=10)
available_models = {item.get("id") for item in models_response.get("data", []) if isinstance(item, dict)}
assert VLLM_MODEL in available_models, f"Expected {VLLM_MODEL!r}; available models: {sorted(available_models)!r}"

chat_payload = {
    "model": VLLM_MODEL,
    "messages": [{"role": "user", "content": "Reply with exactly: vLLM connection OK"}],
    "temperature": 0,
    "top_p": 1.0,
    "max_tokens": 32,
}
request_started = time.perf_counter()
chat_response = _vllm_request(
    f"{VLLM_BASE_URL}/chat/completions", method="POST", payload=chat_payload, timeout=60
)
request_latency_ms = (time.perf_counter() - request_started) * 1_000

response_model = chat_response.get("model")
assert response_model == VLLM_MODEL, f"Unexpected response model: {response_model!r}"
try:
    content = chat_response["choices"][0]["message"]["content"].strip()
except (KeyError, IndexError, TypeError, AttributeError) as exc:
    raise AssertionError("Chat response does not contain choices[0].message.content") from exc
assert content == "vLLM connection OK", f"Unexpected chat response: {content!r}"

usage = chat_response.get("usage") or {}
completion_details = usage.get("completion_tokens_details") or {}
reasoning_tokens = usage.get("reasoning_tokens", completion_details.get("reasoning_tokens"))
fingerprint = chat_response.get("system_fingerprint") or "not provided"

print(
    "vLLM Runtime Check: PASS",
    f"Root URL: {VLLM_ROOT_URL}",
    "API: OpenAI-compatible /v1",
    f"Model: {response_model}",
    f"Health: {'PASS' if health_status == 200 else 'FAIL'}",
    "Model discovery: PASS",
    "Chat completion: PASS",
    f"Response: {content}",
    f"vLLM fingerprint: {fingerprint}",
    (
        "Tokens: "
        f"prompt={usage.get('prompt_tokens', 'n/a')}, "
        f"completion={usage.get('completion_tokens', 'n/a')}, "
        f"total={usage.get('total_tokens', 'n/a')}"
    ),
    f"Reasoning tokens: {reasoning_tokens if reasoning_tokens is not None else 'n/a'}",
    f"Request latency: {request_latency_ms:.2f} ms",
    sep="\n",
)


vLLM Runtime Check: PASS
Root URL: http://vllm-gemma4-12b:8000
API: OpenAI-compatible /v1
Model: gemma-4-12b-it
Health: PASS
Model discovery: PASS
Chat completion: PASS
Response: vLLM connection OK
vLLM fingerprint: vllm-0.29.0-4b5aecce
Tokens: prompt=22, completion=6, total=28
Reasoning tokens: 0
Request latency: 57.08 ms


### 17.2 OpenAI-Compatible LLM Adapter


vLLM exposes an OpenAI-compatible API, so the official OpenAI Python SDK is used here only as the transport/client layer. A thin project-owned adapter isolates the remaining RAG pipeline from SDK-specific response objects and returns a compact normalized result. Prompt construction, ContextPack grounding and answer policy are added in the following subsections.


In [1]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from typing import Any

from openai import OpenAI


@dataclass(frozen=True)
class LLMUsage:
    prompt_tokens: int | None
    completion_tokens: int | None
    total_tokens: int | None
    reasoning_tokens: int | None


@dataclass(frozen=True)
class LLMResult:
    text: str
    model: str
    finish_reason: str | None
    latency_ms: float
    usage: LLMUsage
    system_fingerprint: str | None


class OpenAICompatibleLLM:
    def __init__(
        self,
        *,
        base_url: str,
        model: str,
        api_key: str = "not-needed",
        timeout: float = 60.0,
    ) -> None:
        self.model = model
        self.client = OpenAI(
            base_url=base_url.rstrip("/"),
            api_key=api_key,
            timeout=timeout,
            max_retries=0,
        )

    def generate(
        self,
        messages: Sequence[Mapping[str, Any]],
        *,
        temperature: float = 0.0,
        top_p: float = 1.0,
        max_tokens: int = 512,
    ) -> LLMResult:
        started = time.perf_counter()
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[dict(message) for message in messages],
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        latency_ms = (time.perf_counter() - started) * 1_000

        if not response.choices:
            raise RuntimeError("LLM response contains no choices")
        content = response.choices[0].message.content
        if content is None:
            raise RuntimeError("LLM response choice contains no message content")
        text = content.strip()
        if not text:
            raise RuntimeError("LLM response message content is empty")
        if not response.model:
            raise RuntimeError("LLM response contains no model identity")

        usage = response.usage
        completion_details = getattr(usage, "completion_tokens_details", None)
        return LLMResult(
            text=text,
            model=response.model,
            finish_reason=response.choices[0].finish_reason,
            latency_ms=latency_ms,
            usage=LLMUsage(
                prompt_tokens=getattr(usage, "prompt_tokens", None),
                completion_tokens=getattr(usage, "completion_tokens", None),
                total_tokens=getattr(usage, "total_tokens", None),
                reasoning_tokens=getattr(completion_details, "reasoning_tokens", None),
            ),
            system_fingerprint=getattr(response, "system_fingerprint", None),
        )


llm = OpenAICompatibleLLM(base_url=VLLM_BASE_URL, model=VLLM_MODEL)


In [2]:
adapter_result = llm.generate(
    [{"role": "user", "content": "Reply with exactly: LLM adapter OK"}],
    temperature=0.0,
    top_p=1.0,
    max_tokens=32,
)
assert adapter_result.text == "LLM adapter OK", f"Unexpected response: {adapter_result.text!r}"
assert adapter_result.model == VLLM_MODEL, f"Unexpected model: {adapter_result.model!r}"

print(
    "OpenAI-Compatible LLM Adapter: PASS",
    f"Model: {adapter_result.model}",
    f"Response: {adapter_result.text}",
    f"Finish reason: {adapter_result.finish_reason or 'not provided'}",
    f"Fingerprint: {adapter_result.system_fingerprint or 'not provided'}",
    (
        "Tokens: "
        f"prompt={adapter_result.usage.prompt_tokens}, "
        f"completion={adapter_result.usage.completion_tokens}, "
        f"total={adapter_result.usage.total_tokens}"
    ),
    f"Reasoning tokens: {adapter_result.usage.reasoning_tokens}",
    f"Request latency: {adapter_result.latency_ms:.2f} ms",
    sep="\n",
)


OpenAI-Compatible LLM Adapter: PASS
Model: gemma-4-12b-it
Response: LLM adapter OK
Finish reason: stop
Fingerprint: vllm-0.29.0-4b5aecce
Tokens: prompt=21, completion=5, total=26
Reasoning tokens: 0
Request latency: 1639.32 ms


### 17.3 Grounded Prompt Contract


This prompt boundary consumes a completed `ContextPack` and treats its evidence as the only factual authority. Inline citations are restricted to source IDs carried by that pack, insufficient evidence triggers an explicit canonical abstention, and the model is asked for the final answer without hidden reasoning. Actual LLM generation is implemented in Section 17.4.


In [1]:
import re

GROUNDING_ABSTENTION = "Insufficient evidence in the provided context."

GROUNDED_SYSTEM_PROMPT = (
    "Answer using only the provided evidence. Do not use outside knowledge. Do not invent facts or citations. "
    "Cite each supported factual claim inline with the provided source IDs, such as [S1]. "
    "Never cite a source ID that is not listed as allowed. "
    f"If the evidence is insufficient, respond exactly: {GROUNDING_ABSTENTION} "
    "Return the final answer only. Do not reveal chain-of-thought, hidden reasoning, scratch work, "
    "or internal deliberation."
)


def build_grounded_messages(
    query: str,
    context_pack: ContextPack,
) -> list[dict[str, str]]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a nonempty string")
    if not isinstance(context_pack, ContextPack):
        raise TypeError("context_pack must be a ContextPack")

    evidence = context_pack.context_text
    if not isinstance(evidence, str) or not evidence.strip():
        raise ValueError("context_pack must contain nonempty evidence text")
    if not context_pack.evidence_blocks or not context_pack.sources:
        raise ValueError("context_pack must contain evidence blocks and source IDs")

    source_ids = tuple(block.source_id for block in context_pack.evidence_blocks)
    expected_ids = tuple(f"S{i}" for i in range(1, len(source_ids) + 1))
    if source_ids != expected_ids or any(re.fullmatch(r"S[1-9]\d*", source_id) is None for source_id in source_ids):
        raise ValueError(f"unexpected ContextPack source IDs: {source_ids!r}")
    if tuple(context_pack.sources) != source_ids:
        raise ValueError("ContextPack evidence blocks and sources are inconsistent")

    allowed_citations = tuple(f"[{source_id}]" for source_id in source_ids)
    missing_citations = [citation for citation in allowed_citations if citation not in evidence]
    if missing_citations:
        raise ValueError(f"evidence text is missing source markers: {missing_citations!r}")

    user_prompt = (
        f"QUESTION\n--------\n{query.strip()}\n\n"
        f"EVIDENCE\n--------\n{evidence.strip()}\n\n"
        f"Allowed source IDs: {', '.join(allowed_citations)}\n\n"
        "Answer using only the evidence above."
    )
    return [
        {"role": "system", "content": GROUNDED_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


In [2]:
from dataclasses import replace

prompt_test_key = sorted(context_packs, key=lambda key: (str(key[0]), str(key[1])))[0]
prompt_test_pack = context_packs[prompt_test_key]
prompt_test_query = prompt_test_pack.query
grounded_messages = build_grounded_messages(prompt_test_query, prompt_test_pack)
allowed_source_ids = tuple(block.source_id for block in prompt_test_pack.evidence_blocks)
allowed_citations = tuple(f"[{source_id}]" for source_id in allowed_source_ids)

assert len(grounded_messages) == 2
assert [message["role"] for message in grounded_messages] == ["system", "user"]
system_content, user_content = (message["content"] for message in grounded_messages)
assert prompt_test_query in user_content
assert prompt_test_pack.context_text in user_content
assert all(citation in user_content for citation in allowed_citations)
assert GROUNDING_ABSTENTION in system_content
system_lower = system_content.lower()
assert "only the provided evidence" in system_lower and "do not use outside knowledge" in system_lower
assert "do not invent facts or citations" in system_lower
assert "never cite a source id that is not listed as allowed" in system_lower
assert "do not reveal chain-of-thought" in system_lower

try:
    build_grounded_messages("   ", prompt_test_pack)
except ValueError as exc:
    assert "query" in str(exc)
else:
    raise AssertionError("empty query must raise ValueError")

try:
    build_grounded_messages(prompt_test_query, replace(prompt_test_pack, context_text=""))
except ValueError as exc:
    assert "evidence text" in str(exc)
else:
    raise AssertionError("empty evidence must raise ValueError")

print(
    "Grounded Prompt Contract: PASS",
    f"Messages: {len(grounded_messages)}",
    f"Allowed sources: {', '.join(allowed_citations)}",
    "Abstention policy: PASS",
    "Evidence-only policy: PASS",
    "Citation boundary: PASS",
    "No-chain-of-thought policy: PASS",
    sep="\n",
)


Grounded Prompt Contract: PASS
Messages: 2
Allowed sources: [S1], [S2]
Abstention policy: PASS
Evidence-only policy: PASS
Citation boundary: PASS
No-chain-of-thought policy: PASS


### 17.4 Answer and Citation Generation


This layer connects a completed `ContextPack`, the grounded prompt contract and the LLM adapter to produce a normal grounded natural-language answer. Citations are validated against the source IDs of that specific pack; unknown or malformed source markers are contract violations, while the exact canonical abstention is valid without citations. Broader behavior checks and S0/S1/S2 evaluation remain in Section 17.5 and Section 18.


In [1]:
_CITATION_PATTERN = re.compile(r"\[(S[1-9]\d*)\]")
_CITATION_LIKE_PATTERN = re.compile(r"\[S[^\]\r\n]*\]")


@dataclass(frozen=True)
class GroundedAnswer:
    text: str
    cited_source_ids: tuple[str, ...]
    is_abstention: bool
    llm_result: LLMResult


def extract_cited_source_ids(text: str) -> tuple[str, ...]:
    if not isinstance(text, str):
        raise TypeError("answer text must be a string")
    seen, ordered_source_ids = set(), []
    for source_id in _CITATION_PATTERN.findall(text):
        if source_id not in seen:
            seen.add(source_id)
            ordered_source_ids.append(source_id)
    return tuple(ordered_source_ids)


def validate_grounded_answer(
    text: str,
    context_pack: ContextPack,
) -> tuple[tuple[str, ...], bool]:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("grounded answer must be a nonempty string")
    if not isinstance(context_pack, ContextPack):
        raise TypeError("context_pack must be a ContextPack")

    normalized_text = text.strip()
    if normalized_text == GROUNDING_ABSTENTION:
        return (), True

    citation_like_markers = _CITATION_LIKE_PATTERN.findall(normalized_text)
    malformed_markers = tuple(dict.fromkeys(
        marker for marker in citation_like_markers
        if _CITATION_PATTERN.fullmatch(marker) is None
    ))
    if malformed_markers:
        raise ValueError(f"malformed citation markers: {malformed_markers!r}")

    cited_source_ids = extract_cited_source_ids(normalized_text)
    if not cited_source_ids:
        raise ValueError("non-abstaining grounded answer must contain at least one citation")

    allowed_source_ids = {block.source_id for block in context_pack.evidence_blocks}
    unknown_source_ids = tuple(
        source_id for source_id in cited_source_ids if source_id not in allowed_source_ids
    )
    if unknown_source_ids:
        raise ValueError(
            f"grounded answer cites source IDs outside the ContextPack: {unknown_source_ids!r}"
        )
    return cited_source_ids, False


def generate_grounded_answer(
    query: str,
    context_pack: ContextPack,
    *,
    llm_client: OpenAICompatibleLLM = llm,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
) -> GroundedAnswer:
    messages = build_grounded_messages(query, context_pack)
    llm_result = llm_client.generate(
        messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    cited_source_ids, is_abstention = validate_grounded_answer(
        llm_result.text, context_pack,
    )
    return GroundedAnswer(
        text=llm_result.text,
        cited_source_ids=cited_source_ids,
        is_abstention=is_abstention,
        llm_result=llm_result,
    )


In [2]:
citation_test_pack = prompt_test_pack
citation_test_ids = tuple(block.source_id for block in citation_test_pack.evidence_blocks)
assert len(citation_test_ids) >= 2
first_id, second_id = citation_test_ids[:2]

citation_test_text = f"Fact [{second_id}]. More [{first_id}]. Again [{second_id}]."
assert extract_cited_source_ids(citation_test_text) == (second_id, first_id)
assert validate_grounded_answer(citation_test_text, citation_test_pack) == (
    (second_id, first_id), False,
)
assert validate_grounded_answer(GROUNDING_ABSTENTION, citation_test_pack) == ((), True)

for invalid_text, expected_error in (
    ("Unknown source [S999].", "outside the ContextPack"),
    ("Malformed source [Sx].", "malformed citation"),
    ("Answer without a citation.", "at least one citation"),
    ("   ", "nonempty string"),
):
    try:
        validate_grounded_answer(invalid_text, citation_test_pack)
    except ValueError as exc:
        assert expected_error in str(exc)
    else:
        raise AssertionError(f"expected citation validation failure for {invalid_text!r}")

print(
    "Citation Contract Tests: PASS",
    "Extraction order and deduplication: PASS",
    "Valid citation subset: PASS",
    "Unknown/malformed citation rejection: PASS",
    "Citation-required branch: PASS",
    "Canonical abstention branch: PASS",
    sep="\n",
)


Citation Contract Tests: PASS
Extraction order and deduplication: PASS
Valid citation subset: PASS
Unknown/malformed citation rejection: PASS
Citation-required branch: PASS
Canonical abstention branch: PASS


In [3]:
generation_context_pack = prompt_test_pack
generation_query = prompt_test_query
grounded_answer = generate_grounded_answer(generation_query, generation_context_pack)

allowed_generation_ids = tuple(
    block.source_id for block in generation_context_pack.evidence_blocks
)
assert grounded_answer.text
assert grounded_answer.llm_result.model == VLLM_MODEL
assert grounded_answer.llm_result.finish_reason
if grounded_answer.is_abstention:
    assert grounded_answer.text == GROUNDING_ABSTENTION
    assert grounded_answer.cited_source_ids == ()
else:
    assert grounded_answer.cited_source_ids
    assert set(grounded_answer.cited_source_ids) <= set(allowed_generation_ids)

usage = grounded_answer.llm_result.usage
print(
    "Grounded Answer Generation: PASS",
    f"Model: {grounded_answer.llm_result.model}",
    f"Strategy: {generation_context_pack.strategy}",
    f"Abstention: {'YES' if grounded_answer.is_abstention else 'NO'}",
    (
        "Allowed sources: "
        + ", ".join(f"[{source_id}]" for source_id in allowed_generation_ids)
    ),
    (
        "Cited sources: "
        + (
            ", ".join(
                f"[{source_id}]"
                for source_id in grounded_answer.cited_source_ids
            )
            or "none"
        )
    ),
    "Citation boundary: PASS",
    f"Finish reason: {grounded_answer.llm_result.finish_reason}",
    (
        f"Tokens: prompt={usage.prompt_tokens}, "
        f"completion={usage.completion_tokens}, total={usage.total_tokens}"
    ),
    f"Reasoning tokens: {usage.reasoning_tokens}",
    f"Request latency: {grounded_answer.llm_result.latency_ms:.2f} ms",
    f"\nAnswer:\n{grounded_answer.text}",
    sep="\n",
)


Grounded Answer Generation: PASS
Model: gemma-4-12b-it
Strategy: retrieved_chunks
Abstention: NO
Allowed sources: [S1], [S2], [S3], [S4], [S5], [S6], [S7], [S8], [S9], [S10]
Cited sources: [S10]
Citation boundary: PASS
Finish reason: stop
Tokens: prompt=1094, completion=47, total=1141
Reasoning tokens: 0
Request latency: 964.24 ms

Answer:
When creating a collection, you must declare the vector size and distance [S10]. For example, a collection can be created with a size of 1024 and a distance of COSINE [S10].


#### Semantic scope note

The smoke test above validates the **structural grounding contract**: the answer is non-empty, its citation refers to a source that actually exists in the selected `ContextPack`, and no phantom source IDs are present.

It does **not** establish semantic faithfulness or generalization correctness. In this example, the answer cites `[S10]`, which contains a concrete configuration example using `size=1024` and `Distance.COSINE`. The generated answer correctly attributes those values to `[S10]`, but Section 17.4 does not determine whether an example-specific configuration could be interpreted too broadly as a general requirement.

Semantic support, citation entailment, faithfulness, and answer correctness are intentionally deferred to **Section 17.5 / Section 18 RAG Evaluation**. Therefore, `Citation boundary: PASS` should be read as a structural validation result, not as a semantic quality score.

### 17.5 Generation Sanity Checks

This behavioral smoke check exercises the completed generation pipeline with existing `ContextPack` objects only. Retrieval is not rerun, `final_holdout` is excluded, and the observed answers do not select a winner among S0/S1/S2. Structural citations are resolved back to each pack's real evidence blocks and source URLs; formal semantic and end-to-end RAG evaluation remains in Section 18.


In [1]:
GENERATION_SANITY_STRATEGIES = tuple(CONTEXT_STRATEGIES)
assert GENERATION_SANITY_STRATEGIES == (
    "retrieved_chunks",
    "adaptive_section_expansion",
    "evidence_aware_packing",
)

sanity_split_by_id = dict(
    context_sanity_queries[["query_id", "split"]].itertuples(index=False, name=None)
)
packs_by_query = {}
for (query_id, strategy), pack in context_packs.items():
    if strategy in GENERATION_SANITY_STRATEGIES:
        packs_by_query.setdefault(query_id, {})[strategy] = pack

complete_query_ids = sorted(
    query_id for query_id, packs in packs_by_query.items()
    if set(packs) == set(GENERATION_SANITY_STRATEGIES)
)
eligible_query_ids = [
    query_id for query_id in complete_query_ids
    if sanity_split_by_id.get(query_id) in {"research", "confirmation"}
]
selected_sanity_query_ids = []
for preferred_split in ("research", "confirmation"):
    candidate = next(
        (query_id for query_id in eligible_query_ids
         if sanity_split_by_id[query_id] == preferred_split
         and query_id not in selected_sanity_query_ids),
        None,
    )
    if candidate is not None:
        selected_sanity_query_ids.append(candidate)
for query_id in eligible_query_ids:
    if len(selected_sanity_query_ids) == 2:
        break
    if query_id not in selected_sanity_query_ids:
        selected_sanity_query_ids.append(query_id)
selected_sanity_query_ids = tuple(selected_sanity_query_ids)

assert len(selected_sanity_query_ids) == 2
assert all(sanity_split_by_id[query_id] != "final_holdout" for query_id in selected_sanity_query_ids)
for query_id in selected_sanity_query_ids:
    packs = packs_by_query[query_id]
    assert tuple(packs) == GENERATION_SANITY_STRATEGIES
    assert len({packs[strategy].query for strategy in GENERATION_SANITY_STRATEGIES}) == 1


def run_generation_sanity_case(query_id: str, strategy: str) -> dict:
    context_pack = packs_by_query[query_id][strategy]
    answer = generate_grounded_answer(
        context_pack.query,
        context_pack,
        temperature=0.0,
        top_p=1.0,
        max_tokens=512,
    )
    usage = answer.llm_result.usage
    assert answer.text and answer.llm_result.model == VLLM_MODEL
    assert answer.llm_result.finish_reason and answer.llm_result.latency_ms > 0
    assert all(value is not None for value in (
        usage.prompt_tokens, usage.completion_tokens, usage.total_tokens,
    ))

    blocks_by_id = {block.source_id: block for block in context_pack.evidence_blocks}
    if answer.is_abstention:
        assert answer.text == GROUNDING_ABSTENTION and answer.cited_source_ids == ()
    else:
        assert answer.cited_source_ids
        assert set(answer.cited_source_ids) <= set(blocks_by_id)
    citation_sources = tuple(
        {
            "source_id": source_id,
            "page_title": blocks_by_id[source_id].page_title,
            "section_title": blocks_by_id[source_id].section_title,
            "url": (
                blocks_by_id[source_id].section_url
                or blocks_by_id[source_id].page_url
                or context_pack.sources.get(source_id)
            ),
        }
        for source_id in answer.cited_source_ids
    )
    assert all(source["url"] for source in citation_sources)
    return {
        "query_id": query_id,
        "split": sanity_split_by_id[query_id],
        "query": context_pack.query,
        "strategy": strategy,
        "answer": answer,
        "citation_sources": citation_sources,
    }


selected_query_lines = [
    (
        f"- {query_id} ({sanity_split_by_id[query_id]}): "
        f"{packs_by_query[query_id][GENERATION_SANITY_STRATEGIES[0]].query}"
    )
    for query_id in selected_sanity_query_ids
]
print(
    "Generation sanity selection: PASS",
    f"Strategies: {', '.join(GENERATION_SANITY_STRATEGIES)}",
    *selected_query_lines,
    sep="\n",
)


Generation sanity selection: PASS
Strategies: retrieved_chunks, adaptive_section_expansion, evidence_aware_packing
- h01 (confirmation): Which vector size and distance function must I declare when creating a collection?
- h02 (confirmation): Can one collection store multiple named embeddings with different dimensions and metrics?


In [2]:
supported_generation_results = []
for query_id in selected_sanity_query_ids:
    for strategy in GENERATION_SANITY_STRATEGIES:
        supported_generation_results.append(
            run_generation_sanity_case(query_id, strategy)
        )

assert len(supported_generation_results) == 6
assert {result["strategy"] for result in supported_generation_results} == set(
    GENERATION_SANITY_STRATEGIES
)
supported_summary_df = pd.DataFrame([
    {
        "query_id": result["query_id"],
        "split": result["split"],
        "strategy": result["strategy"],
        "abstention": "YES" if result["answer"].is_abstention else "NO",
        "cited_source_ids": ", ".join(result["answer"].cited_source_ids) or "none",
        "finish_reason": result["answer"].llm_result.finish_reason,
        "prompt_tokens": result["answer"].llm_result.usage.prompt_tokens,
        "completion_tokens": result["answer"].llm_result.usage.completion_tokens,
        "total_tokens": result["answer"].llm_result.usage.total_tokens,
        "latency_ms": result["answer"].llm_result.latency_ms,
        "structural_status": "PASS",
    }
    for result in supported_generation_results
])
print(supported_summary_df.round({"latency_ms": 2}).to_string(index=False))

report_blocks = []
for result in supported_generation_results:
    answer = result["answer"]
    citation_lines = []
    if result["citation_sources"]:
        for source in result["citation_sources"]:
            title = " / ".join(
                value
                for value in (source["page_title"], source["section_title"])
                if value
            )
            citation_lines.append(
                f"[{source['source_id']}] -> {title}: {source['url']}"
            )
    citations = "\n".join(citation_lines) or "none"
    usage = answer.llm_result.usage
    report_blocks.append(
        f"Query ID: {result['query_id']} ({result['split']})\n"
        f"Query: {result['query']}\n"
        f"Strategy: {result['strategy']}\n"
        f"Abstention: {'YES' if answer.is_abstention else 'NO'}\n"
        f"Answer:\n{answer.text}\n"
        f"Citations:\n{citations}\n"
        f"Finish={answer.llm_result.finish_reason}; "
        f"tokens={usage.prompt_tokens}/{usage.completion_tokens}/{usage.total_tokens}; "
        f"reasoning={usage.reasoning_tokens}; latency={answer.llm_result.latency_ms:.2f} ms"
    )

print(*report_blocks, sep="\n\n")


query_id        split                   strategy abstention cited_source_ids finish_reason  prompt_tokens  completion_tokens  total_tokens  latency_ms structural_status
     h01 confirmation           retrieved_chunks         NO           S1, S2          stop            295                 54           349      889.73              PASS
     h01 confirmation adaptive_section_expansion         NO           S1, S2          stop            295                 54           349      388.93              PASS
     h01 confirmation     evidence_aware_packing         NO           S1, S2          stop            295                 54           349      386.40              PASS
     h02 confirmation           retrieved_chunks         NO           S1, S2          stop            283                 56           339      417.41              PASS
     h02 confirmation adaptive_section_expansion         NO           S1, S2          stop            283                 56           339      396.13     

In [3]:
UNSUPPORTED_SANITY_QUERY = "What was the closing price of Apple stock on January 3, 2020?"
unsupported_evidence_query_id = selected_sanity_query_ids[0]
unsupported_evidence_pack = packs_by_query[unsupported_evidence_query_id][
    GENERATION_SANITY_STRATEGIES[0]
]
unsupported_answer = generate_grounded_answer(
    UNSUPPORTED_SANITY_QUERY,
    unsupported_evidence_pack,
    temperature=0.0,
    top_p=1.0,
    max_tokens=512,
)
assert unsupported_answer.is_abstention is True
assert unsupported_answer.text == GROUNDING_ABSTENTION
assert unsupported_answer.cited_source_ids == ()
assert unsupported_answer.llm_result.model == VLLM_MODEL
assert unsupported_answer.llm_result.finish_reason
assert unsupported_answer.llm_result.latency_ms > 0
unsupported_usage = unsupported_answer.llm_result.usage
assert all(value is not None for value in (
    unsupported_usage.prompt_tokens,
    unsupported_usage.completion_tokens,
    unsupported_usage.total_tokens,
))

print(
    "Unsupported Evidence Check: PASS",
    f"Question: {UNSUPPORTED_SANITY_QUERY}",
    (
        f"Strategy/evidence pack: {unsupported_evidence_pack.strategy} "
        f"({unsupported_evidence_query_id}, "
        f"{sanity_split_by_id[unsupported_evidence_query_id]})"
    ),
    "Abstention: YES",
    f"Answer: {unsupported_answer.text}",
    "Citations: none",
    f"Model: {unsupported_answer.llm_result.model}",
    f"Finish reason: {unsupported_answer.llm_result.finish_reason}",
    (
        f"Tokens: prompt={unsupported_usage.prompt_tokens}, "
        f"completion={unsupported_usage.completion_tokens}, "
        f"total={unsupported_usage.total_tokens}"
    ),
    f"Reasoning tokens: {unsupported_usage.reasoning_tokens}",
    f"Request latency: {unsupported_answer.llm_result.latency_ms:.2f} ms",
    sep="\n",
)
print(
    "\nGeneration Sanity Checks: PASS",
    "Supported query groups: 2",
    "Strategies exercised: 3",
    "Supported generation calls: 6",
    "Unsupported-evidence calls: 1",
    "Total LLM calls: 7",
    "Structural citation violations: 0",
    "Phantom citations: 0",
    "Unsupported-evidence abstention: PASS",
    "Final-holdout used: NO",
    "Strategy winner selected: NO",
    sep="\n",
)


Unsupported Evidence Check: PASS
Question: What was the closing price of Apple stock on January 3, 2020?
Strategy/evidence pack: retrieved_chunks (h01, confirmation)
Abstention: YES
Answer: Insufficient evidence in the provided context.
Citations: none
Model: gemma-4-12b-it
Finish reason: stop
Tokens: prompt=300, completion=8, total=308
Reasoning tokens: 0
Request latency: 89.85 ms

Generation Sanity Checks: PASS
Supported query groups: 2
Strategies exercised: 3
Supported generation calls: 6
Unsupported-evidence calls: 1
Total LLM calls: 7
Structural citation violations: 0
Phantom citations: 0
Unsupported-evidence abstention: PASS
Final-holdout used: NO
Strategy winner selected: NO


### 17.6 RAG Generation Conclusions

Section 17 establishes a framework-agnostic generation stack around locally served Gemma 4 (`gemma-4-12b-it`). vLLM exposes the OpenAI-compatible transport boundary, while the project-owned `OpenAICompatibleLLM` adapter normalizes SDK responses into `LLMResult`; the remaining RAG logic does not depend directly on OpenAI SDK objects.

The grounded prompt contract consumes a completed `ContextPack`, treats its evidence as the only factual authority, prohibits outside knowledge, and requires inline citations such as `[S1]`. Only source IDs present in the current pack are allowed; otherwise the model must return the exact canonical abstention, `Insufficient evidence in the provided context.` The contract requests only the final answer and explicitly excludes chain-of-thought, hidden reasoning and internal deliberation.

The implemented path is `ContextPack` → `build_grounded_messages(...)` → `OpenAICompatibleLLM.generate(...)` → `LLMResult` → citation-boundary validation → `GroundedAnswer`. Citation validation is structural: it confirms that citations exist for a non-abstaining answer, belong to the current `ContextPack`, and contain no phantom source IDs. `Citation boundary: PASS` does not establish semantic entailment, factual correctness, complete faithfulness or answer completeness; the `size=1024` / `Distance.COSINE` example illustrates why those claims require separate evaluation.

Section 17.5 exercised `retrieved_chunks`, `adaptive_section_expansion` and `evidence_aware_packing` on deterministic non-final-holdout query groups `h01` and `h02`, both from the `confirmation` split. All six supported generations completed with nonempty answers and captured model identity, finish reason, token usage and request latency. Cited IDs resolved to real `ContextBlock` objects and source URLs, with zero structural citation violations and zero phantom citations. A seventh real LLM call paired unrelated evidence with an unsupported question and returned the exact canonical abstention without citations; `final_holdout` was not used.

Section 17 does **not** select a winning context strategy: it verifies the generation architecture and its structural and behavioral invariants only. Section 18 performs formal RAG evaluation of answer quality, groundedness, citation quality and entailment, abstention behavior, and the S0/S1/S2 comparison that can support final context-strategy selection.

**Execution-state note.** Section 17 was implemented incrementally in separate cells, so saved outputs across its subsections may reflect different kernel execution states; this is not an experimental result. A separate cleanup followed by **Restart Kernel → Run All** will establish one sequential execution state after Section 17 is complete.


## 18. RAG Evaluation


### 18.1 Evaluation Dataset


### 18.2 Answer Quality


### 18.3 Groundedness


### 18.4 Citation Quality


### 18.5 Context Strategy Comparison


## 19. Results and Conclusions


### 19.1 Best Retrieval Configuration

The production retrieval configuration is the immutable decision recorded in `FINAL_CONFIG` in Section 12. It is derived from the existing research + confirmation tables and is not changed by final-holdout evaluation.


### 19.2 Best Context Strategy


### 19.3 Final Architecture

The physical `docs_search_final` schema, HNSW and quantization settings, payload indexes and production-like retrieval entry point are all constructed from `FINAL_CONFIG`. Only the representations selected by the frozen pipeline are stored.


### 19.4 Limitations


### 19.5 Next Steps
